In [ ]:
# @title 1.) Install all dependencies & imports
# Install required packages
%pip install -q rdkit pyscf chempy torch scikit-learn numpy scipy pandas matplotlib seaborn deepchem plotly gdown xgboost shap dgl catboost deepod e3nn ngboost transformers joblib tensorflow keras tqdm

# Suppress verbose logs
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['WANDB_SILENT'] = 'true'

import warnings
warnings.filterwarnings('ignore')

# ---------- Standard library ----------
import sys
import subprocess
import contextlib
import io
import time
import pickle
import json
import logging
from datetime import datetime
from enum import Enum
from typing import Dict, Tuple, List, Optional, Any, Set, Union
from collections import defaultdict
import random
import math
import multiprocessing as mp

# ---------- Numerical & scientific ----------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy.signal import savgol_filter, find_peaks
from scipy.ndimage import shift

# ---------- Machine learning frameworks ----------
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn.functional as F
from torch.nn import Conv1d, BatchNorm1d, ReLU, MaxPool1d, AdaptiveAvgPool1d, Linear, Dropout

import tensorflow as tf
import keras

# ---------- Scikit-learn ----------
from sklearn.preprocessing import RobustScaler, LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.utils import shuffle
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import Ridge, LinearRegression , RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.neural_network import MLPRegressor

# ---------- XGBoost, CatBoost, NGBoost ----------
import xgboost as xgb
from catboost import CatBoostRegressor
from ngboost import NGBRegressor
from ngboost.distns import Normal

# ---------- RDKit ----------
from rdkit import Chem, DataStructs
from rdkit.Chem import (
    Descriptors, rdMolDescriptors, AllChem, rdFingerprintGenerator,
    GetDistanceMatrix, Descriptors3D, MolStandardize, Scaffolds
)
from rdkit.Chem.rdForceFieldHelpers import MMFFOptimizeMolecule
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

# ---------- DeepChem ----------
from deepchem.feat import RDKitDescriptors

# ---------- PySCF ----------
from pyscf import gto, dft, tddft

# ---------- Google Colab & Drive (guarded) ----------
try:
    from google.colab import files, drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    files = None
    drive = None

import gdown

# ---------- SHAP, e3nn, Transformers, joblib ----------
import shap
import e3nn.o3 as o3
from e3nn import nn as e3nn_nn
from e3nn.math import soft_one_hot_linspace
from transformers import AutoTokenizer, AutoModel
import joblib

# ---------- Plotly (conditional) ----------
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

# ---------- scikit-learn availability flag ----------
try:
    from sklearn.ensemble import RandomForestRegressor, IsolationForest
    from sklearn.neighbors import NearestNeighbors
    from sklearn.linear_model import Ridge, LinearRegression
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False

# ---------- XGBoost availability flag ----------
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

print("✅ All libraries loaded.")

✅ All libraries loaded.


In [ ]:
# @title 2.) Configuration
class PlotBackend(Enum):
    PLOTLY = "plotly"
    MATPLOTLIB = "matplotlib"

class Config:
    WL_MIN = 200
    WL_MAX = 800
    WL_STEP = 0.5
    wavelengths = np.arange(WL_MIN, WL_MAX + WL_STEP, WL_STEP)

    PATH_LENGTH = 1.0
    NOISE_LEVEL = 0.0001
    BASELINE_DRIFT = 0.00005
    FINAL_SMOOTHING_WINDOW = 9

    HIDDEN_LAYERS = [512, 256, 128, 64]
    DROPOUT = 0.3
    LEARNING_RATE = 0.001
    EPOCHS = 60
    BATCH_SIZE = 32
    EARLY_STOPPING_PATIENCE = 30
    VALIDATION_SPLIT = 0.2
    USE_CUDA = torch.cuda.is_available()

    MAX_FEEDBACK_STORE = 10000
    RETRAIN_THRESHOLD = 20

    ORIG_FEAT_DIM = 271
    DEEPCHEM_FEAT_DIM = 200
    TOTAL_FEAT_DIM = ORIG_FEAT_DIM + DEEPCHEM_FEAT_DIM
    PADDED_FEAT_DIM = 1024   # DO NOT CHANGE

    DEFAULT_TEMP = 298.15
    DEFAULT_CONC = 0.01
    DEFAULT_PH = 7.0

    TDDFT_OSC_THRESHOLD = 0.05
    INDUCTIVE_SHIFT = 12.0
    STERIC_PENALTY = 15.0
    PUSH_PULL_ENHANCEMENT = 25.0

    PLOT_BACKEND = PlotBackend.PLOTLY if PLOTLY_AVAILABLE else PlotBackend.MATPLOTLIB
    MATPLOTLIB_DPI = 300
    MATPLOTLIB_FIGSIZE = (12, 6)
    PLOTLY_TEMPLATE = "plotly_white"
    SHOW_PEAK_ANNOTATIONS = True

    USE_RESIDUAL_CORRECTION = True
    RESIDUAL_MODEL_TYPE = "random_forest"
    USE_DYNAMIC_SOLVENT = True
    SOLVENT_MODEL_TYPE = "xgboost"
    USE_ENSEMBLE = True
    ENSEMBLE_MODELS = ["random_forest", "xgboost"]
    USE_OUTLIER_DETECTION = True
    OUTLIER_CONTAMINATION = 0.05

    EXTRACT_QUANTUM_DESCRIPTORS = True
    STORE_QUANTUM_DESCRIPTORS = True

    CONFIDENCE_KNN_K = 5
    CONFIDENCE_SIMILARITY_THRESHOLD = 0.7

    CONJUGATION_SHIFT_PER_ATOM = 2.0
    ICT_INTENSITY_BOOST = 1.2

    USE_QUANTUM_CORRECTION = True
    QUANTUM_BASIS_PREFERENCE = ["def2-tzvp", "def2-svp", "sto-3g"]
    QUANTUM_FUNCTIONALS = ["cam-b3lyp", "wb97x-d"]
    QUANTUM_SHIFT_REFERENCE_GAP = 4.0
    QUANTUM_MAX_SHIFT = 50.0

    USE_CONFORMATIONAL_CORRECTION = True
    CONFORMER_NUM_CONFS = 5
    CONFORMER_MAX_ATTEMPTS = 100

    USE_BOLTZMANN_AVERAGING = True
    BOLTZMANN_TEMP = 298.15
    BOLTZMANN_ENERGY_CUTOFF = 3.0
    BOLTZMANN_NUM_CONFS = 10

    USE_SHAP = True
    SHAP_BACKGROUND_SIZE = 100
    SHAP_MAX_DISPLAY = 15

    USE_DRCNN = True
    DRCNN_CHAR_DIM = 64
    DRCNN_MAX_SMILES_LEN = 150
    DRCNN_HIDDEN_FILTERS = [64, 128, 256]
    DRCNN_KERNEL_SIZES = [5, 3, 3]
    DRCNN_DROPOUT = 0.2
    DRCNN_EPOCHS = 5
    DRCNN_BATCH_SIZE = 32
    DRCNN_LEARNING_RATE = 0.001

    VERBOSE_TRAINING = False

    TEST_SIZE = 0.15
    VALIDATION_SIZE = 0.15
    RANDOM_SEED = 42

    MODEL_VERSION = "13.7.3"
    FEATURE_VERSION = "1.0.0"

    # ---- Adaptive Reaction Engine ----
    REACTION_THRESHOLD = 0.5        # 0.4 permissive, 0.7 conservative
    REACTION_ML_FEATURES = 28       # must match ReactionFeasibilityML.N_FEATURES
    REACTION_RETRAIN_THRESHOLD = 20 # real feedback samples before auto-retrain

    # --- Reaction-aware prediction safety (v13.7.3) ---
    REACTION_ENABLED = True            # OPT-IN. Default: pure-parent prediction
    REACTION_SHOW_CANDIDATES = True    # Report candidates without using them
    REACTION_MAX_PRODUCTS = 5
    REACTION_INFER_LABELS_FROM_RESIDUALS = True  # SAFETY: never learn from spectra

# Global verbosity flag
VERBOSE_EVAL = False

print("✅ Configuration loaded.")

✅ Configuration loaded.


In [ ]:
# @title 3.) Chromophore Detector & Solvent Database

# ================================================================
# SolventDatabase (100 solvents, corrected pH)
# ================================================================
class SolventDatabase:
    def __init__(self):
        self.solvents = {
        # ---------- PURE SOLVENTS ----------
        "water":            {"id": "01", "name": "Water", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 7.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "ethanol":          {"id": "02", "name": "Ethanol", "polarity": 0.65, "dielectric": 24.5,
                             "hbond_donor": 0.86, "hbond_acceptor": 0.75, "refractive_index": 1.361,
                             "ph": 7.0, "pi_star": 0.54, "alpha": 0.86, "beta": 0.77},
        "methanol":         {"id": "03", "name": "Methanol", "polarity": 0.76, "dielectric": 32.7,
                             "hbond_donor": 0.98, "hbond_acceptor": 0.66, "refractive_index": 1.328,
                             "ph": 7.0, "pi_star": 0.60, "alpha": 0.98, "beta": 0.66},
        "acetonitrile":     {"id": "04", "name": "Acetonitrile", "polarity": 0.46, "dielectric": 37.5,
                             "hbond_donor": 0.19, "hbond_acceptor": 0.31, "refractive_index": 1.344,
                             "ph": 7.0, "pi_star": 0.75, "alpha": 0.19, "beta": 0.40},
        "dmso":             {"id": "05", "name": "DMSO", "polarity": 0.44, "dielectric": 46.7,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.76, "refractive_index": 1.479,
                             "ph": 7.0, "pi_star": 1.00, "alpha": 0.00, "beta": 0.76},
        "acetone":          {"id": "06", "name": "Acetone", "polarity": 0.36, "dielectric": 20.7,
                             "hbond_donor": 0.08, "hbond_acceptor": 0.43, "refractive_index": 1.359,
                             "ph": 7.0, "pi_star": 0.71, "alpha": 0.08, "beta": 0.48},
        "chloroform":       {"id": "07", "name": "Chloroform", "polarity": 0.26, "dielectric": 4.8,
                             "hbond_donor": 0.44, "hbond_acceptor": 0.00, "refractive_index": 1.446,
                             "ph": 7.0, "pi_star": 0.58, "alpha": 0.44, "beta": 0.00},
        "dichloromethane":  {"id": "08", "name": "DCM", "polarity": 0.31, "dielectric": 9.1,
                             "hbond_donor": 0.10, "hbond_acceptor": 0.10, "refractive_index": 1.424,
                             "ph": 7.0, "pi_star": 0.82, "alpha": 0.13, "beta": 0.10},
        "toluene":          {"id": "09", "name": "Toluene", "polarity": 0.10, "dielectric": 2.4,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.11, "refractive_index": 1.497,
                             "ph": 7.0, "pi_star": 0.54, "alpha": 0.00, "beta": 0.11},
        "hexane":           {"id": "10", "name": "Hexane", "polarity": 0.01, "dielectric": 1.9,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.00, "refractive_index": 1.375,
                             "ph": 7.0, "pi_star": 0.00, "alpha": 0.00, "beta": 0.00},
        "diethyl_ether":    {"id": "11", "name": "Diethyl Ether", "polarity": 0.27, "dielectric": 4.3,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.47, "refractive_index": 1.353,
                             "ph": 7.0, "pi_star": 0.27, "alpha": 0.00, "beta": 0.47},
        "ethyl_acetate":    {"id": "12", "name": "Ethyl Acetate", "polarity": 0.23, "dielectric": 6.0,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.45, "refractive_index": 1.372,
                             "ph": 7.0, "pi_star": 0.55, "alpha": 0.00, "beta": 0.45},
        "isopropanol":      {"id": "13", "name": "Isopropanol", "polarity": 0.55, "dielectric": 18.3,
                             "hbond_donor": 0.76, "hbond_acceptor": 0.84, "refractive_index": 1.377,
                             "ph": 7.0, "pi_star": 0.48, "alpha": 0.76, "beta": 0.84},
        "butanol":          {"id": "14", "name": "Butanol", "polarity": 0.54, "dielectric": 17.8,
                             "hbond_donor": 0.79, "hbond_acceptor": 0.88, "refractive_index": 1.399,
                             "ph": 7.0, "pi_star": 0.47, "alpha": 0.84, "beta": 0.88},
        "pentane":          {"id": "15", "name": "Pentane", "polarity": 0.01, "dielectric": 1.8,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.00, "refractive_index": 1.358,
                             "ph": 7.0, "pi_star": 0.00, "alpha": 0.00, "beta": 0.00},
        "cyclohexane":      {"id": "16", "name": "Cyclohexane", "polarity": 0.02, "dielectric": 2.0,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.00, "refractive_index": 1.426,
                             "ph": 7.0, "pi_star": 0.00, "alpha": 0.00, "beta": 0.00},
        "benzene":          {"id": "17", "name": "Benzene", "polarity": 0.11, "dielectric": 2.3,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.14, "refractive_index": 1.501,
                             "ph": 7.0, "pi_star": 0.59, "alpha": 0.00, "beta": 0.10},
        "thf":              {"id": "18", "name": "Tetrahydrofuran", "polarity": 0.21, "dielectric": 7.6,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.55, "refractive_index": 1.407,
                             "ph": 7.0, "pi_star": 0.58, "alpha": 0.00, "beta": 0.55},
        "dioxane":          {"id": "19", "name": "1,4-Dioxane", "polarity": 0.16, "dielectric": 2.2,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.37, "refractive_index": 1.422,
                             "ph": 7.0, "pi_star": 0.55, "alpha": 0.00, "beta": 0.37},
        "pyridine":         {"id": "20", "name": "Pyridine", "polarity": 0.30, "dielectric": 12.3,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.64, "refractive_index": 1.510,
                             "ph": 7.0, "pi_star": 0.87, "alpha": 0.00, "beta": 0.64},
        "formamide":        {"id": "21", "name": "Formamide", "polarity": 0.80, "dielectric": 111.0,
                             "hbond_donor": 0.75, "hbond_acceptor": 0.65, "refractive_index": 1.447,
                             "ph": 7.0, "pi_star": 0.97, "alpha": 0.71, "beta": 0.48},
        "nitromethane":     {"id": "22", "name": "Nitromethane", "polarity": 0.48, "dielectric": 36.7,
                             "hbond_donor": 0.06, "hbond_acceptor": 0.31, "refractive_index": 1.381,
                             "ph": 7.0, "pi_star": 0.85, "alpha": 0.22, "beta": 0.06},
        "dmf":              {"id": "23", "name": "Dimethylformamide", "polarity": 0.43, "dielectric": 36.7,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.69, "refractive_index": 1.431,
                             "ph": 7.0, "pi_star": 0.88, "alpha": 0.00, "beta": 0.69},
        "mek":              {"id": "24", "name": "Methyl Ethyl Ketone", "polarity": 0.34, "dielectric": 18.5,
                             "hbond_donor": 0.06, "hbond_acceptor": 0.48, "refractive_index": 1.379,
                             "ph": 7.0, "pi_star": 0.65, "alpha": 0.06, "beta": 0.48},
        "n_propanol":       {"id": "25", "name": "n-Propanol", "polarity": 0.62, "dielectric": 20.1,
                             "hbond_donor": 0.84, "hbond_acceptor": 0.78, "refractive_index": 1.384,
                             "ph": 7.0, "pi_star": 0.52, "alpha": 0.84, "beta": 0.78},
        "tert_butanol":     {"id": "26", "name": "tert-Butanol", "polarity": 0.55, "dielectric": 12.5,
                             "hbond_donor": 0.70, "hbond_acceptor": 0.80, "refractive_index": 1.387,
                             "ph": 7.0, "pi_star": 0.41, "alpha": 0.60, "beta": 0.80},
        "dma":              {"id": "27", "name": "Dimethylacetamide", "polarity": 0.41, "dielectric": 37.8,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.76, "refractive_index": 1.438,
                             "ph": 7.0, "pi_star": 0.88, "alpha": 0.00, "beta": 0.76},
        "nmp":              {"id": "28", "name": "N-Methyl-2-pyrrolidone", "polarity": 0.42, "dielectric": 32.2,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.75, "refractive_index": 1.470,
                             "ph": 7.0, "pi_star": 0.85, "alpha": 0.00, "beta": 0.75},
        "dichloroethane":   {"id": "29", "name": "1,2-Dichloroethane", "polarity": 0.31, "dielectric": 10.4,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.10, "refractive_index": 1.444,
                             "ph": 7.0, "pi_star": 0.75, "alpha": 0.00, "beta": 0.10},
        "carbon_tetrachloride": {"id": "30", "name": "Carbon Tetrachloride", "polarity": 0.18, "dielectric": 2.2,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.00, "refractive_index": 1.460,
                             "ph": 7.0, "pi_star": 0.28, "alpha": 0.00, "beta": 0.00},
        "anisole":          {"id": "31", "name": "Anisole", "polarity": 0.20, "dielectric": 4.3,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.20, "refractive_index": 1.517,
                             "ph": 7.0, "pi_star": 0.73, "alpha": 0.00, "beta": 0.22},
        "phenol":           {"id": "32", "name": "Phenol", "polarity": 0.61, "dielectric": 13.0,
                             "hbond_donor": 0.30, "hbond_acceptor": 0.20, "refractive_index": 1.540,
                             "ph": 7.0, "pi_star": 0.70, "alpha": 0.30, "beta": 0.20},
        # ---------- AQUEOUS BUFFERS ----------
        "water_0.1M_HCl":   {"id": "33", "name": "0.1M HCl", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 1.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "water_1.0M_HCl":   {"id": "34", "name": "1.0M HCl", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 0.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "water_0.01M_HCl":  {"id": "35", "name": "0.01M HCl", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 2.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "water_0.1M_HNO3":  {"id": "36", "name": "0.1M HNO3", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 1.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "water_0.1M_H2SO4": {"id": "37", "name": "0.1M H2SO4", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 1.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "water_0.01M_H2SO4": {"id": "38", "name": "0.01M H2SO4", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 1.7, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        # FIX #6: 0.1M acetic acid pH is 2.87, not 4.0
        "water_0.1M_acetic": {"id": "39", "name": "0.1M Acetic Acid", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 2.87, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "water_0.1M_NaOH":  {"id": "40", "name": "0.1M NaOH", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 12.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "water_0.01M_NaOH": {"id": "41", "name": "0.01M NaOH", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 12.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "water_1.0M_NaOH":  {"id": "42", "name": "1.0M NaOH", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 14.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "water_0.1M_KOH":   {"id": "43", "name": "0.1M KOH", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 13.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "water_0.1M_NaHCO3":{"id": "44", "name": "0.1M NaHCO3", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 8.3, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "water_0.1M_Na2CO3":{"id": "45", "name": "0.1M Na2CO3", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 11.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "ammonium_chloride_0.1M": {"id": "46", "name": "0.1M NH4Cl", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 5.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "ammonium_chloride_1.0M": {"id": "47", "name": "1.0M NH4Cl", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 4.6, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "ammonium_hydroxide_0.1M": {"id": "48", "name": "0.1M NH4OH", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 11.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "ammonium_hydroxide_1.0M": {"id": "49", "name": "1.0M NH4OH", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 11.6, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "ammonium_acetate_0.1M": {"id": "50", "name": "0.1M NH4OAc", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 7.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "ammonium_carbonate_0.1M": {"id": "51", "name": "0.1M (NH4)2CO3", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 9.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "phosphate_pH_7":   {"id": "52", "name": "Phosphate pH7", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 7.4, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "phosphate_pH_8":   {"id": "53", "name": "Phosphate pH8", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 8.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "phosphate_pH_9":   {"id": "54", "name": "Phosphate pH9", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 9.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "acetate_pH_4":     {"id": "55", "name": "Acetate pH4", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 4.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "acetate_pH_5":     {"id": "56", "name": "Acetate pH5", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 5.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "tris_pH_7":        {"id": "57", "name": "Tris pH7", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 7.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "tris_pH_8":        {"id": "58", "name": "Tris pH8", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 8.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "tris_pH_9":        {"id": "59", "name": "Tris pH9", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 9.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "citrate_pH_3":     {"id": "60", "name": "Citrate pH3", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 3.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "citrate_pH_5":     {"id": "61", "name": "Citrate pH5", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 5.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "borate_pH_8":      {"id": "62", "name": "Borate pH8", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 8.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "borate_pH_9":      {"id": "63", "name": "Borate pH9", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 9.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "borate_pH_10":     {"id": "64", "name": "Borate pH10", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 10.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "carbonate_pH_9":   {"id": "65", "name": "Carbonate pH9", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 9.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "carbonate_pH_10":  {"id": "66", "name": "Carbonate pH10", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 10.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "nacl_0.9%":        {"id": "67", "name": "0.9% NaCl", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 7.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "nacl_0.1M":        {"id": "68", "name": "0.1M NaCl", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 7.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "kcl_0.1M":         {"id": "69", "name": "0.1M KCl", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 7.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "pbs_ph_7":         {"id": "70", "name": "PBS pH7", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 7.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        "pbs_ph_8":         {"id": "71", "name": "PBS pH8", "polarity": 1.00, "dielectric": 80.1,
                             "hbond_donor": 1.0, "hbond_acceptor": 1.0, "refractive_index": 1.333,
                             "ph": 8.0, "pi_star": 1.09, "alpha": 1.17, "beta": 0.47},
        # ---------- BINARY MIXTURES ----------
        "water_methanol_50_50":     {"id": "72", "name": "Water:Methanol (1:1)", "polarity": 0.88, "dielectric": 56.4,
                             "hbond_donor": 0.99, "hbond_acceptor": 0.83, "refractive_index": 1.331,
                             "ph": 7.0, "pi_star": 0.85, "alpha": 1.05, "beta": 0.57},
        "water_acetonitrile_50_50": {"id": "73", "name": "Water:ACN (1:1)", "polarity": 0.73, "dielectric": 58.8,
                             "hbond_donor": 0.60, "hbond_acceptor": 0.66, "refractive_index": 1.339,
                             "ph": 7.0, "pi_star": 0.90, "alpha": 0.70, "beta": 0.44},
        "water_ethanol_50_50":      {"id": "74", "name": "Water:Ethanol (1:1)", "polarity": 0.83, "dielectric": 52.3,
                             "hbond_donor": 0.93, "hbond_acceptor": 0.88, "refractive_index": 1.347,
                             "ph": 7.0, "pi_star": 0.82, "alpha": 1.00, "beta": 0.62},
        "water_ethanol_25_75":      {"id": "75", "name": "Water:Ethanol (1:3)", "polarity": 0.72, "dielectric": 38.5,
                             "hbond_donor": 0.88, "hbond_acceptor": 0.79, "refractive_index": 1.354,
                             "ph": 7.0, "pi_star": 0.74, "alpha": 0.93, "beta": 0.70},
        "water_ethanol_75_25":      {"id": "76", "name": "Water:Ethanol (3:1)", "polarity": 0.92, "dielectric": 66.2,
                             "hbond_donor": 0.98, "hbond_acceptor": 0.92, "refractive_index": 1.340,
                             "ph": 7.0, "pi_star": 0.92, "alpha": 1.10, "beta": 0.54},
        "ethanol_methanol_50_50":   {"id": "77", "name": "Ethanol:Methanol (1:1)", "polarity": 0.70, "dielectric": 28.6,
                             "hbond_donor": 0.92, "hbond_acceptor": 0.70, "refractive_index": 1.345,
                             "ph": 7.0, "pi_star": 0.57, "alpha": 0.92, "beta": 0.72},
        "water_isopropanol_50_50":  {"id": "78", "name": "Water:Isopropanol (1:1)", "polarity": 0.78, "dielectric": 45.0,
                             "hbond_donor": 0.88, "hbond_acceptor": 0.92, "refractive_index": 1.355,
                             "ph": 7.0, "pi_star": 0.76, "alpha": 0.95, "beta": 0.65},
        "water_acetone_50_50":      {"id": "79", "name": "Water:Acetone (1:1)", "polarity": 0.68, "dielectric": 48.2,
                             "hbond_donor": 0.55, "hbond_acceptor": 0.72, "refractive_index": 1.346,
                             "ph": 7.0, "pi_star": 0.90, "alpha": 0.62, "beta": 0.48},
        "water_dmf_50_50":          {"id": "80", "name": "Water:DMF (1:1)", "polarity": 0.72, "dielectric": 58.0,
                             "hbond_donor": 0.50, "hbond_acceptor": 0.85, "refractive_index": 1.389,
                             "ph": 7.0, "pi_star": 0.98, "alpha": 0.58, "beta": 0.58},
        "water_dmso_50_50":         {"id": "81", "name": "Water:DMSO (1:1)", "polarity": 0.72, "dielectric": 63.0,
                             "hbond_donor": 0.50, "hbond_acceptor": 0.88, "refractive_index": 1.406,
                             "ph": 7.0, "pi_star": 1.04, "alpha": 0.58, "beta": 0.62},
        "water_thf_50_50":          {"id": "82", "name": "Water:THF (1:1)", "polarity": 0.61, "dielectric": 40.0,
                             "hbond_donor": 0.50, "hbond_acceptor": 0.78, "refractive_index": 1.370,
                             "ph": 7.0, "pi_star": 0.83, "alpha": 0.58, "beta": 0.51},
        "methanol_acetonitrile_50_50": {"id": "83", "name": "Methanol:ACN (1:1)", "polarity": 0.61, "dielectric": 35.1,
                             "hbond_donor": 0.58, "hbond_acceptor": 0.49, "refractive_index": 1.336,
                             "ph": 7.0, "pi_star": 0.68, "alpha": 0.58, "beta": 0.53},
        "ethanol_acetonitrile_50_50": {"id": "84", "name": "Ethanol:ACN (1:1)", "polarity": 0.55, "dielectric": 31.0,
                             "hbond_donor": 0.53, "hbond_acceptor": 0.53, "refractive_index": 1.353,
                             "ph": 7.0, "pi_star": 0.65, "alpha": 0.53, "beta": 0.59},
        "water_ethanol_acidic":     {"id": "85", "name": "Water:Ethanol (1:1) + 0.1M HCl", "polarity": 0.83, "dielectric": 52.3,
                             "hbond_donor": 0.95, "hbond_acceptor": 0.88, "refractive_index": 1.347,
                             "ph": 1.0, "pi_star": 0.82, "alpha": 1.10, "beta": 0.62},
        "water_ethanol_basic":      {"id": "86", "name": "Water:Ethanol (1:1) + 0.1M NaOH", "polarity": 0.83, "dielectric": 52.3,
                             "hbond_donor": 0.93, "hbond_acceptor": 0.90, "refractive_index": 1.347,
                             "ph": 12.0, "pi_star": 0.82, "alpha": 1.00, "beta": 0.62},
        "ethanol_acidic":           {"id": "87", "name": "Ethanol + 0.1M HCl", "polarity": 0.65, "dielectric": 24.5,
                             "hbond_donor": 0.90, "hbond_acceptor": 0.75, "refractive_index": 1.361,
                             "ph": 1.0, "pi_star": 0.54, "alpha": 0.90, "beta": 0.77},
        "ethanol_basic":            {"id": "88", "name": "Ethanol + 0.1M NaOH", "polarity": 0.65, "dielectric": 24.5,
                             "hbond_donor": 0.86, "hbond_acceptor": 0.78, "refractive_index": 1.361,
                             "ph": 12.0, "pi_star": 0.54, "alpha": 0.86, "beta": 0.77},
        # ---------- ADDITIONAL SOLVENTS (89–100) ----------
        "95_percent_ethanol":       {"id": "89", "name": "95% Ethanol", "polarity": 0.64, "dielectric": 23.8,
                             "hbond_donor": 0.85, "hbond_acceptor": 0.76, "refractive_index": 1.362,
                             "ph": 7.0, "pi_star": 0.54, "alpha": 0.86, "beta": 0.77},
        "glycerol":                 {"id": "90", "name": "Glycerol", "polarity": 0.90, "dielectric": 42.5,
                             "hbond_donor": 1.00, "hbond_acceptor": 0.80, "refractive_index": 1.474,
                             "ph": 7.0, "pi_star": 0.62, "alpha": 1.00, "beta": 0.80},
        "ethylene_glycol":          {"id": "91", "name": "Ethylene Glycol", "polarity": 0.88, "dielectric": 37.7,
                             "hbond_donor": 0.95, "hbond_acceptor": 0.80, "refractive_index": 1.432,
                             "ph": 7.0, "pi_star": 0.70, "alpha": 0.95, "beta": 0.80},
        "propylene_glycol":         {"id": "92", "name": "Propylene Glycol", "polarity": 0.75, "dielectric": 32.0,
                             "hbond_donor": 0.85, "hbond_acceptor": 0.80, "refractive_index": 1.433,
                             "ph": 7.0, "pi_star": 0.65, "alpha": 0.85, "beta": 0.80},
        "propylene_carbonate":      {"id": "93", "name": "Propylene Carbonate", "polarity": 0.48, "dielectric": 64.9,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.40, "refractive_index": 1.422,
                             "ph": 7.0, "pi_star": 0.83, "alpha": 0.00, "beta": 0.40},
        "gamma_butyrolactone":      {"id": "94", "name": "gamma-Butyrolactone", "polarity": 0.45, "dielectric": 39.0,
                             "hbond_donor": 0.00, "hbond_acceptor": 0.55, "refractive_index": 1.435,
                             "ph": 7.0, "pi_star": 0.80, "alpha": 0.00, "beta": 0.55},
        "2_methoxyethanol":         {"id": "95", "name": "2-Methoxyethanol", "polarity": 0.68, "dielectric": 16.9,
                             "hbond_donor": 0.75, "hbond_acceptor": 0.75, "refractive_index": 1.402,
                             "ph": 7.0, "pi_star": 0.60, "alpha": 0.75, "beta": 0.75},
        "2_ethoxyethanol":          {"id": "96", "name": "2-Ethoxyethanol", "polarity": 0.62, "dielectric": 13.4,
                             "hbond_donor": 0.70, "hbond_acceptor": 0.75, "refractive_index": 1.407,
                             "ph": 7.0, "pi_star": 0.55, "alpha": 0.70, "beta": 0.75},
        "diethylene_glycol":        {"id": "97", "name": "Diethylene Glycol", "polarity": 0.82, "dielectric": 31.7,
                             "hbond_donor": 0.90, "hbond_acceptor": 0.82, "refractive_index": 1.447,
                             "ph": 7.0, "pi_star": 0.68, "alpha": 0.90, "beta": 0.82},
        "triethylene_glycol":       {"id": "98", "name": "Triethylene Glycol", "polarity": 0.80, "dielectric": 23.7,
                             "hbond_donor": 0.88, "hbond_acceptor": 0.85, "refractive_index": 1.455,
                             "ph": 7.0, "pi_star": 0.65, "alpha": 0.88, "beta": 0.85},
        "peg_200":                  {"id": "99", "name": "PEG 200", "polarity": 0.72, "dielectric": 18.0,
                             "hbond_donor": 0.80, "hbond_acceptor": 0.85, "refractive_index": 1.459,
                             "ph": 7.0, "pi_star": 0.60, "alpha": 0.80, "beta": 0.85},
        "peg_400":                  {"id": "100", "name": "PEG 400", "polarity": 0.70, "dielectric": 14.0,
                             "hbond_donor": 0.78, "hbond_acceptor": 0.86, "refractive_index": 1.465,
                             "ph": 7.0, "pi_star": 0.58, "alpha": 0.78, "beta": 0.86},
        }

        # --- Solvent SMILES injection for ReactionEngine ---
        _SOLVENT_SMILES = {
            "water": "O", "ethanol": "CCO", "methanol": "CO",
            "acetonitrile": "CC#N", "dmso": "CS(=O)C", "acetone": "CC(=O)C",
            "chloroform": "ClC(Cl)Cl", "dichloromethane": "ClCCl",
            "toluene": "Cc1ccccc1", "hexane": "CCCCCC",
            "diethyl_ether": "CCOCC", "ethyl_acetate": "CC(=O)OCC",
            "isopropanol": "CC(C)O", "butanol": "CCCCO",
            "pentane": "CCCCC", "cyclohexane": "C1CCCCC1",
            "benzene": "c1ccccc1", "thf": "C1CCOC1",
            "dioxane": "C1COCCO1", "pyridine": "c1ccncc1",
            "formamide": "NC=O", "nitromethane": "C[N+](=O)[O-]",
            "dmf": "CN(C)C=O", "mek": "CCC(C)=O",
            "n_propanol": "CCCO", "tert_butanol": "CC(C)(C)O",
            "dma": "CC(=O)N(C)C", "nmp": "CN1CCCC1=O",
            "dichloroethane": "ClCCCl", "carbon_tetrachloride": "ClC(Cl)(Cl)Cl",
            "anisole": "COc1ccccc1", "phenol": "Oc1ccccc1",
            "glycerol": "OCC(O)CO", "ethylene_glycol": "OCCO",
            "propylene_glycol": "CC(O)CO", "propylene_carbonate": "CC1COC(=O)O1",
            "gamma_butyrolactone": "O=C1CCCO1",
            "2_methoxyethanol": "COCCO", "2_ethoxyethanol": "CCOCCO",
            "diethylene_glycol": "OCCOCCO", "triethylene_glycol": "OCCOCCOCCO",
            "peg_200": "OCCOCCOCCOCCO",
            "peg_400": "OCCOCCOCCOCCOCCOCCOCCOCCO",
            "95_percent_ethanol": "CCO",
        }
        for k in self.solvents:
            if k in _SOLVENT_SMILES:
                continue

            _SALINE_KEYS = {"nacl_0.9%", "nacl_0.1M", "kcl_0.1M", "pbs_ph_7", "pbs_ph_8"}
            if k in _SALINE_KEYS:
                _SOLVENT_SMILES[k] = None          # explicit: no single-molecule SMILES
            elif any(t in k for t in ["water", "buffer", "phosphate", "tris",
                                       "carbonate", "borate", "citrate", "acetate",
                                       "hcl", "naoh", "koh",
                                       "hno3", "h2so4", "nh4", "ammonium", "acetic"]):
                _SOLVENT_SMILES[k] = "O"
            elif "methanol" in k:
                _SOLVENT_SMILES[k] = "CO"
            elif "ethanol" in k:
                _SOLVENT_SMILES[k] = "CCO"
            elif "isopropanol" in k:
                _SOLVENT_SMILES[k] = "CC(C)O"
            elif "acetonitrile" in k:
                _SOLVENT_SMILES[k] = "CC#N"

        for k, v in self.solvents.items():
            v["smiles"] = _SOLVENT_SMILES.get(k, None)
        # --- end SMILES injection ---

        # --- Aqueous vs non-aqueous pH flag (v13.7.3) ---
        _AQUEOUS_TOKENS = (
            "water", "buffer", "phosphate", "tris", "carbonate", "borate",
            "citrate", "acetate", "pbs", "nacl", "kcl", "hcl", "naoh", "koh",
            "hno3", "h2so4", "nh4", "ammonium", "acetic", "_acidic", "_basic"
        )
        _ELECTROLYTE_SPECS = {
            "nacl_0.9%":   {"matrix": "water", "ionic_strength_M": 0.154,
                            "ions": {"Na+": 0.154, "Cl-": 0.154}, "cosolvent": None},
            "nacl_0.1M":   {"matrix": "water", "ionic_strength_M": 0.10,
                            "ions": {"Na+": 0.10, "Cl-": 0.10}, "cosolvent": None},
            "kcl_0.1M":    {"matrix": "water", "ionic_strength_M": 0.10,
                            "ions": {"K+": 0.10, "Cl-": 0.10}, "cosolvent": None},
            "pbs_ph_7":    {"matrix": "water", "ionic_strength_M": 0.15,
                            "ions": {"Na+": 0.15, "Cl-": 0.14, "phosphate": 0.01}, "cosolvent": None},
            "pbs_ph_8":    {"matrix": "water", "ionic_strength_M": 0.15,
                            "ions": {"Na+": 0.15, "Cl-": 0.14, "phosphate": 0.01}, "cosolvent": None},
            "water_0.1M_HCl":  {"matrix": "water", "ionic_strength_M": 0.10,
                                "ions": {"H+": 0.10, "Cl-": 0.10}, "cosolvent": None},
            "water_1.0M_HCl":  {"matrix": "water", "ionic_strength_M": 1.00,
                                "ions": {"H+": 1.00, "Cl-": 1.00}, "cosolvent": None},
            "water_0.01M_HCl": {"matrix": "water", "ionic_strength_M": 0.01,
                                "ions": {"H+": 0.01, "Cl-": 0.01}, "cosolvent": None},
            "water_0.1M_NaOH": {"matrix": "water", "ionic_strength_M": 0.10,
                                "ions": {"Na+": 0.10, "OH-": 0.10}, "cosolvent": None},
            "water_1.0M_NaOH": {"matrix": "water", "ionic_strength_M": 1.00,
                                "ions": {"Na+": 1.00, "OH-": 1.00}, "cosolvent": None},
            "water_0.1M_KOH":  {"matrix": "water", "ionic_strength_M": 0.10,
                                "ions": {"K+": 0.10, "OH-": 0.10}, "cosolvent": None},
            "ammonium_chloride_0.1M": {"matrix": "water", "ionic_strength_M": 0.10,
                                       "ions": {"NH4+": 0.10, "Cl-": 0.10}, "cosolvent": None},
            "ammonium_chloride_1.0M": {"matrix": "water", "ionic_strength_M": 1.00,
                                       "ions": {"NH4+": 1.00, "Cl-": 1.00}, "cosolvent": None},
        }
        for k, v in self.solvents.items():
            is_aqueous = any(tok in k.lower() for tok in _AQUEOUS_TOKENS)
            v["ph_available"] = bool(is_aqueous)
            v["ph_source"] = "aqueous" if is_aqueous else "unavailable"
            # Default: not an electrolyte
            v.setdefault("is_electrolyte", False)
            v.setdefault("ionic_strength_M", 0.0)
            v.setdefault("ions", {})
            v.setdefault("solvent_matrix", k)
            v.setdefault("cosolvent", None)
            if k in _ELECTROLYTE_SPECS:
                spec = _ELECTROLYTE_SPECS[k]
                v["is_electrolyte"]  = True
                v["solvent_matrix"]  = spec["matrix"]
                v["ionic_strength_M"] = spec["ionic_strength_M"]
                v["ions"]            = dict(spec["ions"])
                v["cosolvent"]       = spec["cosolvent"]

        # THIS IS THE MISSING LINE — rebuild the id → key map
        self.id_to_key = {v["id"]: k for k, v in self.solvents.items()}

        self.current_solvent = "water"
        self.current_ph = 7.0

    def is_aqueous(self, code):
        """Return True if the solvent has a physically meaningful pH."""
        props = self.get(code)
        return bool(props.get("ph_available", False))

        self.id_to_key = {v["id"]: k for k, v in self.solvents.items()}
        self.current_solvent = "water"
        self.current_ph = 7.0

    def get(self, code):
        return self.solvents.get(code, self.solvents["water"])

    def list_solvents(self):
        return [f"{k} ({v['id']})" for k, v in sorted(self.solvents.items(), key=lambda kv: int(kv[1]["id"]))]

    def set_solvent(self, code):
        code = code.strip()
        props = None
        if code.isdigit():
            key = self.id_to_key.get(code)
            if key:
                props = self.solvents[key]
                self.current_solvent = key
        if props is None and code in self.solvents:
            props = self.solvents[code]
            self.current_solvent = code
        if props is None:
            matches = [k for k in self.solvents if k.startswith(code.lower())]
            if matches:
                self.current_solvent = matches[0]
                props = self.solvents[self.current_solvent]
                print(f"🔍 Using closest match: {self.current_solvent} ({props['id']})")
        if props is None:
            print(f"❌ Solvent '{code}' not found. Using water.")
            self.current_solvent = "water"
            props = self.solvents["water"]
        self.current_ph = props.get("ph", 7.0)
        print(f"✅ Solvent: {props['name']} (pH={self.current_ph:.1f})")
        return props

    # FIX #9: Kamlet–Taft is now implemented (was a stub returning 0.0)
    def compute_kamlet_taft_shift(self, props, temperature=298.15, chromophore_type="general"):
        pi_star = float(props.get("pi_star", 0.0))
        alpha   = float(props.get("alpha", 0.0))
        beta    = float(props.get("beta", 0.0))
        if chromophore_type == "n→π*":
            s, a, b = -5.0,  5.0,  8.0
        elif chromophore_type == "π→π*":
            s, a, b = 15.0, -8.0, -5.0
        elif chromophore_type == "ICT":
            s, a, b = 25.0, -10.0, -12.0
        else:
            s, a, b = 8.0, -3.0, -2.0
        shift_nm = s * pi_star + a * alpha + b * beta
        temp_delta = float(temperature - 298.15)
        shift_nm += 0.02 * temp_delta
        return float(np.clip(shift_nm, -30.0, 30.0))


# ================================================================
# ChromophoreDetector
# ================================================================
class ChromophoreDetector:
    """Advanced heuristic engine for UV-Vis spectra prediction.
    Third tuple element is molar absorptivity ε (L·mol⁻¹·cm⁻¹);
    normalized to 0–3 intensity at runtime (see evaluate_molecule)."""

    # Reference maximum for ε → intensity normalization (cyanine 5 = 250,000)
    EPS_REF = 250000.0
    INTENSITY_MAX = 3.0

    def __init__(self):
        # ---------- CHROMOPHORES (SMARTS, λmax_nm, ε, bandwidth_nm) ----------
        self.chromophores: dict[str, tuple[Chem.Mol, int, float, int]] = {
            # Aromatic hydrocarbons
            "benzene":                (Chem.MolFromSmarts("c1ccccc1"), 254, 204, 15),
            "naphthalene":            (Chem.MolFromSmarts("c1ccc2ccccc2c1"), 275, 6000, 20),
            "anthracene":             (Chem.MolFromSmarts("c1ccc2cc3ccccc3cc2c1"), 375, 7900, 25),
            "phenanthrene":           (Chem.MolFromSmarts("c1ccc2c(c1)ccc1ccccc21"), 292, 9000, 22),
            "pyrene":                 (Chem.MolFromSmarts("c1cc2ccc3cccc4ccc1c2c34"), 335, 39000, 28),
            "chrysene":               (Chem.MolFromSmarts("c1ccc2c(c1)ccc3c2ccc4ccccc34"), 330, 12000, 24),
            "triphenylene":           (Chem.MolFromSmarts("c1ccc2c(c1)c1ccccc1c1ccccc21"), 258, 11000, 20),
            "perylene":               (Chem.MolFromSmarts("c1ccc2c(c1)c1ccc3c4c1c2ccc4ccc3"), 435, 38000, 30),
            "azulene":                (Chem.MolFromSmarts("c1ccc2cccc2c1"), 350, 9000, 25),
            "fluorene":               (Chem.MolFromSmarts("c1ccc2c(c1)Cc3ccccc23"), 265, 8000, 20),
            "dibenzofuran":           (Chem.MolFromSmarts("c1ccc2c(c1)oc1ccccc21"), 280, 8000, 22),
            "carbazole":              (Chem.MolFromSmarts("c1ccc2c(c1)c3ccccc3[nH]2"), 330, 8000, 22),
            "dibenzothiophene":       (Chem.MolFromSmarts("c1ccc2c(c1)sc1ccccc21"), 285, 7000, 20),
            "coronene":               (Chem.MolFromSmarts("c1cc2ccc3ccc4ccc5ccc6ccc1c1c2c3c4c5c61"), 340, 50000, 35),
            "ovalene":                (Chem.MolFromSmarts("c1cc2ccc3ccc4ccc5ccc6ccc7ccc1c1c2c3c4c5c6c71"), 360, 60000, 38),
            "benzo_a_pyrene":         (Chem.MolFromSmarts("c1ccc2c(c1)cc1ccc3cccc4ccc2c1c34"), 380, 25000, 32),
            # N-heterocycles
            "pyridine":               (Chem.MolFromSmarts("c1ccncc1"), 251, 2000, 18),
            "pyrimidine":             (Chem.MolFromSmarts("c1cncnc1"), 245, 2000, 18),
            "pyrazine":               (Chem.MolFromSmarts("c1cnccn1"), 330, 5000, 18),
            "pyridazine":             (Chem.MolFromSmarts("c1cnncc1"), 250, 2000, 18),
            "triazine":               (Chem.MolFromSmarts("c1ncncn1"), 240, 1500, 16),
            "purine":                 (Chem.MolFromSmarts("c1ncnc2ncnc12"), 265, 9000, 22),
            "quinazoline":            (Chem.MolFromSmarts("c1cnc2ccccc2n1"), 265, 9000, 22),
            "quinoline":              (Chem.MolFromSmarts("c1ccc2ncccc2c1"), 310, 9000, 25),
            "isoquinoline":           (Chem.MolFromSmarts("c1ccc2cnccc2c1"), 310, 9000, 25),
            "quinoxaline":            (Chem.MolFromSmarts("c1ccc2nccnc2c1"), 315, 8000, 22),
            "cinnoline":              (Chem.MolFromSmarts("c1ccc2c(c1)cnnc2"), 305, 7000, 20),
            "phthalazine":            (Chem.MolFromSmarts("c1ccc2c(c1)cnnc2"), 300, 7000, 20),
            "acridine":               (Chem.MolFromSmarts("c1ccc2c(c1)nc1ccccc1c2"), 357, 9000, 25),
            "phenazine":              (Chem.MolFromSmarts("c1ccc2c(c1)nc3ccccc3n2"), 360, 9500, 28),
            "phenanthroline":         (Chem.MolFromSmarts("c1cnc2c(c1)c1cccnc1n2"), 260, 6000, 20),
            "bipyridine":             (Chem.MolFromSmarts("c1ccncc1-c2ccncc2"), 280, 7000, 22),
            "terpyridine":            (Chem.MolFromSmarts("c1ccncc1-c2cc(-c3ccncc3)ccn2"), 290, 8000, 25),
            "pteridine":              (Chem.MolFromSmarts("c1cnc2ncnc2n1"), 280, 8000, 20),
            "naphthyridine":          (Chem.MolFromSmarts("c1ccc2ncccc2n1"), 310, 9000, 24),
            "phenanthridine":         (Chem.MolFromSmarts("c1ccc2c(c1)cc1ccccc1n2"), 350, 10000, 28),
            # Five-membered heterocycles
            "indole":                 (Chem.MolFromSmarts("c1ccc2c(c1)cc[nH]2"), 280, 7000, 20),
            "pyrrole":                (Chem.MolFromSmarts("c1cc[nH]c1"), 210, 5000, 15),
            "imidazole":              (Chem.MolFromSmarts("c1cnc[nH]1"), 210, 5000, 14),
            "benzimidazole":          (Chem.MolFromSmarts("c1ccc2c(c1)ncn2"), 240, 7000, 20),
            "thiazole":               (Chem.MolFromSmarts("c1cscn1"), 240, 4000, 16),
            "oxazole":                (Chem.MolFromSmarts("c1cocn1"), 230, 4000, 16),
            "isoxazole":              (Chem.MolFromSmarts("c1cocn1"), 225, 3000, 12),
            "triazole":               (Chem.MolFromSmarts("c1n[nH]cn1"), 215, 3000, 10),
            "tetrazole":              (Chem.MolFromSmarts("c1nnn[nH]1"), 210, 3000, 15),
            "oxadiazole":             (Chem.MolFromSmarts("c1nocn1"), 235, 4000, 15),
            "thiadiazole":            (Chem.MolFromSmarts("c1nscn1"), 245, 5000, 16),
            "benzothiazole":          (Chem.MolFromSmarts("c1ccc2scnc2c1"), 285, 8000, 20),
            "benzoxazole":            (Chem.MolFromSmarts("c1ccc2ocnc2c1"), 275, 7000, 20),
            "benzisoxazole":          (Chem.MolFromSmarts("c1ccc2onc2c1"), 275, 7000, 20),
            "benzothiophene":         (Chem.MolFromSmarts("c1ccc2c(c1)ccs2"), 260, 6000, 20),
            "benzofuran":             (Chem.MolFromSmarts("c1ccc2c(c1)cco2"), 255, 6000, 20),
            "thiophene":              (Chem.MolFromSmarts("c1ccsc1"), 235, 5000, 15),
            "furan":                  (Chem.MolFromSmarts("c1ccoc1"), 220, 5000, 16),
            "selenophene":            (Chem.MolFromSmarts("c1cc[Se]c1"), 240, 5000, 16),
            "tellurophene":           (Chem.MolFromSmarts("c1cc[Te]c1"), 250, 5000, 16),
            "pyrazole":               (Chem.MolFromSmarts("c1cc[nH]n1"), 210, 5000, 14),
            "indazole":               (Chem.MolFromSmarts("c1ccc2[nH]ncc2c1"), 290, 8000, 22),
            "benzotriazole":          (Chem.MolFromSmarts("c1ccc2[nH]nnc2c1"), 275, 7000, 20),
            # Carbonyl chromophores
            "carbonyl":               (Chem.MolFromSmarts("C=O"), 280, 18, 20),
            "ketone":                 (Chem.MolFromSmarts("[CX3](=O)[#6]"), 285, 20, 21),
            "aldehyde":               (Chem.MolFromSmarts("[CX3H1](=O)"), 290, 30, 22),
            "ester":                  (Chem.MolFromSmarts("C(=O)O"), 210, 60, 18),
            "amide":                  (Chem.MolFromSmarts("C(=O)N"), 220, 100, 19),
            "carboxyl":               (Chem.MolFromSmarts("C(=O)O"), 210, 60, 18),
            "anhydride":              (Chem.MolFromSmarts("C(=O)OC(=O)"), 225, 500, 25),
            "acid_chloride":          (Chem.MolFromSmarts("C(=O)Cl"), 240, 400, 18),
            "acid_bromide":           (Chem.MolFromSmarts("C(=O)Br"), 245, 400, 18),
            "thioamide":              (Chem.MolFromSmarts("C(=S)N"), 270, 15000, 25),
            "thioketone":             (Chem.MolFromSmarts("C=S"), 500, 10, 30),
            "thioester":              (Chem.MolFromSmarts("C(=O)S"), 240, 5000, 22),
            # Conjugated carbonyls (Woodward–Fieser)
            "enone":                  (Chem.MolFromSmarts("C=CC=O"), 320, 7000, 25),
            "dienone":                (Chem.MolFromSmarts("C=CC=CC=O"), 345, 12000, 28),
            "trienone":               (Chem.MolFromSmarts("C=CC=CC=CC=O"), 370, 18000, 30),
            # FIX #10: Woodward–Fieser aldehyde & acid bases
            "enone_aldehyde":         (Chem.MolFromSmarts("[CX3H1](=O)C=C"), 207, 5000, 22),
            "enone_acid":             (Chem.MolFromSmarts("C(=O)OC=C"), 208, 5000, 22),
            # Quinones / anthraquinones
            "quinone":                (Chem.MolFromSmarts("O=C1C=CC(=O)C=C1"), 450, 12500, 30),
            "benzoquinone_deriv":     (Chem.MolFromSmarts("O=C1C=CC(=O)C(C)=C1"), 460, 13000, 30),
            "naphthoquinone":         (Chem.MolFromSmarts("O=C1C=CC(=O)C2=CC=CC=C12"), 480, 15000, 32),
            "juglone":                (Chem.MolFromSmarts("O=C1C=CC(=O)c2cc(O)ccc12"), 430, 9000, 26),
            "lawsone":                (Chem.MolFromSmarts("O=C1C=CC(=O)c2cc(O)ccc12"), 460, 9000, 26),
            "benzoquinone":           (Chem.MolFromSmarts("O=C1C=CC(=O)C=C1"), 260, 5000, 18),
            "anthraquinone":          (Chem.MolFromSmarts("O=C1c2ccccc2C(=O)c3ccccc13"), 390, 9000, 28),
            "aminoanthraquinone":     (Chem.MolFromSmarts("O=C1c2ccccc2C(=O)c3cc(N)ccc13"), 480, 10000, 30),
            "diaminoanthraquinone":   (Chem.MolFromSmarts("Nc1ccc2C(=O)c3ccccc3C(=O)c2c1N"), 560, 12000, 32),
            "hydroxyanthraquinone":   (Chem.MolFromSmarts("O=C1c2ccccc2C(=O)c3cc(O)ccc13"), 420, 9000, 28),
            "dihydroxyanthraquinone": (Chem.MolFromSmarts("O=C1c2ccccc2C(=O)c3c(O)ccc(O)c13"), 490, 11000, 30),
            "alizarin":               (Chem.MolFromSmarts("O=C1c2ccccc2C(=O)c3cc(O)c(O)cc13"), 440, 10000, 28),
            "purpurin":               (Chem.MolFromSmarts("O=C1c2cc(O)c(O)cc2C(=O)c3cc(O)c(O)cc13"), 480, 12000, 30),
            "phenanthrenequinone":    (Chem.MolFromSmarts("O=C1c2ccccc2C(=O)c2ccccc12"), 500, 16000, 28),
            "aminoanthraquinone_general": (Chem.MolFromSmarts("Nc1ccc2C(=O)c3ccccc3C(=O)c2c1"), 500, 10000, 30),
            # Azo / nitro / nitroso
            "azo_trans":              (Chem.MolFromSmarts("c1ccc(/N=N/c2ccccc2)cc1"), 340, 22000, 35),
            "azo_cis":                (Chem.MolFromSmarts("c1ccc(/N=N\\c2ccccc2)cc1"), 280, 5200, 30),
            "azoxy":                  (Chem.MolFromSmarts("N(=O)N"), 350, 8000, 22),
            "diazo":                  (Chem.MolFromSmarts("N=Nc1ccccc1N=N"), 480, 12000, 35),
            "dinitro_aromatic":       (Chem.MolFromSmarts("c1cc([N+](=O)[O-])c([N+](=O)[O-])cc1"), 270, 13000, 28),
            "nitro":                  (Chem.MolFromSmarts("[N+](=O)[O-]"), 240, 3000, 20),
            "nitro_aromatic":         (Chem.MolFromSmarts("c1ccccc1[N+](=O)[O-]"), 250, 12000, 25),
            "nitroaniline":           (Chem.MolFromSmarts("Nc1ccc([N+](=O)[O-])cc1"), 380, 12800, 30),
            "nitrophenol":            (Chem.MolFromSmarts("Oc1ccc([N+](=O)[O-])cc1"), 315, 10000, 28),
            "nitroimidazole":         (Chem.MolFromSmarts("c1c[nH]c([N+](=O)[O-])n1"), 320, 10000, 30),
            "nitrothiophene":         (Chem.MolFromSmarts("c1cc([N+](=O)[O-])sc1"), 350, 12000, 28),
            "nitrofuran":             (Chem.MolFromSmarts("c1cc([N+](=O)[O-])oc1"), 340, 12000, 28),
            "nitropyrrole":           (Chem.MolFromSmarts("c1cc([N+](=O)[O-])[nH]1"), 330, 11000, 28),
            "nitroso":                (Chem.MolFromSmarts("N=O"), 320, 7000, 28),
            "nitrate":                (Chem.MolFromSmarts("[O-][N+](=O)[O-]"), 300, 5000, 20),
            # Substituted aromatics
            "aniline":                (Chem.MolFromSmarts("Nc1ccccc1"), 280, 1430, 22),
            "phenol":                 (Chem.MolFromSmarts("Oc1ccccc1"), 270, 1450, 20),
            "biphenyl":               (Chem.MolFromSmarts("c1ccc(cc1)-c2ccccc2"), 248, 17000, 25),
            "terphenyl":              (Chem.MolFromSmarts("c1ccc(cc1)-c2ccc(-c3ccccc3)cc2"), 250, 20000, 28),
            "stilbene":               (Chem.MolFromSmarts("c1ccc(C=Cc2ccccc2)cc1"), 310, 25000, 22),
            "distyrylbenzene":        (Chem.MolFromSmarts("c1ccc(C=Cc2ccc(C=Cc3ccccc3)cc2)cc1"), 370, 40000, 28),
            "tolan":                  (Chem.MolFromSmarts("c1ccc(C#Cc2ccccc2)cc1"), 290, 20000, 22),
            "styrene":                (Chem.MolFromSmarts("c1ccc(C=C)cc1"), 245, 14000, 18),
            "phenylacetylene":        (Chem.MolFromSmarts("c1ccc(C#C)cc1"), 255, 15000, 18),
            "diphenylmethane":        (Chem.MolFromSmarts("c1ccccc1Cc2ccccc2"), 260, 12000, 22),
            "diphenyl_ether":         (Chem.MolFromSmarts("c1ccccc1Oc2ccccc2"), 270, 15000, 25),
            "diphenylamine":          (Chem.MolFromSmarts("c1ccccc1Nc2ccccc2"), 285, 16000, 26),
            # Dyes
            "fluorescein":            (Chem.MolFromSmarts("O=C1OC2(c3ccc(O)cc3Oc4cc(O)ccc24)c5ccccc15"), 490, 92300, 28),
            "rhodamine":              (Chem.MolFromSmarts("O=C1OC2(c3ccc(N)cc3Oc4cc(N)ccc24)c5ccccc15"), 540, 106000, 25),
            "rhodamine_B":            (Chem.MolFromSmarts("CCN(CC)c1ccc2c(c1)Oc1cc(=[N+](CC)CC)ccc1C21OC(=O)c2ccccc21"), 550, 106000, 28),
            "eosin":                  (Chem.MolFromSmarts("O=C1OC2(c3ccc(Br)c(O)c3BrOc4c(Br)c(O)c(Br)cc24)c5cc(Br)c(Br)c(Br)c15"), 518, 76000, 30),
            "erythrosin":             (Chem.MolFromSmarts("O=C1OC2(c3ccc(I)c(O)c3IOc4c(I)c(O)c(I)cc24)c5cc(I)c(I)c(I)c15"), 530, 80000, 30),
            "triphenylmethane":       (Chem.MolFromSmarts("c1ccc(C(c2ccccc2)(c3ccccc3))cc1"), 420, 25000, 30),
            "malachite_green":        (Chem.MolFromSmarts("CN(C)c1ccc(C(c2ccccc2)=C3C=CC(=C(C)C)C=C3)cc1"), 617, 110000, 35),
            "crystal_violet":         (Chem.MolFromSmarts("CN(C)c1ccc(C(c2ccc(N(C)C)cc2)=c3ccc(N(C)C)cc3)cc1"), 590, 120000, 35),
            "cresyl_violet":          (Chem.MolFromSmarts("CN(C)c1ccc(C(c2ccc(N(C)C)cc2)=c3ccc(N(C)C)cc3)cc1"), 590, 100000, 35),
            "methyl_violet":          (Chem.MolFromSmarts("CN(C)c1ccc(C(c2ccc(N(C)C)cc2)(c3ccc(N(C)C)cc3))cc1"), 585, 100000, 32),
            "bromocresol_green":      (Chem.MolFromSmarts("O=C1OC2(c3c(Br)c(O)c(Br)cc3Oc4c(Br)c(O)c(Br)cc24)c5ccc(S(=O)(=O)O)cc15"), 615, 30000, 35),
            "azobenzene":             (Chem.MolFromSmarts("c1ccc(/N=N/c2ccccc2)cc1"), 340, 22000, 35),
            "methyl_orange":          (Chem.MolFromSmarts("CN(C)c1ccc(N=Nc2ccc(S(=O)(=O)[O-])cc2)cc1"), 465, 28000, 30),
            "disperse_red":           (Chem.MolFromSmarts("Nc1ccc(N=Nc2cc([N+](=O)[O-])ccc2O)cc1"), 480, 35000, 35),
            "disperse_orange":        (Chem.MolFromSmarts("O=[N+]([O-])c1ccc(N=Nc2ccc(O)cc2)cc1"), 440, 25000, 30),
            "disperse_yellow":        (Chem.MolFromSmarts("O=[N+]([O-])c1ccc(N=Nc2ccc(N(C)C)cc2)cc1"), 450, 28000, 30),
            "reactive_black_5":       (Chem.MolFromSmarts("O=S(=O)(O)c1ccc(N=Nc2ccc(N=Nc3ccc(O)c(N)c3)c(O)c2)cc1"), 597, 40000, 40),
            "congo_red":              (Chem.MolFromSmarts("Nc1ccc(N=Nc2ccc(-c3ccc(N=Nc4ccc(N)cc4)cc3)cc2)cc1"), 500, 45000, 38),
            "indigo":                 (Chem.MolFromSmarts("O=C1C=Cc2ccccc2N1C=C3C(=O)C=CC4=C3N4"), 600, 20000, 40),
            "thioindigo":             (Chem.MolFromSmarts("O=C1C=Cc2ccccc2S1C=C3C(=O)C=CC4=C3S4"), 540, 15000, 35),
            "methylene_blue":         (Chem.MolFromSmarts("CN(C)c1ccc2c(c1)sc1cc(=[N+](C)C)ccc1n2"), 664, 95000, 22),
            "indocyanine_green":      (Chem.MolFromSmarts("c1ccc2c(c1)C(C)(C)C=CN2=CC=CC=CC=CC=C3[N+](=C)C4=CC=CC=C4C3(C)C"), 780, 250000, 45),
            "cyanine_cy7":            (Chem.MolFromSmarts("O=S(=O)([O-])c1ccc2c(c1)[N+](CCCC(=O)[O-])=C(C=C2)C=CC=CC=C3N(CCCC(=O)[O-])c4ccc(cc4C3)S(=O)(=O)[O-]"), 750, 250000, 40),
            "nile_red":               (Chem.MolFromSmarts("CCN(CC)c1ccc2nc3ccc(=O)cc3oc2c1"), 550, 38000, 40),
            # Coumarins
            "coumarin":               (Chem.MolFromSmarts("O=c1ccc2ccccc2o1"), 280, 10000, 28),
            "hydroxycoumarin":        (Chem.MolFromSmarts("O=c1c(O)ccc2ccccc12"), 325, 14000, 30),
            "aminocoumarin":          (Chem.MolFromSmarts("O=c1c(N)ccc2ccccc12"), 350, 18000, 32),
            "methoxycoumarin":        (Chem.MolFromSmarts("O=c1c(OC)ccc2ccccc12"), 310, 12000, 28),
            "methylcoumarin":         (Chem.MolFromSmarts("O=c1c(C)ccc2ccccc12"), 295, 10000, 26),
            "coumarin_343":           (Chem.MolFromSmarts("O=c1ccc2c(c1)N(CC)CC2"), 370, 20000, 30),
            "umbelliferone":          (Chem.MolFromSmarts("O=c1ccc2cc(O)ccc2o1"), 325, 14000, 28),
            "scopoletin":             (Chem.MolFromSmarts("O=c1ccc2cc(O)c(OC)cc2o1"), 350, 16000, 30),
            "psoralen":               (Chem.MolFromSmarts("O=c1ccc2cc3ccoc3cc2o1"), 340, 18000, 28),
            # Cyanines
            "cyanine_3":              (Chem.MolFromSmarts("[NX3]=CC=CC=C[NX3]"), 550, 150000, 30),
            "cyanine_5":              (Chem.MolFromSmarts("[NX3]=CC=CC=CC=C[NX3]"), 650, 250000, 35),
            "cyanine_7":              (Chem.MolFromSmarts("[NX3]=CC=CC=CC=CC=C[NX3]"), 750, 250000, 40),
            "hemicyanine":            (Chem.MolFromSmarts("[NX3]=CC=CC=Cc1ccccc1"), 450, 50000, 30),
            "trimethine_cyanine":     (Chem.MolFromSmarts("[NX3]=CC=CC=[NX3]"), 550, 150000, 30),
            "pentamethine_cyanine":   (Chem.MolFromSmarts("[NX3]=CC=CC=CC=[NX3]"), 650, 250000, 35),
            "heptamethine_cyanine":   (Chem.MolFromSmarts("[NX3]=CC=CC=CC=CC=[NX3]"), 750, 250000, 40),
            # Polyenes
            "butadiene":              (Chem.MolFromSmarts("C=CC=C"), 217, 21000, 20),
            "hexatriene":             (Chem.MolFromSmarts("C=CC=CC=C"), 260, 35000, 22),
            "octatetraene":           (Chem.MolFromSmarts("C=CC=CC=CC=C"), 300, 50000, 25),
            "decapentaene":           (Chem.MolFromSmarts("C=CC=CC=CC=CC=C"), 335, 70000, 28),
            "carotene":               (Chem.MolFromSmarts("C=CC=CC=CC=CC=CC=CC=CC=CC=C"), 450, 140000, 35),
            "lycopene":               (Chem.MolFromSmarts("C=CC=CC=CC=CC=CC=CC=CC=CC=CC=C"), 470, 185000, 38),
            "astaxanthin":            (Chem.MolFromSmarts("CC1=C(C=CC(=CC=CC(=CC=CC(=CC=CC(=CC=CC(=C)C(C)(O)C)C)C)C)C)C(C)(C)CCC1O"), 485, 125000, 38),
            "retinal":                (Chem.MolFromSmarts("CC1=C(C=CC(=CC=CC(=CC=CC(=O)C)C)C)C(C)(C)CCC1"), 380, 40000, 30),
            "vitamin_A":              (Chem.MolFromSmarts("CC1=C(C=CC(=CC=CC(=CC=CCO)C)C)C(C)(C)CCC1"), 325, 45000, 28),
            # Porphyrins
            "porphyrin":              (Chem.MolFromSmarts("c1cc2ccc3ccc4ccc5ccc1n5c2n3n4"), 405, 172000, 30),
            "chlorin":                (Chem.MolFromSmarts("c1cc2ccc3c4cc5c6c(cc6)c(cc5)c4c(cc3)c2c1"), 650, 40000, 35),
            "bacteriochlorin":        (Chem.MolFromSmarts("c1cc2c(cc3c4c5c6c7c(c8c9c(c(c1)c2)c3)c4)c5c6c7c8c9"), 750, 80000, 40),
            "phthalocyanine":         (Chem.MolFromSmarts("c1ccc2c3nc(nc4[nH]c(nc5nc(nc6[nH]c(n3)c7ccccc67)c8ccccc58)c9ccccc49)c2c1"), 670, 140000, 35),
            "porphyrin_Zn":           (Chem.MolFromSmarts("[Zn]c1cc2ccc3ccc4ccc5ccc1n5c2n3n4"), 420, 200000, 30),
            "porphyrin_Mg":           (Chem.MolFromSmarts("[Mg]c1cc2ccc3ccc4ccc5ccc1n5c2n3n4"), 410, 180000, 30),
            "heme":                   (Chem.MolFromSmarts("CC1=C(CCC(=O)O)C2=CC3=C(C(C)=C(C=C)C3C)C3=CC4=C(C(C)=C(C=C)C4C)C=C1N2[Fe]N3"), 400, 100000, 35),
            "chlorophyll_a":          (Chem.MolFromSmarts("CCC1=C(C)C2=CC3=C(C(C)=C(C=C)C3C)C3=CC4=C(C(C)=C(C=C)C4C)C=C1N2[Mg]N3"), 660, 89800, 35),
            "chlorophyll_b":          (Chem.MolFromSmarts("CCC1=C(C)C2=CC3=C(C(C)=C(C=O)C3C)C3=CC4=C(C(C)=C(C=C)C4C)C=C1N2[Mg]N3"), 640, 56000, 35),
            # Other chromophores
            "indolinone":             (Chem.MolFromSmarts("O=C1C=Cc2ccccc12"), 300, 12000, 28),
            "carbostyril":            (Chem.MolFromSmarts("O=c1cc2ccccc2[nH]1"), 308, 12000, 25),
            "4H_chromene":            (Chem.MolFromSmarts("O1c2ccccc2C=CC1"), 315, 8000, 22),
            "phthalimide":            (Chem.MolFromSmarts("O=C1c2ccccc2C(=O)N1"), 240, 6000, 22),
            "xanthone":               (Chem.MolFromSmarts("O=C1c2ccccc2Oc2ccccc12"), 330, 10000, 30),
            "xanthene":               (Chem.MolFromSmarts("c1ccc2c(c1)oc3ccccc3c2"), 490, 30000, 28),
            "thioxanthone":           (Chem.MolFromSmarts("O=C1c2ccccc2Sc2ccccc12"), 380, 12000, 30),
            "acridone":               (Chem.MolFromSmarts("O=C1c2ccccc2Nc2ccccc12"), 400, 15000, 30),
            "squaraine":              (Chem.MolFromSmarts("O=C1C(=CC=C2C(=O)C3=C(C=CC=C3)C2=O)C4=C1C=CC=C4"), 650, 250000, 35),
            "croconaine":             (Chem.MolFromSmarts("O=C1C(=CC=CC=C2C(=O)C3=C(C=CC=C3)C2=O)C4=C1C=CC=C4"), 700, 300000, 40),
            "bodipy":                 (Chem.MolFromSmarts("c1cc2n(c1)B(F)(F)n3cccc3=C2"), 505, 94000, 25),
            "phenothiazine":          (Chem.MolFromSmarts("c1ccc2nc3ccccc3sc2c1"), 265, 10000, 35),
            "phenoxazine":            (Chem.MolFromSmarts("c1ccc2oc3ccccc3nc2c1"), 270, 9000, 30),
            "phenoselenazine":        (Chem.MolFromSmarts("c1ccc2[Se]c3ccccc3nc2c1"), 275, 9000, 30),
            "quinophthalone":         (Chem.MolFromSmarts("O=C1N(c2ccccc2C3=Nc4ccccc4C3)C(=O)c5ccccc15"), 445, 12000, 35),
            "diketopyrrolopyrrole":   (Chem.MolFromSmarts("O=C1C=C(c2ccccc2)N(C1=O)c3ccccc3"), 500, 25000, 40),
            "quinacridone":           (Chem.MolFromSmarts("O=C1c2ccccc2Nc3cc4c(cc13)C(=O)c5ccccc5N4"), 480, 20000, 35),
            # Pharmaceutical cores
            "quinolone_core":         (Chem.MolFromSmarts("O=C1C=CN(C)c2ccccc12"), 315, 9000, 25),
            "tetracycline_core":      (Chem.MolFromSmarts("c1cc2c(cc1)cc3c(c2)CC4C(C3)CCC4"), 355, 18000, 30),
            "penam_core":             (Chem.MolFromSmarts("N1C(=O)C2C1SC(C)(C)C2"), 225, 4000, 20),
            "cephem_core":            (Chem.MolFromSmarts("O=C1N2C(C(=O)O)=C(C)CSC12"), 260, 7000, 22),
            "carbapenem_core":        (Chem.MolFromSmarts("O=C1N2C(C(=O)O)=C(C)CC12"), 240, 5000, 20),
            "monobactam_core":        (Chem.MolFromSmarts("O=S(=O)(O)N1C(=O)C=CC1"), 230, 4000, 18),
            "benzodiazepine":         (Chem.MolFromSmarts("c1ccc2c(c1)NC=CN=C2"), 260, 6000, 30),
            "benzothiadiazine":       (Chem.MolFromSmarts("c1ccc2c(c1)S(=O)(=O)NCN2"), 270, 8000, 22),
            "barbiturate_core":       (Chem.MolFromSmarts("O=C1NC(=O)NC(=O)C1"), 240, 5000, 20),
            "hydantoin_core":         (Chem.MolFromSmarts("O=C1NCNC1=O"), 235, 4000, 18),
            "sulfonylurea_core":      (Chem.MolFromSmarts("S(=O)(=O)NC(=O)N"), 230, 3000, 20),
            "biguanide_core":         (Chem.MolFromSmarts("NC(=N)NC(=N)N"), 230, 3000, 18),
            "statine_core":           (Chem.MolFromSmarts("OC(CC(O)CC(=O)O)"), 210, 2000, 15),
            # Biological chromophores
            "tryptophan":             (Chem.MolFromSmarts("c1ccc2c(c1)c(c[nH]2)CC(N)C(=O)O"), 280, 5600, 25),
            "tyrosine":               (Chem.MolFromSmarts("c1ccc(cc1)CC(N)C(=O)O"), 275, 1400, 20),
            "phenylalanine":          (Chem.MolFromSmarts("c1ccc(cc1)CC(N)C(=O)O"), 258, 200, 18),
            "histidine":              (Chem.MolFromSmarts("c1c[nH]cn1CC(N)C(=O)O"), 210, 3000, 14),
            "NADH":                   (Chem.MolFromSmarts("NC(=O)c1cc[nH]c2c1C(C(O)C(O)COP(=O)(O)OP(=O)(O)OCC3OC(n4ccccc4=O)C(O)C3O)C2"), 340, 6200, 30),
            "FAD":                    (Chem.MolFromSmarts("CC1=CC2=C(C3=NC(=O)NC3=O)N(C)C(=O)N2C1=O"), 450, 11300, 35),
            "riboflavin":             (Chem.MolFromSmarts("CC1=CC2=C(C3=NC(=O)NC3=O)N(C)C(=O)N2C1=O"), 445, 12500, 30),
            # Flavonoids
            "flavone":                (Chem.MolFromSmarts("O=C1C=C(c2ccccc2)oc3ccccc13"), 340, 18000, 30),
            "flavonol":               (Chem.MolFromSmarts("O=C1C(O)=C(c2ccccc2)oc3ccccc13"), 370, 22000, 30),
            "quercetin":              (Chem.MolFromSmarts("O=C1C(O)=C(c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12"), 370, 24000, 32),
            "kaempferol":             (Chem.MolFromSmarts("O=C1C(O)=C(c2ccc(O)cc2)oc2cc(O)cc(O)c12"), 365, 20000, 30),
            "myricetin":              (Chem.MolFromSmarts("O=C1C(O)=C(c2cc(O)c(O)c(O)c2)oc2cc(O)cc(O)c12"), 375, 26000, 32),
            "apigenin":               (Chem.MolFromSmarts("O=C1C=C(c2ccc(O)cc2)oc2cc(O)cc(O)c12"), 335, 18000, 28),
            "luteolin":               (Chem.MolFromSmarts("O=C1C=C(c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12"), 350, 20000, 28),
            "catechin":               (Chem.MolFromSmarts("Oc1cc(O)c2c(c1)OC(c1ccc(O)c(O)c1)C2"), 280, 4000, 22),
            "epicatechin":            (Chem.MolFromSmarts("Oc1cc(O)c2c(c1)OC(c1ccc(O)c(O)c1)C2"), 280, 4000, 22),
            "naringenin":             (Chem.MolFromSmarts("O=C1CC(c2ccc(O)cc2)oc2cc(O)cc(O)c12"), 288, 15000, 28),
            "chrysin":                (Chem.MolFromSmarts("O=C1C=C(c2ccccc2)oc2cc(O)cc(O)c12"), 270, 28000, 25),
            "chalcone":               (Chem.MolFromSmarts("O=C(C=Cc1ccccc1)c2ccccc2"), 320, 20000, 25),
            "curcumin":               (Chem.MolFromSmarts("COc1ccc(C=CC(=O)CC(=O)C=Cc2ccc(OC)c(O)c2)cc1O"), 420, 55000, 40),
            # Cyano-benzenes
            "cyano_benzene":          (Chem.MolFromSmarts("c1ccc(C#N)cc1"), 270, 12000, 22),
            "dicyano_benzene":        (Chem.MolFromSmarts("c1ccc(C#N)c(C#N)c1"), 290, 15000, 25),
            # Tropones
            "tropone":                (Chem.MolFromSmarts("O=C1C=CC=CC=C1"), 300, 7000, 22),
            "tropolone":              (Chem.MolFromSmarts("O=c1c(O)cccc1"), 385, 9000, 25),
            "azulene_ketone":         (Chem.MolFromSmarts("O=C1C=CC2=CC=CC=C12"), 350, 8000, 25),
            # Miscellaneous
            "phenolphthalein":        (Chem.MolFromSmarts("O=C1OC2(c3ccc(O)cc3Oc4cc(O)ccc24)c5ccccc15"), 550, 30000, 35),
            # Simple chromophores
            "ethene":                 (Chem.MolFromSmarts("C=C"), 180, 15000, 12),
            "propadiene":             (Chem.MolFromSmarts("C=C=C"), 220, 18000, 15),
            "butatriene":             (Chem.MolFromSmarts("C=C=C=C"), 240, 22000, 18),
            "cyclopropene":           (Chem.MolFromSmarts("C1=CC1"), 190, 10000, 12),
            "cyclobutene":            (Chem.MolFromSmarts("C1=CCC1"), 200, 12000, 14),
            "cyclopentadiene":        (Chem.MolFromSmarts("C1=CC=CC1"), 240, 15000, 16),
            "cyclohexadiene":         (Chem.MolFromSmarts("C1=CC=CCC1"), 260, 18000, 18),
            "cyclooctatetraene":      (Chem.MolFromSmarts("C1=CC=CC=CC=C1"), 300, 25000, 22),
            "fulvene":                (Chem.MolFromSmarts("C=C1C=CC=C1"), 280, 15000, 20),
            "pentalene":              (Chem.MolFromSmarts("c1cc2cccc2c1"), 320, 20000, 22),
            # Polyazines
            "pentazine":              (Chem.MolFromSmarts("n1nnnnc1"), 575, 12000, 40),
            "tetrazine":              (Chem.MolFromSmarts("n1ncnnc1"), 520, 10000, 35),
            # Azo extended
            "azo_general":            (Chem.MolFromSmarts("[#6]-N=N-[#6]"), 450, 8000, 25),
            "hydrazone_azo":          (Chem.MolFromSmarts("C(=O)-N=N-[#6]"), 440, 9000, 28),
            "pyrazolone_azo":         (Chem.MolFromSmarts("O=C1C=C(N=NC2=CC=CC=C2)N(N1)C"), 430, 11000, 30),
            "acetoacetanilide_azo":   (Chem.MolFromSmarts("CC(=O)[CH2]C(=O)Nc1ccccc1"), 400, 8000, 25),
            "azo_anthraquinone":      (Chem.MolFromSmarts("O=C1c2ccccc2C(=O)c3cc(N=Nc4ccccc4)ccc13"), 520, 12000, 35),
            "azo_hydrazone":          (Chem.MolFromSmarts("C(=O)NN=C"), 440, 9000, 28),
            "dimethylamino_azo":      (Chem.MolFromSmarts("CN(C)c1ccc(N=Nc2ccccc2)cc1"), 450, 30000, 30),
            "dma_nitro_azobenzene":   (Chem.MolFromSmarts("CN(C)c1ccc(N=Nc2ccc([N+](=O)[O-])cc2)cc1"), 480, 38000, 35),
            # Organometallic
            "ferrocene":              (Chem.MolFromSmarts("[Fe]"), 440, 100, 50),
            "ferrocene_derivative":   (Chem.MolFromSmarts("[Fe]c1ccccc1"), 450, 200, 55),
            "ruthenocene":            (Chem.MolFromSmarts("[Ru]"), 350, 50, 40),
            "ru_bpy_complex":         (Chem.MolFromSmarts("[Ru]"), 450, 12000, 60),
            "metalloporphyrin":       (Chem.MolFromSmarts("[Fe,Co,Ni,Cu,Zn,Mg]"), 420, 150000, 30),
            "cu_chlorophyll":         (Chem.MolFromSmarts("[Cu]"), 660, 60000, 35),
            "ni_dithiocarbazate":     (Chem.MolFromSmarts("[Ni]"), 480, 8000, 70),
            "cu_thiosemicarbazone":   (Chem.MolFromSmarts("[Cu]"), 450, 6000, 60),
            "au_alkynyl":             (Chem.MolFromSmarts("[Au]C#C"), 380, 10000, 40),
        }

        # ---------- AUXOCHROMES (SMARTS, shift_nm, ε_multiplier) ----------
        # FIX #5: removed duplicate keys n_methylaniline, n_ethylaniline, dma
        self.auxochromes: dict[str, tuple[Chem.Mol, int, float]] = {
            # O donors
            "oh":                       (Chem.MolFromSmarts("[OX2H]"), 12, 1.1),
            "oh_phenol":                (Chem.MolFromSmarts("c1ccccc1O"), 15, 1.2),
            "phenolate_ion":            (Chem.MolFromSmarts("c1ccccc1[O-]"), 35, 1.4),
            "methoxy_aromatic":         (Chem.MolFromSmarts("c1ccccc1OC"), 13, 1.15),
            "ethoxy_aromatic":          (Chem.MolFromSmarts("c1ccccc1OCC"), 14, 1.16),
            "propoxy_aromatic":         (Chem.MolFromSmarts("c1ccccc1OCCC"), 15, 1.17),
            "butoxy_aromatic":          (Chem.MolFromSmarts("c1ccccc1OCCCC"), 16, 1.18),
            "phenoxy_aromatic":         (Chem.MolFromSmarts("c1ccccc1Oc2ccccc2"), 20, 1.25),
            "benzyloxy_aromatic":       (Chem.MolFromSmarts("c1ccccc1OCc2ccccc2"), 22, 1.28),
            "methoxy_alkoxy":           (Chem.MolFromSmarts("OC"), 5, 1.03),
            "ethoxy_alkoxy":            (Chem.MolFromSmarts("OCC"), 6, 1.04),
            "hydroxy_alkali":           (Chem.MolFromSmarts("[OH]"), 12, 1.1),
            "acetoxy_aromatic":         (Chem.MolFromSmarts("c1ccccc1OC(=O)C"), 18, 1.18),
            "benzoyloxy_aromatic":      (Chem.MolFromSmarts("c1ccccc1OC(=O)c2ccccc2"), 22, 1.22),
            "methylenedioxy":           (Chem.MolFromSmarts("c1cc2OCOc2cc1"), 12, 1.12),
            # N donors
            "nh2":                      (Chem.MolFromSmarts("[NX3H2]"), 15, 1.2),
            "nhr":                      (Chem.MolFromSmarts("[NX3H1]"), 18, 1.25),
            "nr2":                      (Chem.MolFromSmarts("[NX3H0]"), 22, 1.3),
            "amino_aromatic":           (Chem.MolFromSmarts("c1ccccc1N"), 26, 1.3),
            "methylamino_aromatic":     (Chem.MolFromSmarts("c1ccccc1NC"), 28, 1.32),
            "dimethylamino_aromatic":   (Chem.MolFromSmarts("CN(C)c1ccccc1"), 30, 1.35),
            "ethylamino_aromatic":      (Chem.MolFromSmarts("c1ccccc1NCC"), 28, 1.33),
            "diethylamino_aromatic":    (Chem.MolFromSmarts("CCN(CC)c1ccccc1"), 32, 1.38),
            "propylamino_aromatic":     (Chem.MolFromSmarts("c1ccccc1NCCC"), 30, 1.35),
            "dipropylamino_aromatic":   (Chem.MolFromSmarts("CCCN(CCC)c1ccccc1"), 34, 1.40),
            "anilino_aromatic":         (Chem.MolFromSmarts("c1ccccc1Nc2ccccc2"), 28, 1.35),
            "hydrazino_aromatic":       (Chem.MolFromSmarts("c1ccccc1NN"), 20, 1.2),
            "guanidine":                (Chem.MolFromSmarts("NC(=N)N"), 10, 1.05),
            "amidine":                  (Chem.MolFromSmarts("C(=N)N"), 8, 1.04),
            "hydrazide":                (Chem.MolFromSmarts("C(=O)NN"), 10, 1.1),
            "azide":                    (Chem.MolFromSmarts("N=[N+]=[N-]"), 25, 1.2),
            "isocyanate":               (Chem.MolFromSmarts("N=C=O"), 15, 1.1),
            "pyrrolidino_aromatic":     (Chem.MolFromSmarts("c1ccccc1N2CCCC2"), 32, 1.38),
            # S donors
            "thioether":                (Chem.MolFromSmarts("[#6][SX2][#6]"), 25, 1.15),
            "sh":                       (Chem.MolFromSmarts("[SX2H]"), 15, 1.1),
            "thioether_aromatic":       (Chem.MolFromSmarts("c1ccccc1SC"), 22, 1.17),
            "ethylthio_aromatic":       (Chem.MolFromSmarts("c1ccccc1SCC"), 24, 1.19),
            "phenylthio_aromatic":      (Chem.MolFromSmarts("c1ccccc1Sc2ccccc2"), 28, 1.22),
            "mercapto_aromatic":        (Chem.MolFromSmarts("c1ccccc1S"), 18, 1.12),
            "sulfoxide":                (Chem.MolFromSmarts("[#6][SX3](=O)[#6]"), 12, 1.08),
            "sulfone":                  (Chem.MolFromSmarts("[#6][SX4](=O)(=O)[#6]"), 8, 1.05),
            "sulfoxide_aromatic":       (Chem.MolFromSmarts("c1ccccc1S(=O)C"), 12, 1.08),
            "sulfone_aromatic":         (Chem.MolFromSmarts("c1ccccc1S(=O)(=O)C"), 8, 1.05),
            "sulfonamide":              (Chem.MolFromSmarts("S(=O)(=O)N"), 5, 1.05),
            "sulfonamide_aromatic":     (Chem.MolFromSmarts("c1ccccc1S(=O)(=O)N"), 5, 1.05),
            "sulfate":                  (Chem.MolFromSmarts("S(=O)(=O)"), 10, 1.05),
            "sulfonate":                (Chem.MolFromSmarts("S(=O)(=O)[O-]"), 2, 1.01),
            "so3h":                     (Chem.MolFromSmarts("S(=O)(=O)O"), 2, 1.03),
            "sulfonyl":                 (Chem.MolFromSmarts("S(=O)(=O)"), 10, 1.05),
            # Halogens
            "cl":                       (Chem.MolFromSmarts("[Cl]"), 5, 1.02),
            "br":                       (Chem.MolFromSmarts("[Br]"), 8, 1.03),
            "i":                        (Chem.MolFromSmarts("[I]"), 15, 1.05),
            "fluoro_aromatic":          (Chem.MolFromSmarts("c1ccccc1F"), 2, 0.98),
            "chloro_aromatic":          (Chem.MolFromSmarts("c1ccccc1Cl"), 5, 1.02),
            "bromo_aromatic":           (Chem.MolFromSmarts("c1ccccc1Br"), 8, 1.03),
            "iodo_aromatic":            (Chem.MolFromSmarts("c1ccccc1I"), 15, 1.05),
            "trifluoromethyl_aromatic": (Chem.MolFromSmarts("FC(F)(F)c1ccccc1"), -2, 0.95),
            "trifluoromethoxy_aromatic":(Chem.MolFromSmarts("FC(F)(F)Oc1ccccc1"), -3, 0.93),
            "chloro_vinyl":             (Chem.MolFromSmarts("ClC=C"), 12, 1.08),
            "fluoro_vinyl":             (Chem.MolFromSmarts("FC=C"), 8, 1.05),
            "bromo_vinyl":              (Chem.MolFromSmarts("BrC=C"), 15, 1.10),
            "iodo_vinyl":               (Chem.MolFromSmarts("IC=C"), 20, 1.12),
            # EWG
            "no2":                      (Chem.MolFromSmarts("[N+](=O)[O-]"), 20, 1.15),
            "nitro_aromatic":           (Chem.MolFromSmarts("c1ccccc1[N+](=O)[O-]"), 14, 1.2),
            "cn":                       (Chem.MolFromSmarts("C#N"), -5, 0.95),
            "cyano_aromatic":           (Chem.MolFromSmarts("c1ccccc1C#N"), 10, 0.95),
            "cooh":                     (Chem.MolFromSmarts("C(=O)[OH]"), 3, 1.02),
            "coo-":                     (Chem.MolFromSmarts("C(=O)[O-]"), 8, 1.08),
            "carboxy_aromatic":         (Chem.MolFromSmarts("c1ccccc1C(=O)O"), 16, 1.02),
            "carboxylate_aromatic":     (Chem.MolFromSmarts("c1ccccc1C(=O)[O-]"), 20, 1.08),
            "acetyl":                   (Chem.MolFromSmarts("C(=O)C"), 8, 1.08),
            "acetyl_aromatic":          (Chem.MolFromSmarts("c1ccccc1C(=O)C"), 31, 1.10),
            "formyl":                   (Chem.MolFromSmarts("[CH]=O"), 10, 1.10),
            "formyl_aromatic":          (Chem.MolFromSmarts("c1ccccc1C=O"), 33, 1.15),
            "amide_aromatic":           (Chem.MolFromSmarts("c1ccccc1C(=O)N"), 15, 1.02),
            "ester_aromatic":           (Chem.MolFromSmarts("c1ccccc1C(=O)O"), 16, 1.02),
            "sulfonyl_aromatic":        (Chem.MolFromSmarts("c1ccccc1S(=O)(=O)"), 8, 1.05),
            "phosphate_aromatic":       (Chem.MolFromSmarts("c1ccccc1OP(=O)(O)O"), 0, 1.0),
            "boronic_acid_aromatic":    (Chem.MolFromSmarts("c1ccccc1B(O)O"), 5, 1.02),
            "primary_amide":            (Chem.MolFromSmarts("C(=O)N[H2]"), 5, 1.02),
            "carbamoyl":                (Chem.MolFromSmarts("C(=O)N"), 5, 1.02),
            "cf3":                      (Chem.MolFromSmarts("C(F)(F)F"), -2, 0.95),
            "trifluoromethoxy":         (Chem.MolFromSmarts("OC(F)(F)F"), -3, 0.93),
            "difluoromethyl":           (Chem.MolFromSmarts("C(F)F"), -1, 0.97),
            "pentafluoroethyl":         (Chem.MolFromSmarts("C(C(F)(F)F)(F)F"), -4, 0.92),
            # Other
            "phosphate":                (Chem.MolFromSmarts("OP(=O)(O)O"), 0, 1.0),
            "boronic_acid":             (Chem.MolFromSmarts("B(O)O"), 5, 1.02),
            "boronate":                 (Chem.MolFromSmarts("B(OC)OC"), 5, 1.02),
            "hydroxamic_acid":          (Chem.MolFromSmarts("C(=O)NO"), 15, 1.1),
            "siloxy":                   (Chem.MolFromSmarts("[Si](C)(C)O[#6]"), 5, 1.02),
            "silyl":                    (Chem.MolFromSmarts("[Si]"), -5, 0.95),
            "germanyl":                 (Chem.MolFromSmarts("[Ge]"), -5, 0.95),
            "stannyl":                  (Chem.MolFromSmarts("[Sn]"), -5, 0.95),
            "phosphine":                (Chem.MolFromSmarts("[PX3]"), 10, 1.05),
            "phosphine_oxide":          (Chem.MolFromSmarts("[PX4](=O)"), 5, 1.02),
            "selenoether":              (Chem.MolFromSmarts("[Se]"), 30, 1.2),
            "telluroether":             (Chem.MolFromSmarts("[Te]"), 35, 1.25),
            "nitro_alkene":             (Chem.MolFromSmarts("[N+](=O)[O-]C=C"), 20, 1.2),
            "cyano_vinyl":              (Chem.MolFromSmarts("C#NC=C"), -10, 0.95),
            "sulfonyl_vinyl":           (Chem.MolFromSmarts("S(=O)(=O)C=C"), 5, 1.05),
            "thiocyanate":              (Chem.MolFromSmarts("SC#N"), 20, 1.15),
            "selenocyanate":            (Chem.MolFromSmarts("[Se]C#N"), 25, 1.2),
            # Alkyls
            "methyl":                   (Chem.MolFromSmarts("[CH3]"), -3, 0.96),
            "ethyl":                    (Chem.MolFromSmarts("[CH2]C"), -2, 0.97),
            "isopropyl":                (Chem.MolFromSmarts("[CH](C)C"), -1, 0.98),
            "tert_butyl":               (Chem.MolFromSmarts("C(C)(C)C"), 0, 0.99),
            "vinyl":                    (Chem.MolFromSmarts("C=C"), 15, 1.15),
            "ethynyl":                  (Chem.MolFromSmarts("C#C"), 20, 1.18),
            "methyl_aromatic":          (Chem.MolFromSmarts("c1ccccc1C"), 2, 0.96),
            "ethyl_aromatic":           (Chem.MolFromSmarts("c1ccccc1CC"), 3, 0.97),
            "isopropyl_aromatic":       (Chem.MolFromSmarts("c1ccccc1C(C)C"), 4, 0.98),
            "tert_butyl_aromatic":      (Chem.MolFromSmarts("c1ccccc1C(C)(C)C"), 5, 0.99),
            "vinyl_aromatic":           (Chem.MolFromSmarts("c1ccccc1C=C"), 15, 1.15),
            "ethynyl_aromatic":         (Chem.MolFromSmarts("c1ccccc1C#C"), 20, 1.18),
            "allyl_aromatic":           (Chem.MolFromSmarts("c1ccccc1CC=C"), 12, 1.12),
            "benzyl_aromatic":          (Chem.MolFromSmarts("c1ccccc1Cc2ccccc2"), 10, 1.10),
            "propyl_aromatic":          (Chem.MolFromSmarts("c1ccccc1CCC"), 4, 0.98),
            "butyl_aromatic":           (Chem.MolFromSmarts("c1ccccc1CCCC"), 5, 0.99),
            "cyclohexyl_aromatic":      (Chem.MolFromSmarts("c1ccccc1C2CCCCC2"), 5, 0.99),
            "cyclopentyl_aromatic":     (Chem.MolFromSmarts("c1ccccc1C2CCCC2"), 4, 0.98),
            # Heterocyclic subs
            "thiophene_sub":            (Chem.MolFromSmarts("c1ccsc1"), 20, 1.15),
            "furan_sub":                (Chem.MolFromSmarts("c1ccoc1"), 18, 1.12),
            "pyrrole_sub":              (Chem.MolFromSmarts("c1cc[nH]1"), 15, 1.1),
            "pyridine_sub":             (Chem.MolFromSmarts("c1ccncc1"), 10, 1.08),
            "imidazole_sub":            (Chem.MolFromSmarts("c1cnc[nH]1"), 12, 1.1),
            "triazole_sub":             (Chem.MolFromSmarts("c1n[nH]cn1"), 10, 1.08),
            "tetrazole_sub":            (Chem.MolFromSmarts("c1nnn[nH]1"), 8, 1.06),
            # Phenoxide & ligands
            "phenoxide":                (Chem.MolFromSmarts("[O-]c1ccccc1"), 35, 1.4),
            "bipyridine_ligand":        (Chem.MolFromSmarts("[n]1ccccc1-c1[n]ccccc1"), 30, 1.2),
            "terpyridine_ligand":       (Chem.MolFromSmarts("[n]1ccccc1-c2[n]cc(-c3[n]cccc3)cc2"), 40, 1.3),
            "cyclopentadienyl":         (Chem.MolFromSmarts("[CH]1[CH][CH][CH][CH]1"), 10, 1.1),
            "thiosemicarbazone":        (Chem.MolFromSmarts("S=C(N)-N"), 20, 1.2),
        }

        # ---------- INDUCTIVE PULL (SMARTS, shift_nm) ----------
        self.inductive_pull: dict[str, tuple[Chem.Mol, int]] = {
            "cf3_inductive":            (Chem.MolFromSmarts("FC(F)(F)Cc1ccccc1"), -2),
            "cl_inductive":             (Chem.MolFromSmarts("ClCc1ccccc1"), 2),
            "difluoromethyl_pull":      (Chem.MolFromSmarts("FC(F)Cc1ccccc1"), -1),
            "trifluoroethyl_pull":      (Chem.MolFromSmarts("FC(F)(F)CCc1ccccc1"), -2),
            "pentafluoroethyl_pull":    (Chem.MolFromSmarts("FC(F)(F)C(F)(F)Cc1ccccc1"), -4),
            "perfluoropropyl_pull":     (Chem.MolFromSmarts("FC(F)(F)C(F)(F)C(F)(F)Cc1ccccc1"), -6),
            "perfluorobutyl_pull":      (Chem.MolFromSmarts("FC(F)(F)C(F)(F)C(F)(F)C(F)(F)Cc1ccccc1"), -8),
            "fluoro_methyl_pull":       (Chem.MolFromSmarts("FCc1ccccc1"), 0),
            "fluoro_ethyl_pull":        (Chem.MolFromSmarts("FCCc1ccccc1"), 0),
            "dichloro_methyl_pull":     (Chem.MolFromSmarts("ClC(Cl)Cc1ccccc1"), 2),
            "trichloro_methyl_pull":    (Chem.MolFromSmarts("ClC(Cl)(Cl)Cc1ccccc1"), 3),
            "chloro_ethyl_pull":        (Chem.MolFromSmarts("ClCCc1ccccc1"), 1),
            "chloro_propyl_pull":       (Chem.MolFromSmarts("ClCCCc1ccccc1"), 1),
            "bromo_methyl_pull":        (Chem.MolFromSmarts("BrCc1ccccc1"), 2),
            "dibromomethyl_pull":       (Chem.MolFromSmarts("BrC(Br)Cc1ccccc1"), 3),
            "tribromomethyl_pull":      (Chem.MolFromSmarts("BrC(Br)(Br)Cc1ccccc1"), 4),
            "iodomethyl_pull":          (Chem.MolFromSmarts("ICc1ccccc1"), 1),
            "diiodomethyl_pull":        (Chem.MolFromSmarts("IC(I)Cc1ccccc1"), 2),
            "triiodomethyl_pull":       (Chem.MolFromSmarts("IC(I)(I)Cc1ccccc1"), 3),
            "chlorodifluoromethyl_pull":(Chem.MolFromSmarts("ClC(F)(F)Cc1ccccc1"), -1),
            "bromodifluoromethyl_pull": (Chem.MolFromSmarts("BrC(F)(F)Cc1ccccc1"), -1),
            "dichlorofluoromethyl_pull":(Chem.MolFromSmarts("ClC(Cl)(F)Cc1ccccc1"), 1),
            "difluoroethyl_pull":       (Chem.MolFromSmarts("FCC(F)(F)c1ccccc1"), -2),
            "hexafluoroisopropyl_pull": (Chem.MolFromSmarts("FC(F)(F)C(F)(F)C(F)(F)Cc1ccccc1"), -8),
            "methoxymethyl_pull":       (Chem.MolFromSmarts("COCc1ccccc1"), 2),
            "ethoxymethyl_pull":        (Chem.MolFromSmarts("CCOCc1ccccc1"), 2),
            "trifluoroethoxy_pull":     (Chem.MolFromSmarts("FC(F)(F)COc1ccccc1"), -3),
            "trifluoromethoxy_pull":    (Chem.MolFromSmarts("FC(F)(F)Oc1ccccc1"), -3),
            "difluoromethoxy_pull":     (Chem.MolFromSmarts("FC(F)Oc1ccccc1"), -2),
            "acetoxy_pull":             (Chem.MolFromSmarts("CC(=O)Oc1ccccc1"), 4),
            "benzoyloxy_pull":          (Chem.MolFromSmarts("c1ccccc1C(=O)Oc1ccccc1"), 4),
            "carbonate_pull":           (Chem.MolFromSmarts("O=C(OC)Oc1ccccc1"), 3),
            "carbamate_pull":           (Chem.MolFromSmarts("O=C(N)OCc1ccccc1"), 3),
            "sulfonyloxy_pull":         (Chem.MolFromSmarts("O=S(=O)(O)Oc1ccccc1"), -2),
            "cyano_methyl_pull":        (Chem.MolFromSmarts("N#CCc1ccccc1"), -3),
            "cyano_ethyl_pull":         (Chem.MolFromSmarts("N#CCCc1ccccc1"), -2),
            "dicyano_methyl_pull":      (Chem.MolFromSmarts("N#CC(C#N)c1ccccc1"), -6),
            "nitro_methyl_pull":        (Chem.MolFromSmarts("[N+](=O)[O-]Cc1ccccc1"), -5),
            "nitro_ethyl_pull":         (Chem.MolFromSmarts("[N+](=O)[O-]CCc1ccccc1"), -3),
            "sulfonyl_methyl_pull":     (Chem.MolFromSmarts("CS(=O)(=O)Cc1ccccc1"), -2),
            "sulfonyl_ethyl_pull":      (Chem.MolFromSmarts("CS(=O)(=O)CCc1ccccc1"), -1),
            "sulfamoyl_pull":           (Chem.MolFromSmarts("NS(=O)(=O)c1ccccc1"), -2),
            "sulfonamide_methyl_pull":  (Chem.MolFromSmarts("CS(=O)(=O)NCc1ccccc1"), -1),
            "azido_methyl_pull":        (Chem.MolFromSmarts("N=[N+]=[N-]Cc1ccccc1"), -2),
            "isocyano_pull":            (Chem.MolFromSmarts("[N+]#[C-]c1ccccc1"), -2),
            "isothiocyanato_pull":      (Chem.MolFromSmarts("N=C=Sc1ccccc1"), -2),
            "thiocyanato_pull":         (Chem.MolFromSmarts("N#CSc1ccccc1"), -2),
            "selenocyanato_pull":       (Chem.MolFromSmarts("N#C[Se]c1ccccc1"), -3),
            "trifluoromethyl_aryl":     (Chem.MolFromSmarts("FC(F)(F)c1ccccc1"), -2),
            "pentafluoroethyl_aryl_pull":(Chem.MolFromSmarts("FC(F)(F)C(F)(F)c1ccccc1"), -4),
            "trifluoromethoxy_aryl_pull":(Chem.MolFromSmarts("FC(F)(F)Oc1ccccc1"), -3),
            "pentafluorosulfanyl_aryl_pull":(Chem.MolFromSmarts("FS(=O)(=O)(F)(F)c1ccccc1"), -6),
            "cyano_aryl":               (Chem.MolFromSmarts("N#Cc1ccccc1"), -5),
            "dicyano_aryl_pull":        (Chem.MolFromSmarts("N#Cc1ccc(C#N)cc1"), -8),
            "nitro_aryl":               (Chem.MolFromSmarts("[N+](=O)[O-]c1ccccc1"), -5),
            "dinitro_aryl_pull":        (Chem.MolFromSmarts("[N+](=O)[O-]c1ccc([N+](=O)[O-])cc1"), -8),
            "formyl_aryl":              (Chem.MolFromSmarts("O=Cc1ccccc1"), 3),
            "acetyl_aryl":              (Chem.MolFromSmarts("CC(=O)c1ccccc1"), 2),
            "benzoyl_aryl_pull":        (Chem.MolFromSmarts("O=C(c1ccccc1)c1ccccc1"), 2),
            "carboxy_aryl":             (Chem.MolFromSmarts("OC(=O)c1ccccc1"), 1),
            "carboxylate_aryl_pull":    (Chem.MolFromSmarts("[O-]C(=O)c1ccccc1"), 3),
            "ester_aryl_pull":          (Chem.MolFromSmarts("COC(=O)c1ccccc1"), 2),
            "amide_aryl_pull":          (Chem.MolFromSmarts("NC(=O)c1ccccc1"), 1),
            "sulfonamide_aryl":         (Chem.MolFromSmarts("S(=O)(=O)Nc1ccccc1"), -1),
            "sulfonyl_aryl_pull":       (Chem.MolFromSmarts("S(=O)(=O)c1ccccc1"), -3),
            "phosphonate_aryl_pull":    (Chem.MolFromSmarts("P(=O)(O)(O)c1ccccc1"), 2),
            "boronic_acid_aryl_pull":   (Chem.MolFromSmarts("B(O)Oc1ccccc1"), 2),
            "chloro_aryl":              (Chem.MolFromSmarts("Clc1ccccc1"), 5),
            "fluoro_aryl":              (Chem.MolFromSmarts("Fc1ccccc1"), 2),
            "bromo_aryl":               (Chem.MolFromSmarts("Brc1ccccc1"), 8),
            "iodo_aryl":                (Chem.MolFromSmarts("Ic1ccccc1"), 15),
            "ethynyl_aryl_pull":        (Chem.MolFromSmarts("C#Cc1ccccc1"), -3),
            "vinyl_aryl_pull":          (Chem.MolFromSmarts("C=Cc1ccccc1"), -2),
            "nitroso_aryl":             (Chem.MolFromSmarts("N=Oc1ccccc1"), -3),
            "acrylonitrile_pull":       (Chem.MolFromSmarts("N#CC=C"), -5),
            "acrylate_pull":            (Chem.MolFromSmarts("C=CC(=O)O"), -3),
            "acrylamide_pull":          (Chem.MolFromSmarts("C=CC(=O)N"), -2),
            "nitroethylene_pull":       (Chem.MolFromSmarts("[N+](=O)[O-]C=C"), -6),
            "vinyl_sulfone_pull":       (Chem.MolFromSmarts("S(=O)(=O)C=C"), -4),
            "vinyl_sulfoxide_pull":     (Chem.MolFromSmarts("S(=O)C=C"), -3),
            "vinyl_phosphonate_pull":   (Chem.MolFromSmarts("P(=O)(O)(O)C=C"), -3),
            "ethynyl_carbonyl_pull":    (Chem.MolFromSmarts("C#CC(=O)"), -4),
            "ethynyl_nitrile_pull":     (Chem.MolFromSmarts("C#CC#N"), -6),
            "sulfonylurea_core":        (Chem.MolFromSmarts("S(=O)(=O)NC(=O)N"), -5),
            "aryl_sulfonylurea":        (Chem.MolFromSmarts("c1ccccc1S(=O)(=O)NC(=O)N"), -3),
            "glimepiride_urethane_core":(Chem.MolFromSmarts("NC(=O)OC"), -3),
            "cyanoguanidine_core":      (Chem.MolFromSmarts("N#CNC(=N)N"), -6),
            "biguanide_core":           (Chem.MolFromSmarts("NC(=N)NC(=N)N"), -4),
            "thiourea_core":            (Chem.MolFromSmarts("NC(=S)N"), -2),
            "aryl_thiourea":            (Chem.MolFromSmarts("c1ccccc1NC(=S)N"), -2),
            "thioamide_core":           (Chem.MolFromSmarts("C(=S)N"), -2),
            "dithiocarbamate_core":     (Chem.MolFromSmarts("S=C(S)N"), -3),
            "beta_lactam_carbonyl":     (Chem.MolFromSmarts("N1C(=O)CC1"), -6),
            "penam_carbonyl_pull":      (Chem.MolFromSmarts("N1C(=O)C2C1SC(C)(C)C2"), -5),
            "cephem_carbonyl_pull":     (Chem.MolFromSmarts("O=C1N2C(C(=O)O)=C(C)CSC12"), -4),
            "monobactam_carbonyl_pull": (Chem.MolFromSmarts("O=S(=O)(O)N1C(=O)C=CC1"), -4),
            "statin_dihydroxy_acid":    (Chem.MolFromSmarts("OC(CC(O)CC(=O)O)"), -4),
            "statin_lactone":           (Chem.MolFromSmarts("O=C1CC(O)CC(=O)O1"), -4),
            "fluoroquinolone_core":     (Chem.MolFromSmarts("O=C1C=CN(C)c2cc(F)c(cc12)"), -6),
            "quinolone_carboxyl":       (Chem.MolFromSmarts("O=C1C=CN(C)c2ccccc12C(=O)O"), -4),
            "sulfanilamide_core":       (Chem.MolFromSmarts("Nc1ccc(S(=O)(=O)N)cc1"), -2),
            "aryl_sulfonamide_bulky":   (Chem.MolFromSmarts("c1ccccc1S(=O)(=O)N(c1ccccc1)"), -3),
            "heteroaryl_sulfonamide":   (Chem.MolFromSmarts("c1ccncc1S(=O)(=O)N"), -3),
            "aryl_carbamate_pull":      (Chem.MolFromSmarts("c1ccccc1OC(=O)N"), -3),
            "n_aryl_carbamate":         (Chem.MolFromSmarts("c1ccccc1NC(=O)O"), -2),
            "aryl_propionic_acid":      (Chem.MolFromSmarts("CC(C(=O)O)c1ccccc1"), -2),
            "aryl_acetic_acid":         (Chem.MolFromSmarts("OC(=O)Cc1ccccc1"), -2),
            "salicylate_core":          (Chem.MolFromSmarts("OC(=O)c1ccccc1O"), 3),
            "aryl_urea_kinase":         (Chem.MolFromSmarts("c1ccccc1NC(=O)Nc1ccccc1"), -4),
            "heteroaryl_urea_kinase":   (Chem.MolFromSmarts("c1ccncc1NC(=O)N"), -4),
            "aminopyrimidine_kinase":   (Chem.MolFromSmarts("Nc1ncccn1"), 3),
            "aminopyridine_kinase":     (Chem.MolFromSmarts("Nc1ccccn1"), 2),
            "phosphate_monoester":      (Chem.MolFromSmarts("OP(=O)(O)O"), -2),
            "phosphate_diester":        (Chem.MolFromSmarts("OP(=O)(O)OC"), -3),
            "phosphonate_ester":        (Chem.MolFromSmarts("CP(=O)(O)O"), -2),
            "acid_chloride_pull":       (Chem.MolFromSmarts("C(=O)Cl"), -5),
            "acid_bromide_pull":        (Chem.MolFromSmarts("C(=O)Br"), -4),
            "acid_fluoride_pull":       (Chem.MolFromSmarts("C(=O)F"), -6),
            "mixed_anhydride_pull":     (Chem.MolFromSmarts("C(=O)OC(=O)"), -4),
            "triflate_pull":            (Chem.MolFromSmarts("OS(=O)(=O)C(F)(F)F"), -8),
            "nonaflate_pull":           (Chem.MolFromSmarts("OS(=O)(=O)C(F)(F)C(F)(F)C(F)(F)C(F)(F)F"), -10),
            "gem_dicyano":              (Chem.MolFromSmarts("C(C#N)(C#N)"), -8),
            "gem_dinitro":              (Chem.MolFromSmarts("C([N+](=O)[O-])([N+](=O)[O-])"), -10),
            "cyano_nitro_gem":          (Chem.MolFromSmarts("C(C#N)([N+](=O)[O-])"), -9),
            "keto_nitrile":             (Chem.MolFromSmarts("N#CC(=O)"), -6),
            "keto_ester":               (Chem.MolFromSmarts("C(=O)C(=O)O"), -5),
            "diketone":                 (Chem.MolFromSmarts("C(=O)C(=O)"), -5),
            "beta_keto_amide":          (Chem.MolFromSmarts("C(=O)C(=O)N"), -4),
        }

        # ---------- INSULATORS (SMARTS, cap_nm, penalty_weight) ----------
        # FIX #4 / #8: penalty_weight is now actively used in evaluate_molecule
        self.insulators: dict[str, tuple[Chem.Mol, int, int]] = {
            "aliphatic_peptide":        (Chem.MolFromSmarts("NC(=O)CCC(=O)NC(C)C"), 258, 15),
            "peptide_backbone_iso":     (Chem.MolFromSmarts("CC(=O)NC(C)C(=O)NC"), 258, 10),
            "urethane_core_iso":        (Chem.MolFromSmarts("NC(=O)OC"), 250, 10),
            "piperazine_core_iso":      (Chem.MolFromSmarts("N1CCNCC1"), 260, 15),
            "pyrrolidine_core_iso":     (Chem.MolFromSmarts("C1CCNC1"), 230, 10),
            "morpholine_core_iso":      (Chem.MolFromSmarts("O1CCNCC1"), 240, 12),
            "piperidine_core_iso":      (Chem.MolFromSmarts("C1CCNCC1"), 240, 12),
            "tetrahydrofuran_core":     (Chem.MolFromSmarts("O1CCCC1"), 220, 10),
            "tetrahydropyran_core":     (Chem.MolFromSmarts("O1CCCCC1"), 230, 10),
            "azetidine_core_iso":       (Chem.MolFromSmarts("C1CNC1"), 225, 10),
            "oxetane_core_iso":         (Chem.MolFromSmarts("C1COC1"), 220, 10),
            "dioxane_core_iso":         (Chem.MolFromSmarts("O1CCOCC1"), 225, 10),
            "dithiane_core_iso":        (Chem.MolFromSmarts("S1CCSCC1"), 240, 12),
            "cyclopropane_iso":         (Chem.MolFromSmarts("C1CC1"), 210, 8),
            "cyclobutane_iso":          (Chem.MolFromSmarts("C1CCC1"), 215, 8),
            "methylene_bridge_iso":     (Chem.MolFromSmarts("c1ccccc1Cc1ccccc1"), 265, 12),
            "ester_bridge_iso":         (Chem.MolFromSmarts("c1ccccc1C(=O)OCC"), 275, 15),
            "sulfide_bridge_iso":       (Chem.MolFromSmarts("c1ccccc1Sc1ccccc1"), 280, 15),
            "ether_bridge_iso":         (Chem.MolFromSmarts("c1ccccc1Oc1ccccc1"), 270, 10),
            "urea_bridge_iso":          (Chem.MolFromSmarts("c1ccccc1NC(=O)Nc2ccccc2"), 270, 15),
            "sulfonamide_bridge_iso":   (Chem.MolFromSmarts("c1ccccc1S(=O)(=O)Nc2ccccc2"), 275, 15),
            "carbonate_bridge":         (Chem.MolFromSmarts("c1ccccc1OC(=O)Oc2ccccc2"), 270, 15),
            "anhydride_bridge_iso":     (Chem.MolFromSmarts("c1ccccc1C(=O)OC(=O)c2ccccc2"), 280, 15),
            "sulfone_bridge_iso":       (Chem.MolFromSmarts("c1ccccc1S(=O)(=O)c1ccccc1"), 285, 15),
            "amine_bridge_iso":         (Chem.MolFromSmarts("c1ccccc1Nc1ccccc1"), 290, 15),
            "amide_bridge_iso":         (Chem.MolFromSmarts("c1ccccc1C(=O)Nc1ccccc1"), 280, 15),
            "ketone_bridge_iso":        (Chem.MolFromSmarts("c1ccccc1C(=O)c1ccccc1"), 275, 12),
            "alkene_bridge_iso":        (Chem.MolFromSmarts("c1ccccc1C=Cc1ccccc1"), 285, 12),
            "alkyl_chain_long":         (Chem.MolFromSmarts("C~C~C~C~C"), 260, 10),
            "alkyl_chain_short":        (Chem.MolFromSmarts("C~C~C~C"), 250, 10),
            "alkyl_chain_medium":       (Chem.MolFromSmarts("C~C~C~C~C~C"), 255, 10),
            "siloxane_bridge":          (Chem.MolFromSmarts("c1ccccc1[Si](C)(C)O[Si](C)(C)c2ccccc2"), 265, 12),
            "polyether_link_iso":       (Chem.MolFromSmarts("c1ccccc1OCCOCc2ccccc2"), 265, 10),
            "oxetane_bridge":           (Chem.MolFromSmarts("c1ccccc1C2OCC2c3ccccc3"), 260, 12),
            "azetidine_bridge":         (Chem.MolFromSmarts("c1ccccc1C2CCN2c3ccccc3"), 255, 10),
            "thiirane_bridge":          (Chem.MolFromSmarts("c1ccccc1C2SC2c3ccccc3"), 265, 15),
            "aziridine_bridge":         (Chem.MolFromSmarts("c1ccccc1C2CN2c3ccccc3"), 255, 10),
            "dioxolane_bridge":         (Chem.MolFromSmarts("c1ccccc1C2OCOC2c3ccccc3"), 265, 15),
            "dithiolane_bridge":        (Chem.MolFromSmarts("c1ccccc1C2SCSC2c3ccccc3"), 270, 15),
            "oxazolidine_bridge":       (Chem.MolFromSmarts("c1ccccc1C2NCOC2c3ccccc3"), 260, 12),
            "thiazolidine_bridge":      (Chem.MolFromSmarts("c1ccccc1C2NCSC2c3ccccc3"), 265, 12),
            "tetrahydrofuran_bridge":   (Chem.MolFromSmarts("c1ccccc1C2CCCO2c3ccccc3"), 255, 12),
            "tetrahydropyran_bridge":   (Chem.MolFromSmarts("c1ccccc1C2CCCCO2c3ccccc3"), 258, 12),
            "cyclohexyl_bridge_iso":    (Chem.MolFromSmarts("c1ccccc1C2CCCCC2c3ccccc3"), 260, 10),
            "cyclopropyl_bridge":       (Chem.MolFromSmarts("c1ccccc1C2CC2c3ccccc3"), 255, 10),
            "cyclobutyl_bridge":        (Chem.MolFromSmarts("c1ccccc1C2CCC2c3ccccc3"), 255, 10),
            "cyclopentyl_bridge":       (Chem.MolFromSmarts("c1ccccc1C2CCCC2c3ccccc3"), 258, 10),
            "pyrimidine_core_iso":      (Chem.MolFromSmarts("n1c(=O)[nH]c(=O)nc1"), 270, 20),
            "triazole_thiol_iso":       (Chem.MolFromSmarts("c1nncs1"), 285, 15),
            "adamantane_spacer_iso":    (Chem.MolFromSmarts("c1ccccc1C23CC4CC(CC(C4)C2)C3c5ccccc5"), 255, 10),
            "spiro_cyclic_iso":         (Chem.MolFromSmarts("c1ccccc1C2(CCCCC2)c3ccccc3"), 260, 10),
            "alkyl_spacer":             (Chem.MolFromSmarts("C~C~C~C~C"), 260, 10),
            "amide_linker":             (Chem.MolFromSmarts("NC(=O)"), 270, 15),
            "ester_linker":             (Chem.MolFromSmarts("C(=O)O"), 280, 15),
            "sulfone_linker":           (Chem.MolFromSmarts("S(=O)(=O)"), 285, 15),
            "alkyne_linker":            (Chem.MolFromSmarts("C#C"), 280, 15),
            "diene_linker":             (Chem.MolFromSmarts("C=CC=C"), 270, 12),
            "penam_core_iso":           (Chem.MolFromSmarts("N1C(=O)C2C1SC(C)(C)C2"), 225, 20),
            "cephem_core_iso":          (Chem.MolFromSmarts("O=C1N2C(C(=O)O)=C(C)CSC12"), 260, 22),
            "carbapenem_core_iso":      (Chem.MolFromSmarts("O=C1N2C(C(=O)O)=C(C)CC12"), 240, 20),
            "monobactam_core_iso":      (Chem.MolFromSmarts("O=S(=O)(O)N1C(=O)C=CC1"), 230, 18),
            "benzodiazepine_iso":       (Chem.MolFromSmarts("c1ccc2c(c1)NC=CN=C2"), 260, 30),
            "barbiturate_iso":          (Chem.MolFromSmarts("O=C1NC(=O)NC(=O)C1"), 240, 20),
            "hydantoin_iso":            (Chem.MolFromSmarts("O=C1NCNC1=O"), 235, 18),
            "oxazolidinone_iso":        (Chem.MolFromSmarts("O=C1OCCN1"), 240, 18),
            "imidazolidinone_iso":      (Chem.MolFromSmarts("O=C1NCCN1"), 235, 18),
            "sulfonylurea_iso":         (Chem.MolFromSmarts("S(=O)(=O)NC(=O)N"), 230, 15),
            "biguanide_iso":            (Chem.MolFromSmarts("NC(=N)NC(=N)N"), 230, 12),
            "guanidine_iso":            (Chem.MolFromSmarts("NC(=N)N"), 220, 10),
            "thiourea_iso":             (Chem.MolFromSmarts("NC(=S)N"), 250, 12),
        }

        # ---------- STERIC PENALTIES (SMARTS, shift_nm, ε_multiplier) ----------
        self.steric_penalties: dict[str, tuple[Chem.Mol, int, float]] = {
            "ortho_methyl_biphenyl":        (Chem.MolFromSmarts("Cc1c(-c2ccccc2)cccc1"), -15, 0.7),
            "ortho_dimethyl_biphenyl":      (Chem.MolFromSmarts("Cc1c(c(C)ccc1)-c1ccccc1"), -25, 0.5),
            "ortho_di_tert_butyl_biphenyl": (Chem.MolFromSmarts("CC(C)(C)c1c(-c2ccccc2)c(C(C)(C)C)ccc1"), -40, 0.4),
            "ortho_methoxy_biphenyl":       (Chem.MolFromSmarts("COc1c(-c2ccccc2)cccc1"), -18, 0.6),
            "ortho_chloro_biphenyl":        (Chem.MolFromSmarts("Clc1c(-c2ccccc2)cccc1"), -12, 0.7),
            "ortho_iodo_biphenyl":          (Chem.MolFromSmarts("Ic1c(-c2ccccc2)cccc1"), -20, 0.5),
            "ortho_nitro_biphenyl":         (Chem.MolFromSmarts("[N+](=O)[O-]c1c(-c2ccccc2)cccc1"), -20, 0.6),
            "ortho_amino_biphenyl":         (Chem.MolFromSmarts("Nc1c(-c2ccccc2)cccc1"), -12, 0.7),
            "ortho_carboxy_biphenyl":       (Chem.MolFromSmarts("OC(=O)c1c(-c2ccccc2)cccc1"), -15, 0.65),
            "ortho_cyano_biphenyl":         (Chem.MolFromSmarts("N#Cc1c(-c2ccccc2)cccc1"), -15, 0.65),
            "ortho_fluoro_biphenyl":        (Chem.MolFromSmarts("Fc1c(-c2ccccc2)cccc1"), -10, 0.75),
            "ortho_trifluoromethyl_biphenyl":(Chem.MolFromSmarts("FC(F)(F)c1c(-c2ccccc2)cccc1"), -15, 0.65),
            "ortho_sulfonyl_biphenyl":      (Chem.MolFromSmarts("O=S(=O)(O)c1c(-c2ccccc2)cccc1"), -20, 0.6),
            "ortho_dimethoxy_biphenyl":     (Chem.MolFromSmarts("COc1c(-c2ccccc2)cccc1OC"), -18, 0.6),
            "ortho_dichloro_biphenyl":      (Chem.MolFromSmarts("Clc1c(-c2ccccc2)cccc1Cl"), -15, 0.65),
            "di_ortho_methyl_biphenyl":     (Chem.MolFromSmarts("Cc1c(C)c(C)ccc1-c2ccccc2"), -30, 0.4),
            "tri_ortho_tert_butyl":         (Chem.MolFromSmarts("CC(C)(C)c1c(C(C)(C)C)c(C(C)(C)C)ccc1"), -45, 0.3),
            "tetra_ortho_bi_phenyl":        (Chem.MolFromSmarts("c1(c(C)c(C)ccc1)-c1c(C)c(C)ccc1"), -35, 0.4),
            "ortho_ethyl_biphenyl":         (Chem.MolFromSmarts("CCc1c(-c2ccccc2)cccc1"), -18, 0.65),
            "ortho_isopropyl_biphenyl":     (Chem.MolFromSmarts("CC(C)c1c(-c2ccccc2)cccc1"), -22, 0.55),
            "ortho_tert_butyl_biphenyl":    (Chem.MolFromSmarts("CC(C)(C)c1c(-c2ccccc2)cccc1"), -30, 0.45),
            "ortho_diethyl_biphenyl":       (Chem.MolFromSmarts("CCc1c(-c2ccccc2)cccc1CC"), -30, 0.45),
            "ortho_diisopropyl_biphenyl":   (Chem.MolFromSmarts("CC(C)c1c(-c2ccccc2)cccc1C(C)C"), -40, 0.35),
            "ortho_ethoxy_biphenyl":        (Chem.MolFromSmarts("CCOc1c(-c2ccccc2)cccc1"), -20, 0.58),
            "ortho_difluoro_biphenyl":      (Chem.MolFromSmarts("Fc1c(-c2ccccc2)cccc1F"), -14, 0.70),
            "ortho_bromo_biphenyl":         (Chem.MolFromSmarts("Brc1c(-c2ccccc2)cccc1"), -16, 0.65),
            "ortho_cf3_biphenyl":           (Chem.MolFromSmarts("FC(F)(F)c1c(-c2ccccc2)cccc1"), -18, 0.60),
            "ortho_di_cf3_biphenyl":        (Chem.MolFromSmarts("FC(F)(F)c1c(-c2ccccc2)cccc1C(F)(F)F"), -28, 0.42),
            "ortho_hydroxy_biphenyl":       (Chem.MolFromSmarts("Oc1c(-c2ccccc2)cccc1"), -15, 0.68),
            "twisted_amide_tert_butyl":     (Chem.MolFromSmarts("C(=O)N(C(C)(C)C)"), -12, 0.60),
            "twisted_amide_isopropyl":      (Chem.MolFromSmarts("C(=O)N(C(C)C)"), -10, 0.65),
            "n_tert_butyl_benzamide":       (Chem.MolFromSmarts("O=C(c1ccccc1)NC(C)(C)C"), -15, 0.60),
            "n_n_di_isopropyl_amide":       (Chem.MolFromSmarts("O=C(c1ccccc1)N(C(C)C)C(C)C"), -20, 0.50),
            "n_cyclohexyl_amide_twist":     (Chem.MolFromSmarts("O=C(c1ccccc1)NC1CCCCC1"), -10, 0.70),
            "n_aryl_amide_twisted":         (Chem.MolFromSmarts("O=C(c1ccccc1)N(c1ccccc1)"), -12, 0.65),
            "lactam_ring_constrained":      (Chem.MolFromSmarts("O=C1N(C)CCC1"), -8, 0.75),
            "trisubstituted_double_bond":   (Chem.MolFromSmarts("C=C(C)C(C)(C)"), -8, 0.80),
            "tetrasubstituted_alkene":      (Chem.MolFromSmarts("C=C(C)(C)"), -12, 0.75),
            "bulky_diene_twist":            (Chem.MolFromSmarts("C=C(C)C(C)=C"), -10, 0.70),
            "trisubstituted_alkene":        (Chem.MolFromSmarts("C=C(C)C"), -8, 0.85),
            "tetraaryl_ethylene":           (Chem.MolFromSmarts("c1ccccc1C(=C(c1ccccc1)c1ccccc1)c1ccccc1"), -25, 0.45),
            "1_1_diaryl_ethylene":          (Chem.MolFromSmarts("C=C(c1ccccc1)c1ccccc1"), -15, 0.65),
            "triaryl_ethylene":             (Chem.MolFromSmarts("c1ccccc1C(=C(c1ccccc1))c1ccccc1"), -20, 0.55),
            "gem_dimethyl":                 (Chem.MolFromSmarts("C(=C)(C)C"), -10, 0.78),
            "gem_diphenyl":                 (Chem.MolFromSmarts("C(=C)(c1ccccc1)c1ccccc1"), -20, 0.55),
            "tert_butyl_cumulene":          (Chem.MolFromSmarts("C(C)(C)C=C=C"), -15, 0.60),
            "ortho_di_sub_benzene":         (Chem.MolFromSmarts("c1c([#6,#7,#8])c([#6,#7,#8])ccc1"), -10, 0.80),
            "tri_ortho_sub_phenyl":         (Chem.MolFromSmarts("c1c(C)c(C)c(C)cc1"), -12, 0.70),
            "ortho_substituted_pyridine":   (Chem.MolFromSmarts("c1c(C)nccc1"), -10, 0.70),
            "ortho_substituted_styrene":    (Chem.MolFromSmarts("c1c(C=Cc2ccccc2)c(C)cccc1"), -15, 0.60),
            "crowded_stilbene":             (Chem.MolFromSmarts("C1=C(C)C(=C(C)C)C=C1C=Cc2ccccc2"), -20, 0.50),
            "bulky_beta_styrene":           (Chem.MolFromSmarts("c1ccccc1C=C(C(C)(C)C)"), -15, 0.60),
            "ortho_methyl_styrene":         (Chem.MolFromSmarts("Cc1ccccc1C=C"), -12, 0.70),
            "ortho_tert_butyl_styrene":     (Chem.MolFromSmarts("CC(C)(C)c1ccccc1C=C"), -22, 0.55),
            "ortho_halo_styrene":           (Chem.MolFromSmarts("Clc1ccccc1C=C"), -10, 0.72),
            "ortho_substituted_stilbene":   (Chem.MolFromSmarts("c1(C=Cc2ccccc2)c(C)cccc1"), -15, 0.60),
            "tetraaryl_stilbene":           (Chem.MolFromSmarts("c1ccccc1C(=C(c1ccccc1)c1ccccc1)c1ccccc1"), -25, 0.45),
            "n_alkyl_carbazole_twist":      (Chem.MolFromSmarts("c1ccc2c(c1)n(C(C)C)c3ccccc32"), -10, 0.70),
            "n_tert_butyl_indole":          (Chem.MolFromSmarts("CC(C)(C)n1ccc2ccccc21"), -15, 0.60),
            "n_tert_butyl_carbazole":       (Chem.MolFromSmarts("CC(C)(C)n1c2ccccc2c2ccccc21"), -18, 0.55),
            "n_aryl_carbazole_twist":       (Chem.MolFromSmarts("c1ccc2c(c1)n(C(C)C)c3ccccc32"), -10, 0.70),
            "n_bulky_pyrrole":              (Chem.MolFromSmarts("CC(C)(C)n1cccc1"), -12, 0.65),
            "n_bulky_pyrazole":             (Chem.MolFromSmarts("CC(C)(C)n1cccn1"), -10, 0.70),
            "n_bulky_imidazole":            (Chem.MolFromSmarts("CC(C)(C)n1ccnc1"), -10, 0.70),
            "peri_substituted_naphthalene": (Chem.MolFromSmarts("c1ccc2c(c1)cccc2C"), -12, 0.70),
            "1_8_disubstituted_naphthalene":(Chem.MolFromSmarts("Cc1cccc2cccc(C)c12"), -20, 0.58),
            "peri_substituted_anthracene":  (Chem.MolFromSmarts("c1ccc2c(c1)cc3ccccc3c2C"), -18, 0.60),
            "crowded_pyrene":               (Chem.MolFromSmarts("Cc1ccc2ccc3cccc4ccc1c2c34"), -15, 0.65),
            "hexaalkylbenzene":             (Chem.MolFromSmarts("c1c(C)c(C)c(C)c(C)c1C"), -30, 0.50),
            "biphenyl_2_2_diyl":            (Chem.MolFromSmarts("c1ccc2c(c1)c1ccccc1-c2"), -20, 0.60),
            "boc_protected_amine":          (Chem.MolFromSmarts("CC(C)(C)OC(=O)N"), -8, 0.75),
            "cbz_protected_amine":          (Chem.MolFromSmarts("c1ccccc1COC(=O)N"), -10, 0.72),
            "fmoc_protected_amine":         (Chem.MolFromSmarts("c1ccc2c(c1)c1ccccc1C2COC(=O)N"), -15, 0.65),
            "tosyl_protected_amine":        (Chem.MolFromSmarts("Cc1ccc(S(=O)(=O)N)cc1"), -8, 0.78),
            "trityl_protected":             (Chem.MolFromSmarts("C(c1ccccc1)(c1ccccc1)(c1ccccc1)N"), -18, 0.60),
            "benzyl_ether_protected":       (Chem.MolFromSmarts("c1ccccc1CO"), -5, 0.85),
            "silyl_ether_protected":        (Chem.MolFromSmarts("C[Si](C)(C)O"), -8, 0.75),
            "acetonide_protected":          (Chem.MolFromSmarts("CC1(C)OCO1"), -5, 0.85),
            "aryl_glycoside":               (Chem.MolFromSmarts("c1ccccc1OC1OC(CO)C(O)C(O)C1O"), -12, 0.65),
            "aryl_glucuronide":             (Chem.MolFromSmarts("c1ccccc1OC1OC(C(=O)O)C(O)C(O)C1O"), -15, 0.60),
            "n_glycoside":                  (Chem.MolFromSmarts("c1ccccc1N1OC(CO)C(O)C(O)C1O"), -12, 0.65),
            "c_glycoside":                  (Chem.MolFromSmarts("c1ccccc1C1OC(CO)C(O)C(O)C1O"), -10, 0.70),
            "proline_residue":              (Chem.MolFromSmarts("N1CCCC1C(=O)"), -5, 0.85),
            "pipecolic_acid":               (Chem.MolFromSmarts("N1CCCCC1C(=O)"), -5, 0.85),
            "macrocycle_constraint":        (Chem.MolFromSmarts("C1CCCCCCCCCC1"), -10, 0.70),
            "cyclophane_like":              (Chem.MolFromSmarts("c1ccccc1CCc1ccccc1"), -15, 0.60),
            "spiro_cyclic_constraint":      (Chem.MolFromSmarts("c1ccccc1C2(CCCCC2)c3ccccc3"), -10, 0.70),
            "adamantane_constraint":        (Chem.MolFromSmarts("c1ccccc1C23CC4CC(CC(C4)C2)C3c5ccccc5"), -15, 0.60),
            "ortho_substituted_quinoline":  (Chem.MolFromSmarts("c1ccc2ncccc2c1C"), -12, 0.68),
            "2_6_disubstituted_pyridine":   (Chem.MolFromSmarts("Cc1cccc(C)n1"), -12, 0.65),
            "2_6_disubstituted_pyrimidine": (Chem.MolFromSmarts("Cc1ccnc(C)n1"), -12, 0.65),
            "ortho_substituted_phenyl_hydrazone": (Chem.MolFromSmarts("c1c(C)cccc1=NN"), -15, 0.60),
            "ortho_di_phenyl_substitution": (Chem.MolFromSmarts("c1c(c2ccccc2)c(c2ccccc2)ccc1"), -30, 0.40),
        }

        # ---------- PUSH-PULL OVERRIDES (SMARTS, λmax, intensity, bandwidth) ----------
        self.push_pull_overrides: dict[str, tuple[Chem.Mol, int, float, int]] = {
            "p_nitroaniline":           (Chem.MolFromSmarts("Nc1ccc([N+](=O)[O-])cc1"), 380, 1.6, 40),
            "p_nitrophenol":            (Chem.MolFromSmarts("Oc1ccc([N+](=O)[O-])cc1"), 315, 1.4, 35),
            "dimethylamino_nitro":      (Chem.MolFromSmarts("CN(C)c1ccc([N+](=O)[O-])cc1"), 390, 1.6, 40),
            "diethylamino_nitro":       (Chem.MolFromSmarts("CCN(CC)c1ccc([N+](=O)[O-])cc1"), 395, 1.6, 42),
            "azo_dye_core":             (Chem.MolFromSmarts("c1ccc(/N=N/c2ccccc2)cc1"), 420, 1.5, 35),
            "n_n_dimethyl_azo":         (Chem.MolFromSmarts("CN(C)c1ccc(N=Nc2ccccc2)cc1"), 450, 1.8, 45),
            "dma_nitro_azobenzene_pp":  (Chem.MolFromSmarts("CN(C)c1ccc(N=Nc2ccc([N+](=O)[O-])cc2)cc1"), 480, 2.0, 50),
            "donor_acceptor_azobenzene":(Chem.MolFromSmarts("Nc1ccc(N=Nc2ccc([N+](=O)[O-])cc2)cc1"), 450, 1.8, 50),
            "amino_azobenzene":         (Chem.MolFromSmarts("Nc1ccc(N=Nc2ccccc2)cc1"), 420, 1.6, 40),
            "hydroxy_azobenzene":       (Chem.MolFromSmarts("Oc1ccc(N=Nc2ccccc2)cc1"), 410, 1.5, 40),
            "dma_azobenzene":           (Chem.MolFromSmarts("CN(C)c1ccc(N=Nc2ccccc2)cc1"), 440, 1.8, 45),
            "amino_nitro_azobenzene_pp":(Chem.MolFromSmarts("Nc1ccc(N=Nc2ccc([N+](=O)[O-])cc2)cc1"), 460, 1.8, 48),
            "hydroxy_nitro_azobenzene": (Chem.MolFromSmarts("Oc1ccc(N=Nc2ccc([N+](=O)[O-])cc2)cc1"), 450, 1.7, 46),
            "push_pull_azobenzene":     (Chem.MolFromSmarts("Nc1ccc(N=Nc2ccc([N+](=O)[O-])cc2)cc1"), 450, 1.8, 50),
            "stilbene_donor_acceptor":  (Chem.MolFromSmarts("Nc1ccc(C=Cc2ccc([N+](=O)[O-])cc2)cc1"), 410, 1.7, 45),
            "dma_nitro_stilbene":       (Chem.MolFromSmarts("CN(C)c1ccc(C=Cc2ccc([N+](=O)[O-])cc2)cc1"), 430, 1.9, 45),
            "donor_acceptor_stilbene":  (Chem.MolFromSmarts("Nc1ccc(C=Cc2ccc([N+](=O)[O-])cc2)cc1"), 410, 1.7, 45),
            "dma_cyano_stilbene":       (Chem.MolFromSmarts("CN(C)c1ccc(C=Cc2ccc(C#N)cc2)cc1"), 410, 1.8, 42),
            "amino_cyano_stilbene":     (Chem.MolFromSmarts("Nc1ccc(C=Cc2ccc(C#N)cc2)cc1"), 390, 1.6, 40),
            "hydroxy_cyano_stilbene":   (Chem.MolFromSmarts("Oc1ccc(C=Cc2ccc(C#N)cc2)cc1"), 380, 1.5, 40),
            "push_pull_stilbene":       (Chem.MolFromSmarts("Nc1ccc(C=Cc2ccc([N+](=O)[O-])cc2)cc1"), 410, 1.7, 45),
            "merocyanine_core":         (Chem.MolFromSmarts("O=C1C=C(N(C)C)C=C1"), 420, 1.8, 45),
            "merocyanine_dye":          (Chem.MolFromSmarts("O=C1C=CC(=C2CCCCN2)C=C1"), 560, 1.8, 45),
            "hemicyanine_da":           (Chem.MolFromSmarts("[NX3]C=CC=CC=Cc1ccc([N+](=O)[O-])cc1"), 460, 1.8, 45),
            "cyanine_indolenine":       (Chem.MolFromSmarts("c1ccc2c(c1)C(C)(C)C=CN2=CC=CC=C3[N+](=C)C4=CC=CC=C4C3(C)C"), 650, 1.9, 50),
            "benzothiazole_push_pull":  (Chem.MolFromSmarts("Nc1ccc(c2nc3ccccc3s2)cc1"), 360, 1.4, 35),
            "dma_benzothiazole":        (Chem.MolFromSmarts("CN(C)c1ccc2sc(nc2c1)c3ccc([N+](=O)[O-])cc3"), 450, 1.8, 45),
            "D_A_benzothiazole":        (Chem.MolFromSmarts("Nc1ccc2sc(c3ccc([N+](=O)[O-])cc3)nc2c1"), 440, 1.6, 45),
            "benzothiazole_donor":      (Chem.MolFromSmarts("Nc1ccc2sc(nc2c1)c1ccc([N+](=O)[O-])cc1"), 450, 1.8, 45),
            "benzoxazole_donor":        (Chem.MolFromSmarts("Nc1ccc2oc(nc2c1)c1ccc([N+](=O)[O-])cc1"), 440, 1.7, 45),
            "quinoline_donor":          (Chem.MolFromSmarts("Nc1ccc2nccc(c2c1)C(=O)c1ccc([N+](=O)[O-])cc1"), 420, 1.6, 40),
            "pyridine_donor":           (Chem.MolFromSmarts("Nc1ccc(nc1)c1ccc([N+](=O)[O-])cc1"), 380, 1.5, 35),
            "pyrimidine_donor":         (Chem.MolFromSmarts("Nc1ccc(nc1)c1ccc([N+](=O)[O-])cc1"), 370, 1.4, 35),
            "thiophene_donor_acceptor": (Chem.MolFromSmarts("Nc1ccc(c2cc([N+](=O)[O-])sc2)cc1"), 410, 1.6, 45),
            "furan_donor_acceptor":     (Chem.MolFromSmarts("Nc1ccc(c2cc([N+](=O)[O-])oc2)cc1"), 400, 1.5, 40),
            "pyrrole_donor_acceptor":   (Chem.MolFromSmarts("Nc1ccc(c2cc([N+](=O)[O-])[nH]2)cc1"), 390, 1.5, 40),
            "D_A_quinoline":            (Chem.MolFromSmarts("Nc1ccc2nccc(c2c1)C(=O)c3ccc([N+](=O)[O-])cc3"), 420, 1.5, 40),
            "D_A_phenothiazine":        (Chem.MolFromSmarts("Nc1ccc2c(c1)Sc1ccc(N)cc1N2C(=O)c3ccc([N+](=O)[O-])cc3"), 480, 1.7, 50),
            "donor_acceptor_thiophene": (Chem.MolFromSmarts("Nc1ccc(c2cc([N+](=O)[O-])sc2)cc1"), 410, 1.6, 45),
            "donor_acceptor_bithiophene":(Chem.MolFromSmarts("Nc1ccc(c2ccc(s2)c3ccc([N+](=O)[O-])cc3)cc1"), 460, 1.8, 45),
            "dicyano_benzene_donor":    (Chem.MolFromSmarts("Nc1ccc(C#N)c(C#N)c1"), 400, 1.5, 40),
            "cyano_acrylate_donor":     (Chem.MolFromSmarts("Nc1ccc(C=C(C#N)C(=O)O)cc1"), 395, 1.6, 40),
            "cyano_styrene_donor":      (Chem.MolFromSmarts("Nc1ccc(C=C(C#N)C#N)cc1"), 430, 1.8, 40),
            "dicyanovinyl_aniline":     (Chem.MolFromSmarts("Nc1ccc(C=C(C#N)C#N)cc1"), 430, 1.8, 40),
            "tricyanovinyl_aniline":    (Chem.MolFromSmarts("Nc1ccc(C=C(C#N)C#N)cc1"), 440, 1.9, 45),
            "dicyano_donor":            (Chem.MolFromSmarts("NC1=CC=C(C=C(C#N)C#N)C=C1"), 440, 1.9, 50),
            "cyano_nitro_aniline":      (Chem.MolFromSmarts("Nc1ccc(C#N)cc1[N+](=O)[O-]"), 420, 1.7, 42),
            "diaminodicyano_benzene":   (Chem.MolFromSmarts("Nc1cc(N)c(C#N)c(C#N)c1"), 410, 1.6, 40),
            "dicyano_benzene_da":       (Chem.MolFromSmarts("N#Cc1ccc(C#N)c(N)c1"), 400, 1.5, 40),
            "amino_chalcone":           (Chem.MolFromSmarts("Nc1ccc(C=CC(=O)c2ccccc2)cc1"), 375, 1.5, 40),
            "amino_benzophenone":       (Chem.MolFromSmarts("Nc1ccc(C(=O)c2ccccc2)cc1"), 330, 1.2, 30),
            "amino_acetophenone":       (Chem.MolFromSmarts("Nc1ccc(C(C)=O)cc1"), 320, 1.2, 30),
            "amino_benzaldehyde":       (Chem.MolFromSmarts("Nc1ccc(C=O)cc1"), 340, 1.3, 35),
            "hydroxy_benzaldehyde":     (Chem.MolFromSmarts("Oc1ccc(C=O)cc1"), 330, 1.2, 30),
            "hydroxy_acetophenone":     (Chem.MolFromSmarts("Oc1ccc(C(C)=O)cc1"), 320, 1.2, 30),
            "push_pull_chalcone":       (Chem.MolFromSmarts("Nc1ccc(C=CC(=O)c2ccccc2)cc1"), 375, 1.5, 40),
            "push_pull_indenone":       (Chem.MolFromSmarts("Nc1ccc(C=C2C(=O)c3ccccc3C2=O)cc1"), 420, 1.6, 40),
            "push_pull_fulvene":        (Chem.MolFromSmarts("Nc1ccc(C=C2C=CC(=O)C=C2)cc1"), 380, 1.5, 35),
            "push_pull_diene":          (Chem.MolFromSmarts("[NX3]C=CC=CC(=O)"), 350, 1.5, 35),
            "push_pull_polyene":        (Chem.MolFromSmarts("Nc1ccc(C=CC=CC=C[N+](=O)[O-])cc1"), 480, 2.0, 55),
            "amino_phenyl_butadiene":   (Chem.MolFromSmarts("Nc1ccc(C=CC=C[N+](=O)[O-])cc1"), 440, 1.8, 50),
            "donor_acceptor_ethyne":    (Chem.MolFromSmarts("Nc1ccc(C#Cc2ccc([N+](=O)[O-])cc2)cc1"), 360, 1.4, 35),
            "amino_anthraquinone":      (Chem.MolFromSmarts("Nc1ccc2C(=O)c3ccccc3C(=O)c2c1"), 500, 1.9, 50),
            "hydrazone_ct":             (Chem.MolFromSmarts("NN=C1C(=O)Nc2ccccc12"), 435, 1.2, 40),
            "tricyclic_bent_alkene":    (Chem.MolFromSmarts("C1=Cc2ccccc2CCc3ccccc13"), 240, 0.7, 20),
            "vinyl_nitro_donor":        (Chem.MolFromSmarts("COc1ccc(C=C[N+](=O)[O-])cc1"), 375, 1.5, 40),
            "dicyanomethylene_pyran":   (Chem.MolFromSmarts("CC1=CC=C(C=C(C#N)C#N)C=C1"), 460, 1.9, 50),
            "quinolyl_vinyl_nitro":     (Chem.MolFromSmarts("c1ccc2ncccc2c1C=C[N+](=O)[O-]"), 415, 1.7, 45),
            "aminopyridine_nitro":      (Chem.MolFromSmarts("Nc1ccncc1[N+](=O)[O-]"), 365, 1.5, 35),
            "hydroxy_styryl_pyridine":  (Chem.MolFromSmarts("Oc1ccc(C=Cc2ccccn2)cc1"), 385, 1.6, 40),
            "nitro_thiophene_donor":    (Chem.MolFromSmarts("Nc1ccc(c2cc([N+](=O)[O-])sc2)cc1"), 410, 1.6, 45),
            "nitro_thiophene_aniline":  (Chem.MolFromSmarts("Nc1ccc(c2cc([N+](=O)[O-])sc2)cc1"), 410, 1.6, 45),
            "nitro_pyrrole_da":         (Chem.MolFromSmarts("O=[N+]([O-])c1ccc(N)n1"), 350, 1.4, 35),
            "amino_cyano_biphenyl":     (Chem.MolFromSmarts("Nc1ccc(-c2ccc(C#N)cc2)cc1"), 370, 1.5, 38),
            "amino_nitro_biphenyl":     (Chem.MolFromSmarts("Nc1ccc(-c2ccc([N+](=O)[O-])cc2)cc1"), 400, 1.6, 42),
            "hydroxy_nitro_biphenyl":   (Chem.MolFromSmarts("Oc1ccc(-c2ccc([N+](=O)[O-])cc2)cc1"), 390, 1.5, 40),
            "benzothiadiazole_da":      (Chem.MolFromSmarts("c1cc2snc(c2c1)-c3cc4ccccc4n3"), 450, 1.6, 45),
            "amino_benzothiadiazole":   (Chem.MolFromSmarts("Nc1ccc2nsnc2c1"), 450, 1.6, 45),
            "au_alkynyl_push_pull":     (Chem.MolFromSmarts("[Au]C#C-c1[n]ccccc1"), 450, 1.8, 45),
            "nitro_coumarin":           (Chem.MolFromSmarts("O=c1ccc2cc([N+](=O)[O-])ccc2o1"), 400, 1.7, 45),
            "dma_coumarin":             (Chem.MolFromSmarts("O=c1ccc2cc(CN(C)C)ccc2o1"), 400, 1.7, 45),
        }

        # FIX #7: SMARTS validation at end of __init__
        self._validate_smarts()

    # ---------------- INTERNAL HELPERS ----------------
    @staticmethod
    def _normalize_epsilon(eps):
        # FIX #1: ε → intensity in 0.1–3.0 range
        return max(0.1, min(3.0, 3.0 * float(eps) / ChromophoreDetector.EPS_REF))

    def _validate_smarts(self):
        # FIX #7: catch silent SMARTS failures
        for dname, d in [("chromophores", self.chromophores),
                         ("auxochromes", self.auxochromes),
                         ("inductive_pull", self.inductive_pull),
                         ("insulators", self.insulators),
                         ("steric_penalties", self.steric_penalties),
                         ("push_pull_overrides", self.push_pull_overrides)]:
            bad = [k for k, v in d.items() if v[0] is None]
            if bad:
                print(f"⚠️ Invalid SMARTS in {dname}: {bad}")

    def _filter_td_dft_peaks(self, peaks, f_threshold=0.01):
        if not peaks: return []
        return [p for p in peaks if p[1] >= f_threshold]

    # ---------------- MAIN EVALUATOR ----------------
    def evaluate_molecule(self, smiles: str, td_dft_peaks=None):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return {"error": "Invalid SMILES", "final_verified_lambda": 254,
                    "intensity": 0.5, "width": 20, "active_overrides": ["invalid_structure"]}

        results = {"smiles": smiles, "system_type": "additive", "active_overrides": [],
                   "intensity": 0.5, "width": 20, "td_dft_verdict": "pending",
                   "insulator_cap": None, "insulator_penalty": 0.0,
                   "conjugation_length": 0, "ict_score": 0.0}

        # ---- Step 1: Insulators ----
        best_cap = None
        total_penalty = 0.0
        for name, (pat, cap, weight) in self.insulators.items():
            if pat and mol.HasSubstructMatch(pat):
                if best_cap is None or cap < best_cap:
                    best_cap = cap
                total_penalty += weight
                results["active_overrides"].append(f"Insulator: {name}")
        results["insulator_cap"] = best_cap
        # FIX #4 / #8: penalty damped and stored
        results["insulator_penalty"] = total_penalty * 0.3

        # ---- Step 2: Push–pull overrides ----
        best_pp_lam, best_pp_int, best_pp_wid = 0, 0.5, 20
        donors = self._get_donor_atoms(mol)
        acceptors = self._get_acceptor_atoms(mol)
        for name, (pat, over_lam, intensity, width) in self.push_pull_overrides.items():
            if pat and mol.HasSubstructMatch(pat):
                connected = any(self.verify_conjugated_path(mol, d, a)
                                for d in donors for a in acceptors)
                if connected and over_lam > best_pp_lam:
                    best_pp_lam, best_pp_int, best_pp_wid = over_lam, intensity, width
                    results["active_overrides"].append(f"ICT Synergy: {name}")

        # ---- Step 3: Conjugation length ----
        conj_len = ConjugationAnalyzer.longest_conjugated_path_with_hbond(mol)
        results["conjugation_length"] = conj_len
        if conj_len == 0:
            results["active_overrides"].append("no_conjugation")
            results["system_type"] = "no_conjugation"
            results.update({"final_verified_lambda": 0, "intensity": 0.0,
                            "width": 0, "ict_score": 0.0})
            return results

        # ---- Step 4: ICT score ----
        ict_score = self.detect_push_pull_general(mol)[3]
        results["ict_score"] = ict_score

        # ---- Step 5: Base λmax and intensity ----
        if best_pp_lam > 0:
            base_lambda = best_pp_lam
            # FIX #3: push-pull intensity on comparable scale
            results["intensity"] = max(0.1, min(3.0, best_pp_int))
            results["width"] = best_pp_wid
            results["system_type"] = "push-pull-ict"
        else:
            best_lam, best_eps, best_wid = 0, 0.0, 20
            for name, (pat, l_max, eps, width) in self.chromophores.items():
                if pat and mol.HasSubstructMatch(pat):
                    if l_max > best_lam:
                        best_lam, best_eps, best_wid = l_max, eps, width
                        results["active_overrides"].append(f"Base: {name}")
            base_lambda = best_lam
            # FIX #1: normalize ε instead of using raw ε
            results["intensity"] = self._normalize_epsilon(best_eps) if best_eps > 0 else 0.5
            results["width"] = best_wid

            if base_lambda > 0:
                # --- Auxochromes (FIX #2: additive) ---
                total_auxo_shift = 0.0
                for name, (pat, shift, inten_mult) in self.auxochromes.items():
                    if pat:
                        matches = len(mol.GetSubstructMatches(pat))
                        if matches > 0:
                            total_auxo_shift += shift * matches
                            # Additive damped increment on intensity
                            results["intensity"] += 0.15 * (inten_mult - 1.0) * min(matches, 3)
                total_auxo_shift = float(np.clip(total_auxo_shift, -80.0, 80.0))
                base_lambda += total_auxo_shift

                # --- Steric penalties ---
                for name, (pat, penalty, inten_mult) in self.steric_penalties.items():
                    if pat and mol.HasSubstructMatch(pat):
                        base_lambda += penalty
                        results["intensity"] *= inten_mult
                        results["active_overrides"].append(f"Steric Penalty: {name}")

                # --- Inductive pull ---
                for name, (pat, shift) in self.inductive_pull.items():
                    if pat and mol.HasSubstructMatch(pat):
                        base_lambda += shift
                        results["active_overrides"].append(f"Inductive Effect: {name}")

                # --- Conjugation length term ---
                if conj_len > 6:
                    extra = conj_len - 6
                    ring_info = mol.GetRingInfo()
                    arom_rings = sum(1 for ring in ring_info.AtomRings()
                                     if all(mol.GetAtomWithIdx(a).GetIsAromatic() for a in ring))
                    shift = 80.0 * np.log(1 + extra) + 5.0 * arom_rings
                    base_lambda += shift
                    results["active_overrides"].append(
                        f"Conjugation (log): +{extra} atoms, {arom_rings} rings, shift={shift:.1f} nm")

                # --- ICT boost ---
                if ict_score > 0.7:
                    base_lambda += 15
                    results["intensity"] *= 1.2
                    results["active_overrides"].append("ICT Boost: high")

        # ---- Step 6: Insulator penalty + cap (FIX #4) ----
        if base_lambda == 0:
            base_lambda = 200
            results["active_overrides"].append("no_chromophore_detected")
        base_lambda -= results["insulator_penalty"]
        if results["insulator_cap"] is not None:
            base_lambda = min(base_lambda, results["insulator_cap"])
        base_lambda = max(base_lambda, 200)

        # ---- Step 7: TD-DFT verification ----
        if td_dft_peaks:
            valid = self._filter_td_dft_peaks(td_dft_peaks, f_threshold=0.015)
            if valid:
                top = max(valid, key=lambda x: x[1])[0]
                if abs(top - base_lambda) <= 15:
                    results["td_dft_verdict"] = "y"
                    results["final_verified_lambda"] = top
                else:
                    results["td_dft_verdict"] = "n"
                    results["final_verified_lambda"] = base_lambda
            else:
                results["td_dft_verdict"] = "n (No valid peaks)"
                results["final_verified_lambda"] = base_lambda
        else:
            results["final_verified_lambda"] = base_lambda

        results["final_verified_lambda"] = min(max(results["final_verified_lambda"], 200), 800)
        results["intensity"] = min(max(results["intensity"], 0.1), 3.0)
        return results

    # ---------------- PUBLIC DETECT ----------------
    def detect(self, mol, name, pH, td_dft_peaks=None):
        try:
            nl = name.lower()
            # Named-compound overrides
            if "methylene blue" in nl or "methylene_blue" in nl:
                return [(664, 1.0, 22), (610, 0.35, 18), (292, 0.3, 16)], ["methylene_blue"]
            if "phenolphthalein" in nl:
                return ([(550, 1.2, 35)] if pH > 8.5 else [(275, 0.8, 20)]), ["phenolphthalein"]
            if "methyl orange" in nl:
                return ([(465, 0.9, 30)] if pH > 4.4 else [(505, 1.0, 35)]), ["methyl_orange"]
            if "tartrazine" in nl:
                return [(425, 1.0, 25), (255, 0.4, 15)], ["tartrazine"]
            if "beta carotene" in nl or "beta-carotene" in nl or "5280489" in nl:
                return [(450, 1.2, 35), (475, 0.9, 30)], ["beta_carotene"]
            if "riboflavin" in nl or "vitamin b2" in nl:
                return [(370, 1.0, 30), (445, 0.8, 28), (475, 0.7, 28)], ["riboflavin"]
            if "curcumin" in nl:
                return [(420, 1.3, 40)], ["curcumin"]

            res = self.evaluate_molecule(Chem.MolToSmiles(mol), td_dft_peaks=td_dft_peaks)
            if "error" in res:
                return [(254, 0.5, 20)], ["invalid_structure"]
            if res.get("final_verified_lambda") == 0:
                return [], ["no_absorption"]
            classes = res["active_overrides"] or ["default_heuristic"]
            return [(res["final_verified_lambda"], res["intensity"], res["width"])], classes
        except Exception as e:
            print(f"⚠️ Chromophore detection error for {name}: {e}")
            return [(254, 0.5, 20)], ["default_error"]

    # ---------------- D-A DETECTION HELPERS ----------------
    def detect_push_pull_general(self, mol):
        donor_smarts = ["[NX3]", "[n]", "[OX2]", "[SX2]", "c1ccccc1O", "c1ccccc1N"]
        acceptor_smarts = ["[N+](=O)[O-]", "C#N", "C(=O)", "C(=O)O", "C(=O)N", "S(=O)(=O)"]
        donors, acceptors = [], []
        for s in donor_smarts:
            p = Chem.MolFromSmarts(s)
            if p:
                donors.extend(m[0] for m in mol.GetSubstructMatches(p))
        for s in acceptor_smarts:
            p = Chem.MolFromSmarts(s)
            if p:
                acceptors.extend(m[0] for m in mol.GetSubstructMatches(p))
        donors, acceptors = list(set(donors)), list(set(acceptors))
        if not donors or not acceptors:
            return 0, 0, 999, 0.0
        dm = GetDistanceMatrix(mol)
        min_d = min(dm[d][a] for d in donors for a in acceptors)
        return len(donors), len(acceptors), min_d, (1.0 / (1.0 + min_d) if min_d > 0 else 1.0)

    def _get_donor_atoms(self, mol):
        atoms = []
        for s in ["[NX3]", "[n]", "[OX2]", "[SX2]", "c1ccccc1O", "c1ccccc1N"]:
            p = Chem.MolFromSmarts(s)
            if p:
                atoms.extend(m[0] for m in mol.GetSubstructMatches(p))
        return list(set(atoms))

    def _get_acceptor_atoms(self, mol):
        atoms = []
        for s in ["[N+](=O)[O-]", "C#N", "C(=O)", "C(=O)O", "C(=O)N", "S(=O)(=O)"]:
            p = Chem.MolFromSmarts(s)
            if p:
                atoms.extend(m[0] for m in mol.GetSubstructMatches(p))
        return list(set(atoms))

    def verify_conjugated_path(self, mol, donor_atom, acceptor_atom):
        conj_len = ConjugationAnalyzer.longest_conjugated_path(mol)
        dm = GetDistanceMatrix(mol)
        return dm[donor_atom][acceptor_atom] <= conj_len


print("✅ ChromophoreDetector + SolventDatabase loaded.")

✅ ChromophoreDetector + SolventDatabase loaded.


In [ ]:
# @title 4.) Thermodynamic Reaction Engine + Adaptive Learner
import os, pickle
import numpy as np

# ---------- Bond Dissociation Energies (kJ/mol, 298 K) ----------
BDE_TABLE = {
    ("C", "Cl"): 328, ("C", "Br"): 285, ("C", "I"): 234, ("C", "F"): 485,
    ("C", "O"): 358, ("C=O", "O"): 350, ("O", "H"): 463,
    ("C", "N"): 305, ("N", "H"): 391, ("C", "C"): 348,
    ("C=", "C"): 614, ("C", "S"): 259, ("S", "H"): 339,
    ("C", "O="): 745, ("C#", "N"): 891,
    ("S", "S"): 240,
    ("P", "O"): 335,
    ("C", "P"): 264,
    ("C", "N="): 615,
    ("C=", "O"): 745,
}

class ReactionEngine:
    """SMARTS-based solvolysis transforms (21 reaction classes)."""
    def __init__(self):
        self.reactions = []
        R = self._add
        # --- Original 6 ---
        R("[CX3:1](=[OX1:2])[OX2:3][CX3:4](=[OX1:5]).[OX2H1:6][CX4:7]>>[CX3:1](=[OX1:2])[OX2:3][CX4:7].[CX3:4](=[OX1:5])[OX2H1:6]",
          "anhydride_alcohol", "anhydride + alcohol → monoester + acid")
        R("[CX3:1](=[OX1:2])[Cl:3].[OX2H1:4][CX4:5]>>[CX3:1](=[OX1:2])[OX2:4][CX4:5]",
          "acid_chloride_alcohol", "acid chloride + alcohol → ester")
        R("[CX3:1](=[OX1:2])[OX2:3][CX4:4].[OX2H1:5][CX4:6]>>[CX3:1](=[OX1:2])[OX2:5][CX4:6].[OX2H1:3][CX4:4]",
          "ester_alcohol", "ester + alcohol → transesterification")
        R("[CX3:1](=[OX1:2])[OX2:3][CX4:4][CX4:5].[OX2H1:6][CX4:7]>>[CX3:1](=[OX1:2])[OX2:6][CX4:7].[OX2H1:3][CX4:4][CX4:5]",
          "lactone_alcohol", "lactone + alcohol → hydroxy-ester")
        R("[CX4:1]1[OX2:2][CX4:3]1.[OX2H1:4][CX4:5]>>[CX4:1]([OX2H1:4][CX4:5])[CX4:3][OX2H1:2]",
          "epoxide_alcohol", "epoxide + alcohol → β-hydroxy ether")
        R("[NX2:1]=[CX3:2]=[OX1:3].[OX2H1:4][CX4:5]>>[NX2:1]([CX3:2](=[OX1:3])[OX2H1:4])[CX4:5]",
          "isocyanate_alcohol", "isocyanate + alcohol → carbamate")
        # --- Expansion 15 ---
        R("[CX3H1:1](=[OX1:2]).[NX3H2:3][CX4:4]>>[CX3H1:1]=[NX3:3][CX4:4]",
          "aldehyde_amine", "aldehyde + amine → imine")
        R("[#6:5][CX3:1](=[OX1:2]).[NX3H2:3][CX4:4]>>[#6:5][CX3:1](=[NX3:3][CX4:4])",
          "ketone_amine", "ketone + amine → imine")
        R("[CX3H1:1](=[OX1:2]).[OX2H1:3][CX4:4]>>[CX3H1:1]([OX2H1:2])[OX2:3][CX4:4]",
          "aldehyde_alcohol", "aldehyde + alcohol → hemiacetal")
        R("[#6:5][CX3:1](=[OX1:2]).[OX2H1:3][CX4:4]>>[#6:5][CX3:1]([OX2H1:2])[OX2:3][CX4:4]",
          "ketone_alcohol", "ketone + alcohol → hemiketal")
        R("[CX3:1](=[OX1:2])[Cl:3].[OX2H2:4]>>[CX3:1](=[OX1:2])[OX2H1:4]",
          "acid_chloride_water", "acid chloride + water → acid")
        R("[CX3:1](=[OX1:2])[OX2:3][CX3:4](=[OX1:5]).[OX2H2:6]>>[CX3:1](=[OX1:2])[OX2H1:6].[CX3:4](=[OX1:5])[OX2H1:3]",
          "anhydride_water", "anhydride + water → 2 acids")
        R("[CX3:1](=[OX1:2])[OX2:3][CX4:4].[OX2H2:5]>>[CX3:1](=[OX1:2])[OX2H1:5].[OX2H1:3][CX4:4]",
          "ester_water", "ester + water → acid + alcohol")
        R("[CX3:1](=[OX1:2])[NX3:3].[OX2H2:4]>>[CX3:1](=[OX1:2])[OX2H1:4].[NX3H1:3]",
          "amide_water", "amide + water → acid + amine")
        R("[NX3:1]1[CX3:2](=[OX1:3])[CX4:4][CX4:5]1.[OX2H2:6]>>[NX3H1:1][CX3:2](=[OX1:3])[CX4:4][CX4:5][OX2H1:6]",
          "lactam_water", "β-lactam + water → ring-opened acid")
        R("[NX3:1]1[CX3:2](=[OX1:3])[CX4:4][CX4:5]1.[NX3H2:6][CX4:7]>>[NX3H1:1][CX3:2](=[OX1:3])[CX4:4][CX4:5][NX3H1:6][CX4:7]",
          "lactam_amine", "β-lactam + amine → ring-opened amide")
        R("[CX3:1](=[OX1:2])[OX2:3][CX3:4](=[OX1:5]).[NX3H2:6]>>[CX3:1](=[OX1:2])[NX3H1:6].[CX3:4](=[OX1:5])[OX2H1:3]",
          "anhydride_amine", "anhydride + amine → amide + acid")
        R("[CX3:1](=[OX1:2])[Cl:3].[NX3H2:4][CX4:5]>>[CX3:1](=[OX1:2])[NX3H1:4][CX4:5]",
          "acid_chloride_amine", "acid chloride + amine → amide")
        R("[NX2:1]=[CX3:2]=[OX1:3].[NX3H2:4][CX4:5]>>[NX2:1]([CX3:2](=[OX1:3])[NX3H1:4])[CX4:5]",
          "isocyanate_amine", "isocyanate + amine → urea")
        R("[CX3H1:1]=[CX3H1:2][CX3:3](=[OX1:4]).[SX2H1:5][CX4:6]>>[CX3H2:1][CX3H1:2]([SX2:5][CX4:6])[CX3:3](=[OX1:4])",
          "enone_thiol", "Michael: enone + thiol")

        R("[CX3H1:1]=[CX3H1:2][CX3:3](=[OX1:4]).[NX3H2:5][CX4:6]>>[CX3H2:1][CX3H1:2]([NX3H1:5][CX4:6])[CX3:3](=[OX1:4])",
          "enone_amine", "Michael: enone + amine")

        # =================================================================
        # v13.7.4 — Expansion to 50 rules (29 new transformations)
        # =================================================================

        # ---- Group 1: Pharmaceutical stability (5) ----
        R("[#6:5][OX2:4][CX3:2](=[OX1:3])[NX3:1].[OX2H2:6]>>[#6:5][OX2H1:4].[NX3H1:1].[CX3:2](=[OX1:3])=[OX1:6]",
          "carbamate_hydrolysis", "carbamate (urethane) + water → alcohol + amine + CO2")
        R("[PX4:1](=[OX1:2])([OX2:3])([OX2:4])[OX2:5][#6:6].[OX2H2:7]>>[PX4:1](=[OX1:2])([OX2:3])([OX2:4])[OX2H1:7].[OX2H1:5][#6:6]",
          "phosphate_ester_hydrolysis", "phosphate ester + water → phosphate + alcohol")
        R("[SX4:1](=[OX1:2])(=[OX1:3])([OX2:4])[OX2:5][#6:6].[OX2H2:7]>>[SX4:1](=[OX1:2])(=[OX1:3])([OX2:4])[OX2H1:7].[OX2H1:5][#6:6]",
          "sulfate_ester_hydrolysis", "sulfate ester + water → sulfate + alcohol")
        R("[SX4:1](=[OX1:2])(=[OX1:3])([#6:4])[OX2:5][#6:6].[OX2H2:7]>>[SX4:1](=[OX1:2])(=[OX1:3])([#6:4])[OX2H1:7].[OX2H1:5][#6:6]",
          "sulfonate_ester_hydrolysis", "sulfonate ester + water → sulfonic acid + alcohol")
        R("[NX3:1][CX3:2](=[OX1:3])[NX3:4].[OX2H2:5]>>[NX3H1:1].[NX3H1:4].[CX3:2](=[OX1:3])=[OX1:5]",
          "urea_hydrolysis", "urea + water → 2 amines + CO2")

        # ---- Group 2: Bioconjugation (5) ----
        R("[CX3:1](=[OX1:2])[OX2:3][NX3:4][CX3:5](=[OX1:6])[#6:7].[NX3H2:8][#6:9]>>[CX3:1](=[OX1:2])[NX3H1:8][#6:9].[OX2H1:3][NX3:4][CX3:5](=[OX1:6])[#6:7]",
          "nhs_ester_amine", "NHS ester + amine → amide + NHS-OH (protein labelling)")
        R("[OX1:1]=[CX3:2]1[CX3H1:3]=[CX3H1:4][CX3:5](=[OX1:6])[NX3:7]1.[SX2H1:8][#6:9]>>[OX1:1]=[CX3:2]1[CX3H1:3]([SX2:8][#6:9])[CX3H1:4][CX3:5](=[OX1:6])[NX3:7]1",
          "maleimide_thiol", "maleimide + thiol → succinimide thioether (cysteine labelling)")
        R("[NX2:1]=[CX3:2]=[SX1:3].[NX3H2:4][#6:5]>>[NX2:1]([CX3:2](=[SX1:3])[NX3H1:4])[#6:5]",
          "isothiocyanate_amine", "isothiocyanate + amine → thiourea (FITC labelling)")
        R("[CX3H1:1](=[OX1:2]).[CX3:3](=[OX1:4])[NX3H1:5][NX3H2:6]>>[CX3H1:1]=[NX3:6][NX3H1:5][CX3:3](=[OX1:4])",
          "aldehyde_hydrazide", "aldehyde + hydrazide → hydrazone (ADC linker)")
        R("[#6:1][SX2:2][SX2:3][#6:4].[SX2H1:5][#6:6]>>[#6:1][SX2:2][SX2:5][#6:6].[SX2H1:3][#6:4]",
          "thiol_disulfide_exchange", "thiol + disulfide → new disulfide + thiol")

        # ---- Group 3: Carbonyl condensations (6) ----
        R("[CX3H1:1](=[OX1:2]).[NX3H2:3][NX3H2:4]>>[CX3H1:1]=[NX3:3][NX3H2:4]",
          "aldehyde_hydrazine", "aldehyde + hydrazine → hydrazone")
        R("[#6:1][CX3:2](=[OX1:3])[#6:4].[NX3H2:5][NX3H2:6]>>[#6:1][CX3:2](=[NX3:5][NX3H2:6])[#6:4]",
          "ketone_hydrazine", "ketone + hydrazine → hydrazone")
        R("[CX3H1:1](=[OX1:2]).[NX3H2:3][OX2H1:4]>>[CX3H1:1]=[NX3:3][OX2H1:4]",
          "aldehyde_hydroxylamine", "aldehyde + hydroxylamine → oxime")
        R("[#6:1][CX3:2](=[OX1:3])[#6:4].[NX3H2:5][OX2H1:6]>>[#6:1][CX3:2](=[NX3:5][OX2H1:6])[#6:4]",
          "ketone_hydroxylamine", "ketone + hydroxylamine → oxime")
        R("[C:1]=O.[OX2H1:2][C:3][C:4][OX2H1:5]>>[C:3]1[OX2:2][C:1][OX2:5][C:4]1",
          "aldehyde_diol_acetal", "aldehyde + 1,2-diol → cyclic acetal")
        R("[C:1](=O)([#6:2])[#6:3].[OX2H1:4][C:5][C:6][OX2H1:7]>>[C:5]1[OX2:4][C:1]([#6:2])([#6:3])[OX2:7][C:6]1",
          "ketone_diol_ketal", "ketone + 1,2-diol → cyclic ketal")

        # ---- Group 4: Organic synthesis (6) ----
        R("[CX3H1:1]=O.[CX4H2:2][CX3H1:3]=O>>[CX4H1:1](O)[CX4H1:2][CX3H1:3]=O",
          "aldol_addition", "aldol addition → β-hydroxy carbonyl")
        R("[CX3H1:1]=O.[CX4H2:2][CX3H1:3]=O>>[CX3H1:1]=[CX3H1:2][CX3H1:3]=O",
          "aldol_condensation", "aldol condensation → α,β-unsaturated carbonyl")
        R("[CX3:1](=O)[OX2:2][#6:3].[CX4H2:4][CX3:5](=O)[OX2:6][#6:7]>>[CX3:1](=O)[CX4H1:4]([CX3:5](=O)[OX2:6][#6:7])[OX2:2][#6:3]",
          "claisen_condensation", "Claisen condensation → β-keto ester")
        R("[CX3H1:1]=O.[CX4H2:2]([CX3:3]#[NX2:4])[CX3:5]#[NX2:6]>>[CX3H1:1]=[CX3:2]([CX3:3]#[NX2:4])[CX3:5]#[NX2:6]",
          "knoevenagel_condensation", "Knoevenagel: aldehyde + active-methylene → enone")
        R("[CX3H1:1]=O.[PX4+:2]([c:3])([c:4])([c:5])=[CX4H2:6]>>[CX3H1:1]=[CX4H2:6].[PX4:2]([c:3])([c:4])([c:5])=O",
          "wittig_reaction", "Wittig: aldehyde + phosphonium ylide → alkene + Ph3P=O")
        R("[CX3H1:1]=[CX3H1:2][CX3:3](=O).[CX4H2:4][CX3:5](=O)[#6:6]>>[CX3H2:1][CX3H1:2]([CX4H1:4][CX3:5](=O)[#6:6])[CX3:3](=O)",
          "michael_addition_carbon", "Michael: enone + enolate carbon → 1,4-diketone")

        # ---- Group 5: Amine chemistry (3) ----
        R("[NX3H2:1][#6:2].[OX1:3]=[CX3:4]=[OX1:5]>>[NX3H1:1]([CX3:4](=[OX1:3])[OX2H:5])[#6:2]",
          "amine_co2_carbamate", "amine + CO2 → carbamate (CO2 capture)")
        R("[#6:1][CX3:2](=O)[CX4H2:3].[NX3H1:4][#6:5][#6:6]>>[#6:1][CX3:2]([CX4H1:3])=[NX3+:4]([#6:5][#6:6])",
          "enamine_formation", "ketone + secondary amine → enamine / iminium")
        R("[CX3H1:1]=O.[NX3H1:2][#6:3][#6:4]>>[CX3H1:1]=[NX3+:2]([#6:3][#6:4])",
          "iminium_formation", "aldehyde + secondary amine → iminium ion")

        # ---- Group 6: SN2 / ring opening / aminolysis (4) ----
        R("[CX4H2:1][Cl,Br,I:2].[NX3H2:3][#6:4]>>[CX4H2:1][NX3H1:3][#6:4].[ClH,BrH,IH:2]",
          "alkyl_halide_amine", "SN2: alkyl halide + amine → secondary amine + HX")
        R("[CX4H2:1][Cl,Br,I:2].[SX2H1:3][#6:4]>>[CX4H2:1][SX2:3][#6:4].[ClH,BrH,IH:2]",
          "alkyl_halide_thiol", "SN2: alkyl halide + thiol → thioether + HX")
        R("[CX4:1]1[OX2:2][CX4:3]1.[NX3H2:4][#6:5]>>[CX4:1]([OX2H1:2])[CX4:3][NX3H1:4][#6:5]",
          "epoxide_amine", "epoxide + amine → β-amino alcohol")
        R("[CX3:1](=O)[OX2:2][#6:3].[NX3H2:4][#6:5]>>[CX3:1](=O)[NX3H1:4][#6:5].[OX2H:2][#6:3]",
          "ester_aminolysis", "ester + amine → amide + alcohol")

        print(f"✅ ReactionEngine loaded ({len(self.reactions)} rules).")

    def _add(self, smarts, key, name):
        try:
            rxn = AllChem.ReactionFromSmarts(smarts)
            if rxn is not None:
                self.reactions.append({"rxn": rxn, "key": key, "name": name})
        except Exception as e:
            print(f"⚠️ SMARTS error [{key}]: {e}")

    def apply(self, solute_mol, solvent_smiles, only_key=None, max_products=5):
        if solute_mol is None or not solvent_smiles:
            return []
        solv = Chem.MolFromSmiles(solvent_smiles)
        if solv is None:
            return []

        solute_smi = Chem.MolToSmiles(solute_mol, canonical=True)
        solv_smi = Chem.MolToSmiles(solv, canonical=True)

        products = []
        for r in self.reactions:
            if only_key and r["key"] != only_key:
                continue

            prod_tuples = []
            # Method A: separate reactants (standard RDKit practice)
            try:
                prod_tuples = r["rxn"].RunReactants((solute_mol, solv))
            except Exception:
                prod_tuples = []
            # Method B: bundled fallback
            if not prod_tuples:
                try:
                    bundle = Chem.CombineMols(solute_mol, solv)
                    prod_tuples = r["rxn"].RunReactants((bundle,))
                except Exception:
                    prod_tuples = []

            for prod_tuple in prod_tuples:
                for prod in prod_tuple:
                    try:
                        Chem.SanitizeMol(prod)
                        prod_smi = Chem.MolToSmiles(prod, canonical=True)
                        # Skip no-ops
                        if prod_smi in (solute_smi, solv_smi):
                            continue
                        products.append((prod, r["name"], r["key"]))
                    except Exception:
                        continue

        seen = set()
        uniq = []
        for p, n, k in products:
            s = Chem.MolToSmiles(p, canonical=True)
            if s not in seen:
                seen.add(s)
                uniq.append((p, n, k))
        return uniq[:max_products]

    def matches(self, key, solute_mol, solvent_smiles):
        """Return True if the reaction SMARTS for 'key' can actually match.

        Does NOT generate products; only pattern-matches. Cheap pre-check.
        """
        if solute_mol is None or not solvent_smiles:
            return False
        solv = Chem.MolFromSmiles(solvent_smiles)
        if solv is None:
            return False
        for r in self.reactions:
            if r["key"] != key:
                continue
            try:
                tmpl = r["rxn"].GetReactantTemplate(0)
                if tmpl is None:
                    return False
                bundle = Chem.CombineMols(solute_mol, solv)
                return len(bundle.GetSubstructMatches(tmpl)) > 0
            except Exception:
                return False
        return False


class ReactionThermodynamics:
    """Physics teacher: ΔG gate with pH / T / ε modulation."""
    RULES = {
        "anhydride_alcohol":     {"broken": [("C=O","O"),("O","H")], "formed": [("C","O"),("O","H")], "base_dG": -15.0, "optimal_pH": (3.0, 9.5)},
        "acid_chloride_alcohol": {"broken": [("C","Cl"),("O","H")],  "formed": [("C","O"),("O","H")], "base_dG": -45.0, "optimal_pH": (4.0, 9.5)},
        "ester_alcohol":         {"broken": [("C=O","O"),("O","H")], "formed": [("C","O"),("O","H")], "base_dG":  -3.0, "optimal_pH": (4.0,10.0)},
        "lactone_alcohol":       {"broken": [("C=O","O"),("O","H")], "formed": [("C","O"),("O","H")], "base_dG":  -8.0, "optimal_pH": (4.0,10.0)},
        "epoxide_alcohol":       {"broken": [("C","O"),("O","H")],   "formed": [("C","O"),("O","H")], "base_dG": -10.0, "optimal_pH": (5.0, 9.5)},
        "isocyanate_alcohol":    {"broken": [("C=","C"),("O","H")],  "formed": [("C","N"),("O","H")], "base_dG": -55.0, "optimal_pH": (4.0, 9.5)},
        "aldehyde_amine":        {"broken": [("C=O","O"),("N","H")], "formed": [("C","N")],           "base_dG":  -8.0, "optimal_pH": (3.0, 8.0)},
        "ketone_amine":          {"broken": [("C=O","O"),("N","H")], "formed": [("C","N")],           "base_dG":  -5.0, "optimal_pH": (3.0, 8.0)},
        "aldehyde_alcohol":      {"broken": [("C=O","O"),("O","H")], "formed": [("C","O"),("O","H")], "base_dG":  -2.0, "optimal_pH": (2.0, 6.0)},
        "ketone_alcohol":        {"broken": [("C=O","O"),("O","H")], "formed": [("C","O"),("O","H")], "base_dG":  -1.0, "optimal_pH": (2.0, 6.0)},
        "acid_chloride_water":   {"broken": [("C","Cl"),("O","H")],  "formed": [("C","O"),("O","H")], "base_dG": -50.0, "optimal_pH": (0.0,12.0)},
        "anhydride_water":       {"broken": [("C=O","O"),("O","H")], "formed": [("C","O"),("O","H")], "base_dG": -25.0, "optimal_pH": (0.0,12.0)},
        "ester_water":           {"broken": [("C=O","O"),("O","H")], "formed": [("C","O"),("O","H")], "base_dG":  +2.0, "optimal_pH": (0.0, 3.0)},
        "amide_water":           {"broken": [("C","N"),("O","H")],   "formed": [("C","O"),("O","H")], "base_dG":  +5.0, "optimal_pH": (0.0, 2.0)},
        "lactam_water":          {"broken": [("C","N"),("O","H")],   "formed": [("C","O"),("O","H")], "base_dG":  -5.0, "optimal_pH": (3.0, 9.5)},
        "lactam_amine":          {"broken": [("C","N"),("N","H")],   "formed": [("C","N")],           "base_dG": -12.0, "optimal_pH": (5.0, 9.5)},
        "anhydride_amine":       {"broken": [("C=O","O"),("N","H")], "formed": [("C","N")],           "base_dG": -35.0, "optimal_pH": (4.0, 9.5)},
        "acid_chloride_amine":   {"broken": [("C","Cl"),("N","H")],  "formed": [("C","N")],           "base_dG": -55.0, "optimal_pH": (4.0, 9.5)},
        "isocyanate_amine":      {"broken": [("C=","C"),("N","H")],  "formed": [("C","N")],           "base_dG": -45.0, "optimal_pH": (4.0, 9.5)},
        "enone_thiol":           {"broken": [("C=","C"),("S","H")],  "formed": [("C","S")],           "base_dG": -10.0, "optimal_pH": (6.0,10.0)},
        "enone_amine":           {"broken": [("C=","C"),("N","H")],  "formed": [("C","N")],           "base_dG":  -6.0, "optimal_pH": (6.0,10.0)},

        # --- Group 1: Pharmaceutical stability ---
        "carbamate_hydrolysis":      {"broken": [("C","N"),("C","O"),("O","H")], "formed": [("C","O"),("O","H"),("C","O=")], "base_dG": -12.0, "optimal_pH": (10.0,14.0)},
        "phosphate_ester_hydrolysis":{"broken": [("P","O"),("O","H")],          "formed": [("P","O"),("O","H")],            "base_dG":  -8.0, "optimal_pH": (7.0,12.0)},
        "sulfate_ester_hydrolysis":  {"broken": [("S","O"),("O","H")],          "formed": [("S","O"),("O","H")],            "base_dG":  -6.0, "optimal_pH": (5.0,12.0)},
        "sulfonate_ester_hydrolysis":{"broken": [("S","O"),("O","H")],          "formed": [("S","O"),("O","H")],            "base_dG":  -4.0, "optimal_pH": (5.0,12.0)},
        "urea_hydrolysis":           {"broken": [("C","N"),("C","N"),("O","H")],"formed": [("N","H"),("N","H"),("C","O=")], "base_dG":  +8.0, "optimal_pH": (0.0, 2.0)},

        # --- Group 2: Bioconjugation ---
        "nhs_ester_amine":           {"broken": [("C","O"),("N","H")],          "formed": [("C","N"),("O","H")],            "base_dG": -30.0, "optimal_pH": (7.0, 9.0)},
        "maleimide_thiol":           {"broken": [("C=","C"),("S","H")],         "formed": [("C","S"),("C","H")],            "base_dG": -25.0, "optimal_pH": (6.5, 7.5)},
        "isothiocyanate_amine":      {"broken": [("C=","C"),("N","H")],         "formed": [("C","N")],                      "base_dG": -40.0, "optimal_pH": (7.5, 9.0)},
        "aldehyde_hydrazide":        {"broken": [("C=","O"),("N","H")],         "formed": [("C","N")],                      "base_dG": -14.0, "optimal_pH": (4.5, 7.0)},
        "thiol_disulfide_exchange":  {"broken": [("S","S"),("S","H")],          "formed": [("S","S"),("S","H")],            "base_dG":  -2.0, "optimal_pH": (6.0, 9.0)},

        # --- Group 3: Carbonyl condensations ---
        "aldehyde_hydrazine":        {"broken": [("C=","O"),("N","H")],         "formed": [("C","N=")],                     "base_dG": -12.0, "optimal_pH": (4.0, 8.0)},
        "ketone_hydrazine":          {"broken": [("C=","O"),("N","H")],         "formed": [("C","N=")],                     "base_dG":  -8.0, "optimal_pH": (3.0, 7.0)},
        "aldehyde_hydroxylamine":    {"broken": [("C=","O"),("N","H")],         "formed": [("C","N=")],                     "base_dG": -10.0, "optimal_pH": (4.0, 6.5)},
        "ketone_hydroxylamine":      {"broken": [("C=","O"),("N","H")],         "formed": [("C","N=")],                     "base_dG":  -6.0, "optimal_pH": (3.5, 6.0)},
        "aldehyde_diol_acetal":      {"broken": [("C=","O"),("O","H"),("O","H")],"formed": [("C","O"),("C","O")],            "base_dG":  -4.0, "optimal_pH": (1.5, 4.5)},
        "ketone_diol_ketal":         {"broken": [("C=","O"),("O","H"),("O","H")],"formed": [("C","O"),("C","O")],            "base_dG":  -2.0, "optimal_pH": (1.0, 4.0)},

        # --- Group 4: Organic synthesis ---
        "aldol_addition":            {"broken": [("C=","O"),("C","H")],         "formed": [("C","C"),("O","H")],            "base_dG":  -6.0, "optimal_pH": (2.0,12.0)},
        "aldol_condensation":        {"broken": [("C=","O"),("C","H"),("C","H")],"formed": [("C=","C")],                       "base_dG":  -8.0, "optimal_pH": (2.0,12.0)},
        "claisen_condensation":      {"broken": [("C=","O"),("C","H"),("C","O")],"formed": [("C","C"),("C=O","O")],          "base_dG":  -5.0, "optimal_pH": (10.0,14.0)},
        "knoevenagel_condensation":  {"broken": [("C=","O"),("C","H")],         "formed": [("C=","C")],                     "base_dG": -10.0, "optimal_pH": (8.0,11.0)},
        "wittig_reaction":           {"broken": [("C=","O"),("C","P")],         "formed": [("C=","C"),("C=","O")],          "base_dG": -20.0, "optimal_pH": (5.0, 9.0)},
        "michael_addition_carbon":   {"broken": [("C=","C"),("C","H")],         "formed": [("C","C")],                      "base_dG":  -9.0, "optimal_pH": (9.0,13.0)},

        # --- Group 5: Amine chemistry ---
        "amine_co2_carbamate":       {"broken": [("N","H"),("C=","O")],         "formed": [("C","N"),("C","O")],            "base_dG":  -3.0, "optimal_pH": (8.0,11.0)},
        "enamine_formation":         {"broken": [("C=","O"),("N","H")],         "formed": [("C","N="),("O","H")],           "base_dG":  -4.0, "optimal_pH": (3.0, 6.0)},
        "iminium_formation":         {"broken": [("C=","O"),("N","H")],         "formed": [("C","N=")],                     "base_dG":  -3.0, "optimal_pH": (3.0, 5.5)},

        # --- Group 6: SN2 / ring opening ---
        "alkyl_halide_amine":        {"broken": [("C","Cl"),("N","H")],         "formed": [("C","N")],                      "base_dG": -16.0, "optimal_pH": (8.0,11.0)},
        "alkyl_halide_thiol":        {"broken": [("C","Cl"),("S","H")],         "formed": [("C","S")],                      "base_dG": -18.0, "optimal_pH": (7.0,10.0)},
        "epoxide_amine":             {"broken": [("C","O"),("N","H")],          "formed": [("C","N"),("O","H")],            "base_dG": -14.0, "optimal_pH": (5.0, 9.5)},
        "ester_aminolysis":          {"broken": [("C=O","O"),("N","H")],        "formed": [("C","N"),("O","H")],            "base_dG":  -6.0, "optimal_pH": (7.0,10.0)},
    }
    def _bde(self, pair): return BDE_TABLE.get(pair, 350.0)
    def _ph(self, pH, lo, hi):
        if lo <= pH <= hi: return 1.0
        return max(0.2, 1.0 - ((lo - pH) * 0.25 if pH < lo else (pH - hi) * 0.25))
    def _T(self, T): return float(np.exp((T - 298.15) / 30.0))
    def _eps(self, eps): return float(np.clip(0.5 + 0.02 * eps, 0.5, 2.0))
    def evaluate(self, solute_mol, solvent_props, pH, temp_K):
        out = []
        for key, rule in self.RULES.items():
            dH = sum(self._bde(b) for b in rule["broken"]) - sum(self._bde(f) for f in rule["formed"]) + rule["base_dG"]
            dS = -0.05
            dG = dH - temp_K * dS
            lo, hi = rule["optimal_pH"]
            f = self._ph(pH, lo, hi) * self._T(temp_K) * self._eps(solvent_props.get("dielectric", 30.0))
            dG_eff = dG / max(f, 0.05)
            prob = 1.0 / (1.0 + np.exp((dG_eff + 5.0) / 5.0))
            out.append({"key": key, "delta_G": float(dG_eff), "probability": float(prob)})
        return out


class ReactionFeasibilityML:
    """RandomForest surrogate of the physics engine. Persists to .pkl."""
    MODEL_PATH = "reaction_feasibility.pkl"
    N_FEATURES = 28

    def __init__(self, model_path=None):
        self.model_path = model_path or self.MODEL_PATH
        self.model = None; self.scaler = None; self.fitted = False
        self.thermo = ReactionThermodynamics()
        self._load_or_train()

    def _vec(self, mol, sp, pH, T):
        from rdkit.Chem import Descriptors
        try:
            mw = Descriptors.MolWt(mol); logp = Descriptors.MolLogP(mol); tpsa = Descriptors.TPSA(mol)
        except Exception:
            mw, logp, tpsa = 200.0, 0.0, 40.0
        fg_smarts = ["[CX3](=O)[OX2][CX3]=O", "[CX3](=O)[Cl]", "[CX3](=O)[OX2][CX4]",
                     "[CX4]1[OX2][CX4]1", "[NX2]=[CX3]=O", "[CX3H1](=O)",
                     "[CX3](=O)[NX3]", "[CX3](=O)[OX2H1]",
                     "[NX3]1[CX3](=O)[CX4][CX4]1", "[CX3H1]=[CX3H1][CX3]=O",
                     "[#6][CX3](=O)[#6]", "[NX3H2][CX4]", "[SX2H1]", "[OX2H2]",
                     "[#6]N=C=O", "[CX4]1[OX2][CX4]1"]
        fg = []
        for sm in fg_smarts:
            p = Chem.MolFromSmarts(sm)
            fg.append(min(len(mol.GetSubstructMatches(p)) if p else 0, 4) / 4.0)
        v = [mw/500.0, logp/5.0, tpsa/150.0, *fg,
             sp.get("dielectric",30.0)/80.0, sp.get("polarity",0.5),
             sp.get("hbond_donor",0.5), sp.get("hbond_acceptor",0.5),
             pH/14.0, T/373.0, float(np.clip((T-298.15)/100.0,-1,1)),
             sp.get("pi_star",0.5)]
        v = v[:self.N_FEATURES]
        while len(v) < self.N_FEATURES: v.append(0.0)
        return np.array(v, dtype=np.float32)

    def _synthetic(self, n=3000):
        import random; random.seed(42); np.random.seed(42)
        smi_pool = [
            "O=C1C=CC(=O)O1","O=C1OC(=O)c2ccccc21","O=C1CCC(=O)O1","O=C(Cl)c1ccccc1",
            "O=C1OCc2ccccc21","O=C1CCc2ccccc21","CC1=CC(=O)OC1","C1OC1c1ccccc1",
            "O=C=Nc1ccccc1","CC(=O)Oc1ccccc1C(=O)O","O=Cc1ccccc1","Nc1ccccc1",
            "O=C(N)c1ccccc1","CC(=O)OC","O=C(O)c1ccccc1","c1ccc2c(c1)oc1ccccc12",
            "CC(=O)C","c1ccccc1","CC(=O)Nc1ccccc1","O=C1Nc2ccccc21",
            "CC(=O)C=Cc1ccccc1","O=C(C=Cc1ccccc1)c1ccccc1","CN","CCN","OCCN",
            "CC(N)=O","NC(=O)C=C","O=C1CC(=O)NC1","c1ccccc1CS","CCS","OCCS",
            "O=C1N(C)C(=O)NC1=O","CC(=O)CC(C)=O","O=C1CCC(=O)N1","NCC(=O)O",
            "O=Cc1ccc(N)cc1","O=Cc1ccc(O)cc1","c1ccccc1C(=O)Cl","O=C1OC(=O)c2ccccc21",
            "O=C1NC(=O)CC1","O=C1NCC(=O)N1","NC(=O)N","O=C=NCC","CC(=O)ON=C",
        ]
        solv_pool = [
            {"dielectric":80.1,"polarity":1.0,"hbond_donor":1.0,"hbond_acceptor":1.0,"pi_star":1.09},
            {"dielectric":24.5,"polarity":0.65,"hbond_donor":0.86,"hbond_acceptor":0.75,"pi_star":0.54},
            {"dielectric":32.7,"polarity":0.76,"hbond_donor":0.98,"hbond_acceptor":0.66,"pi_star":0.60},
            {"dielectric":37.5,"polarity":0.46,"hbond_donor":0.19,"hbond_acceptor":0.31,"pi_star":0.75},
            {"dielectric":46.7,"polarity":0.44,"hbond_donor":0.0,"hbond_acceptor":0.76,"pi_star":1.0},
            {"dielectric":20.7,"polarity":0.36,"hbond_donor":0.08,"hbond_acceptor":0.43,"pi_star":0.71},
            {"dielectric":4.8,"polarity":0.26,"hbond_donor":0.44,"hbond_acceptor":0.0,"pi_star":0.58},
            {"dielectric":2.4,"polarity":0.1,"hbond_donor":0.0,"hbond_acceptor":0.11,"pi_star":0.54},
            {"dielectric":1.9,"polarity":0.01,"hbond_donor":0.0,"hbond_acceptor":0.0,"pi_star":0.0},
        ]
        X, y = [], []
        for _ in range(n):
            m = Chem.MolFromSmiles(random.choice(smi_pool))
            if m is None: continue
            sp = random.choice(solv_pool)
            pH = float(np.random.uniform(0,14)); T = float(np.random.uniform(273.15,353.15))
            X.append(self._vec(m, sp, pH, T))
            res = self.thermo.evaluate(m, sp, pH, T)
            y.append(max(r["probability"] for r in res) if res else 0.05)
        return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

    def _load_or_train(self):
        if os.path.exists(self.model_path):
            try:
                with open(self.model_path,"rb") as f: b = pickle.load(f)
                if b.get("N_FEATURES", 20) != self.N_FEATURES:
                    print(f"⚠️ N_FEATURES mismatch → retraining")
                else:
                    self.model = b["model"]; self.scaler = b.get("scaler"); self.fitted = True
                    print(f"✅ ReactionFeasibilityML loaded"); return
            except Exception as e:
                print(f"⚠️ Load failed ({e}) — retraining.")
        print("🧪 Training ReactionFeasibilityML")
        X, y = self._synthetic(3000)
        if len(X) < 100:
            print("⚠️ Too few samples — ML disabled."); return
        from sklearn.ensemble import RandomForestRegressor
        from sklearn.preprocessing import StandardScaler
        self.scaler = StandardScaler().fit(X)
        self.model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
        self.model.fit(self.scaler.transform(X), y)
        self.fitted = True
        with open(self.model_path,"wb") as f:
            pickle.dump({"model": self.model, "scaler": self.scaler, "N_FEATURES": self.N_FEATURES}, f)
        print(f"✅ ReactionFeasibilityML trained + saved")

    def predict(self, mol, solvent_props, pH, T, engine=None):
        if engine is None:
            engine = ReactionEngine()

        solvent_smiles = solvent_props.get("smiles")
        if not solvent_smiles:
            return 0.05, None

        # Compute the heuristic score (via ML or thermo fallback)
        if not self.fitted or self.model is None:
            res = self.thermo.evaluate(mol, solvent_props, pH, T)
            matching = [r for r in res if engine.matches(r["key"], mol, solvent_smiles)]
            if not matching:
                return 0.05, None
            best = max(matching, key=lambda r: r["probability"])
            return float(best["probability"]), best["key"]

        v = self._vec(mol, solvent_props, pH, T).reshape(1, -1)
        if self.scaler is not None:
            v = self.scaler.transform(v)
        score = float(np.clip(self.model.predict(v)[0], 0.0, 1.0))

        res = self.thermo.evaluate(mol, solvent_props, pH, T)
        matching = [r for r in res if engine.matches(r["key"], mol, solvent_smiles)]
        if not matching:
            return score, None
        best_key = max(matching, key=lambda r: r["probability"])["key"]
        return score, best_key


class ReactionLearner:
    """Adaptive wrapper — learns from user feedback, retrains every 20 samples."""
    MODEL_VERSION = "2.0"
    RETRAIN_THRESHOLD = 20
    MIN_REAL_SAMPLES = 10

    def __init__(self, model_path="reaction_feasibility.pkl", engine=None, chromo=None):
        self.model_path = model_path
        self.engine = engine or ReactionEngine()
        self.chromo = chromo or ChromophoreDetector()
        self.base_ml = ReactionFeasibilityML(model_path=model_path)
        self._product_lambda_cache = {}
        self.new_count = 0
        self.real_X, self.real_y, self.real_keys = [], [], []
        self._load_history()

    def _load_history(self):
        if not os.path.exists(self.model_path): return
        try:
            with open(self.model_path, "rb") as f: b = pickle.load(f)
            if b.get("version") == self.MODEL_VERSION:
                self.real_X = [np.array(v, dtype=np.float32) for v in b.get("real_X", [])]
                self.real_y = list(b.get("real_y", []))
                self.real_keys = list(b.get("real_keys", []))
                print(f"✅ ReactionLearner history restored: {len(self.real_X)} real samples")
        except Exception:
            pass

    def _get_product_lambda(self, mol, pH):
        smi = Chem.MolToSmiles(mol)
        key = (smi, round(pH, 1))
        if key in self._product_lambda_cache:
            return self._product_lambda_cache[key]
        try:
            peaks, _ = self.chromo.detect(mol, "product", pH)
            lam = peaks[0][0] if peaks else None
        except Exception:
            lam = None
        self._product_lambda_cache[key] = lam
        return lam

    def infer_from_feedback(self, mol, solvent_props, pH, T, actual_lambda, predicted_lambda):
        delta = abs(actual_lambda - predicted_lambda)
        if delta < 15.0:
            return 0.05, None, 1.0 - delta / 15.0
        solvent_smiles = solvent_props.get("smiles")
        if not solvent_smiles:
            return 0.5, None, 0.3
        products = self.engine.apply(mol, solvent_smiles)
        if not products:
            return min(1.0, delta / 100.0), None, 0.4
        best_key, best_match = None, 1e9
        for prod, _, key in products:
            prod_lam = self._get_product_lambda(prod, pH)
            if prod_lam is None: continue
            diff = abs(prod_lam - actual_lambda)
            if diff < best_match:
                best_match = diff; best_key = key
        if best_key is None or best_match > 50.0:
            return min(0.9, delta / 100.0), None, 0.3
        confidence = max(0.3, 1.0 - best_match / 50.0)
        return min(0.95, 0.5 + 0.5 * confidence), best_key, confidence

    def record_feedback(self, mol, solvent_props, pH, T,
                        actual_lambda=None, predicted_lambda=None,
                        confirmed_reaction_key=None, confirmed=False):

        key = None
        prob = 0.0
        conf = 0.0

        if confirmed and confirmed_reaction_key:
            prob = 1.0
            key = confirmed_reaction_key
            conf = 1.0
        elif Config.REACTION_INFER_LABELS_FROM_RESIDUALS and \
             actual_lambda is not None and predicted_lambda is not None:
            prob, key, conf = self.infer_from_feedback(
                mol, solvent_props, pH, T, actual_lambda, predicted_lambda
            )
        else:
            print("🔬 ReactionLearner: skipping (no explicit reaction label)")
            return

        if conf < 0.5:
            print(f"   🔬 ReactionLearner: low confidence ({conf:.2f}), skipping")
            return

        vec = self.base_ml._vec(mol, solvent_props, pH, T)
        self.real_X.append(vec)
        self.real_y.append(prob)
        self.real_keys.append(key)
        self.new_count += 1
        print(f"   🔬 ReactionLearner recorded: prob={prob:.2f} key={key} "
              f"({len(self.real_X)} total, {self.new_count}/{self.RETRAIN_THRESHOLD})")
        if self.new_count >= self.RETRAIN_THRESHOLD:
            self.retrain()
            self.new_count = 0

    def retrain(self):
        if len(self.real_X) < self.MIN_REAL_SAMPLES:
            print(f"⚠️ Need ≥{self.MIN_REAL_SAMPLES} samples, have {len(self.real_X)}")
            return
        print(f"🧪 ReactionLearner retraining on {len(self.real_X)} real + 3000 synthetic...")
        X_syn, y_syn = self.base_ml._synthetic(n=3000)
        X_real = np.vstack(self.real_X)
        y_real = np.array(self.real_y, dtype=np.float32)
        X_all = np.vstack([X_syn] + [X_real] * 3)
        y_all = np.concatenate([y_syn] + [y_real] * 3)
        from sklearn.ensemble import RandomForestRegressor
        from sklearn.preprocessing import StandardScaler
        self.base_ml.scaler = StandardScaler().fit(X_all)
        self.base_ml.model = RandomForestRegressor(n_estimators=100, max_depth=12,
                                                   random_state=42, n_jobs=-1)
        self.base_ml.model.fit(self.base_ml.scaler.transform(X_all), y_all)
        self.base_ml.fitted = True
        self.save()
        print(f"   ✅ ReactionLearner retrained and saved ({len(self.real_X)} real ×3 weight)")

    def predict(self, mol, solvent_props, pH, T, engine=None):
        if engine is None:
            engine = self.engine
        return self.base_ml.predict(mol, solvent_props, pH, T, engine=engine)

    def save(self):
        with open(self.model_path, "wb") as f:
            pickle.dump({
                "version": self.MODEL_VERSION,
                "model": self.base_ml.model,
                "scaler": self.base_ml.scaler,
                "N_FEATURES": self.base_ml.N_FEATURES,
                "real_X": [v.tolist() for v in self.real_X],
                "real_y": list(self.real_y),
                "real_keys": list(self.real_keys),
            }, f)
        print(f"✅ ReactionLearner saved ")

print("✅ ReactionEngine + Thermodynamics + ML + Adaptive Learner loaded.")

✅ ReactionEngine + Thermodynamics + ML + Adaptive Learner loaded.


In [ ]:
# @title 5.) Core Classes for UV‑Vis Prediction

# ---------- SpectraVisualizer ----------
class SpectraVisualizer:
    def __init__(self, wavelengths: np.ndarray, backend: PlotBackend = Config.PLOT_BACKEND):
        self.wavelengths = wavelengths
        self.backend = backend
        self._plt = None
        self._sns = None
        self._go = None
        self._plotly_available = False
        self._initialize_backend()

    def _initialize_backend(self) -> None:
        if self.backend == PlotBackend.PLOTLY:
            try:
                import plotly.graph_objects as go
                self._go = go
                self._plotly_available = True
            except ImportError:
                self.backend = PlotBackend.MATPLOTLIB
        if self.backend == PlotBackend.MATPLOTLIB:
            import matplotlib.pyplot as plt
            import seaborn as sns
            self._plt = plt
            self._sns = sns
            plt.rcParams.update({
                "font.size": 11,
                "axes.labelsize": 12,
                "axes.titlesize": 13,
                "xtick.labelsize": 10,
                "ytick.labelsize": 10,
                "legend.fontsize": 10,
                "figure.dpi": Config.MATPLOTLIB_DPI,
                "savefig.dpi": Config.MATPLOTLIB_DPI,
                "savefig.bbox": "tight",
                "mathtext.default": "regular",
            })
            sns.set_style("ticks")
            sns.set_context("paper")

    @staticmethod
    def _extract_top_peaks(y: np.ndarray, n_peaks: int = 3, height_ratio: float = 0.01) -> List[Tuple[float, float]]:
        max_y = np.max(y)
        if max_y <= 0:
            return []
        height_thresh = max_y * height_ratio
        peaks, props = find_peaks(y, height=height_thresh, distance=max(1, int(20 / Config.WL_STEP)))
        if len(peaks) == 0:
            return []
        heights = props["peak_heights"]
        sorted_idx = np.argsort(heights)[::-1][:n_peaks]
        return [(peaks[i], heights[i]) for i in sorted_idx]

    def plot_spectra(self, spectra: np.ndarray, labels: List[str],
                     mixture: Optional[np.ndarray] = None,
                     title: str = "UV-Vis Spectra",
                     xlabel: str = "Wavelength (nm)",
                     ylabel: str = "Absorbance (a.u.)",
                     save_path: Optional[str] = None,
                     show_peaks: bool = Config.SHOW_PEAK_ANNOTATIONS) -> None:
        if len(spectra) == 0:
            return
        if self.backend == PlotBackend.PLOTLY:
            self._plot_spectra_plotly(spectra, labels, mixture, title, xlabel, ylabel, show_peaks)
        else:
            self._plot_spectra_matplotlib(spectra, labels, mixture, title, xlabel, ylabel, save_path, show_peaks)

    def _plot_spectra_plotly(self, spectra, labels, mixture, title, xlabel, ylabel, show_peaks):
        fig = self._go.Figure()
        for idx, (spec, name) in enumerate(zip(spectra, labels)):
            fig.add_trace(self._go.Scatter(
                x=self.wavelengths, y=spec, mode="lines", name=name,
                line=dict(width=2),
                hovertemplate=f"<b>{name}</b><br>λ: %{{x:.0f}} nm<br>A: %{{y:.4f}}<extra></extra>",
            ))
        if mixture is not None:
            fig.add_trace(self._go.Scatter(
                x=self.wavelengths, y=mixture, mode="lines", name="Mixture",
                line=dict(width=3.5, dash="dash", color="black"),
                hovertemplate="<b>Mixture</b><br>λ: %{x:.0f} nm<br>A: %{y:.4f}<extra></extra>",
            ))
            if show_peaks:
                top = self._extract_top_peaks(mixture)
                for wl_idx, height in top:
                    wl = self.wavelengths[wl_idx]
                    fig.add_annotation(
                        x=wl, y=height, text=f"λ = {wl:.0f} nm",
                        showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1,
                        arrowcolor="darkred", font=dict(size=10, color="darkred"), yshift=10,
                    )
        fig.update_layout(
            title=dict(text=title, font=dict(size=16)),
            xaxis=dict(title=xlabel, range=[Config.WL_MIN, Config.WL_MAX], zeroline=False),
            yaxis=dict(title=ylabel, zeroline=False),
            template=Config.PLOTLY_TEMPLATE,
            hovermode="x unified",
            legend=dict(orientation="v", x=1.02, y=1, xanchor="left"),
            margin=dict(l=60, r=120, t=80, b=60),
        )
        fig.show()

    def _plot_spectra_matplotlib(self, spectra, labels, mixture, title, xlabel, ylabel, save_path, show_peaks):
        fig, ax = self._plt.subplots(figsize=Config.MATPLOTLIB_FIGSIZE)
        n = len(spectra)
        cmap = self._sns.color_palette("viridis", n) if n > 10 else self._sns.color_palette("tab10", n)
        for i, (spec, name) in enumerate(zip(spectra, labels)):
            ax.plot(self.wavelengths, spec, color=cmap[i], linewidth=1.8, label=name, alpha=0.85)
        if mixture is not None:
            ax.plot(self.wavelengths, mixture, color="black", linestyle="--", linewidth=2.8, label="Mixture", zorder=10)
            if show_peaks:
                top = self._extract_top_peaks(mixture)
                for wl_idx, height in top:
                    wl = self.wavelengths[wl_idx]
                    ax.annotate(f"{wl:.0f} nm", xy=(wl, height),
                                xytext=(wl, height + 0.02 * np.max(mixture)),
                                ha="center", fontsize=9, color="darkred",
                                arrowprops=dict(arrowstyle="->", color="darkred", lw=0.8))
        ax.set_xlabel(xlabel, fontweight="bold")
        ax.set_ylabel(ylabel, fontweight="bold")
        ax.set_title(title, fontweight="bold")
        ax.set_xlim(Config.WL_MIN, Config.WL_MAX)
        ax.grid(True, linestyle=":", alpha=0.6)
        ax.legend(loc="upper right", frameon=True, fancybox=True, shadow=True)
        self._sns.despine(ax=ax, trim=True)
        if save_path:
            self._plt.savefig(save_path, dpi=Config.MATPLOTLIB_DPI)
        self._plt.show()

    def plot_learning_curve(self, error_history: List[float], save_path: Optional[str] = None) -> None:
        if not error_history:
            return
        if self.backend == PlotBackend.PLOTLY:
            fig = self._go.Figure()
            fig.add_trace(self._go.Scatter(
                x=list(range(len(error_history))), y=error_history,
                mode="lines+markers", marker=dict(size=6, color="royalblue"),
                line=dict(width=2, color="royalblue"),
                hovertemplate="Feedback #%{x}<br>Error: %{y:.2f} nm<extra></extra>",
            ))
            if len(error_history) > 5:
                window = min(5, len(error_history) // 2)
                moving_avg = np.convolve(error_history, np.ones(window)/window, mode='valid')
                fig.add_trace(self._go.Scatter(
                    x=list(range(window-1, len(error_history))), y=moving_avg,
                    mode="lines", name="Moving Avg",
                    line=dict(width=2, color="darkred", dash="dot"),
                ))
            fig.update_layout(
                title="Learning Progress: Prediction Error vs. Feedback",
                xaxis_title="Feedback Iteration", yaxis_title="Absolute Error (nm)",
                template=Config.PLOTLY_TEMPLATE, hovermode="x",
            )
            fig.show()
        else:
            fig, ax = self._plt.subplots(figsize=(10, 4))
            ax.plot(error_history, "o-", color="royalblue", markersize=6, linewidth=1.8, label="Error")
            if len(error_history) > 5:
                window = min(5, len(error_history) // 2)
                moving_avg = np.convolve(error_history, np.ones(window)/window, mode='valid')
                ax.plot(range(window-1, len(error_history)), moving_avg, "--", color="darkred", linewidth=2, label="Moving Avg")
            ax.set_xlabel("Feedback #", fontweight="bold")
            ax.set_ylabel("Error (nm)", fontweight="bold")
            ax.set_title("Learning Progress", fontweight="bold")
            ax.grid(True, linestyle=":", alpha=0.6)
            ax.legend()
            self._sns.despine(ax=ax, trim=True)
            if save_path:
                self._plt.savefig(save_path, dpi=Config.MATPLOTLIB_DPI)
            self._plt.show()

    def plot_individual_spectrum(self, spectrum: np.ndarray, name: str,
                                 confidence: float = 0.5, save_path: Optional[str] = None) -> None:
        if self.backend == PlotBackend.PLOTLY:
            fig = self._go.Figure()
            fig.add_trace(self._go.Scatter(
                x=self.wavelengths, y=spectrum, mode="lines", name=name,
                line=dict(width=2.5, color="#2E86C1"),
                hovertemplate="λ: %{x:.0f} nm<br>A: %{y:.4f}<extra></extra>",
            ))
            peak_info = self._extract_top_peaks(spectrum, n_peaks=1)
            if peak_info:
                wl_idx, height = peak_info[0]
                wl = self.wavelengths[wl_idx]
                fig.add_annotation(x=wl, y=height, text=f"λ_max = {wl:.0f} nm",
                                   showarrow=True, arrowhead=2)
            fig.update_layout(
                title=f"{name} (Uncertainty: {1-confidence:.2f})",
                xaxis_title="Wavelength (nm)", yaxis_title="Absorbance",
                template=Config.PLOTLY_TEMPLATE, hovermode="x",
            )
            fig.show()
        else:
            fig, ax = self._plt.subplots(figsize=(8, 3.5))
            ax.plot(self.wavelengths, spectrum, color="#2E86C1", linewidth=2)
            peak_info = self._extract_top_peaks(spectrum, n_peaks=1)
            if peak_info:
                wl_idx, height = peak_info[0]
                wl = self.wavelengths[wl_idx]
                ax.annotate(f"λ_max = {wl:.0f} nm", xy=(wl, height), xytext=(wl, height * 0.85),
                            ha="center", fontsize=10, color="darkred",
                            arrowprops=dict(arrowstyle="->", color="darkred"))
            ax.set_title(f"{name} (Uncertainty: {1-confidence:.2f})", fontweight="bold")
            ax.set_xlabel("Wavelength (nm)", fontweight="bold")
            ax.set_ylabel("Absorbance", fontweight="bold")
            ax.grid(True, linestyle=":", alpha=0.6)
            self._sns.despine(ax=ax, trim=True)
            if save_path:
                self._plt.savefig(save_path, dpi=Config.MATPLOTLIB_DPI)
            self._plt.show()


# ---------- FeatureExtractor ----------
class FeatureExtractor:
    def __init__(self):
        self.dc_featurizer = RDKitDescriptors()
        self.target_dim = Config.PADDED_FEAT_DIM

    def extract_original(self, mol, pH):
        try:
            feat = []
            feat.append(Descriptors.MolWt(mol))
            feat.append(Descriptors.MolLogP(mol))
            feat.append(Descriptors.NumHDonors(mol))
            feat.append(Descriptors.NumHAcceptors(mol))
            feat.append(Descriptors.TPSA(mol))
            feat.append(Descriptors.NumRotatableBonds(mol))
            feat.append(mol.GetNumAtoms())
            feat.append(mol.GetNumHeavyAtoms())
            feat.append(rdMolDescriptors.CalcNumAromaticRings(mol))
            feat.append(rdMolDescriptors.CalcNumAliphaticRings(mol))
            feat.append(rdMolDescriptors.CalcNumHeterocycles(mol))
            n_double = sum(1 for b in mol.GetBonds() if b.GetBondTypeAsDouble() == 2.0)
            n_triple = sum(1 for b in mol.GetBonds() if b.GetBondTypeAsDouble() == 3.0)
            n_aromatic = sum(1 for b in mol.GetBonds() if b.GetIsAromatic())
            feat.extend([n_double, n_triple, n_aromatic])
            mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=256)
            fp = mfpgen.GetFingerprint(mol)
            arr = np.zeros(256)
            DataStructs.ConvertToNumpyArray(fp, arr)
            feat.extend(arr)
            feat.append(pH)
            arr_feat = np.array(feat, dtype=np.float32)
            if np.any(~np.isfinite(arr_feat)):
                print("⚠️ NaN/Inf detected in original features. Returning zeros.")
                return np.zeros(Config.ORIG_FEAT_DIM, dtype=np.float32)
            return arr_feat
        except Exception as e:
            print(f"⚠️ extract_original failed: {e}")
            return np.zeros(Config.ORIG_FEAT_DIM, dtype=np.float32)

    def extract_deepchem(self, mol):
        try:
            smiles = Chem.MolToSmiles(mol)
            desc = self.dc_featurizer.featurize([smiles])[0]
            if desc is None:
                return np.zeros(Config.DEEPCHEM_FEAT_DIM, dtype=np.float32)
            desc = np.asarray(desc, dtype=np.float32).reshape(-1)
            desc = np.nan_to_num(desc, nan=0.0, posinf=0.0, neginf=0.0)
            return desc
        except Exception as e:
            print(f"⚠️ extract_deepchem failed: {e}")
            return np.zeros(Config.DEEPCHEM_FEAT_DIM, dtype=np.float32)

    def extract(self, mol, pH):
        orig = self.extract_original(mol, pH)
        if len(orig) != Config.ORIG_FEAT_DIM:
            if len(orig) < Config.ORIG_FEAT_DIM:
                orig = np.pad(orig, (0, Config.ORIG_FEAT_DIM - len(orig)))
            else:
                orig = orig[:Config.ORIG_FEAT_DIM]
        dc_vec = self.extract_deepchem(mol)
        if len(dc_vec) != Config.DEEPCHEM_FEAT_DIM:
            if len(dc_vec) < Config.DEEPCHEM_FEAT_DIM:
                dc_vec = np.pad(dc_vec, (0, Config.DEEPCHEM_FEAT_DIM - len(dc_vec)))
            else:
                dc_vec = dc_vec[:Config.DEEPCHEM_FEAT_DIM]
        combined = np.concatenate([orig, dc_vec])
        if len(combined) < self.target_dim:
            combined = np.pad(combined, (0, self.target_dim - len(combined)))
        else:
            combined = combined[:self.target_dim]
        # --- FINAL SAFETY NET ---
        combined = np.asarray(combined, dtype=np.float32)
        combined = np.nan_to_num(combined, nan=0.0, posinf=0.0, neginf=0.0)
        return combined


# ---------- UVVisNN ----------
class UVVisNN(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)])
            prev = h
        self.backbone = nn.Sequential(*layers)
        self.lambda_head = nn.Linear(prev, 1)

    def forward(self, x):
        features = self.backbone(x)
        lambda_pred = self.lambda_head(features).squeeze(-1)
        return lambda_pred


# ---------- ConjugationAnalyzer ----------
class ConjugationAnalyzer:
    _SMARTS_DICT: Optional[Dict[str, Tuple[str, int]]] = None
    _SMARTS_PKL_FILENAME = 'chromophore_smarts.pkl'

    @classmethod
    def _get_smarts_dict(cls, save_dir: str = '/content/drive/MyDrive/UVVis_Learner/') -> Dict[str, Tuple[str, int]]:
        if cls._SMARTS_DICT is not None:
            return cls._SMARTS_DICT
        pkl_path = os.path.join(save_dir, cls._SMARTS_PKL_FILENAME)
        if os.path.exists(pkl_path):
            try:
                with open(pkl_path, 'rb') as f:
                    cls._SMARTS_DICT = pickle.load(f)
                print(f"✅ Loaded SMARTS dict from {pkl_path} ({len(cls._SMARTS_DICT)} entries)")
                return cls._SMARTS_DICT
            except Exception as e:
                print(f"⚠️ Failed to load SMARTS dict: {e}. Rebuilding...")
        cls._SMARTS_DICT = {
            "benzene": ("c1ccccc1", 6),
            "pyridine": ("c1ccncc1", 6),
            "aniline": ("Nc1ccccc1", 7),
            "nitrobenzene": ("[N+](=O)[O-]c1ccccc1", 8),
        }
        try:
            os.makedirs(save_dir, exist_ok=True)
            with open(pkl_path, 'wb') as f:
                pickle.dump(cls._SMARTS_DICT, f)
            print(f"✅ Saved SMARTS dict to {pkl_path} ({len(cls._SMARTS_DICT)} entries)")
        except Exception:
            pass
        return cls._SMARTS_DICT

    @classmethod
    def get_isolated_chromophores(cls, mol: Chem.Mol) -> List[Chem.Mol]:
        bonds_to_break = []
        for bond in mol.GetBonds():
            if bond.IsInRing():
                continue
            if bond.GetBondType() == Chem.rdchem.BondType.SINGLE:
                a1 = bond.GetBeginAtom()
                a2 = bond.GetEndAtom()
                if a1.GetHybridization() == Chem.rdchem.HybridizationType.SP3 or \
                   a2.GetHybridization() == Chem.rdchem.HybridizationType.SP3:
                    bonds_to_break.append(bond.GetIdx())
        if not bonds_to_break:
            return [mol]
        shattered = Chem.FragmentOnBonds(mol, bonds_to_break, addDummies=False)
        fragments = Chem.GetMolFrags(shattered, asMols=True)
        return [frag for frag in fragments if frag.GetNumHeavyAtoms() > 3]

    @classmethod
    def longest_conjugated_path(cls, mol: Chem.Mol) -> int:
        if hasattr(Chem.rdmolops, 'GetConjugatedAtoms'):
            try:
                fragments = cls.get_isolated_chromophores(mol)
                longest = 0
                for frag in fragments:
                    atoms = set(Chem.rdmolops.GetConjugatedAtoms(frag))
                    longest = max(longest, len(atoms))
                if longest > 0:
                    return longest
            except Exception:
                pass
        fragments = cls.get_isolated_chromophores(mol)
        longest = 0
        for frag in fragments:
            atoms = cls._detect_conjugated_atoms_manual(frag)
            longest = max(longest, len(atoms))
        if longest == 0:
            smarts_dict = cls._get_smarts_dict()
            best = 0
            for name, (smarts, length) in smarts_dict.items():
                patt = Chem.MolFromSmarts(smarts)
                if patt is not None and mol.HasSubstructMatch(patt):
                    if length > best:
                        best = length
            return best
        return longest

    @classmethod
    def _detect_conjugated_atoms_manual(cls, mol: Chem.Mol) -> Set[int]:
        if mol.GetNumAtoms() == 0:
            return set()
        conj_adj = defaultdict(set)
        for bond in mol.GetBonds():
            u, v = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            if bond.GetIsAromatic() or bond.GetBondType() in (Chem.rdchem.BondType.DOUBLE, Chem.rdchem.BondType.TRIPLE):
                conj_adj[u].add(v)
                conj_adj[v].add(u)
            elif bond.GetBondType() == Chem.rdchem.BondType.SINGLE:
                a1, a2 = bond.GetBeginAtom(), bond.GetEndAtom()
                is_pi1 = a1.GetHybridization() in (Chem.rdchem.HybridizationType.SP2, Chem.rdchem.HybridizationType.SP)
                is_pi2 = a2.GetHybridization() in (Chem.rdchem.HybridizationType.SP2, Chem.rdchem.HybridizationType.SP)
                has_lp1 = a1.GetAtomicNum() in (7, 8, 16)
                has_lp2 = a2.GetAtomicNum() in (7, 8, 16)
                if (is_pi1 and has_lp2) or (is_pi2 and has_lp1):
                    conj_adj[u].add(v)
                    conj_adj[v].add(u)
        seeds = set()
        for atom in mol.GetAtoms():
            idx = atom.GetIdx()
            if atom.GetIsAromatic() or any(b.GetBondType() in (Chem.rdchem.BondType.DOUBLE, Chem.rdchem.BondType.TRIPLE)
                                           for b in atom.GetBonds()):
                seeds.add(idx)
        if not seeds:
            return set()
        visited = set()
        for seed in seeds:
            if seed in visited:
                continue
            stack = [seed]
            while stack:
                u = stack.pop()
                if u in visited:
                    continue
                visited.add(u)
                for v in conj_adj.get(u, []):
                    if v not in visited:
                        stack.append(v)
        return visited

    @classmethod
    def effective_conjugation_length(cls, mol: Chem.Mol) -> int:
        return cls.longest_conjugated_path(mol)

    @classmethod
    def get_conjugation_report(cls, mol: Chem.Mol) -> Dict:
        fragments = cls.get_isolated_chromophores(mol)
        report = {'num_fragments': len(fragments), 'fragment_sizes': [],
                  'longest': 0, 'total_conjugated_atoms': 0}
        for frag in fragments:
            atoms = cls._detect_conjugated_atoms_manual(frag) if not hasattr(Chem.rdmolops, 'GetConjugatedAtoms') else set(Chem.rdmolops.GetConjugatedAtoms(frag))
            size = len(atoms)
            report['fragment_sizes'].append(size)
            report['total_conjugated_atoms'] += size
            if size > report['longest']:
                report['longest'] = size
        if report['num_fragments'] > 0:
            report['average_fragment_size'] = report['total_conjugated_atoms'] / report['num_fragments']
        return report

    @classmethod
    def _detect_intramolecular_hbond_pseudoring(cls, mol: Chem.Mol) -> bool:
        patterns = [
            "[O,N]~C=C~C~[O,N]",
            "[OH]C=C-C=[O,N]",
            "[NH]C=C-C=[O,N]",
            "O=C-C=C-[OH]",
            "O=C-C=C-[NH]",
            "[OH]C=C-C(=O)-[OH]",
        ]
        for smarts in patterns:
            patt = Chem.MolFromSmarts(smarts)
            if patt and mol.HasSubstructMatch(patt):
                return True
        return False

    @classmethod
    def longest_conjugated_path_with_hbond(cls, mol: Chem.Mol) -> int:
        base_len = cls.longest_conjugated_path(mol)
        if cls._detect_intramolecular_hbond_pseudoring(mol):
            base_len += 6
        return base_len


# ---------- ChromophoreFamilyClassifier ----------
class ChromophoreFamilyClassifier:
    DEFAULT_PROTOTYPES = {
        "cyanine": {"mean": [20.0, 0.15, 0.25, 2.5, 1.0, 0.75, 0.85, 3.0, 2.0, 4.0], "std": [5.0, 0.10, 0.10, 1.0, 0.5, 0.15, 0.10, 1.0, 0.8, 1.5]},
        "coumarin": {"mean": [12.0, 0.25, 0.30, 1.0, 1.5, 0.30, 0.25, 2.5, 1.0, 2.0], "std": [3.0, 0.08, 0.08, 0.5, 0.5, 0.10, 0.10, 0.8, 0.5, 0.8]},
        "porphyrin": {"mean": [18.0, 0.90, 0.20, 4.0, 0.0, 0.0, 0.0, 4.0, 4.0, 0.0], "std": [2.0, 0.05, 0.05, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 0.5]},
        "azo": {"mean": [14.0, 0.30, 0.30, 1.0, 1.5, 0.60, 0.50, 2.5, 1.5, 3.0], "std": [4.0, 0.10, 0.10, 0.5, 0.5, 0.15, 0.15, 0.8, 0.8, 1.0]},
        "flavonoid": {"mean": [16.0, 0.40, 0.35, 1.0, 1.5, 0.40, 0.40, 3.0, 1.5, 2.5], "std": [4.0, 0.08, 0.08, 0.5, 0.5, 0.10, 0.10, 0.8, 0.5, 0.8]},
        "donor_acceptor_dye": {"mean": [18.0, 0.20, 0.20, 2.0, 2.0, 0.80, 0.90, 2.0, 1.0, 3.0], "std": [5.0, 0.10, 0.10, 1.0, 1.0, 0.15, 0.10, 1.0, 0.5, 1.0]},
        "polycyclic_aromatic": {"mean": [25.0, 0.60, 0.10, 0.0, 0.0, 0.95, 0.10, 4.0, 0.0, 1.0], "std": [6.0, 0.15, 0.05, 0.5, 0.5, 0.05, 0.05, 1.5, 0.5, 0.5]},
        "simple_aromatic": {"mean": [6.0, 0.50, 0.15, 0.0, 0.0, 0.20, 0.10, 1.0, 0.0, 1.0], "std": [2.0, 0.15, 0.05, 0.5, 0.5, 0.10, 0.05, 0.5, 0.5, 0.5]},
        "aliphatic": {"mean": [0.0, 0.0, 0.50, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0], "std": [1.0, 0.02, 0.10, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 2.0]},
        "chlorin": {"mean": [18.0, 0.75, 0.25, 4.0, 0.0, 0.0, 0.0, 5.0, 5.0, 0.0], "std": [2.0, 0.10, 0.05, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 0.5]},
        "bacteriochlorin": {"mean": [20.0, 0.70, 0.25, 4.0, 0.0, 0.0, 0.0, 5.0, 5.0, 0.0], "std": [2.0, 0.10, 0.05, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 0.5]},
        "phthalocyanine": {"mean": [24.0, 0.85, 0.15, 4.0, 0.0, 0.0, 0.0, 6.0, 6.0, 0.0], "std": [3.0, 0.05, 0.05, 0.5, 0.5, 0.05, 0.05, 1.0, 1.0, 0.5]},
        "xanthene": {"mean": [14.0, 0.60, 0.30, 2.0, 1.0, 0.20, 0.10, 4.0, 1.0, 2.0], "std": [3.0, 0.10, 0.10, 1.0, 0.5, 0.10, 0.10, 0.5, 0.5, 1.0]},
        "acridine": {"mean": [14.0, 0.70, 0.15, 1.0, 0.0, 0.0, 0.0, 4.0, 2.0, 1.0], "std": [2.0, 0.10, 0.05, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 0.5]},
        "phenothiazine": {"mean": [12.0, 0.60, 0.25, 1.0, 0.0, 0.0, 0.0, 3.0, 2.0, 1.0], "std": [2.0, 0.10, 0.10, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 0.5]},
        "carbazole": {"mean": [12.0, 0.70, 0.10, 1.0, 0.0, 0.0, 0.0, 3.0, 1.0, 1.0], "std": [2.0, 0.10, 0.05, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 0.5]},
        "bodipy": {"mean": [10.0, 0.40, 0.30, 2.0, 0.0, 0.50, 0.20, 3.0, 2.0, 2.0], "std": [2.0, 0.10, 0.10, 0.5, 0.5, 0.10, 0.10, 0.5, 0.5, 1.0]},
        "squaraine": {"mean": [16.0, 0.50, 0.20, 2.0, 2.0, 0.70, 0.80, 4.0, 1.0, 3.0], "std": [3.0, 0.10, 0.10, 0.5, 0.5, 0.10, 0.10, 0.5, 0.5, 1.0]},
        "stilbene": {"mean": [8.0, 0.60, 0.00, 0.0, 0.0, 0.30, 0.20, 2.0, 0.0, 2.0], "std": [2.0, 0.10, 0.05, 0.5, 0.5, 0.10, 0.10, 0.5, 0.5, 1.0]},
        "chalcone": {"mean": [10.0, 0.50, 0.10, 0.0, 1.0, 0.40, 0.30, 3.0, 0.0, 3.0], "std": [2.0, 0.10, 0.05, 0.5, 0.5, 0.10, 0.10, 0.5, 0.5, 1.0]},
        "curcuminoid": {"mean": [12.0, 0.40, 0.20, 0.0, 1.0, 0.30, 0.20, 2.0, 0.0, 5.0], "std": [2.0, 0.10, 0.10, 0.5, 0.5, 0.10, 0.10, 0.5, 0.5, 1.0]},
        "steroid": {"mean": [6.0, 0.00, 0.00, 0.0, 1.0, 0.0, 0.0, 4.0, 0.0, 3.0], "std": [2.0, 0.05, 0.05, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 1.0]},
        "beta_lactam": {"mean": [4.0, 0.00, 0.30, 1.0, 1.0, 0.0, 0.0, 2.0, 1.0, 2.0], "std": [1.0, 0.05, 0.10, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 1.0]},
        "tetracycline": {"mean": [14.0, 0.60, 0.30, 2.0, 2.0, 0.30, 0.20, 4.0, 1.0, 3.0], "std": [2.0, 0.10, 0.10, 0.5, 0.5, 0.10, 0.10, 0.5, 0.5, 1.0]},
        "quinolone": {"mean": [10.0, 0.50, 0.20, 1.0, 1.0, 0.30, 0.20, 3.0, 1.0, 2.0], "std": [2.0, 0.10, 0.10, 0.5, 0.5, 0.10, 0.10, 0.5, 0.5, 1.0]},
        "benzodiazepine": {"mean": [8.0, 0.50, 0.20, 1.0, 0.0, 0.0, 0.0, 3.0, 1.0, 2.0], "std": [2.0, 0.10, 0.10, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 1.0]},
        "barbiturate": {"mean": [4.0, 0.00, 0.40, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 3.0], "std": [1.0, 0.05, 0.10, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 1.0]},
        "purine": {"mean": [10.0, 0.70, 0.30, 1.0, 0.0, 0.0, 0.0, 2.0, 2.0, 1.0], "std": [2.0, 0.10, 0.10, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 0.5]},
        "pteridine": {"mean": [10.0, 0.60, 0.35, 1.0, 0.0, 0.0, 0.0, 2.0, 2.0, 1.0], "std": [2.0, 0.10, 0.10, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 0.5]},
        "indole": {"mean": [8.0, 0.60, 0.10, 1.0, 0.0, 0.0, 0.0, 2.0, 1.0, 1.0], "std": [2.0, 0.10, 0.05, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 0.5]},
        "carotenoid": {"mean": [24.0, 0.00, 0.00, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.0], "std": [4.0, 0.05, 0.05, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 2.0]},
        "retinoid": {"mean": [16.0, 0.20, 0.00, 0.0, 1.0, 0.20, 0.10, 1.0, 0.0, 4.0], "std": [3.0, 0.10, 0.05, 0.5, 0.5, 0.10, 0.10, 0.5, 0.5, 1.0]},
        "anthraquinone": {"mean": [14.0, 0.70, 0.20, 0.0, 2.0, 0.20, 0.10, 3.0, 0.0, 1.0], "std": [2.0, 0.10, 0.10, 0.5, 0.5, 0.10, 0.10, 0.5, 0.5, 0.5]},
        "triphenylmethane": {"mean": [10.0, 0.50, 0.00, 1.0, 0.0, 0.20, 0.10, 3.0, 0.0, 4.0], "std": [2.0, 0.10, 0.05, 0.5, 0.5, 0.10, 0.10, 0.5, 0.5, 1.0]},
        "thiazole": {"mean": [4.0, 0.30, 0.30, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0], "std": [1.0, 0.10, 0.10, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 0.5]},
        "benzothiazole": {"mean": [8.0, 0.50, 0.20, 1.0, 0.0, 0.0, 0.0, 2.0, 1.0, 1.0], "std": [2.0, 0.10, 0.10, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 0.5]},
        "hydrazone": {"mean": [6.0, 0.30, 0.20, 1.0, 1.0, 0.20, 0.10, 1.0, 0.0, 3.0], "std": [2.0, 0.10, 0.10, 0.5, 0.5, 0.10, 0.10, 0.5, 0.5, 1.0]},
        "pyridine": {"mean": [4.0, 0.50, 0.15, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0], "std": [1.0, 0.10, 0.05, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 0.5]},
        "quinoline": {"mean": [8.0, 0.60, 0.10, 1.0, 0.0, 0.0, 0.0, 2.0, 1.0, 1.0], "std": [2.0, 0.10, 0.05, 0.5, 0.5, 0.05, 0.05, 0.5, 0.5, 0.5]},
    }
    DESCRIPTOR_NAMES = ["conjugation_length", "aromatic_fraction", "heteroatom_density",
                        "donor_count", "acceptor_count", "ict_score", "push_pull_score",
                        "ring_count", "heterocycle_count", "rotatable_bonds"]

    def __init__(self, prototype_path=None, threshold=0.70):
        self.threshold = threshold
        if prototype_path and os.path.exists(prototype_path):
            with open(prototype_path, 'r') as f:
                self.prototypes = json.load(f)
        else:
            self.prototypes = self.DEFAULT_PROTOTYPES
        self.family_names = list(self.prototypes.keys())
        self.means = np.array([self.prototypes[f]['mean'] for f in self.family_names])
        self.stds = np.array([self.prototypes[f]['std'] for f in self.family_names])

    def compute_descriptors(self, mol):
        conj_len = ConjugationAnalyzer.longest_conjugated_path(mol)
        heavy_atoms = mol.GetNumHeavyAtoms()
        aromatic_atoms = sum(1 for a in mol.GetAtoms() if a.GetIsAromatic())
        arom_frac = aromatic_atoms / (heavy_atoms + 1e-8)
        hetero_atoms = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() in (7,8,15,16,9,17,35,53))
        hetero_frac = hetero_atoms / (heavy_atoms + 1e-8)
        cd = ChromophoreDetector()
        d_count, a_count, min_dist, ict_score = cd.detect_push_pull_general(mol)
        if ict_score > 0:
            approx_dist = 1.0/ict_score - 1.0
        else:
            approx_dist = 999
        push_pull_score = d_count * a_count / (approx_dist + 1)
        ring_info = mol.GetRingInfo()
        total_rings = len(ring_info.AtomRings())
        hetero_rings = 0
        for ring in ring_info.AtomRings():
            if any(mol.GetAtomWithIdx(a).GetAtomicNum() in (7,8,16) for a in ring):
                hetero_rings += 1
        rot_bonds = Descriptors.NumRotatableBonds(mol)
        return np.array([conj_len, arom_frac, hetero_frac,
                         d_count, a_count, ict_score, push_pull_score,
                         total_rings, hetero_rings, rot_bonds], dtype=float)

    def classify(self, mol):
        v = self.compute_descriptors(mol)
        similarities = []
        for i, name in enumerate(self.family_names):
            diff = (v - self.means[i]) / (self.stds[i] + 1e-8)
            dist = np.sqrt(np.sum(diff**2))
            sim = 1.0 / (1.0 + dist)
            similarities.append((name, sim))
        similarities.sort(key=lambda x: -x[1])
        best_name, best_sim = similarities[0]
        if best_sim >= self.threshold:
            family = best_name
        else:
            family = "novel/unknown"
        novelty = 1.0 - best_sim
        return family, similarities, novelty

    def get_summary(self, mol):
        family, sims, novelty = self.classify(mol)
        top3 = sims[:3]
        return {
            "family": family,
            "novelty": novelty,
            "top_similarities": top3,
            "descriptors": dict(zip(self.DESCRIPTOR_NAMES, self.compute_descriptors(mol)))
        }


# ---------- SimilarityIndex ----------
class SimilarityIndex:
    def __init__(self):
        self.fingerprints = []
        self.smiles_list = []
        self.lambdas = []
        self.names = []
        self.nn = None
        self.is_built = False

    def add_molecule(self, mol, lam=None, name=None):
        fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=256).GetFingerprint(mol)
        arr = np.zeros(256)
        DataStructs.ConvertToNumpyArray(fp, arr)
        self.fingerprints.append(arr)
        self.smiles_list.append(Chem.MolToSmiles(mol))
        self.lambdas.append(lam if lam is not None else np.nan)
        if name is None:
            name = mol.GetProp('_Name') if mol.HasProp('_Name') else 'unknown'
        self.names.append(name)
        self.is_built = False

    def build(self):
        if len(self.fingerprints) == 0:
            return
        X = np.array(self.fingerprints)
        self.nn = NearestNeighbors(n_neighbors=min(5, len(X)), metric='cosine')
        self.nn.fit(X)
        self.is_built = True

    def ensure_built(self):
        if len(self.fingerprints) > 0 and not self.is_built:
            self.build()
        return self.is_built

    def query(self, mol, k=5):
        if not self.is_built or self.nn is None or len(self.fingerprints) == 0:
            return []
        n_available = len(self.fingerprints)
        if k > n_available:
            k = n_available
        if k == 0:
            return []
        fp_arr = np.zeros(256)
        fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=256).GetFingerprint(mol)
        DataStructs.ConvertToNumpyArray(fp, fp_arr)
        distances, indices = self.nn.kneighbors([fp_arr], n_neighbors=k)
        results = []
        for i, dist in zip(indices[0], distances[0]):
            sim = 1 - dist
            lam = self.lambdas[i]
            smi = self.smiles_list[i]
            name = self.names[i]
            results.append((smi, lam, sim, name))
        return results

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump({
                'fingerprints': self.fingerprints,
                'smiles_list': self.smiles_list,
                'lambdas': self.lambdas,
                'names': self.names,
                'is_built': self.is_built
            }, f)

    def load(self, path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
            self.fingerprints = data.get('fingerprints', [])
            self.smiles_list = data.get('smiles_list', [])
            self.lambdas = data.get('lambdas', [])
            self.names = data.get('names', [])
            self.is_built = data.get('is_built', False)
        if len(self.fingerprints) > 0:
            self.build()


# ---------- ChemistryMemory ----------
class ChemistryMemory:
    def __init__(self):
        self.solvent_stats = {}
        self.class_stats = {}
        self.global_stats = {'count': 0, 'mae': 0.0, 'rmse': 0.0}
        self.historical_errors = []

    def update(self, predicted_lambda, actual_lambda, solvent, chromophore_classes):
        error = abs(predicted_lambda - actual_lambda)
        self._update_stats(self.global_stats, error)
        self._update_solvent(solvent, error)
        for cls in chromophore_classes:
            self._update_class(cls, error)
        self.historical_errors.append(error)
        if len(self.historical_errors) > 10000:
            self.historical_errors = self.historical_errors[-10000:]

    def _update_stats(self, stats_dict, error):
        n = stats_dict.get('count', 0)
        old_mae = stats_dict.get('mae', 0.0)
        old_rmse = stats_dict.get('rmse', 0.0)
        new_count = n + 1
        new_mae = old_mae + (error - old_mae) / new_count
        if n == 0:
            new_rmse = error
        else:
            sum_sq = old_rmse**2 * n
            sum_sq_new = sum_sq + error**2
            new_rmse = np.sqrt(sum_sq_new / new_count)
        stats_dict['count'] = new_count
        stats_dict['mae'] = new_mae
        stats_dict['rmse'] = new_rmse

    def _update_solvent(self, solvent, error):
        if solvent not in self.solvent_stats:
            self.solvent_stats[solvent] = {'count': 0, 'mae': 0.0, 'rmse': 0.0}
        self._update_stats(self.solvent_stats[solvent], error)

    def _update_class(self, cls, error):
        if cls not in self.class_stats:
            self.class_stats[cls] = {'count': 0, 'mae': 0.0, 'rmse': 0.0}
        self._update_stats(self.class_stats[cls], error)

    def get_familiarity_score(self, solvent, chromophore_classes):
        scores = []
        if solvent in self.solvent_stats:
            n_sol = self.solvent_stats[solvent].get('count', 0)
            sol_score = min(1.0, n_sol / 50)
            scores.append(sol_score)
        else:
            scores.append(0.0)
        if chromophore_classes:
            class_scores = []
            for cls in chromophore_classes:
                if cls in self.class_stats:
                    n_cls = self.class_stats[cls].get('count', 0)
                    class_scores.append(min(1.0, n_cls / 20))
                else:
                    class_scores.append(0.0)
            scores.append(np.mean(class_scores))
        else:
            scores.append(0.0)
        global_n = self.global_stats.get('count', 0)
        scores.append(min(1.0, global_n / 100))
        return np.mean(scores)

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump({
                'solvent_stats': self.solvent_stats,
                'class_stats': self.class_stats,
                'global_stats': self.global_stats,
                'historical_errors': self.historical_errors
            }, f)

    def load(self, path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
            self.solvent_stats = data.get('solvent_stats', {})
            self.class_stats = data.get('class_stats', {})
            self.global_stats = data.get('global_stats', {'count': 0, 'mae': 0.0, 'rmse': 0.0})
            self.historical_errors = data.get('historical_errors', [])


# ---------- ResidualCorrector ----------
class ResidualCorrector:
    def __init__(self, model_type="random_forest"):
        self.model_type = model_type
        self.model = None

    def train(self, features, errors):
        if len(features) < 20:
            print("⚠️  Not enough data to train residual corrector (need ≥20).")
            return
        X = np.array(features)
        y = np.array(errors)
        y = np.clip(y, -200, 200)
        if self.model_type == "random_forest":
            if not SKLEARN_AVAILABLE:
                print("⚠️  scikit-learn not available; cannot train residual model.")
                return
            self.model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
        elif self.model_type == "mlp":
            from sklearn.neural_network import MLPRegressor
            self.model = MLPRegressor(hidden_layer_sizes=(64, 32), activation='relu',
                                      max_iter=200, random_state=42)
        else:
            raise ValueError(f"Unsupported residual model type: {self.model_type}")
        self.model.fit(X, y)
        print("✅ Residual corrector trained.")

    def predict(self, features):
        if self.model is None:
            return np.zeros(len(features))
        return self.model.predict(features)

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump({'model': self.model, 'model_type': self.model_type}, f)

    def load(self, path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
            self.model = data['model']
            self.model_type = data.get('model_type', self.model_type)


# ---------- SolventLearner ----------
class SolventLearner:
    def __init__(self, model_type="xgboost"):
        self.model_type = model_type
        self.model = None
        self.solvent_encoder = None

    def train(self, features, solvent_codes, shifts):
        if len(features) < 20:
            print("⚠️  Not enough data for solvent learner.")
            return
        self.solvent_encoder = LabelEncoder()
        sol_encoded = self.solvent_encoder.fit_transform(solvent_codes).reshape(-1, 1)
        X = np.hstack([features, sol_encoded])
        y = np.array(shifts)
        if self.model_type == "xgboost":
            if not XGB_AVAILABLE:
                print("⚠️  XGBoost not installed; falling back to RandomForest.")
                self.model_type = "random_forest"
                if SKLEARN_AVAILABLE:
                    self.model = RandomForestRegressor(n_estimators=50, max_depth=8, random_state=42)
                else:
                    print("⚠️  scikit-learn not available; solvent learner disabled.")
                    return
            else:
                self.model = xgb.XGBRegressor(n_estimators=50, max_depth=4, random_state=42)
        elif self.model_type == "random_forest":
            if SKLEARN_AVAILABLE:
                self.model = RandomForestRegressor(n_estimators=50, max_depth=8, random_state=42)
            else:
                print("⚠️  scikit-learn not available; solvent learner disabled.")
                return
        self.model.fit(X, y)
        print("✅ Solvent learner trained.")

    def predict_shift(self, features, solvent_code):
        if self.model is None or self.solvent_encoder is None:
            return 0.0
        try:
            sol_enc = self.solvent_encoder.transform([solvent_code])[0]
        except ValueError:
            return 0.0
        X = np.hstack([features, np.array([[sol_enc]])])
        return float(self.model.predict(X)[0])

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump({'model': self.model, 'encoder': self.solvent_encoder,
                         'model_type': self.model_type}, f)

    def load(self, path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
            self.model = data['model']
            self.solvent_encoder = data['encoder']
            self.model_type = data.get('model_type', self.model_type)


# ---------- EnsemblePredictor ----------
class EnsemblePredictor:
    def __init__(self, primary_model, feature_dim=1024):
        self.primary = primary_model
        self.rf = None
        self.xgb = None
        self.cat = None
        self.trained = False
        self.feature_dim = feature_dim

    def train(self, X, y_lam):
        if len(X) < 20:
            print("⚠️ Not enough data for ensemble training.")
            return
        y_lam = np.asarray(y_lam, dtype=np.float32).ravel()
        if SKLEARN_AVAILABLE:
            self.rf = RandomForestRegressor(n_estimators=15, max_depth=10, random_state=42)
            self.rf.fit(X, y_lam)
        else:
            print("⚠️ scikit-learn not available; skipping RandomForest.")
        if XGB_AVAILABLE:
            self.xgb = xgb.XGBRegressor(n_estimators=15, max_depth=6, random_state=42)
            self.xgb.fit(X, y_lam)
        else:
            print("⚠️ XGBoost not installed; skipping.")
        try:
            from catboost import CatBoostRegressor
            self.cat = CatBoostRegressor(iterations=15, depth=6, learning_rate=0.1,
                                         verbose=False, random_seed=42)
            self.cat.fit(X, y_lam)
            print("✅ CatBoost trained.")
        except ImportError:
            print("⚠️ CatBoost not installed; skipping.")
            self.cat = None
        self.trained = (self.rf is not None or self.xgb is not None or self.cat is not None)

    def predict(self, X):
        if not self.trained:
            return None, None
        preds = []
        if self.rf is not None:
            preds.append(np.asarray(self.rf.predict(X), dtype=np.float32))
        if self.xgb is not None:
            preds.append(np.asarray(self.xgb.predict(X), dtype=np.float32))
        if self.cat is not None:
            preds.append(np.asarray(self.cat.predict(X), dtype=np.float32))
        if not preds:
            return None, None
        preds = np.asarray(preds, dtype=np.float32)
        mean_lambda = np.mean(preds, axis=0)
        std_lambda = np.std(preds, axis=0)
        return mean_lambda, std_lambda

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump({'rf': self.rf, 'xgb': self.xgb, 'cat': self.cat,
                         'trained': self.trained, 'feature_dim': self.feature_dim}, f)

    def load(self, path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
            self.rf = data.get('rf')
            self.xgb = data.get('xgb')
            self.cat = data.get('cat')
            self.trained = data.get('trained', False)
            self.feature_dim = data.get('feature_dim', self.feature_dim)


# ---------- NGBoostPredictor ----------
class NGBoostPredictor:
    def __init__(self, n_estimators=50, learning_rate=0.01, verbose=False):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.verbose = verbose
        self.model = None
        self.fitted = False

    def train(self, X, y):
        if len(X) < 20:
            print("⚠️ Not enough data to train NGBoost (need ≥20).")
            return
        try:
            from ngboost import NGBRegressor
            from ngboost.distns import Normal
            self.model = NGBRegressor(n_estimators=self.n_estimators,
                                      learning_rate=self.learning_rate,
                                      Dist=Normal, verbose=self.verbose,
                                      random_state=42)
            self.model.fit(X, y)
            self.fitted = True
            print("✅ NGBoost trained.")
        except ImportError:
            print("⚠️ NGBoost not installed. Skipping.")
            self.fitted = False

    def predict(self, X):
        if not self.fitted or self.model is None:
            return None, None
        try:
            dist = self.model.pred_dist(X)
            mean = dist.loc.numpy().flatten() if hasattr(dist, 'loc') else self.model.predict(X)
            std = dist.scale.numpy().flatten() if hasattr(dist, 'scale') else np.ones_like(mean) * 10.0
            return mean, std
        except:
            mean = self.model.predict(X)
            std = np.ones_like(mean) * 10.0
            return mean, std

    def save(self, path):
        if self.fitted and self.model is not None:
            import joblib
            joblib.dump({'model': self.model}, path)

    def load(self, path):
        import joblib
        data = joblib.load(path)
        self.model = data['model']
        self.fitted = True


# ---------- MISTPredictor ----------
class MISTPredictor:
    def __init__(self, model_name="ibm/MIST-small", device=None):
        self.model_name = model_name
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = None
        self.model = None
        self.regressor = None
        self.fitted = False

    def load_model(self):
        if self.tokenizer is None:
            try:
                self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
                self.model = AutoModel.from_pretrained(self.model_name).to(self.device)
                self.model.eval()
                with torch.no_grad():
                    dummy = self.tokenizer("C", return_tensors="pt")
                    dummy_input = {k: v.to(self.device) for k, v in dummy.items()}
                    output = self.model(**dummy_input)
                    self.embed_dim = output.last_hidden_state.shape[-1]
                print(f"✅ MIST model loaded (embed_dim={self.embed_dim})")
            except Exception as e:
                print(f"⚠️ Failed to load MIST model: {e}")
                self.tokenizer = None
                self.model = None
                self.embed_dim = 768

    def get_embedding(self, mol):
        if self.tokenizer is None or self.model is None:
            self.load_model()
        if self.tokenizer is None or self.model is None:
            return None
        try:
            smi = Chem.MolToSmiles(mol)
            inputs = self.tokenizer(smi, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = self.model(**inputs)
            emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            return emb[0]
        except Exception as e:
            return None

    def train(self, mols, y_vals):
        from sklearn.linear_model import Ridge
        if len(mols) < 10:
            print("⚠️ Not enough data to train MIST regressor (need ≥10).")
            return
        self.load_model()
        if self.tokenizer is None or self.model is None:
            print("⚠️ MIST model not available.")
            return
        X_list, y_list = [], []
        for mol, y in zip(mols, y_vals):
            emb = self.get_embedding(mol)
            if emb is not None:
                X_list.append(emb)
                y_list.append(y)
        if len(X_list) < 10:
            print("⚠️ Not enough valid embeddings for MIST training.")
            return
        X = np.array(X_list)
        y = np.array(y_list)
        self.regressor = Ridge(alpha=1.0)
        self.regressor.fit(X, y)
        self.fitted = True
        print(f"✅ MIST regressor trained on {len(X_list)} samples.")

    def predict(self, mol):
        if not self.fitted or self.regressor is None:
            return None, None
        emb = self.get_embedding(mol)
        if emb is None:
            return None, None
        pred = float(self.regressor.predict([emb])[0])
        conf = min(0.8, len(self.regressor.coef_) / 5000)
        return pred, conf

    def save(self, path):
        if self.fitted and self.regressor is not None:
            import joblib
            joblib.dump({'regressor': self.regressor, 'embed_dim': self.embed_dim}, path)

    def load(self, path):
        import joblib
        data = joblib.load(path)
        self.regressor = data['regressor']
        self.embed_dim = data.get('embed_dim', 768)
        self.fitted = True
        self.load_model()


# ---------- MetaCombiner ----------
class MetaCombiner:
    """Accuracy-first convex stacking of OOF model predictions.

    The combiner is deliberately constrained to a convex combination:
        prediction = sum(weight_i * model_i_prediction)
        weight_i >= 0 and sum(weight_i) = 1

    Therefore it cannot create an unsupported extrapolation such as 390 nm
    when every component prediction is around 224-291 nm.  The weights are
    learned from out-of-fold predictions against experimental lambda-max using
    robust Huber loss. No intercept and no feature scaling are used.
    """

    def __init__(self, alpha=1.0):
        self.model = None
        self.scaler = None
        self.fitted = False
        self.weights = None
        self.fallback_weights = None
        self.fallback_normalized = True
        self.n_features = 0
        self.training_mae = None
        self.training_rmse = None
        self.training_bias = None

    def _clean(self, X_preds, y_true=None):
        X = np.asarray(X_preds, dtype=np.float64)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        X = np.nan_to_num(X, nan=np.nan, posinf=np.nan, neginf=np.nan)
        if y_true is None:
            return X
        y = np.asarray(y_true, dtype=np.float64).ravel()
        y = np.nan_to_num(y, nan=np.nan, posinf=np.nan, neginf=np.nan)
        mask = np.isfinite(y) & np.all(np.isfinite(X), axis=1)
        X = X[mask]
        y = y[mask]
        y = np.clip(y, Config.WL_MIN, Config.WL_MAX)
        return X, y

    @staticmethod
    def _mae(y_true, y_pred):
        return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

    def _inverse_mae_weights(self, X, y):
        maes = np.array([self._mae(y, X[:, i]) for i in range(X.shape[1])], dtype=float)
        inv = 1.0 / np.maximum(maes, 1e-6)
        w = inv / np.sum(inv)
        return w

    def train(self, X_preds, y_true):
        X, y = self._clean(X_preds, y_true)
        if X.shape[0] < 10:
            print("⚠️ Not enough samples to train MetaCombiner (need ≥10).")
            self.fitted = False
            return
        if X.shape[1] < 1:
            print("⚠️ MetaCombiner received no prediction features.")
            self.fitted = False
            return

        self.n_features = X.shape[1]
        fallback = self._inverse_mae_weights(X, y)
        self.fallback_weights = fallback.copy()

        try:
            from scipy.optimize import minimize

            def huber_loss(w):
                pred = np.dot(X, w)
                err = pred - y
                delta = 10.0
                abs_err = np.abs(err)
                loss = np.where(
                    abs_err <= delta,
                    0.5 * err ** 2,
                    delta * (abs_err - 0.5 * delta)
                )
                return float(np.mean(loss))

            constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)
            bounds = [(0.0, 1.0)] * self.n_features
            x0 = fallback.copy()

            result = minimize(
                huber_loss,
                x0,
                method="SLSQP",
                bounds=bounds,
                constraints=constraints,
                options={"maxiter": 1000, "ftol": 1e-10, "disp": False}
            )

            if result.success and np.all(np.isfinite(result.x)):
                weights = np.asarray(result.x, dtype=float)
                weights = np.maximum(weights, 0.0)
                weights = weights / np.sum(weights)
            else:
                print(f"⚠️ MetaCombiner optimizer did not converge: {result.message}. Using inverse-MAE weights.")
                weights = fallback
        except Exception as e:
            print(f"⚠️ MetaCombiner optimizer unavailable/failed: {e}. Using inverse-MAE weights.")
            weights = fallback

        self.weights = weights
        self.fallback_weights = weights.copy()
        self.fitted = True

        oof_pred = np.clip(np.dot(X, self.weights), Config.WL_MIN, Config.WL_MAX)
        self.training_mae = self._mae(y, oof_pred)
        self.training_rmse = float(np.sqrt(np.mean((y - oof_pred) ** 2)))
        self.training_bias = float(np.mean(oof_pred - y))

        print("✅ Accuracy-first MetaCombiner trained.")
        if Config.VERBOSE_TRAINING:
          print(
              "   Weights: " + ", ".join(
                  f"{w:.3f}" for w in self.weights
              )
          )
          print(
              f"   OOF Meta MAE={self.training_mae:.2f} nm | "
              f"RMSE={self.training_rmse:.2f} nm | Bias={self.training_bias:+.2f} nm"
          )

    def predict(self, X_preds):
        X = self._clean(X_preds)
        if self.n_features and X.shape[1] != self.n_features:
            raise ValueError(
                f"MetaCombiner expects {self.n_features} features, got {X.shape[1]}."
            )

        if self.fitted and self.weights is not None:
            weights = np.asarray(self.weights, dtype=float)
        elif self.fallback_weights is not None and len(self.fallback_weights) == X.shape[1]:
            weights = np.asarray(self.fallback_weights, dtype=float)
        else:
            weights = np.ones(X.shape[1], dtype=float) / X.shape[1]

        weights = np.maximum(weights, 0.0)
        if np.sum(weights) <= 0:
            weights = np.ones(X.shape[1], dtype=float) / X.shape[1]
        else:
            weights = weights / np.sum(weights)

        pred = np.dot(X, weights)
        pred = np.clip(pred, Config.WL_MIN, Config.WL_MAX)
        return np.asarray(pred, dtype=float)

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump({
                'weights': self.weights,
                'fitted': self.fitted,
                'fallback_weights': self.fallback_weights,
                'n_features': self.n_features,
                'training_mae': self.training_mae,
                'training_rmse': self.training_rmse,
                'training_bias': self.training_bias
            }, f)

    def load(self, path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
        self.weights = data.get('weights', None)
        self.fitted = bool(data.get('fitted', self.weights is not None))
        self.fallback_weights = data.get('fallback_weights', self.weights)
        self.n_features = int(data.get(
            'n_features',
            len(self.weights) if self.weights is not None else 0
        ))
        self.training_mae = data.get('training_mae')
        self.training_rmse = data.get('training_rmse')
        self.training_bias = data.get('training_bias')
        self.model = None
        self.scaler = None


# ---------- RadialMLPPredictor (e3nn) ----------
class RadialMLPPredictor:
    def __init__(self, device=None):
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.fc1 = None
        self.fc2 = None
        self.fc3 = None
        self.radial_basis = None
        self.fitted = False

    def _build_model(self, num_basis=20):
        import torch.nn as nn
        self.radial_basis = nn.Sequential(
            nn.Linear(1, 32), nn.SiLU(), nn.Linear(32, num_basis)
        )
        self.fc1 = nn.Linear(num_basis, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)
        self.dropout = nn.Dropout(0.1)
        self.relu = nn.ReLU()
        self.activation = nn.SiLU()
        self.to(self.device)

    def to(self, device):
        if self.radial_basis is not None:
            self.radial_basis.to(device)
        if self.fc1 is not None:
            self.fc1.to(device)
        if self.fc2 is not None:
            self.fc2.to(device)
        if self.fc3 is not None:
            self.fc3.to(device)
        self.device = device
        return self

    def _featurize(self, mol, conf_id=0):
        if mol.GetNumConformers() == 0:
            return None, None
        conf = mol.GetConformer(conf_id)
        atoms = [atom.GetAtomicNum() for atom in mol.GetAtoms()]
        coords = np.array([conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())], dtype=np.float32)
        return np.array(atoms), coords

    def train(self, mols, y_vals):
        import torch.optim as optim
        from torch.optim.lr_scheduler import ReduceLROnPlateau
        if len(mols) < 10:
            print("⚠️ Not enough data for RadialMLP training (need ≥10).")
            return
        self._build_model()
        raw_distances, all_targets = [], []
        for mol, target in zip(mols, y_vals):
            atoms, coords = self._featurize(mol)
            if atoms is None or coords is None:
                continue
            n = len(atoms)
            if n < 2:
                continue
            coords_t = torch.tensor(coords, dtype=torch.float32).to(self.device)
            dist_matrix = torch.cdist(coords_t, coords_t)
            triu = torch.triu_indices(n, n, offset=1)
            distances = dist_matrix[triu[0], triu[1]]
            if len(distances) == 0:
                continue
            raw_distances.append(distances.detach())
            all_targets.append(float(target))
        if len(raw_distances) < 10:
            print("⚠️ Not enough valid data for RadialMLP training.")
            return
        optimizer = optim.Adam(
            list(self.radial_basis.parameters()) +
            list(self.fc1.parameters()) + list(self.fc2.parameters()) + list(self.fc3.parameters()),
            lr=0.001
        )
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
        criterion = nn.HuberLoss(delta=10.0)
        self.radial_basis.train(); self.fc1.train(); self.fc2.train(); self.fc3.train()
        best_loss = float('inf')
        epochs_no_improve = 0
        max_epochs = 5
        patience = 8
        for epoch in range(max_epochs):
            total_loss = 0.0
            for distances, target in zip(raw_distances, all_targets):
                optimizer.zero_grad()
                basis = self.radial_basis(distances.reshape(-1, 1))
                feat = torch.mean(basis, dim=0)
                out = self.fc1(feat.unsqueeze(0))
                out = self.activation(out)
                out = self.dropout(out)
                out = self.fc2(out)
                out = self.activation(out)
                out = self.fc3(out)
                target_tensor = torch.tensor([target], dtype=torch.float32).to(self.device)
                loss = criterion(out.squeeze(), target_tensor.squeeze())
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            avg_loss = total_loss / len(raw_distances)
            scheduler.step(avg_loss)
            if avg_loss < best_loss:
                best_loss = avg_loss
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    break
        self.fitted = True
        self.radial_basis.eval(); self.fc1.eval(); self.fc2.eval(); self.fc3.eval()
        print(f"✅ RadialMLP trained on {len(raw_distances)} molecules.")

    def predict(self, mol):
        if not self.fitted:
            return None, None
        try:
            atoms, coords = self._featurize(mol)
            if atoms is None or coords is None:
                return None, None
            n = len(atoms)
            if n < 2:
                return None, None
            coords_t = torch.tensor(coords, dtype=torch.float32)
            dist_matrix = torch.cdist(coords_t, coords_t)
            triu = torch.triu_indices(n, n, offset=1)
            distances = dist_matrix[triu[0], triu[1]]
            with torch.no_grad():
                basis = self.radial_basis(distances.reshape(-1, 1))
                feat = torch.sum(basis, dim=0) / len(distances)
                pred = self.fc3(self.relu(self.fc2(self.relu(self.fc1(feat.unsqueeze(0)))))).item()
            conf = min(0.8, len(self.fc1.weight) / 1000)
            return float(pred), conf
        except Exception as e:
            print(f"⚠️ RadialMLP prediction failed: {e}")
            return None, None

    def save(self, path):
        if self.fitted:
            torch.save({
                'fc1_state': self.fc1.state_dict(),
                'fc2_state': self.fc2.state_dict(),
                'fc3_state': self.fc3.state_dict(),
                'radial_basis_state': self.radial_basis.state_dict(),
                'fitted': self.fitted
            }, path)

    def load(self, path):
        data = torch.load(path, map_location=self.device)
        self._build_model()
        self.fc1.load_state_dict(data['fc1_state'])
        self.fc2.load_state_dict(data['fc2_state'])
        self.fc3.load_state_dict(data['fc3_state'])
        self.radial_basis.load_state_dict(data['radial_basis_state'])
        self.fitted = data['fitted']
        self.to(self.device)
        self.fc1.eval(); self.fc2.eval(); self.fc3.eval(); self.radial_basis.eval()
        return self


# ---------- Hückel Calculator and Predictor ----------
class HuckelCalculator:
    def __init__(self):
        self.hetero_params = {
            7: {'h': 1.5, 'k': 0.8},
            8: {'h': 2.0, 'k': 0.6},
            16: {'h': 1.0, 'k': 0.7},
            9: {'h': 3.0, 'k': 0.5},
            17: {'h': 2.5, 'k': 0.5},
            35: {'h': 2.0, 'k': 0.6},
        }

    def _is_conjugated(self, mol):
        conj = set()
        for bond in mol.GetBonds():
            if bond.GetIsAromatic() or bond.GetBondType() in (Chem.rdchem.BondType.DOUBLE, Chem.rdchem.BondType.TRIPLE):
                conj.add(bond.GetBeginAtomIdx()); conj.add(bond.GetEndAtomIdx())
        changed = True
        while changed:
            changed = False
            for bond in mol.GetBonds():
                if bond.GetBondType() == Chem.rdchem.BondType.SINGLE:
                    a1, a2 = bond.GetBeginAtom(), bond.GetEndAtom()
                    if (a1.GetIdx() in conj) or (a2.GetIdx() in conj):
                        for atom in [a1, a2]:
                            if atom.GetIdx() not in conj:
                                if atom.GetHybridization() in (Chem.rdchem.HybridizationType.SP2, Chem.rdchem.HybridizationType.SP) and atom.GetAtomicNum() in (6,7,8,16):
                                    conj.add(atom.GetIdx()); changed = True
        return conj

    def compute_descriptors(self, mol):
        conj_atoms = self._is_conjugated(mol)
        if len(conj_atoms) < 2:
            return {'homo_lumo_gap': 0.0, 'deloc_energy': 0.0, 'n_pi_electrons': 0, 'n_conj_atoms': 0}
        atom_indices = sorted(conj_atoms)
        n = len(atom_indices)
        H = np.zeros((n, n))
        for i, old_i in enumerate(atom_indices):
            atom_i = mol.GetAtomWithIdx(old_i)
            Z_i = atom_i.GetAtomicNum()
            h_i = 0.0 if Z_i == 6 else self.hetero_params.get(Z_i, {'h': 0.0})['h']
            H[i,i] = h_i
        for i, old_i in enumerate(atom_indices):
            for j, old_j in enumerate(atom_indices):
                if i >= j: continue
                bond = mol.GetBondBetweenAtoms(old_i, old_j)
                if bond is None: continue
                if not (bond.GetIsAromatic() or bond.GetBondType() == Chem.rdchem.BondType.DOUBLE): continue
                atom_i = mol.GetAtomWithIdx(old_i); atom_j = mol.GetAtomWithIdx(old_j)
                Z_i, Z_j = atom_i.GetAtomicNum(), atom_j.GetAtomicNum()
                k_i = 1.0 if Z_i == 6 else self.hetero_params.get(Z_i, {'k': 1.0})['k']
                k_j = 1.0 if Z_j == 6 else self.hetero_params.get(Z_j, {'k': 1.0})['k']
                k = (k_i + k_j) / 2.0
                H[i,j] = H[j,i] = k
        if n == 0:
            return {'homo_lumo_gap': 0.0, 'deloc_energy': 0.0, 'n_pi_electrons': 0, 'n_conj_atoms': 0}
        eigenvals = np.linalg.eigvalsh(H)
        eigenvals = np.sort(eigenvals)
        n_pi_electrons = 0
        for old_i in atom_indices:
            atom = mol.GetAtomWithIdx(old_i)
            Z = atom.GetAtomicNum()
            if Z == 6: n_pi_electrons += 1
            elif Z in (7,8,16): n_pi_electrons += 2
            else: n_pi_electrons += 1
        n_occ = n_pi_electrons // 2
        if n_occ > 0 and n_occ <= len(eigenvals):
            homo = eigenvals[n_occ - 1]
            lumo = eigenvals[n_occ] if n_occ < len(eigenvals) else eigenvals[-1] + 1.0
            gap = lumo - homo
        else:
            gap = 0.0
        total_energy = 2.0 * sum(eigenvals[:n_occ]) if n_occ > 0 else 0.0
        n_double = sum(1 for bond in mol.GetBonds() if bond.GetIsAromatic() or bond.GetBondType() == Chem.rdchem.BondType.DOUBLE)
        loc_energy = 2.0 * n_double
        deloc_energy = total_energy - loc_energy
        beta_ev = 2.5
        gap_ev = gap * beta_ev
        deloc_ev = deloc_energy * beta_ev
        return {'homo_lumo_gap': float(gap_ev), 'deloc_energy': float(deloc_ev),
                'n_pi_electrons': n_pi_electrons, 'n_conj_atoms': n}

class HuckelPredictor:
    def __init__(self):
        self.calculator = HuckelCalculator()
        self.regressor = None
        self.fitted = False

    def get_features(self, mol):
        desc = self.calculator.compute_descriptors(mol)
        return np.array([desc['homo_lumo_gap'], desc['deloc_energy'],
                         desc['n_pi_electrons'], desc['n_conj_atoms']], dtype=np.float32)

    def train(self, mols, y_vals):
        from sklearn.linear_model import Ridge
        if len(mols) < 10:
            print("⚠️ Not enough data for Hückel predictor (need ≥10).")
            return
        X_list, y_list = [], []
        for mol, y in zip(mols, y_vals):
            feat = self.get_features(mol)
            if np.all(feat == 0):
                continue
            X_list.append(feat); y_list.append(y)
        if len(X_list) < 10:
            print("⚠️ Not enough valid Hückel features for training.")
            return
        X = np.array(X_list); y = np.array(y_list)
        self.regressor = Ridge(alpha=1.0)
        self.regressor.fit(X, y)
        self.fitted = True
        print(f"✅ Hückel predictor trained on {len(X_list)} samples.")

    def predict(self, mol):
        if not self.fitted or self.regressor is None:
            return None, None
        feat = self.get_features(mol)
        if np.all(feat == 0):
            return None, None
        pred = float(self.regressor.predict([feat])[0])
        conf = min(0.8, len(self.regressor.coef_) / 1000)
        return pred, conf

    def save(self, path):
        import joblib
        if self.fitted and self.regressor is not None:
            joblib.dump({'regressor': self.regressor}, path)

    def load(self, path):
        import joblib
        data = joblib.load(path)
        self.regressor = data['regressor']
        self.fitted = True


# ---------- ConformerHandler ----------
class ConformerHandler:
    @staticmethod
    def get_lowest_energy_conformer(mol, num_confs=Config.CONFORMER_NUM_CONFS,
                                    max_attempts=Config.CONFORMER_MAX_ATTEMPTS):
        mol_h = Chem.AddHs(mol)
        from rdkit.Chem import AllChem
        try:
            params = AllChem.ETKDGv2()
            params.randomSeed = 42
            params.useExpTorsionAnglePreferences = True
            params.useBasicKnowledge = True

            cids = AllChem.EmbedMultipleConfs(mol_h, numConfs=int(num_confs),
                                              maxAttempts=max_attempts, params=params)
        except Exception:
            cids = AllChem.EmbedMultipleConfs(mol_h, numConfs=int(num_confs),
                                              maxAttempts=max_attempts, randomSeed=42)
        if not cids:
            AllChem.EmbedMolecule(mol_h, randomSeed=42)
            return mol_h, 0
        energies = []
        for cid in cids:
            if MMFFOptimizeMolecule(mol_h, confId=cid, maxIters=50) == 0:
                mmff_props = AllChem.MMFFGetMoleculeProperties(mol_h)
                energy = AllChem.MMFFGetMoleculeForceField(mol_h, mmff_props, confId=cid).CalcEnergy()
                energies.append((cid, energy))
            else:
                energies.append((cid, float('inf')))
        best_cid = min(energies, key=lambda x: x[1])[0]
        return mol_h, best_cid

    @staticmethod
    def get_boltzmann_ensemble(mol, num_confs=Config.BOLTZMANN_NUM_CONFS,
                               temp=Config.BOLTZMANN_TEMP,
                               energy_cutoff=Config.BOLTZMANN_ENERGY_CUTOFF):
        import math
        mol_h = Chem.AddHs(mol)
        cids = AllChem.EmbedMultipleConfs(mol_h, numConfs=num_confs,
                                          maxAttempts=Config.CONFORMER_MAX_ATTEMPTS,
                                          randomSeed=42)
        if not cids:
            AllChem.EmbedMolecule(mol_h, randomSeed=42)
            cids = [0]
        energies = []
        for cid in cids:
            if MMFFOptimizeMolecule(mol_h, confId=cid, maxIters=50) == 0:
                mmff_props = AllChem.MMFFGetMoleculeProperties(mol_h)
                energy = AllChem.MMFFGetMoleculeForceField(mol_h, mmff_props, confId=cid).CalcEnergy()
                energies.append((cid, energy))
            else:
                continue
        if not energies:
            return [(mol_h, 0, 0.0, 1.0)]
        min_energy = min(e for _, e in energies)
        filtered = [(cid, e) for cid, e in energies if e - min_energy <= energy_cutoff]
        if not filtered:
            filtered = [(cid, e) for cid, e in energies if e == min_energy]
        R = 0.0019872041
        beta = 1.0 / (R * temp)
        weights = []
        total = 0.0
        for cid, e in filtered:
            w = math.exp(-beta * (e - min_energy))
            weights.append(w); total += w
        weights = [w / total for w in weights]
        result = []
        for (cid, e), w in zip(filtered, weights):
            result.append((mol_h, cid, e, w))
        return result

    @staticmethod
    def compute_3d_descriptors(mol, conf_id):
        desc = {}
        try:
            desc['dipole'] = Descriptors3D.CalcMolecularDipole(mol, confId=conf_id)
            desc['radius_gyration'] = Descriptors3D.CalcRadiusOfGyration(mol, confId=conf_id)
            desc['asphericity'] = Descriptors3D.CalcAsphericity(mol, confId=conf_id)
            desc['eccentricity'] = Descriptors3D.CalcEccentricity(mol, confId=conf_id)
            desc['inertial_shape'] = Descriptors3D.CalcInertialShapeFactor(mol, confId=conf_id)
            desc['sasa'] = Descriptors3D.CalcLabuteASA(mol, confId=conf_id)
        except:
            desc = {k: 0.0 for k in ['dipole', 'radius_gyration', 'asphericity',
                                     'eccentricity', 'inertial_shape', 'sasa']}
        return desc

    # Alternative numeric version (used in code)
    @staticmethod
    def compute_3d_descriptors_numeric(mol, conf_id=0):
        if mol is None:
            raise ValueError("Invalid molecule")
        if mol.GetNumConformers() == 0:
            raise ValueError("Molecule has no 3D conformer")
        conf = mol.GetConformer(conf_id)
        coords = np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())], dtype=np.float64)
        centroid = np.mean(coords, axis=0)
        centered = coords - centroid
        distances = np.linalg.norm(centered, axis=1)
        if len(coords) > 1:
            diff = coords[:, None, :] - coords[None, :, :]
            pairwise = np.linalg.norm(diff, axis=-1)
            upper = pairwise[np.triu_indices(len(coords), k=1)]
        else:
            upper = np.array([0.0])
        if len(coords) >= 3:
            cov = np.cov(centered.T)
            eigvals = np.linalg.eigvalsh(cov)
            eigvals = np.sort(np.maximum(eigvals, 0.0))[::-1]
            p1 = float(np.sqrt(eigvals[0]))
            p2 = float(np.sqrt(eigvals[1]))
            p3 = float(np.sqrt(eigvals[2]))
        else:
            p1 = p2 = p3 = 0.0
        return np.array([float(np.mean(distances)), float(np.std(distances)),
                         float(np.max(distances)), float(np.mean(upper)),
                         float(np.std(upper)), float(np.max(upper)),
                         p1, p2, p3], dtype=np.float32)


# ---------- ConformationalCorrector ----------
class ConformationalCorrector:
    def __init__(self):
        self.model = None
        self.scaler = RobustScaler()

    def train(self, X_3d, shifts):
        if len(X_3d) < 5:
            print("⚠️ Not enough 3D data to train (need ≥5).")
            return
        keys = ['dipole', 'radius_gyration', 'asphericity',
                'eccentricity', 'inertial_shape', 'sasa']
        X = np.array([[d.get(k, 0.0) for k in keys] for d in X_3d])
        X = self.scaler.fit_transform(X)
        y = np.array(shifts)
        from sklearn.ensemble import RandomForestRegressor
        self.model = RandomForestRegressor(n_estimators=50, max_depth=6, random_state=42)
        self.model.fit(X, y)
        print("✅ Conformational corrector trained.")

    def predict_shift(self, desc_dict):
        if self.model is None:
            return 0.0
        keys = ['dipole', 'radius_gyration', 'asphericity',
                'eccentricity', 'inertial_shape', 'sasa']
        X = np.array([[desc_dict.get(k, 0.0) for k in keys]])
        X = self.scaler.transform(X)
        return float(self.model.predict(X)[0])

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump({'model': self.model, 'scaler': self.scaler}, f)

    def load(self, path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
            self.model = data['model']
            self.scaler = data['scaler']


# ---------- ApplicabilityDomain ----------
class ApplicabilityDomain:
    def __init__(self, feedback_db=None, fingerprint_size=256, radius=2):
        self.fingerprint_size = fingerprint_size
        self.radius = radius
        self.train_fps = None
        self.nn = None
        self.fitted = False
        self.isolation_forest = None
        self.iso_min = None
        self.iso_max = None
        if feedback_db is not None:
            self.fit(feedback_db)

    def _compute_fp(self, mol):
        fp = rdFingerprintGenerator.GetMorganGenerator(radius=self.radius, fpSize=self.fingerprint_size).GetFingerprint(mol)
        arr = np.zeros(self.fingerprint_size)
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr

    def fit(self, feedback_db):
        fps = []
        for entry in feedback_db:
            mol = entry.get('mol')
            if mol is None:
                smi = entry.get('smiles')
                if smi:
                    mol = Chem.MolFromSmiles(smi)
            if mol is not None:
                fps.append(self._compute_fp(mol))
        if len(fps) < 5:
            print("⚠️  Not enough molecules for AD fitting (need ≥5).")
            return
        self.train_fps = np.array(fps)
        self.nn = NearestNeighbors(n_neighbors=min(5, len(fps)), metric='jaccard')
        self.nn.fit(self.train_fps)
        if SKLEARN_AVAILABLE and len(fps) > 10:
            self.isolation_forest = IsolationForest(contamination=0.05, random_state=42)
            self.isolation_forest.fit(fps)
            scores = self.isolation_forest.score_samples(fps)
            self.iso_min = np.min(scores)
            self.iso_max = np.max(scores)
            self.fitted = True
        print(f"✅ Applicability Domain fitted on {len(fps)} molecules.")

    def compute_similarity(self, mol):
        if not self.fitted or self.nn is None:
            return 0.0
        fp = self._compute_fp(mol)
        distances, _ = self.nn.kneighbors([fp], n_neighbors=1)
        tanimoto = 1 - distances[0][0]
        return tanimoto

    def compute_isolation_score(self, mol):
        if self.isolation_forest is None or self.iso_min is None or self.iso_max is None:
            return 0.5
        fp = self._compute_fp(mol)
        score = self.isolation_forest.score_samples([fp])[0]
        novelty = (self.iso_max - score) / (self.iso_max - self.iso_min + 1e-8)
        return float(np.clip(novelty, 0.0, 1.0))

    def evaluate(self, mol):
        if not self.fitted:
            return 0.0, 1.0, 'Unknown'
        tan = self.compute_similarity(mol)
        iso = self.compute_isolation_score(mol)
        if tan > 0.6 and iso < 0.6:
            status = 'High'
        elif tan > 0.4 and iso < 0.8:
            status = 'Moderate'
        else:
            status = 'Low'
        return tan, iso, status


# ---------- KamletTaftModel ----------
class KamletTaftModel:
    def __init__(self):
        self.coefficients = None
        self.fitted = False
        self.model = None

    def fit(self, X, y, alpha_reg=0.1):
        X = np.asarray(X)
        y = np.asarray(y)
        if X.shape[0] < 10:
            print("⚠️  Not enough data to fit Kamlet–Taft (need ≥10).")
            return
        X_aug = np.hstack([np.ones((X.shape[0], 1)), X])
        self.model = Ridge(alpha=alpha_reg, fit_intercept=False)
        self.model.fit(X_aug, y)
        self.coefficients = self.model.coef_
        self.fitted = True
        print("✅ Kamlet–Taft model fitted.")

    def predict(self, props, temperature=298.15):
        if not self.fitted or self.coefficients is None:
            return None
        pi = props.get('pi_star', 0.5)
        alp = props.get('alpha', 0.5)
        bet = props.get('beta', 0.5)
        if len(self.coefficients) >= 5:
            temp = props.get('temperature', temperature)
            X = np.array([1, pi, alp, bet, temp])
        else:
            X = np.array([1, pi, alp, bet])
        return np.dot(self.coefficients, X)

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump({'coefficients': self.coefficients, 'fitted': self.fitted,
                         'model': self.model}, f)

    def load(self, path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
            self.coefficients = data['coefficients']
            self.fitted = data['fitted']
            self.model = data.get('model')


# ---------- PySCFPredictor, QuantumIntegrator, InstrumentParameters, SpectrumGenerator ----------
class PySCFPredictor:
    @staticmethod
    def predict(mol, solvent_eps=None, extract_descriptors=False,
                basis_list=None, functionals=None):
        if basis_list is None:
            basis_list = ["sto-3g", "def2-svp", "def2-tzvp"]
        if functionals is None:
            functionals = ["b3lyp", "cam-b3lyp", "wb97x-d"]
        if mol.GetNumHeavyAtoms() > 80:
            print(f"   ⏩ Skipping TD‑DFT: {mol.GetNumHeavyAtoms()} heavy atoms (too large)")
            return [], {}, False
        smiles = Chem.MolToSmiles(mol)
        result_queue = mp.Queue()

        def _run_in_child():
            from pyscf import gto, dft, tddft
            mol_child = Chem.MolFromSmiles(smiles)
            if mol_child is None:
                result_queue.put(([], {}, False)); return
            try:
                mol_h, conf_id = ConformerHandler.get_lowest_energy_conformer(mol_child)
                conf = mol_h.GetConformer(conf_id)
                atoms = []
                for atom in mol_h.GetAtoms():
                    pos = conf.GetAtomPosition(atom.GetIdx())
                    atoms.append(f"{atom.GetSymbol()} {pos.x} {pos.y} {pos.z}")
                geom = "; ".join(atoms)
                total_charge = Chem.GetFormalCharge(mol_h)
                n_elec = sum(atom.GetAtomicNum() for atom in mol_h.GetAtoms()) - total_charge
                spin = 0 if n_elec % 2 == 0 else 1
                method_class = dft.RKS if spin == 0 else dft.UKS
                n_heavy = mol_child.GetNumHeavyAtoms()
                if n_heavy <= 10: nstates = 8
                elif n_heavy <= 20: nstates = 12
                elif n_heavy <= 30: nstates = 16
                elif n_heavy <= 40: nstates = 20
                elif n_heavy <= 50: nstates = 25
                elif n_heavy <= 70: nstates = 30
                else: nstates = 8
                peaks = []; descriptors = {}; success = False
                for basis in basis_list:
                    if success: break
                    for xc in functionals:
                        try:
                            pyscf_mol = gto.M(atom=geom, basis=basis,
                                               charge=total_charge, spin=spin, verbose=0)
                            mf = method_class(pyscf_mol); mf.xc = xc
                            mf.max_cycle = 800; mf.diis = True; mf.diis_start_cycle = 5
                            mf.diis_space = 20; mf.conv_tol = 1e-7; mf.level_shift = 0.2
                            mf.damp = 0.8; mf.init_guess = 'minao'
                            if solvent_eps is not None:
                                from pyscf import solvent
                                mf = solvent.ddCOSMO(mf); mf.with_solvent.eps = solvent_eps
                            mf.grids.level = 4
                            mf.kernel(max_cycle=800)
                            if not mf.converged:
                                mf.conv_tol = 1e-6; mf.kernel(max_cycle=400)
                                if not mf.converged: continue
                            if extract_descriptors:
                                try:
                                    mo_energy = mf.mo_energy; occ = mf.mo_occ
                                    homo_idx = np.where(occ > 0)[0][-1] if np.any(occ > 0) else 0
                                    lumo_idx = np.where(occ == 0)[0][0] if np.any(occ == 0) else len(mo_energy)-1
                                    descriptors['homo'] = mo_energy[homo_idx] * 27.2114
                                    descriptors['lumo'] = mo_energy[lumo_idx] * 27.2114
                                    descriptors['gap'] = descriptors['lumo'] - descriptors['homo']
                                    descriptors['basis'] = basis; descriptors['functional'] = xc
                                except: pass
                            td = tddft.TDDFT(mf); td.nstates = nstates; td.kernel()
                            osc = td.oscillator_strength() if hasattr(td, 'oscillator_strength') else [0.0]*len(td.e)
                            for i, e in enumerate(td.e):
                                ev = e * 27.2114; lam = 1240 / ev
                                f = osc[i] if i < len(osc) else 0.0
                                if Config.WL_MIN < lam < Config.WL_MAX and f >= Config.TDDFT_OSC_THRESHOLD:
                                    peaks.append((lam, f * 0.8, 20))
                            if peaks: success = True; break
                        except Exception:
                            continue
                result_queue.put((peaks, descriptors, success))
            except Exception:
                result_queue.put(([], {}, False))

        p = mp.Process(target=_run_in_child); p.start(); p.join(timeout=90)
        if p.is_alive(): p.terminate(); p.join(); print("⏱️ TD‑DFT timed out after 90 seconds. Falling back to heuristics."); return [], {}, False
        try: peaks, descriptors, success = result_queue.get_nowait()
        except: peaks, descriptors, success = [], {}, False
        return peaks, descriptors, success


class QuantumIntegrator:
    def __init__(self):
        self.shift_ref_gap = Config.QUANTUM_SHIFT_REFERENCE_GAP
        self.max_shift = Config.QUANTUM_MAX_SHIFT
    def compute_shift(self, gap_eV, prediction_lambda):
        if gap_eV is None or gap_eV <= 0: return 0.0
        delta = self.shift_ref_gap - gap_eV
        shift_nm = delta * 30.0
        shift_nm = np.clip(shift_nm, -self.max_shift, self.max_shift)
        return shift_nm
    def adjust_spectrum(self, spectrum, quantum_desc):
        gap = quantum_desc.get('gap')
        if gap is None: return spectrum
        shift_amount = self.compute_shift(gap, None)
        if abs(shift_amount) < 0.5: return spectrum
        shift_steps = shift_amount / Config.WL_STEP
        shifted = shift(spectrum, shift_steps, mode='nearest')
        return np.maximum(shifted, 0)
    def compute_quantum_confidence(self, quantum_desc, predicted_lambda):
        gap = quantum_desc.get('gap')
        if gap is None: return 0.0
        estimated_lam = 1240 / gap if gap > 0 else 500
        diff = abs(predicted_lambda - estimated_lam)
        confidence_factor = 1.0 - min(diff / 100.0, 1.0)
        return confidence_factor * 0.5


class InstrumentParameters:
    def __init__(self):
        self.path_length = Config.PATH_LENGTH
        self.noise_level = Config.NOISE_LEVEL
        self.baseline_drift = Config.BASELINE_DRIFT
        self.wavelengths = Config.wavelengths

    def apply_instrument_effects(self, spectrum):
        noise = np.random.normal(0, self.noise_level, len(spectrum))
        drift = self.baseline_drift * np.linspace(0, 1, len(spectrum))
        savgol_win = int(round(2 / Config.WL_STEP)) | 1
        smoothed = savgol_filter(spectrum + noise + drift, savgol_win, 3)
        return np.clip(smoothed, 0, None)

class SpectrumGenerator:
    def __init__(self):
        self.wavelengths = Config.wavelengths
    def from_peaks(self, peaks):
        spectrum = np.zeros(len(self.wavelengths))
        for wl, intensity, width in peaks:
            spectrum += intensity * np.exp(-((self.wavelengths - wl)**2) / (2 * width**2))
        return spectrum
    def apply_beer_lambert(self, spectrum, concentration, path_length):
        return spectrum * concentration * path_length


# ---------- ConfidenceScorer ----------
class ConfidenceScorer:
    def __init__(self, feature_extractor, feedback_db, ensemble=None,
                 outlier_detector=None, deep_svdd=None, knn_k=5):
        self.feature_extractor = feature_extractor
        self.feedback_db = feedback_db
        self.ensemble = ensemble
        self.outlier_detector = outlier_detector
        self.deep_svdd = deep_svdd
        self.knn_k = knn_k

    def compute_confidence(self, mol, pH=7.0, ad_score=None,
                           similarity_score=None, ensemble_uncertainty=None,
                           **kwargs):
        scores = []; details = {}
        ad_score = float(np.clip(ad_score if ad_score is not None else 0.5, 0.0, 1.0))
        scores.append(ad_score); details["AD"] = ad_score
        sim = float(np.clip(similarity_score if similarity_score is not None else 0.5, 0.0, 1.0))
        scores.append(sim); details["similarity"] = sim
        if ensemble_uncertainty is not None:
            ens_score = float(np.exp(-float(ensemble_uncertainty) / 25.0))
        else:
            ens_score = 0.5
        ens_score = float(np.clip(ens_score, 0.0, 1.0))
        scores.append(ens_score); details["ensemble"] = ens_score

        iso_score = 0.5
        if self.outlier_detector is not None and self.outlier_detector.fitted:
            try:
                feat = self.feature_extractor.extract(mol, pH)
                iso_score = self.outlier_detector.compute_isolation_score(feat)
            except: iso_score = 0.5
        deep_score = 0.5
        if self.deep_svdd is not None and self.deep_svdd.is_trained:
            try:
                feat = self.feature_extractor.extract(mol, pH)
                deep_score = self.deep_svdd.compute_outlier_score(feat)
            except: deep_score = 0.5
        outlier_score = 0.5 * iso_score + 0.5 * deep_score
        outlier_score = float(np.clip(outlier_score, 0.0, 1.0))
        scores.append(outlier_score); details["outlier"] = outlier_score

        confidence = float(np.mean(scores) * 100.0)
        if confidence >= 80: category, color = "High", "Blue"
        elif confidence >= 60: category, color = "Good", "Green"
        elif confidence >= 40: category, color = "Moderate", "Yellow"
        elif confidence >= 20: category, color = "Low", "Orange"
        else: category, color = "Very Low", "Red"
        return confidence, category, color, details


# ---------- OutlierDetector ----------
class OutlierDetector:
    def __init__(self, contamination=0.05):
        self.contamination = contamination
        self.model = None
        self.fitted = False

    def train(self, X):
        if X is None or len(X) < 10:
            self.model = None; self.fitted = False; return
        X = np.asarray(X, dtype=np.float32)
        try:
            from sklearn.ensemble import IsolationForest
            self.model = IsolationForest(contamination=self.contamination,
                                         random_state=Config.RANDOM_SEED)
            self.model.fit(X); self.fitted = True
        except Exception as e:
            print(f"⚠️ Outlier detector training failed: {e}")
            self.model = None; self.fitted = False

    def predict(self, X):
        if not self.fitted or self.model is None:
            return np.ones(len(np.atleast_2d(X)), dtype=int)
        X = np.asarray(X, dtype=np.float32)
        return self.model.predict(X)

    def compute_isolation_score(self, X):
        if not self.fitted or self.model is None:
            return 0.5
        X = np.asarray(X, dtype=np.float32).reshape(1, -1)
        try:
            score = self.model.decision_function(X)[0]
            return float(1.0 / (1.0 + np.exp(-score)))
        except Exception:
            return 0.5

    def save(self, path):
        import joblib
        joblib.dump({"model": self.model, "contamination": self.contamination,
                     "fitted": self.fitted}, path)

    def load(self, path):
        import joblib
        data = joblib.load(path)
        if isinstance(data, dict):
            self.model = data.get("model", None)
            self.contamination = data.get("contamination", self.contamination)
            self.fitted = data.get("fitted", self.model is not None)
        else:
            self.model = data
            self.fitted = self.model is not None
        return self


# ---------- DeepSVDDDetector ----------
class DeepSVDDDetector:
    def __init__(self, input_dim, hidden_dims=[128, 64], latent_dim=32,
                 nu=0.05, lr=0.001, epochs=50, batch_size=64):
        self.input_dim = input_dim; self.hidden_dims = hidden_dims
        self.latent_dim = latent_dim; self.nu = nu; self.lr = lr
        self.epochs = epochs; self.batch_size = batch_size
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.net = None; self.center = None; self.is_trained = False

    def _build_network(self):
        layers = []
        prev = self.input_dim
        for h in self.hidden_dims:
            layers.append(nn.Linear(prev, h)); layers.append(nn.ReLU()); layers.append(nn.Dropout(0.2))
            prev = h
        layers.append(nn.Linear(prev, self.latent_dim))
        return nn.Sequential(*layers).to(self.device)

    def train(self, X):
        if X is None or len(X) < 10:
            print("⚠️ Not enough data to train Deep SVDD (need ≥10).")
            return
        X = np.asarray(X, dtype=np.float32); n_samples = len(X)
        self.net = self._build_network()
        self.net.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            latent = self.net(X_tensor)
            self.center = torch.mean(latent, dim=0, keepdim=True).detach().to(self.device)
        dataset = TensorDataset(torch.FloatTensor(X))
        dataloader = DataLoader(dataset, batch_size=min(self.batch_size, n_samples), shuffle=True)
        optimizer = optim.Adam(self.net.parameters(), lr=self.lr)
        criterion = nn.MSELoss()
        self.net.train()
        for epoch in range(self.epochs):
            epoch_loss = 0.0
            for batch in dataloader:
                batch_X = batch[0].to(self.device)
                latent = self.net(batch_X)
                dist = torch.sum((latent - self.center) ** 2, dim=1)
                loss = torch.mean(dist)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                epoch_loss += loss.item()
        self.is_trained = True
        print("✅ Deep SVDD trained.")

    def compute_outlier_score(self, X):
        if not self.is_trained or self.net is None or self.center is None:
            return 0.5
        X = np.asarray(X, dtype=np.float32).reshape(1, -1)
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            latent = self.net(X_tensor)
            dist = torch.sum((latent - self.center) ** 2, dim=1).cpu().numpy()[0]
        score = 1.0 / (1.0 + np.exp(-(dist - 0.5)))
        return float(np.clip(score, 0.0, 1.0))

    def save(self, path):
        if self.is_trained:
            torch.save({
                'net_state': self.net.state_dict(),
                'center': self.center.cpu().numpy(),
                'input_dim': self.input_dim,
                'hidden_dims': self.hidden_dims,
                'latent_dim': self.latent_dim,
                'nu': self.nu
            }, path)

    def load(self, path):
        checkpoint = torch.load(path, map_location=self.device, weights_only=False)
        self.input_dim = checkpoint['input_dim']
        self.hidden_dims = checkpoint['hidden_dims']
        self.latent_dim = checkpoint['latent_dim']
        self.nu = checkpoint['nu']
        self.net = self._build_network()
        self.net.load_state_dict(checkpoint['net_state'])
        self.center = torch.FloatTensor(checkpoint['center']).to(self.device)
        self.is_trained = True
        self.net.eval()
        return self


# ---------- SMILES tokenizer for DR‑CNN ----------
class SMILESTokenizer:
    def __init__(self):
        self.chars = set("".join(["()[]{}", "0123456789", "ABCDEFGHIJKLMNOPQRSTUVWXYZ",
                                  "abcdefghijklmnopqrstuvwxyz", "=#@+-.:", "\\/", " ", "*"]))
        self.char_to_idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx_to_char = {0: '<PAD>', 1: '<UNK>'}
        self._next_idx = 2

    def fit(self, smiles_list):
        for smi in smiles_list:
            for ch in smi:
                if ch not in self.char_to_idx:
                    self.char_to_idx[ch] = self._next_idx
                    self.idx_to_char[self._next_idx] = ch
                    self._next_idx += 1
        self.vocab_size = len(self.char_to_idx)

    def encode(self, smi, max_len=150):
        indices = []
        for ch in smi:
            indices.append(self.char_to_idx.get(ch, self.char_to_idx['<UNK>']))
        if len(indices) > max_len:
            indices = indices[:max_len]
        else:
            indices += [0] * (max_len - len(indices))
        return np.array(indices, dtype=np.int64)

    def decode(self, indices):
        return "".join([self.idx_to_char.get(i, '?') for i in indices if i != 0])


class SMILESDataset(Dataset):
    def __init__(self, smiles_list, lambdas, tokenizer, max_len=150):
        self.smiles = smiles_list
        self.lambdas = np.array(lambdas, dtype=np.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.smiles)

    def __getitem__(self, idx):
        enc = self.tokenizer.encode(self.smiles[idx], self.max_len)
        lam = self.lambdas[idx]
        return torch.tensor(enc, dtype=torch.long), torch.tensor(lam, dtype=torch.float32)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dropout=0.2):
        super().__init__()
        self.conv1 = Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2)
        self.bn1 = BatchNorm1d(out_channels)
        self.relu = ReLU(inplace=True)
        self.conv2 = Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2)
        self.bn2 = BatchNorm1d(out_channels)
        self.dropout = Dropout(dropout)
        self.shortcut = Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.conv1(x); out = self.bn1(out); out = self.relu(out); out = self.dropout(out)
        out = self.conv2(out); out = self.bn2(out); out += identity; out = self.relu(out)
        return out


class DeepResidualCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, filters=[64,128,256],
                 kernels=[5,3,3], dropout=0.2, max_len=150):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv_init = Conv1d(embed_dim, filters[0], 1)
        blocks = []
        in_ch = filters[0]
        for i, (out_ch, k) in enumerate(zip(filters, kernels)):
            blocks.append(ResidualBlock(in_ch, out_ch, k, dropout))
            in_ch = out_ch
            if i < len(filters)-1:
                blocks.append(MaxPool1d(2))
        self.res_blocks = nn.Sequential(*blocks)
        self.global_pool = AdaptiveAvgPool1d(1)
        self.fc1 = Linear(in_ch, 128)
        self.relu = ReLU()
        self.dropout = Dropout(dropout)
        self.fc2 = Linear(128, 1)

    def forward(self, x):
        x = self.embedding(x)                 # (batch, seq_len, embed_dim)
        x = x.permute(0, 2, 1)                # (batch, embed_dim, seq_len)
        x = self.conv_init(x)
        x = self.res_blocks(x)
        x = self.global_pool(x)               # (batch, last_filters, 1)
        x = x.squeeze(-1)                     # (batch, last_filters)
        x = self.fc1(x); x = self.relu(x); x = self.dropout(x)
        x = self.fc2(x)
        return x.squeeze(-1)


# ---------- SelfLearningPredictor (MODIFIED) ----------
class SelfLearningPredictor:
    def __init__(self, feature_extractor):
        self.device = torch.device("cuda" if Config.USE_CUDA else "cpu")
        self.feat_extractor = feature_extractor
        self.model = None
        self.scaler = RobustScaler()
        self.is_trained = False
        self.feedback_db = []
        self.error_history = []
        self.metrics_log = []
        self.current_feature_dim = Config.PADDED_FEAT_DIM
        self.new_feedback_count = 0
        self.solvent_offsets = {}

        self.residual_corrector = ResidualCorrector(Config.RESIDUAL_MODEL_TYPE)
        self.solvent_learner = SolventLearner(Config.SOLVENT_MODEL_TYPE)
        self.ensemble = EnsemblePredictor(self.model)
        self.drcnn_model = None
        self.drcnn_tokenizer = None
        self.use_drcnn = Config.USE_DRCNN
        self.outlier_detector = OutlierDetector(Config.OUTLIER_CONTAMINATION)
        self.deep_svdd = DeepSVDDDetector(
            input_dim=self.current_feature_dim,
            hidden_dims=[128, 64],
            latent_dim=32,
            epochs=30
        )
        self.deep_svdd_trained = False
        self.confidence_scorer = None
        self.use_residual = Config.USE_RESIDUAL_CORRECTION
        self.use_dynamic_solvent = Config.USE_DYNAMIC_SOLVENT
        self.use_ensemble = Config.USE_ENSEMBLE
        self.use_outlier = Config.USE_OUTLIER_DETECTION
        self.use_conform = Config.USE_CONFORMATIONAL_CORRECTION
        self.conform_corrector = ConformationalCorrector()
        self.similarity_index = None
        self.chemistry_memory = None
        self.test_indices = set()
        self.validation_indices = set()
        self.meta_combiner = MetaCombiner(alpha=1.0)
        self.ngboost = NGBoostPredictor(n_estimators=30, learning_rate=0.01)
        self.ngboost_calibrator = None
        self.e3nn_predictor = RadialMLPPredictor()
        self.e3nn_trained = False
        self.huckel_predictor = HuckelPredictor()
        self.huckel_trained = False
        self.frozen_test_smiles = set()
        self.reaction_learner = None
        self.sim_ref = None

    def featurize(self, mol, pH=None):
        if pH is None: pH = Config.DEFAULT_PH
        feat = self.feat_extractor.extract(mol, pH)
        feat = np.asarray(feat, dtype=np.float32).reshape(-1)
        if len(feat) != self.current_feature_dim:
            raise ValueError(f"Feature dimension mismatch: expected {self.current_feature_dim}, got {len(feat)}")
        return feat

    def _clean_features(self, X):
        X = np.asarray(X, dtype=np.float32)
        X = np.nan_to_num(X, nan=np.nan, posinf=np.nan, neginf=np.nan)
        imp = SimpleImputer(strategy='median')
        X_clean = imp.fit_transform(X)
        return X_clean, imp

    def train(self, X, y_lam, X_val=None, y_lam_val=None,
              epochs=None, lr=None, patience=None):
        if epochs is None: epochs = Config.EPOCHS
        if lr is None: lr = Config.LEARNING_RATE
        if patience is None: patience = Config.EARLY_STOPPING_PATIENCE
        n_samples = len(X)
        if n_samples < 20:
            print(f"⚠️ Not enough training samples (need ≥20, got {n_samples}).")
            return

        # ---------- SANITISE INPUTS ----------
        X = np.asarray(X, dtype=np.float32)
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        y_lam = np.asarray(y_lam, dtype=np.float32).ravel()
        y_lam = np.nan_to_num(y_lam, nan=300.0, posinf=800.0, neginf=200.0)
        y_lam = np.clip(y_lam, Config.WL_MIN, Config.WL_MAX)

        if X_val is not None and len(X_val) > 0:
            X_val = np.asarray(X_val, dtype=np.float32)
            X_val = np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0)
            y_lam_val = np.asarray(y_lam_val, dtype=np.float32).ravel()
            y_lam_val = np.nan_to_num(y_lam_val, nan=300.0, posinf=800.0, neginf=200.0)
            y_lam_val = np.clip(y_lam_val, Config.WL_MIN, Config.WL_MAX)
        # -------------------------------------

        X_scaled = self.scaler.fit_transform(X)

        # ---------- CHECK SCALER HEALTH ----------
        if np.any(~np.isfinite(self.scaler.center_)) or np.any(~np.isfinite(self.scaler.scale_)):
            print("⚠️ Scaler produced non-finite statistics; refitting clean.")
            self.scaler = RobustScaler()
            self.scaler.fit(X)
            X_scaled = self.scaler.transform(X)
        # ----------------------------------------

        if X_val is not None and len(X_val) > 0:
            X_val_scaled = self.scaler.transform(X_val)
            X_val_scaled = np.nan_to_num(X_val_scaled, nan=0.0, posinf=0.0, neginf=0.0)
        else:
            X_val_scaled = None

        train_dataset = TensorDataset(torch.FloatTensor(X_scaled), torch.FloatTensor(y_lam))
        batch_size = min(Config.BATCH_SIZE, len(X_scaled))
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        self.model = UVVisNN(self.current_feature_dim, Config.HIDDEN_LAYERS, Config.DROPOUT).to(self.device)
        criterion_lam = nn.HuberLoss(delta=10.0)
        optimizer = optim.Adam(self.model.parameters(), lr=lr)
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
        best_val_loss = float('inf')
        epochs_no_improve = 0
        best_state = None

        for epoch in range(epochs):
            self.model.train()
            train_loss_lam = 0.0
            batch_count = 0
            for bx, by_lam in train_loader:
                bx, by_lam = bx.to(self.device), by_lam.to(self.device)
                optimizer.zero_grad()
                lam_out = self.model(bx)
                loss_lam = criterion_lam(lam_out, by_lam)
                # --- SKIP non-finite losses ---
                if not torch.isfinite(loss_lam):
                    continue
                loss_lam.backward()
                # --- GRADIENT CLIPPING ---
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)
                optimizer.step()
                train_loss_lam += loss_lam.item()
                batch_count += 1

            if batch_count == 0:
                print(f"⚠️ All batches produced non-finite loss at epoch {epoch+1}; aborting training.")
                break
            train_loss_lam /= batch_count

            val_loss = None
            if X_val_scaled is not None and len(X_val_scaled) > 0:
                self.model.eval()
                val_loss_lam = 0.0
                val_batch_count = 0
                val_dataset = TensorDataset(torch.FloatTensor(X_val_scaled), torch.FloatTensor(y_lam_val))
                val_loader = DataLoader(val_dataset, batch_size=min(Config.BATCH_SIZE, len(X_val_scaled)), shuffle=False)
                with torch.no_grad():
                    for bx, by_lam in val_loader:
                        bx, by_lam = bx.to(self.device), by_lam.to(self.device)
                        lam_out = self.model(bx)
                        loss_lam = criterion_lam(lam_out, by_lam)
                        if not torch.isfinite(loss_lam):
                            continue
                        val_loss_lam += loss_lam.item()
                        val_batch_count += 1
                if val_batch_count == 0:
                    val_loss = float('inf')
                else:
                    val_loss = val_loss_lam / val_batch_count
                scheduler.step(val_loss)
                if np.isfinite(val_loss) and val_loss < best_val_loss:
                    best_val_loss = val_loss
                    epochs_no_improve = 0
                    import copy
                    best_state = copy.deepcopy(self.model.state_dict())
                else:
                    epochs_no_improve += 1
                    if epochs_no_improve >= patience:
                        print(f"Early stopping at epoch {epoch + 1}")
                        break
            else:
                import copy
                best_state = copy.deepcopy(self.model.state_dict())

            if Config.VERBOSE_TRAINING and (epoch + 1) % 10 == 0:
                val_str = f", Val Loss (λmax): {val_loss:.6f}" if val_loss is not None else ""
                print(f"Epoch {epoch + 1}/{epochs}, Train Loss (λmax): {train_loss_lam:.6f}{val_str}")

        if best_state is not None:
            self.model.load_state_dict(best_state)
        self.model.eval()

        # ---------- FINAL SANITY CHECK ----------
        with torch.no_grad():
            probe = torch.FloatTensor(X_scaled[:min(5, len(X_scaled))]).to(self.device)
            out = self.model(probe)
            if not torch.all(torch.isfinite(out)):
                print("⚠️ Model still produces non-finite output after training. Marking as untrained.")
                self.is_trained = False
                self.model = None
                return
        # ----------------------------------------

        self.is_trained = True
        if self.use_ensemble and self.model is not None:
            self.ensemble = EnsemblePredictor(self.model, self.current_feature_dim)

    def predict(self, mol, pH):
        if not self.is_trained or self.model is None:
            return None, None
        try:
            feat = self.feat_extractor.extract(mol, pH).reshape(1, -1)
            feat = np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0)
            feat_scaled = self.scaler.transform(feat)
            if np.any(~np.isfinite(feat_scaled)):
                print("   ⚠️ Scaled features contain non-finite values; replacing with zeros.")
                feat_scaled = np.nan_to_num(feat_scaled, nan=0.0, posinf=0.0, neginf=0.0)
            with torch.no_grad():
                out = self.model(torch.FloatTensor(feat_scaled).to(self.device)).cpu().item()
            if not np.isfinite(out):
                print("   ⚠️ NN forward pass produced non-finite output; returning None.")
                return None, None
            lam_pred = float(out)
            if self.use_residual and self.residual_corrector.model is not None:
                res = self.residual_corrector.predict(feat_scaled)[0]
                if np.isfinite(res):
                    lam_pred += float(res)
            lam_pred = float(np.clip(lam_pred, Config.WL_MIN, Config.WL_MAX))
            return None, lam_pred
        except Exception as e:
            print(f"⚠️ ML prediction failed: {e}")
            return None, None

    def explain_prediction(self, mol, pH=None, background_mols=None, nsamples=100):
        if not self.is_trained or self.model is None:
            return None, None, None, None
        if pH is None: pH = Config.DEFAULT_PH
        feat = self.feat_extractor.extract(mol, pH).reshape(1, -1)
        feat_scaled = self.scaler.transform(feat)
        if background_mols is None:
            bg_mols = []
            for fb in self.feedback_db:
                if fb.get('data_role') == 'train' and fb.get('training_eligible', True):
                    smi = fb.get('smiles')
                    if smi:
                        m = Chem.MolFromSmiles(smi)
                        if m is not None:
                            bg_mols.append(m)
                    if len(bg_mols) >= Config.SHAP_BACKGROUND_SIZE:
                        break
            if len(bg_mols) < 2:
                bg_mols = [mol]
        else:
            bg_mols = background_mols
        bg_feats = []
        for m in bg_mols[:Config.SHAP_BACKGROUND_SIZE]:
            try:
                f = self.feat_extractor.extract(m, pH).reshape(1, -1)
                f_scaled = self.scaler.transform(f)
                bg_feats.append(f_scaled[0])
            except:
                continue
        if len(bg_feats) < 2:
            bg_feats = np.random.normal(0, 1, (5, self.current_feature_dim))
        bg_feats = np.array(bg_feats)
        def model_predict(X):
            with torch.no_grad():
                X_torch = torch.FloatTensor(X).to(self.device)
                return self.model(X_torch).cpu().numpy()
        explainer = shap.GradientExplainer(model_predict, bg_feats)
        shap_values = explainer.shap_values(feat_scaled, nsamples=nsamples)
        expected_value = explainer.expected_value
        feature_names = [f"Feature_{i}" for i in range(self.current_feature_dim)]
        shap_abs = np.abs(shap_values[0])
        top_indices = np.argsort(shap_abs)[::-1][:Config.SHAP_MAX_DISPLAY]
        top_features = [(feature_names[i], shap_values[0][i]) for i in top_indices]
        return shap_values[0], expected_value, feature_names, top_features

    # ---------- MODIFIED: _get_feedback_arrays returns smiles_list ----------
    def _get_feedback_arrays(self, role='train'):
        X, y_actual, y_pred, smiles_list = [], [], [], []
        for fb in self.feedback_db:
            if fb.get('data_role') != role: continue
            if not fb.get('training_eligible', True): continue
            features = fb.get('features')
            actual = fb.get('actual')
            predicted = fb.get('predicted')
            smiles = fb.get('smiles')
            if features is None or actual is None or predicted is None or smiles is None: continue
            features = np.asarray(features, dtype=np.float32).reshape(-1)
            if len(features) != self.current_feature_dim: continue
            if not np.all(np.isfinite(features)): continue
            if not np.isfinite(actual): continue
            if not np.isfinite(predicted): continue
            X.append(features)
            y_actual.append(actual)
            y_pred.append(predicted)
            smiles_list.append(smiles)
        return np.asarray(X), np.asarray(y_actual), np.asarray(y_pred), smiles_list

    def retrain(self):
        X_train, y_actual, y_pred, train_smiles_all = self._get_feedback_arrays(role='train')
        if len(X_train) < 20:
            print("Not enough training data for retraining.")
            return
        if len(self.frozen_test_smiles) > 0:
            print(f"⚠️ {len(self.frozen_test_smiles)} TEST molecule(s) are frozen. Retraining only on TRAIN data.")
        X_val, y_actual_val, _, _ = self._get_feedback_arrays(role='validation')
        use_val = len(X_val) >= 10

        # Build training arrays and collect smiles
        X, y_lam, train_smiles_clean = [], [], []
        for i in range(len(X_train)):
            feat = X_train[i]
            if len(feat) == self.current_feature_dim:
                X.append(feat)
                y_lam.append(y_actual[i])
                train_smiles_clean.append(train_smiles_all[i])
        if len(X) < 20:
            return
        X = np.array(X); y_lam = np.array(y_lam, dtype=np.float32)
        shuffle_idx = np.random.permutation(len(X))
        X = X[shuffle_idx]; y_lam = y_lam[shuffle_idx]
        train_smiles_clean = [train_smiles_clean[i] for i in shuffle_idx]

        X_val_use, y_lam_val_use = None, None
        if use_val:
            X_val_use, y_lam_val_use = [], []
            for i in range(len(X_val)):
                feat = X_val[i]
                if len(feat) == self.current_feature_dim:
                    X_val_use.append(feat); y_lam_val_use.append(y_actual_val[i])
            if len(X_val_use) >= 10:
                X_val_use = np.array(X_val_use); y_lam_val_use = np.array(y_lam_val_use, dtype=np.float32)
            else:
                X_val_use = None; y_lam_val_use = None

        # Retrain primary NN
        self.train(X, y_lam, X_val=X_val_use, y_lam_val=y_lam_val_use,
                   epochs=Config.EPOCHS, lr=0.0003, patience=Config.EARLY_STOPPING_PATIENCE)
        print("✅ Primary NN retrained.")

        if len(X_train) >= 20:
            X_fb_raw = X_train
            y_actual_train = y_actual
            y_pred_train = y_pred
            X_fb_scaled = self.scaler.transform(X_fb_raw) if self.scaler else X_fb_raw

            # Solvent learner
            if self.use_dynamic_solvent:
                train_solvents = [fb.get('solvent', 'unknown') for fb in self.feedback_db
                                  if fb.get('data_role') == 'train' and fb.get('training_eligible', True)]
                if len(train_solvents) == len(y_actual_train):
                    shifts = y_actual_train - y_pred_train
                    self.solvent_learner.train(X_fb_scaled, train_solvents, shifts)

            # Ensemble
            if self.use_ensemble and (SKLEARN_AVAILABLE or XGB_AVAILABLE):
                self.ensemble.train(X_fb_scaled, np.asarray(y_actual_train, dtype=np.float32))

            # Outliers
            if self.use_outlier and SKLEARN_AVAILABLE:
                try:
                    self.deep_svdd.train(X_fb_raw)
                    self.deep_svdd_trained = True
                except Exception as e:
                    print(f"⚠️ Deep SVDD failed: {e}")
                self.outlier_detector.train(X_fb_raw)

            self.confidence_scorer = ConfidenceScorer(
                self.feat_extractor, self.feedback_db, self.ensemble,
                self.outlier_detector, self.deep_svdd, knn_k=Config.CONFIDENCE_KNN_K
            )

            # ---- Calibrate NGBoost ----
            if self.ngboost.fitted and len(X_fb_scaled) >= 50:
                try:
                    from sklearn.isotonic import IsotonicRegression
                    from sklearn.model_selection import KFold
                    from ngboost import NGBRegressor
                    from ngboost.distns import Normal
                    oof_ngb = np.zeros(len(X_fb_scaled))
                    kf = KFold(n_splits=5, shuffle=True, random_state=Config.RANDOM_SEED)
                    for train_idx, val_idx in kf.split(X_fb_scaled):
                        X_train_fold = X_fb_scaled[train_idx]
                        y_train_fold = y_actual_train[train_idx]
                        X_val_fold = X_fb_scaled[val_idx]
                        temp_ngb = NGBRegressor(n_estimators=30, learning_rate=0.01,
                                                Dist=Normal, verbose=False, random_state=42)
                        temp_ngb.fit(X_train_fold, y_train_fold)
                        oof_ngb[val_idx] = temp_ngb.predict(X_val_fold).flatten()
                    self.ngboost_calibrator = IsotonicRegression(out_of_bounds='clip')
                    self.ngboost_calibrator.fit(oof_ngb, y_actual_train)
                    print("✅ NGBoost calibrator trained on SCALED features.")
                except Exception as e:
                    print(f"⚠️ NGBoost calibration failed: {e}")
                    self.ngboost_calibrator = None

            # Hückel
            if len(X_train) >= 20:
                print("🧪 Training Hückel predictor...")
                try:
                    train_mols_huck, train_y_huck = [], []
                    for fb in self.feedback_db:
                        if fb.get('data_role') == 'train' and fb.get('training_eligible', True):
                            mol = Chem.MolFromSmiles(fb.get('smiles'))
                            if mol is not None and fb.get('actual') is not None:
                                train_mols_huck.append(mol); train_y_huck.append(fb['actual'])
                    if len(train_mols_huck) >= 10:
                        self.huckel_predictor.train(train_mols_huck, train_y_huck)
                        self.huckel_trained = True
                except Exception as e:
                    print(f"⚠️ Hückel failed: {e}")

            # RadialMLP
            if len(X_train) >= 20:
                print("🧪 Training RadialMLPPredictor...")
                try:
                    train_mols_3d, train_y_3d = [], []
                    for fb in self.feedback_db:
                        if fb.get('data_role') == 'train' and fb.get('training_eligible', True):
                            mol = Chem.MolFromSmiles(fb.get('smiles'))
                            if mol is not None and fb.get('actual') is not None:
                                if mol.GetNumConformers() == 0:
                                    try:
                                        mol_h, _ = ConformerHandler.get_lowest_energy_conformer(mol, num_confs=1)
                                        train_mols_3d.append(mol_h)
                                    except Exception:
                                        continue
                                else:
                                    train_mols_3d.append(mol)
                                train_y_3d.append(fb['actual'])
                    if len(train_mols_3d) >= 10:
                        self.e3nn_predictor.train(train_mols_3d, train_y_3d)
                        if self.e3nn_predictor.fitted:
                            self.e3nn_trained = True
                        else:
                            print("⚠️ RadialMLPPredictor fitted=False.")
                    else:
                        print(f"⚠️ Not enough conformers: {len(train_mols_3d)} (need ≥10).")
                except Exception as e:
                    print(f"⚠️ e3nn failed: {e}")

            # ---- OOF for MetaCombiner and ResidualCorrector (MODIFIED) ----
            print("🔄 Generating OOF predictions...")
            oof_meta_X, _ = self._generate_oof_predictions(
                X, y_lam, train_smiles_clean, n_folds=5
            )

            # Sanity check
            oof_meta_X = np.asarray(oof_meta_X, dtype=np.float64)
            if not np.all(np.isfinite(oof_meta_X)):
                print("⚠️ OOF predictions contain non-finite values; cleaning with column medians.")
                for j in range(oof_meta_X.shape[1]):
                    col = oof_meta_X[:, j]
                    finite = np.isfinite(col)
                    fill = float(np.median(col[finite])) if np.any(finite) else 250.0
                    col[~finite] = fill
                    oof_meta_X[:, j] = np.clip(col, Config.WL_MIN, Config.WL_MAX)

            # ---------------------------------------------------------
            # ACCURACY-FIRST META COMBINER
            # Train only from OOF predictions vs experimental lambda-max.
            # ---------------------------------------------------------
            if oof_meta_X.shape[0] >= 10:
                try:
                    self.meta_combiner.train(oof_meta_X, y_lam)
                    meta_oof_pred = self.meta_combiner.predict(oof_meta_X)
                    meta_oof_pred = np.clip(
                        meta_oof_pred, Config.WL_MIN, Config.WL_MAX
                    )

                    model_names = [
                        "NN", "Ensemble", "DR-CNN", "e3nn", "NGBoost"
                    ]

                    meta_mae = mean_absolute_error(y_lam, meta_oof_pred)
                    meta_rmse = float(np.sqrt(mean_squared_error(y_lam, meta_oof_pred)))
                    meta_bias = float(np.mean(meta_oof_pred - y_lam))
                    best_base_mae = min(
                        mean_absolute_error(y_lam, oof_meta_X[:, j])
                        for j in range(oof_meta_X.shape[1])
                    )

                    if Config.VERBOSE_TRAINING:
                        print("📊 OOF accuracy comparison:")
                        for j, name in enumerate(model_names):
                            p = oof_meta_X[:, j]
                            mae_j = mean_absolute_error(y_lam, p)
                            rmse_j = float(np.sqrt(mean_squared_error(y_lam, p)))
                            bias_j = float(np.mean(p - y_lam))
                            print(
                                f"   {name:8s}: MAE={mae_j:.2f} nm | "
                                f"RMSE={rmse_j:.2f} nm | Bias={bias_j:+.2f} nm"
                            )
                        print(
                            f"   META     : MAE={meta_mae:.2f} nm | "
                            f"RMSE={meta_rmse:.2f} nm | Bias={meta_bias:+.2f} nm"
                        )

                    if meta_mae <= best_base_mae:
                        print("✅ MetaCombiner retained: OOF MAE is at least as good as the best base model.")
                    else:
                        print("⚠️ Meta OOF MAE is worse than the best base model; weights remain the optimizer result, but inspect the OOF comparison.")

                    print("✅ MetaCombiner trained on OOF (5 features).")
                except Exception as e:
                    print(f"⚠️ MetaCombiner training failed: {e}. Using inverse-MAE fallback.")
                    self.meta_combiner.fitted = False
                    self.meta_combiner.fallback_weights = None
            else:
                print("⚠️ Not enough OOF samples for MetaCombiner.")

            # ---------------------------------------------------------
            # ResidualCorrector learns META prediction error, NOT NN error.
            # ---------------------------------------------------------
            if (
                oof_meta_X.shape[0] >= 20
                and self.use_residual
                and self.meta_combiner.fitted
            ):
                try:
                    meta_oof_pred = self.meta_combiner.predict(oof_meta_X)
                    meta_oof_residuals = np.asarray(y_lam, dtype=np.float64) - np.asarray(meta_oof_pred, dtype=np.float64)
                    meta_oof_residuals = np.nan_to_num(
                        meta_oof_residuals,
                        nan=0.0,
                        posinf=0.0,
                        neginf=0.0
                    )
                    meta_oof_residuals = np.clip(meta_oof_residuals, -200.0, 200.0)

                    if hasattr(self, 'X_fb_scaled') and self.X_fb_scaled is not None:
                        self.residual_corrector.train(
                            self.X_fb_scaled, meta_oof_residuals
                        )
                    else:
                        self.residual_corrector.train(
                            X_fb_scaled, meta_oof_residuals
                        )
                    print("✅ ResidualCorrector trained on OOF META residuals (actual − meta).")
                except Exception as e:
                    print(f"⚠️ ResidualCorrector training failed: {e}")
            else:
                print("⚠️ ResidualCorrector not trained: insufficient OOF data or MetaCombiner unavailable.")

            # DR‑CNN
            if self.use_drcnn:
                self.train_drcnn()

            # Conformational corrector
            if self.use_conform:
                X_3d, shifts_3d = [], []
                for fb in self.feedback_db:
                    if fb.get('data_role') == 'train':
                        d3 = fb.get('3d_descriptors')
                        if d3 is not None:
                            X_3d.append(d3); shifts_3d.append(fb['actual'] - fb['predicted'])
                if len(X_3d) >= 5:
                    self.conform_corrector.train(X_3d, shifts_3d)
                    print(f"✅ Conformational corrector trained on {len(X_3d)} shapes.")

    def get_frozen_data_role(self, mol):
        canon_smi = Chem.MolToSmiles(mol, canonical=True)
        if canon_smi in self.frozen_test_smiles:
            return 'test'
        for fb in self.feedback_db:
            if fb.get('smiles') == canon_smi:
                return fb.get('data_role')
        return None

    def receive_feedback(self, mol, predicted_spec, predicted_lambda, actual_lambda, pH, solvent,
                         chromophore_classes=None, quantum_desc=None, data_role='train'):
        canon_smi = Chem.MolToSmiles(mol, canonical=True)
        # Handle TEST lock override
        if canon_smi in self.frozen_test_smiles:
            if data_role != 'test':
                print(f"⚠️ WARNING: You are moving molecule {canon_smi} from TEST to {data_role.upper()}.")
                print("   This will cause DATA LEAKAGE if you later evaluate on the test set.")
                print("   Proceed only if you are absolutely sure.")
                self.frozen_test_smiles.remove(canon_smi)
                idx_to_remove = None
                for i, fb in enumerate(self.feedback_db):
                    if fb.get('smiles') == canon_smi:
                        idx_to_remove = i; break
                if idx_to_remove is not None and idx_to_remove in self.test_indices:
                    self.test_indices.remove(idx_to_remove)
                if idx_to_remove is not None and idx_to_remove in self.validation_indices:
                    self.validation_indices.remove(idx_to_remove)
                print(f"✅ Removed {canon_smi} from TEST lock and indices.")
            else:
                print(f"ℹ️ Molecule {canon_smi} is already TEST. Proceeding with TEST feedback.")
        error = abs(predicted_lambda - actual_lambda)
        desc_3d = None
        try:
            mol_h, conf_id = ConformerHandler.get_lowest_energy_conformer(mol)
            desc_3d = ConformerHandler.compute_3d_descriptors(mol_h, conf_id)
        except Exception as e:
            print(f"⚠️ 3D descriptor computation failed: {e}")
        entry = {
            'smiles': canon_smi,
            'name': mol.GetProp('_Name') if mol.HasProp('_Name') else 'unknown',
            'pH': float(pH),
            'features': self.feat_extractor.extract(mol, pH),
            'predicted': predicted_lambda,
            'actual': actual_lambda,
            'error': error,
            'timestamp': datetime.now().isoformat(),
            'chromophores': chromophore_classes if chromophore_classes is not None else [],
            'solvent': solvent,
            'quantum': quantum_desc if quantum_desc else {},
            '3d_descriptors': desc_3d,
            'data_role': data_role,
            'used_for_training': data_role == 'train',
            'used_for_benchmark': data_role == 'test',
            'model_version': Config.MODEL_VERSION,
            'feature_version': Config.FEATURE_VERSION,
            'training_eligible': data_role == 'train'
        }
        self.feedback_db.append(entry)
        idx = len(self.feedback_db) - 1
        if data_role == 'test':
            self.test_indices.add(idx)
            self.frozen_test_smiles.add(canon_smi)
        elif data_role == 'validation':
            self.validation_indices.add(idx)
        else:  # train
            self.new_feedback_count += 1
            self.error_history.append(error)

            if getattr(self, 'reaction_learner', None) is not None \
               and getattr(self, 'sim_ref', None) is not None:
                confirmed_key = None
                if isinstance(locals().get('confirmed_reaction_key'), str):
                    confirmed_key = confirmed_reaction_key
                try:
                    sp = self.sim_ref.solvent_db.get(solvent)
                    _T = float(getattr(self.sim_ref, 'current_temp', 298.15))
                    self.reaction_learner.record_feedback(
                        mol, sp, pH, _T,
                        actual_lambda=actual_lambda,
                        predicted_lambda=predicted_lambda,
                        confirmed_reaction_key=confirmed_key,
                        confirmed=(confirmed_key is not None),
                    )
                except Exception as _e:
                    print(f"   ⚠️ ReactionLearner feedback failed: {_e}")

            if solvent:
                if solvent not in self.solvent_offsets:
                    self.solvent_offsets[solvent] = 0.0
                delta = float(actual_lambda) - float(predicted_lambda)
                delta = float(np.clip(delta, -50.0, 50.0))
                self.solvent_offsets[solvent] = 0.9 * self.solvent_offsets[solvent] + 0.1 * delta
                self.solvent_offsets[solvent] = float(np.clip(self.solvent_offsets[solvent], -30.0, 30.0))
            if self.chemistry_memory is not None:
                self.chemistry_memory.update(predicted_lambda, actual_lambda, solvent,
                                             chromophore_classes if chromophore_classes else [])
            if self.similarity_index is not None:
                self.similarity_index.add_molecule(mol, lam=actual_lambda, name=entry['name'])
                if len(self.similarity_index.fingerprints) % 10 == 0:
                    self.similarity_index.build()
            if self.new_feedback_count >= Config.RETRAIN_THRESHOLD:
                self.retrain()
                self.new_feedback_count = 0
        print(f"📝 Feedback recorded: error = {error:.1f} nm (role={data_role})")
        self.log_metrics()
        return error

    def compute_full_metrics(self, n_last=None):
        db = self.feedback_db[-n_last:] if (n_last is not None and n_last > 0) else self.feedback_db
        if len(db) < 2:
            return {'n': len(db), 'MAE': None, 'RMSE': None, 'R2': None,
                    'SEP': None, 'RPD': None, 'CCC': None, 'ME': None,
                    'normalized_MAE_score': None}
        y_true = np.array([fb['actual'] for fb in db])
        y_pred = np.array([fb['predicted'] for fb in db])
        n = len(y_true)
        mae = np.mean(np.abs(y_true - y_pred))
        rmse = np.sqrt(np.mean((y_true - y_pred)**2))
        ss_res = np.sum((y_true - y_pred)**2)
        ss_tot = np.sum((y_true - np.mean(y_true))**2)
        r2 = 1.0 - ss_res/ss_tot if ss_tot != 0 else (1.0 if ss_res==0 else 0.0)
        sep = np.sqrt(ss_res / (n - 1)) if n > 1 else None
        sd = np.std(y_true, ddof=1)
        rpd = sd / sep if (sep is not None and sep != 0 and sd != 0) else None
        mean_true = np.mean(y_true); mean_pred = np.mean(y_pred)
        var_true = np.var(y_true, ddof=1); var_pred = np.var(y_pred, ddof=1)
        cov = np.cov(y_true, y_pred, ddof=1)[0,1] if n > 1 else 0
        numerator = 2 * cov
        denominator = var_true + var_pred + (mean_true - mean_pred)**2
        ccc = numerator / denominator if denominator != 0 else (1.0 if numerator==0 else 0.0)
        me = np.mean(y_pred - y_true)
        mae_clipped = min(mae, 400.0)
        norm_score = 1.0 - (mae_clipped / 400.0)
        return {
            'n': n, 'MAE': mae, 'RMSE': rmse, 'R2': r2,
            'SEP': sep, 'RPD': rpd, 'CCC': ccc, 'ME': me,
            'normalized_MAE_score': max(0.0, min(1.0, norm_score))
        }

    def log_metrics(self, n_last=20):
        full = self.compute_full_metrics(n_last)
        if full['MAE'] is None:
            print("Not enough feedback to compute metrics.")
            return
        print(f"[Operational Metrics] MAE: {full['MAE']:.2f} nm, RMSE: {full['RMSE']:.2f} nm, R²: {full['R2']:.3f}, SEP: {full['SEP']:.2f}, RPD: {full['RPD']:.2f}, CCC: {full['CCC']:.3f}, ME: {full['ME']:.2f} nm (last {n_last} feedback records)")
        self.metrics_log.append({
            'timestamp': datetime.now().isoformat(),
            'n_feedbacks': len(self.error_history),
            'mae': full['MAE'], 'rmse': full['RMSE'], 'R2': full['R2'],
            'SEP': full['SEP'], 'RPD': full['RPD'], 'CCC': full['CCC'], 'ME': full['ME'],
            'normalized_MAE_score': full['normalized_MAE_score'],
            'last_n': n_last,
            'model_version': Config.MODEL_VERSION,
            'feature_version': Config.FEATURE_VERSION
        })

    def compute_confidence(self, mol, pH, predicted_spectrum=None, family_analysis=None,
                           solvent='unknown', chromophore_classes=None, quantum_confidence=None):
        if self.confidence_scorer is None:
            return 50.0, "Unknown", "Gray", {}
        return self.confidence_scorer.compute_confidence(
            mol, pH, predicted_spectrum, family_analysis,
            solvent=solvent, chromophore_classes=chromophore_classes,
            similarity_index=self.similarity_index,
            chemistry_memory=self.chemistry_memory,
            quantum_confidence=quantum_confidence,
            ad_score=None
        )

    def evaluate_by_solvent(self):
        results = {}
        eval_entries = [fb for idx, fb in enumerate(self.feedback_db) if idx in self.test_indices or idx in self.validation_indices]
        if not eval_entries: return {}
        for fb in eval_entries:
            sol = fb.get('solvent', 'unknown')
            if sol not in results: results[sol] = {'y_true': [], 'y_pred': []}
            results[sol]['y_true'].append(fb['actual']); results[sol]['y_pred'].append(fb['predicted'])
        metrics_by_sol = {}
        for sol, data in results.items():
            yt = np.array(data['y_true']); yp = np.array(data['y_pred'])
            if len(yt) < 2: continue
            n = len(yt); mae = np.mean(np.abs(yt-yp)); rmse = np.sqrt(np.mean((yt-yp)**2))
            ss_res = np.sum((yt-yp)**2); ss_tot = np.sum((yt-np.mean(yt))**2)
            r2 = 1.0 - ss_res/ss_tot if ss_tot != 0 else 0.0
            sep = np.sqrt(ss_res/(n-1)) if n>1 else None
            sd = np.std(yt, ddof=1); rpd = sd/sep if (sep is not None and sep != 0 and sd != 0) else None
            mean_true, mean_pred = np.mean(yt), np.mean(yp)
            var_true, var_pred = np.var(yt, ddof=1), np.var(yp, ddof=1)
            cov = np.cov(yt, yp, ddof=1)[0,1] if n>1 else 0
            numerator = 2*cov
            denom = var_true + var_pred + (mean_true - mean_pred)**2
            ccc = numerator/denom if denom != 0 else 0.0
            me = np.mean(yp - yt)
            metrics_by_sol[sol] = {'n': n, 'MAE': mae, 'RMSE': rmse, 'R2': r2,
                                   'SEP': sep, 'RPD': rpd, 'CCC': ccc, 'ME': me}
        return metrics_by_sol

    def evaluate_by_wavelength_region(self, bins=[200,300,400,500,600,700,800]):
        eval_entries = [fb for idx, fb in enumerate(self.feedback_db) if idx in self.test_indices or idx in self.validation_indices]
        if not eval_entries: return {}
        results = defaultdict(lambda: {'y_true': [], 'y_pred': []})
        for fb in eval_entries:
            lam = fb['actual']
            for i in range(len(bins)-1):
                if bins[i] <= lam < bins[i+1]:
                    label = f"{bins[i]}-{bins[i+1]}"
                    results[label]['y_true'].append(fb['actual']); results[label]['y_pred'].append(fb['predicted'])
                    break
        metrics_by_reg = {}
        for label, data in results.items():
            yt = np.array(data['y_true']); yp = np.array(data['y_pred'])
            if len(yt) < 2: continue
            n = len(yt); mae = np.mean(np.abs(yt-yp)); rmse = np.sqrt(np.mean((yt-yp)**2))
            ss_res = np.sum((yt-yp)**2); ss_tot = np.sum((yt-np.mean(yt))**2)
            r2 = 1.0 - ss_res/ss_tot if ss_tot != 0 else 0.0
            sep = np.sqrt(ss_res/(n-1)) if n>1 else None
            sd = np.std(yt, ddof=1); rpd = sd/sep if (sep is not None and sep != 0 and sd != 0) else None
            me = np.mean(yp - yt)
            metrics_by_reg[label] = {'n': n, 'MAE': mae, 'RMSE': rmse, 'R2': r2,
                                     'SEP': sep, 'RPD': rpd, 'ME': me}
        return metrics_by_reg

    def evaluate_by_class(self):
        eval_entries = [fb for idx, fb in enumerate(self.feedback_db) if idx in self.test_indices or idx in self.validation_indices]
        if not eval_entries: return {}
        results = defaultdict(lambda: {'y_true': [], 'y_pred': []})
        for fb in eval_entries:
            classes = fb.get('chromophores', [])
            if not classes: classes = ['unknown']
            for cls in classes:
                results[cls]['y_true'].append(fb['actual']); results[cls]['y_pred'].append(fb['predicted'])
        metrics_by_class = {}
        for cls, data in results.items():
            yt = np.array(data['y_true']); yp = np.array(data['y_pred'])
            if len(yt) < 2: continue
            n = len(yt); mae = np.mean(np.abs(yt-yp)); rmse = np.sqrt(np.mean((yt-yp)**2))
            ss_res = np.sum((yt-yp)**2); ss_tot = np.sum((yt-np.mean(yt))**2)
            r2 = 1.0 - ss_res/ss_tot if ss_tot != 0 else 0.0
            sep = np.sqrt(ss_res/(n-1)) if n>1 else None
            sd = np.std(yt, ddof=1); rpd = sd/sep if (sep is not None and sep != 0 and sd != 0) else None
            me = np.mean(yp - yt)
            metrics_by_class[cls] = {'n': n, 'MAE': mae, 'RMSE': rmse, 'R2': r2,
                                     'SEP': sep, 'RPD': rpd, 'ME': me}
        return metrics_by_class

    def train_drcnn(self):
        if not self.use_drcnn: return
        smiles_list, lambdas = [], []
        for fb in self.feedback_db:
            if fb.get('data_role') == 'train' and fb.get('training_eligible', True):
                smi = fb.get('smiles')
                if smi and fb.get('actual') is not None:
                    smiles_list.append(smi); lambdas.append(fb['actual'])
        if len(smiles_list) < 20:
            print("⚠️ Not enough data to train DR‑CNN (need ≥20).")
            return
        self.drcnn_tokenizer = SMILESTokenizer()
        self.drcnn_tokenizer.fit(smiles_list)
        dataset = SMILESDataset(smiles_list, lambdas, self.drcnn_tokenizer, Config.DRCNN_MAX_SMILES_LEN)
        train_size = int(0.85 * len(dataset))
        val_size = len(dataset) - train_size
        train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
        train_loader = DataLoader(train_dataset, batch_size=Config.DRCNN_BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=Config.DRCNN_BATCH_SIZE, shuffle=False)
        vocab_size = self.drcnn_tokenizer.vocab_size
        self.drcnn_model = DeepResidualCNN(
            vocab_size=vocab_size,
            embed_dim=Config.DRCNN_CHAR_DIM,
            filters=Config.DRCNN_HIDDEN_FILTERS,
            kernels=Config.DRCNN_KERNEL_SIZES,
            dropout=Config.DRCNN_DROPOUT,
            max_len=Config.DRCNN_MAX_SMILES_LEN
        ).to(self.device)
        optimizer = optim.Adam(self.drcnn_model.parameters(), lr=Config.DRCNN_LEARNING_RATE)
        criterion = nn.HuberLoss(delta=10.0)
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
        best_val_loss = float('inf')
        for epoch in range(Config.DRCNN_EPOCHS):
            self.drcnn_model.train()
            train_loss = 0.0
            for enc, lam in train_loader:
                enc, lam = enc.to(self.device), lam.to(self.device)
                optimizer.zero_grad()
                pred = self.drcnn_model(enc)
                loss = criterion(pred, lam)
                loss.backward(); optimizer.step()
                train_loss += loss.item()
            train_loss /= len(train_loader)
            self.drcnn_model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for enc, lam in val_loader:
                    enc, lam = enc.to(self.device), lam.to(self.device)
                    pred = self.drcnn_model(enc)
                    loss = criterion(pred, lam)
                    val_loss += loss.item()
            val_loss /= len(val_loader) if len(val_loader) > 0 else 1
            scheduler.step(val_loss)
            if Config.VERBOSE_TRAINING and (epoch+1) % 10 == 0:
                print(f"DR‑CNN Epoch {epoch+1}: Train Loss {train_loss:.4f}, Val Loss {val_loss:.4f}")
            if val_loss < best_val_loss:
                best_val_loss = val_loss
        print("✅ DR‑CNN trained successfully.")

    def predict_drcnn(self, mol):
        if not self.use_drcnn or self.drcnn_model is None or self.drcnn_tokenizer is None:
            return None, None
        try:
            smi = Chem.MolToSmiles(mol)
            enc = self.drcnn_tokenizer.encode(smi, Config.DRCNN_MAX_SMILES_LEN)
            enc_t = torch.tensor(enc, dtype=torch.long).unsqueeze(0).to(self.device)
            with torch.no_grad():
                pred = self.drcnn_model(enc_t).item()
            conf = min(0.8, len(self.feedback_db) / 1000)
            return pred, conf
        except Exception as e:
            print(f"⚠️ DR‑CNN prediction failed: {e}")
            return None, None

    # ---------- MODIFIED: _generate_oof_predictions with e3nn & NGBoost OOF ----------
    def _generate_oof_predictions(self, X, y, smiles_list, n_folds=3):
        from sklearn.model_selection import KFold
        from sklearn.preprocessing import RobustScaler
        from sklearn.ensemble import RandomForestRegressor
        import xgboost as xgb
        try:
            from catboost import CatBoostRegressor
        except ImportError:
            CatBoostRegressor = None

        X_clean, imp = self._clean_features(X)
        y = np.asarray(y, dtype=np.float32).ravel()
        n_samples = len(X_clean)

        oof_nn = np.full(n_samples, np.nan)
        oof_nn_raw = np.full(n_samples, np.nan)
        oof_ensemble = np.full(n_samples, np.nan)
        oof_drcnn = np.full(n_samples, np.nan)
        oof_e3nn = np.full(n_samples, np.nan)
        oof_ngboost = np.full(n_samples, np.nan)

        kf = KFold(n_splits=n_folds, shuffle=True, random_state=Config.RANDOM_SEED)

        for train_idx, val_idx in kf.split(X_clean):
            X_train = X_clean[train_idx]
            X_val = X_clean[val_idx]
            y_train = y[train_idx]

            # ---- Scale fold ----
            scaler_fold = RobustScaler()
            X_train_scaled = scaler_fold.fit_transform(X_train + 1e-9 * np.random.randn(*X_train.shape))
            X_val_scaled = scaler_fold.transform(X_val + 1e-9 * np.random.randn(*X_val.shape))

            # ---- Neural Network fold ----
            fold_model = UVVisNN(self.current_feature_dim, Config.HIDDEN_LAYERS, Config.DROPOUT).to(self.device)
            optimizer = optim.Adam(fold_model.parameters(), lr=0.001)
            criterion = nn.HuberLoss(delta=10.0)
            dataset = TensorDataset(torch.FloatTensor(X_train_scaled), torch.FloatTensor(y_train))
            loader = DataLoader(dataset, batch_size=min(Config.BATCH_SIZE, len(X_train)), shuffle=True)
            fold_model.train()
            try:
                for _ in range(10):
                    for bx, by in loader:
                        bx, by = bx.to(self.device), by.to(self.device)
                        optimizer.zero_grad()
                        loss = criterion(fold_model(bx), by)
                        loss.backward(); optimizer.step()
            except Exception:
                pass

            fold_model.eval()
            # ---------------------------------------------------------
            # RAW NN OOF PREDICTION
            # ---------------------------------------------------------
            with torch.no_grad():
                try:
                    nn_pred_raw = (
                        fold_model(
                            torch.FloatTensor(X_val_scaled).to(self.device)
                        )
                        .cpu()
                        .numpy()
                        .flatten()
                    )
                except Exception:
                    nn_pred_raw = np.full(len(X_val), np.nan)

            oof_nn_raw[val_idx] = nn_pred_raw

            # ---------------------------------------------------------
            # IMPORTANT: MetaCombiner learns the RAW NN OOF prediction.
            # Residual correction is applied AFTER MetaCombiner at inference.
            # This keeps training and inference mathematically aligned.
            # ---------------------------------------------------------
            nn_pred_raw = np.nan_to_num(
                nn_pred_raw,
                nan=np.nan,
                posinf=np.nan,
                neginf=np.nan
            )
            oof_nn[val_idx] = np.clip(
                nn_pred_raw,
                Config.WL_MIN,
                Config.WL_MAX
            )

            # ---- Ensemble fold ----
            rf = RandomForestRegressor(n_estimators=30, max_depth=10, random_state=42)
            xgb_model = xgb.XGBRegressor(n_estimators=30, max_depth=6, random_state=42)
            cat = CatBoostRegressor(iterations=30, depth=6, learning_rate=0.1, verbose=False, random_seed=42) if CatBoostRegressor else None
            try:
                rf.fit(X_train_scaled, y_train)
                xgb_model.fit(X_train_scaled, y_train)
                if cat:
                    cat.fit(X_train_scaled, y_train)
            except Exception:
                pass

            preds = []
            try:
                preds.append(rf.predict(X_val_scaled))
            except: pass
            try:
                preds.append(xgb_model.predict(X_val_scaled))
            except: pass
            if cat:
                try:
                    preds.append(cat.predict(X_val_scaled))
                except: pass
            if preds:
                ens_pred = np.mean(preds, axis=0)
            else:
                ens_pred = np.full(len(X_val), np.nan)
            oof_ensemble[val_idx] = ens_pred

            # ---- DR‑CNN fold ----
            try:
                train_smiles = [smiles_list[i] for i in train_idx if i < len(smiles_list)]
                val_smiles = [smiles_list[i] for i in val_idx if i < len(smiles_list)]
                fold_tokenizer = SMILESTokenizer()
                fold_tokenizer.fit(train_smiles)
                vocab_size = fold_tokenizer.vocab_size
                fold_drcnn = DeepResidualCNN(
                    vocab_size=vocab_size,
                    embed_dim=Config.DRCNN_CHAR_DIM,
                    filters=Config.DRCNN_HIDDEN_FILTERS,
                    kernels=Config.DRCNN_KERNEL_SIZES,
                    dropout=Config.DRCNN_DROPOUT,
                    max_len=Config.DRCNN_MAX_SMILES_LEN
                ).to(self.device)
                optimizer_drcnn = optim.Adam(fold_drcnn.parameters(), lr=0.001)
                criterion_drcnn = nn.HuberLoss(delta=10.0)
                fold_drcnn.train()
                for epoch in range(3):
                    for smi, target in zip(train_smiles, y_train[:len(train_smiles)]):
                        enc = fold_tokenizer.encode(smi, Config.DRCNN_MAX_SMILES_LEN)
                        enc_t = torch.tensor(enc, dtype=torch.long).unsqueeze(0).to(self.device)
                        target_t = torch.tensor([target], dtype=torch.float32).to(self.device)
                        optimizer_drcnn.zero_grad()
                        pred = fold_drcnn(enc_t)
                        loss = criterion_drcnn(pred, target_t)
                        loss.backward(); optimizer_drcnn.step()
                fold_drcnn.eval()
                drcnn_preds = []
                for smi in val_smiles:
                    enc = fold_tokenizer.encode(smi, Config.DRCNN_MAX_SMILES_LEN)
                    enc_t = torch.tensor(enc, dtype=torch.long).unsqueeze(0).to(self.device)
                    with torch.no_grad():
                        pred = fold_drcnn(enc_t).cpu().item()
                    drcnn_preds.append(pred)
                oof_drcnn[val_idx] = np.array(drcnn_preds)
            except Exception as e:
                print(f"   ⚠️ OOF DR-CNN fold failed: {e}")
                oof_drcnn[val_idx] = np.nan

            # ---- e3nn fold ----
            try:
                # Prepare training molecules for this fold
                train_mols_e3nn = []
                train_targets_e3nn = []
                for pos, idx_fold in enumerate(train_idx):
                    if idx_fold < len(smiles_list):
                        mol = Chem.MolFromSmiles(smiles_list[idx_fold])
                        if mol is not None:
                            if mol.GetNumConformers() == 0:
                                try:
                                    mol_h, _ = ConformerHandler.get_lowest_energy_conformer(mol, num_confs=1)
                                    train_mols_e3nn.append(mol_h)
                                    train_targets_e3nn.append(y_train[pos])
                                except Exception:
                                    continue
                            else:
                                train_mols_e3nn.append(mol)
                                train_targets_e3nn.append(y_train[pos])
                if len(train_mols_e3nn) >= 5:
                    fold_e3nn = RadialMLPPredictor()
                    fold_e3nn.train(train_mols_e3nn, train_targets_e3nn)
                    if fold_e3nn.fitted:
                        e3nn_preds = []
                        for idx_fold in val_idx:
                            if idx_fold < len(smiles_list):
                                mol = Chem.MolFromSmiles(smiles_list[idx_fold])
                                if mol is not None:
                                    if mol.GetNumConformers() == 0:
                                        try:
                                            mol_h, _ = ConformerHandler.get_lowest_energy_conformer(mol, num_confs=1)
                                            pred, _ = fold_e3nn.predict(mol_h)
                                        except Exception:
                                            pred = np.nan
                                    else:
                                        pred, _ = fold_e3nn.predict(mol)
                                    e3nn_preds.append(pred if pred is not None else np.nan)
                                else:
                                    e3nn_preds.append(np.nan)
                            else:
                                e3nn_preds.append(np.nan)
                        oof_e3nn[val_idx] = np.array(e3nn_preds)
                    else:
                        oof_e3nn[val_idx] = np.nan
                else:
                    oof_e3nn[val_idx] = np.nan
            except Exception as e:
                print(f"   ⚠️ OOF e3nn fold failed: {e}")
                oof_e3nn[val_idx] = np.nan

            # ---- NGBoost fold ----
            try:
                from ngboost import NGBRegressor
                from ngboost.distns import Normal
                temp_ngb = NGBRegressor(n_estimators=30, learning_rate=0.01,
                                        Dist=Normal, verbose=False, random_state=42)
                temp_ngb.fit(X_train_scaled, y_train)
                ngb_pred = temp_ngb.predict(X_val_scaled).flatten()
                oof_ngboost[val_idx] = ngb_pred
            except Exception as e:
                print(f"   ⚠️ OOF NGBoost fold failed: {e}")
                oof_ngboost[val_idx] = np.nan

        # ---- Replace any NaN with column mean ----
        for col in [oof_nn, oof_ensemble, oof_drcnn, oof_e3nn, oof_ngboost]:
            nan_mask = np.isnan(col)
            if np.any(nan_mask):
                mean_val = np.nanmean(col)
                col[nan_mask] = mean_val if not np.isnan(mean_val) else 0.0

        # ---------------------------------------------------------
        # BUILD META FEATURES
        # Exactly five OOF base-model predictions.
        # Analogue prediction is NOT a training feature.
        # ---------------------------------------------------------
        oof_meta_X = np.column_stack([
            oof_nn,
            oof_ensemble,
            oof_drcnn,
            oof_e3nn,
            oof_ngboost
        ])

        oof_meta_X = np.nan_to_num(
            oof_meta_X,
            nan=np.nan,
            posinf=np.nan,
            neginf=np.nan
        )

        # Return only OOF base-model predictions. Residuals are calculated
        # AFTER MetaCombiner is trained so ResidualCorrector learns the
        # actual error of the deployed meta prediction.
        return oof_meta_X, None

# ---------- HydrogenBondAnalyzer ----------
class HydrogenBondAnalyzer:

    DONOR_PATTERNS = [
        ("alcohol_OH",   Chem.MolFromSmarts("[OX2H]"),                 "strong"),
        ("phenol_OH",    Chem.MolFromSmarts("c1ccccc1[OX2H]"),         "moderate"),
        ("amine_NH",     Chem.MolFromSmarts("[NX3;H1,H2;!$(NC=O)]"),   "strong"),
        ("amide_NH",     Chem.MolFromSmarts("[NX3;H1;$(NC=O)]"),       "weak"),
        ("thiol_SH",     Chem.MolFromSmarts("[SX2H]"),                  "moderate"),
        ("carboxyl_OH",  Chem.MolFromSmarts("C(=O)[OX2H]"),             "strong"),
        ("sulfonamide",  Chem.MolFromSmarts("S(=O)(=O)[NX3;H1,H2]"),   "moderate"),
    ]
    ACCEPTOR_PATTERNS = [
        ("carbonyl_O",   Chem.MolFromSmarts("[OX1]=C"),                 "strong"),
        ("ether_O",      Chem.MolFromSmarts("[OX2;!$([OX2H])]"),        "moderate"),
        ("pyridine_N",   Chem.MolFromSmarts("[n]"),                     "moderate"),
        ("tertiary_N",   Chem.MolFromSmarts("[NX3;H0;!$(NC=O)]"),       "strong"),
        ("nitro_O",      Chem.MolFromSmarts("[O-][N+](=O)"),            "weak"),
        ("sulfoxide_O",  Chem.MolFromSmarts("[SX3]=O"),                 "moderate"),
        ("thioether_S",  Chem.MolFromSmarts("[SX2;!$([SX2H])]"),        "weak"),
    ]

    @classmethod
    def analyze(cls, mol: Chem.Mol, solvent_props: dict) -> dict:
        donor_hits, acceptor_hits = [], []
        for name, patt, strength in cls.DONOR_PATTERNS:
            if patt is not None and mol.HasSubstructMatch(patt):
                donor_hits.append((name, strength))
        for name, patt, strength in cls.ACCEPTOR_PATTERNS:
            if patt is not None and mol.HasSubstructMatch(patt):
                acceptor_hits.append((name, strength))

        solv_d = float(solvent_props.get("hbond_donor", 0.0))
        solv_a = float(solvent_props.get("hbond_acceptor", 0.0))

        # Conditional: BOTH sides must have capability
        donor_activity    = min(1.0, solv_a) if donor_hits    else 0.0
        acceptor_activity = min(1.0, solv_d) if acceptor_hits else 0.0
        hbond_activity    = 0.5 * (donor_activity + acceptor_activity)

        return {
            "solute_donors":               [n for n, _ in donor_hits],
            "solute_acceptors":            [n for n, _ in acceptor_hits],
            "solvent_donor_capability":    solv_d,
            "solvent_acceptor_capability": solv_a,
            "hbond_activity":              float(np.clip(hbond_activity, 0.0, 1.0)),
            "is_non_covalent":             True,
            "assumed_reaction":            False,
        }

# ---------- UVVisSimulator (MODIFIED meta-combiner fallback) ----------
class UVVisSimulator:
    def __init__(self):
        self.instr = InstrumentParameters()
        self.solvent_db = SolventDatabase()
        self.chromo = ChromophoreDetector()
        self.spectrum_gen = SpectrumGenerator()
        self.feature_extractor = FeatureExtractor()
        self.predictor = SelfLearningPredictor(self.feature_extractor)
        self.predictor.sim_ref = self
        self.predictor.reaction_learner = None   # created lazily after model load
        self.current_solvent = "water"
        self.current_ph = Config.DEFAULT_PH
        self.current_temp = Config.DEFAULT_TEMP
        self.current_conc = Config.DEFAULT_CONC
        self.blank_spectrum = None
        self.blank_need_update = True
        self.update_blank()
        self.family_classifier = None
        self._last_family_analysis = None
        self.similarity_index = SimilarityIndex()
        self.chemistry_memory = ChemistryMemory()
        self._last_quantum_confidence = 0.0
        self._last_3d_desc = None
        self._last_primary_peak = None
        self._last_interaction_report = None
        self._last_solvent_shift_mechanism = {}
        self.kt_model = KamletTaftModel()
        self.ad_model = ApplicabilityDomain()
        self.predictor.similarity_index = self.similarity_index
        self.empirical_corrector = EmpiricalRuleCorrector()
        self.predictor.chemistry_memory = self.chemistry_memory
        self._conformer_cache = {}
        self._chromophore_cache = {}
        self._feat_cache = {}

    def update_blank(self):
        zero_spec = np.zeros_like(Config.wavelengths)
        self.blank_spectrum = self.instr.apply_instrument_effects(zero_spec)
        self.blank_need_update = False

    def set_solvent(self, code):
        props = self.solvent_db.set_solvent(code)
        self.current_solvent = self.solvent_db.current_solvent
        self.current_ph = self.solvent_db.current_ph

    def shift_spectrum(self, spec, shift_nm):
        if abs(shift_nm) < 0.5: return spec
        shift_steps = shift_nm / Config.WL_STEP
        shifted = shift(spec, shift_steps, mode='nearest')
        return np.maximum(shifted, 0)

    def _fit_kt_model_from_feedback(self):
        fb = self.predictor.feedback_db
        if len(fb) < 10: return
        X_list, y_list = [], []
        for entry in fb:
            if entry.get('data_role') != 'train': continue
            solvent = entry.get('solvent', 'water')
            props = self.solvent_db.get(solvent)
            if props is None: continue
            pi = props.get('pi_star'); alp = props.get('alpha'); bet = props.get('beta')
            if pi is None or alp is None or bet is None: continue
            shift_target = entry['actual'] - entry['predicted']
            X_list.append([pi, alp, bet]); y_list.append(shift_target)
        if len(X_list) >= 10:
            X = np.array(X_list); y = np.array(y_list)
            self.kt_model.fit(X, y, alpha_reg=0.1)
            print("✅ Kamlet–Taft model fitted on training data.")

    def _fit_ad_model_from_feedback(self):
        train_fb = [fb for fb in self.predictor.feedback_db if fb.get('data_role') == 'train']
        if len(train_fb) >= 5:
            self.ad_model.fit(train_fb)
            print("✅ Applicability Domain fitted on training data.")

    def build_interaction_report(self, mol, solvent_props=None, hbond=None, shift_mech=None):

        if solvent_props is None:
            solvent_props = self.solvent_db.get(self.current_solvent)

        dipole_D = None
        if self._last_3d_desc is not None:
            dipole_D = self._last_3d_desc.get("dipole", None)
        dielectric      = float(solvent_props.get("dielectric", 80.1))
        is_electrolyte  = bool(solvent_props.get("is_electrolyte", False))
        ionic_strength  = float(solvent_props.get("ionic_strength_M", 0.0))

        hb = hbond if hbond is not None else {"hbond_activity": 0.0,
                                              "solute_donors": [],
                                              "solute_acceptors": [],
                                              "is_non_covalent": True,
                                              "assumed_reaction": False}
        sm = shift_mech if shift_mech is not None else {}

        modeled, approximated, omitted = [], [], []

        # H-bond (conditional, non-covalent)
        if hb.get("hbond_activity", 0.0) > 0.05:
            modeled.append(
                f"H-bond (non-covalent): solute donors={hb.get('solute_donors')} "
                f"acceptors={hb.get('solute_acceptors')}; solvent D="
                f"{hb.get('solvent_donor_capability'):.2f} A="
                f"{hb.get('solvent_acceptor_capability'):.2f}"
            )
        else:
            omitted.append("H-bond: no compatible solute/solvent donor–acceptor "
                           "pair (conditional — not assumed).")

        # Electronic solvation
        if sm.get("kt_applied"):
            modeled.append("Electronic solvation: Kamlet–Taft (π*, α, β).")
        else:
            omitted.append("Electronic solvation: Kamlet–Taft shift ≈ 0.")

        # Residual solvent effects
        if sm.get("ml_learner_applied"):
            modeled.append("Solvent residual (H-bond-specific / conformational): "
                           "ML solvent learner.")
        elif sm.get("feedback_offset_applied"):
            approximated.append("Solvent residual: feedback offset "
                                "(running mean, no ML learner yet).")

        # Conformational sampling
        if self.predictor.conform_corrector.model is not None:
            modeled.append(
                f"Conformation: Boltzmann-averaged ensemble "
                f"(N={Config.BOLTZMANN_NUM_CONFS}, T={Config.BOLTZMANN_TEMP} K)."
            )
        else:
            omitted.append("Conformation: corrector not yet trained — "
                           "single-conformer prediction.")

        # Electrolyte
        if is_electrolyte:
            approximated.append(
                f"Electrolyte: {solvent_props.get('name')} treated as water "
                f"matrix (I={ionic_strength:.3f} M; ions={solvent_props.get('ions', {})}). "
                f"Ion-specific λmax terms are NOT in the active model."
            )

        # Dipole
        if dipole_D is not None:
            modeled.append(f"Dipole moment: {float(dipole_D):.2f} D "
                           f"(dielectric ε={dielectric:.1f}).")
        else:
            approximated.append("Dipole: 3D descriptor unavailable — "
                                "electrostatic response approximated via ε only.")

        # Dispersion
        if self.predictor.conform_corrector.model is not None or self.kt_model.fitted:
            modeled.append("Dispersion / non-covalent: captured by trained "
                           "corrector + KT prior.")
        else:
            omitted.append("Dispersion: not yet modelled (train more correctors).")

        # Reaction pathway
        if not Config.REACTION_ENABLED:
            omitted.append("Chemical reaction: disabled by config "
                           "(non-reactive pathway only).")

        n_m, n_o = len(modeled), len(omitted)
        n_t = max(1, n_m + n_o + len(approximated))
        reliability = float(np.clip(0.5 + 0.5 * (n_m - n_o) / n_t, 0.0, 1.0))

        return {
            "solvent": self.current_solvent,
            "solvent_name": solvent_props.get("name", self.current_solvent),
            "is_electrolyte": is_electrolyte,
            "ionic_strength_M": ionic_strength,
            "hydrogen_bond": hb,
            "solvent_shift_mechanism": sm,
            "modeled": modeled,
            "approximated": approximated,
            "omitted": omitted,
            "reliability": reliability,
        }

    def predict(self, mol, name, use_tddft=False, extract_quantum=False):
        if self.family_classifier is None:
            self.family_classifier = ChromophoreFamilyClassifier()
        try:
            if mol is None:
                return np.zeros_like(Config.wavelengths), "error", [], [], {}, None

            # Take largest fragment
            frags = Chem.GetMolFrags(mol, asMols=True)
            if len(frags) > 1:
                mol = max(frags, key=lambda m: m.GetNumAtoms())

            # --- SOLVOLYSIS (adaptive reaction model) ---
            self._last_reaction_candidates = []
            if Config.REACTION_ENABLED:
                try:
                    if getattr(self.predictor, 'reaction_learner', None) is None:
                        self.predictor.reaction_learner = ReactionLearner(
                            model_path="reaction_feasibility.pkl",
                            engine=ReactionEngine(),
                            chromo=self.chromo
                        )
                    _sp = self.solvent_db.get(self.current_solvent)
                    _score, _rxn_key = self.predictor.reaction_learner.predict(
                        mol, _sp, self.current_ph, self.current_temp,
                        engine=self.predictor.reaction_learner.engine
                    )
                    if _score >= Config.REACTION_THRESHOLD and _rxn_key:
                        _prods = self.predictor.reaction_learner.engine.apply(
                            mol, _sp.get("smiles"), only_key=_rxn_key,
                            max_products=Config.REACTION_MAX_PRODUCTS
                        )
                        self._last_reaction_candidates = [
                            {"mol": p, "name": n, "key": k, "score": _score}
                            for (p, n, k) in _prods
                        ]
                        if _prods:
                            print(f"   🧪 Possible reaction (heuristic score={_score:.2f}): "
                                  f"{_prods[0][1]} (parent preserved)")
                        else:
                            print(f"   ℹ️ Score {_score:.2f} but SMARTS produced no product — ignoring.")
                    else:
                        print(f"   ℹ️ No reaction candidate (score={_score:.2f}).")
                except Exception as _e:
                    print(f"   ⚠️ Reaction analysis failed: {_e}")
            # --- end reaction analysis ---

            # --- Conditional H-bond analysis (non-covalent ---
            _solvent_props_for_hb = self.solvent_db.get(self.current_solvent)
            try:
                hbond_info = HydrogenBondAnalyzer.analyze(mol, _solvent_props_for_hb)
            except Exception:
                hbond_info = {"hbond_activity": 0.0, "solute_donors": [],
                              "solute_acceptors": [], "is_non_covalent": True,
                              "assumed_reaction": False}

            # TDDFT
            tddft_peaks, quantum_desc, td_success = [], {}, False
            if use_tddft:
                props = self.solvent_db.get(self.current_solvent)
                eps = props.get('dielectric', None)
                tddft_peaks, quantum_desc, td_success = PySCFPredictor.predict(
                    mol, solvent_eps=eps,
                    extract_descriptors=extract_quantum or Config.EXTRACT_QUANTUM_DESCRIPTORS
                )
            # Analogue
            self.similarity_index.ensure_built()
            analogues = self.similarity_index.query(mol, k=5)
            self._last_analogues = analogues
            heavy_atoms = mol.GetNumHeavyAtoms()
            filtered_analogues = []
            for smi, lam, sim, name_ana in analogues:
                try:
                    ana_mol = Chem.MolFromSmiles(smi)
                    if ana_mol is not None:
                        ana_heavy = ana_mol.GetNumHeavyAtoms()
                        if sim > 0.4 or (0.6 * heavy_atoms <= ana_heavy <= 1.5 * heavy_atoms):
                            filtered_analogues.append((smi, lam, sim, name_ana))
                except:
                    continue
            analogue_prediction, analogue_similarity, analogue_name = None, 0.0, None
            if filtered_analogues and filtered_analogues[0][2] > 0.400:
                analogue_similarity = filtered_analogues[0][2]
                analogue_lambda = filtered_analogues[0][1]
                self._last_analogue_lambda = analogue_lambda
                analogue_name = filtered_analogues[0][3] if filtered_analogues[0][3] != 'unknown' else None
                analogue_prediction = analogue_lambda
            # Chromophore detection
            canon_smi = Chem.MolToSmiles(mol, canonical=True)
            if canon_smi in self._chromophore_cache:
                peaks, chrom_classes = self._chromophore_cache[canon_smi]
            else:
                peaks, chrom_classes = self.chromo.detect(mol, name, self.current_ph,
                                                          td_dft_peaks=tddft_peaks if td_success else None)
                self._chromophore_cache[canon_smi] = (peaks, chrom_classes)

            if not peaks:
                spec = np.full_like(Config.wavelengths, 1e-7, dtype=float)
                return spec, "no_absorption", [], chrom_classes, {}, 0
            # Chemistry Memory correction
            memory_correction = 0.0
            if self.chemistry_memory is not None and self.chemistry_memory.global_stats.get('count', 0) > 5:
                mem_classes = []
                for c in chrom_classes:
                    if 'Base:' in c or 'Conjugation' in c or 'ICT' in c:
                        mem_classes.append(c)
                if not mem_classes:
                    mem_classes = ['conjugated'] if ConjugationAnalyzer.longest_conjugated_path(mol) > 3 else ['simple']
                familiarity = self.chemistry_memory.get_familiarity_score(self.current_solvent, mem_classes)
                if analogue_similarity < 0.75 and familiarity > 0.3:
                    if ConjugationAnalyzer.longest_conjugated_path(mol) > 4:
                        memory_correction = 15.0 * familiarity
                        print(f"   🧠 Chemistry Memory: +{memory_correction:.0f} nm (familiarity={familiarity:.2f})")
            # ML prediction
            ml_spec, ml_lambda = self.predictor.predict(mol, self.current_ph)
            ml_peak = None
            if ml_spec is not None:
                ml_peak = Config.WL_MIN + np.argmax(ml_spec) * Config.WL_STEP
                if ml_lambda is not None and 200 <= ml_lambda <= 800:
                    pass
                if analogue_prediction is None or analogue_similarity < 0.4:
                    peaks.append((ml_peak, 0.6, 30))
                    chrom_classes.append("ml_predicted")
            if analogue_prediction is not None and analogue_similarity > 0.4:
                peaks.append((analogue_prediction, 1.2, 30))
                chrom_classes.append(f"analogue_{analogue_name or 'unknown'}")
                print(f"   ✅ Using analogue λmax: {analogue_prediction:.0f} nm (sim={analogue_similarity:.3f})")
            family_analysis = self.family_classifier.get_summary(mol)
            self._last_family_analysis = family_analysis
            # Solvent shifts
            chrom_type = "general"
            if any(c in chrom_classes for c in ["benzene", "naphthalene", "anthracene", "conjugated_system"]):
                chrom_type = "π→π*"
            elif any(c in chrom_classes for c in ["carbonyl", "ketone", "aldehyde", "nitro", "azo"]):
                chrom_type = "n→π*"
            else:
                chrom_type = "general"
            props = self.solvent_db.get(self.current_solvent)
            pred_shift = self.kt_model.predict(props, self.current_temp)
            if pred_shift is not None:
                kt_shift = pred_shift
            else:
                kt_shift = self.solvent_db.compute_kamlet_taft_shift(props, self.current_temp, chrom_type)
            if chrom_type == "n→π*":
                solvent_scale = 0.6
            elif chrom_type == "ICT" or any('ICT' in c for c in chrom_classes):
                solvent_scale = 1.2
            else:
                solvent_scale = 1.0

            _shift_mech = {"kt_applied": False,
                           "ml_learner_applied": False,
                           "feedback_offset_applied": False}

            total_shift = float(kt_shift)
            _shift_mech["kt_applied"] = abs(kt_shift) > 0.01

            residual_shift = 0.0
            if (self.predictor.use_dynamic_solvent
                    and self.predictor.solvent_learner.model is not None):
                try:
                    feat = self.feature_extractor.extract(mol, self.current_ph).reshape(1, -1)
                    feat_scaled = self.predictor.scaler.transform(feat) if self.predictor.scaler else feat
                    dyn_shift = self.predictor.solvent_learner.predict_shift(
                        feat_scaled, self.current_solvent
                    )
                    if np.isfinite(dyn_shift):
                        residual_shift = float(dyn_shift)
                        _shift_mech["ml_learner_applied"] = abs(residual_shift) > 0.01
                except Exception:
                    residual_shift = 0.0
            elif self.current_solvent in self.predictor.solvent_offsets:
                offset_raw = float(self.predictor.solvent_offsets.get(self.current_solvent, 0.0))
                residual_shift = offset_raw * float(solvent_scale)
                _shift_mech["feedback_offset_applied"] = abs(residual_shift) > 0.01

            total_shift += residual_shift
            self._last_solvent_shift_mechanism = _shift_mech
            conj_len = ConjugationAnalyzer.longest_conjugated_path_with_hbond(mol)
            max_secondary_shift = min(15.0, 2.0 * np.sqrt(max(1, conj_len)))
            total_shift = np.clip(total_shift, -max_secondary_shift, max_secondary_shift)
            shifted = [(wl + total_shift, inten, width) for wl, inten, width in peaks]
            # Candidate ranking
            candidates = []
            if ml_spec is not None and np.max(ml_spec) > 1e-6:
                nn_peak = Config.WL_MIN + np.argmax(ml_spec)  * Config.WL_STEP
                nn_peak_shifted = nn_peak + total_shift
                candidates.append({'source': 'NN_spectrum', 'lambda': nn_peak_shifted, 'confidence': 0.7})
            if ml_lambda is not None and 200 <= ml_lambda <= 800:
                nn_lambda_shifted = ml_lambda + total_shift
                candidates.append({'source': 'NN_lambda', 'lambda': nn_lambda_shifted, 'confidence': 0.7})

            if self.predictor.use_ensemble and self.predictor.ensemble.trained:
                if 'feat_scaled' not in locals():
                  canon_smi = Chem.MolToSmiles(mol, canonical=True)
                  if canon_smi in self._feat_cache:
                      feat_scaled = self._feat_cache[canon_smi]
                  else:
                      feat = self.feature_extractor.extract(mol, self.current_ph).reshape(1, -1)
                      feat = np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0)
                      feat_scaled = self.predictor.scaler.transform(feat) if self.predictor.scaler else feat
                      self._feat_cache[canon_smi] = feat_scaled

                if np.any(np.isnan(feat_scaled)):
                  feat_scaled = np.nan_to_num(feat_scaled, nan=0.0)

                if self.predictor.ensemble.rf is not None:
                  try:
                      # RF is a REGRESSOR producing one scalar lambda value.
                      # Do NOT treat the scalar as a wavelength spectrum.
                      rf_lambda = float(
                          np.asarray(
                              self.predictor.ensemble.rf.predict(feat_scaled)
                          ).ravel()[0]
                      )

                      if np.isfinite(rf_lambda):
                          rf_lambda = float(
                              np.clip(
                                  rf_lambda,
                                  Config.WL_MIN,
                                  Config.WL_MAX
                              )
                          )

                          rf_lambda_shifted = rf_lambda + total_shift
                          rf_lambda_shifted = float(
                              np.clip(
                                  rf_lambda_shifted,
                                  Config.WL_MIN,
                                  Config.WL_MAX
                              )
                          )

                          candidates.append({
                              'source': 'RF',
                              'lambda': rf_lambda_shifted,
                              'confidence': 0.6
                          })

                  except Exception as e:
                      print(f"   ⚠️ RF candidate failed: {e}")


                if self.predictor.ensemble.xgb is not None:
                  try:
                      # XGBoost is also a REGRESSOR producing one scalar lambda value.
                      # Do NOT use np.argmax() here.
                      xgb_lambda = float(
                          np.asarray(
                              self.predictor.ensemble.xgb.predict(feat_scaled)
                          ).ravel()[0]
                      )

                      if np.isfinite(xgb_lambda):
                          xgb_lambda = float(
                              np.clip(
                                  xgb_lambda,
                                  Config.WL_MIN,
                                  Config.WL_MAX
                              )
                          )

                          xgb_lambda_shifted = xgb_lambda + total_shift
                          xgb_lambda_shifted = float(
                              np.clip(
                                  xgb_lambda_shifted,
                                  Config.WL_MIN,
                                  Config.WL_MAX
                              )
                          )

                          candidates.append({
                              'source': 'XGB',
                              'lambda': xgb_lambda_shifted,
                              'confidence': 0.6
                          })

                  except Exception as e:
                      print(f"   ⚠️ XGB candidate failed: {e}")

                # ---- NGBoost prediction ----
                if self.predictor.ngboost.fitted:
                    ngb_mean, ngb_std = self.predictor.ngboost.predict(feat_scaled)
                    if ngb_mean is not None:
                        ngb_raw = float(ngb_mean[0])
                        if self.predictor.ngboost_calibrator is not None:
                            try:
                                ngb_calibrated = float(self.predictor.ngboost_calibrator.predict([[ngb_raw]])[0])
                            except Exception:
                                ngb_calibrated = ngb_raw
                        else:
                            ngb_calibrated = ngb_raw
                        ngb_lambda = np.clip(ngb_calibrated, 200, 800)
                        if 200 <= ngb_lambda <= 800:
                            conf = max(0.5, 1.0 - float(ngb_std[0]) / 50.0)
                            candidates.append({'source': 'ngboost', 'lambda': ngb_lambda, 'confidence': conf})
                            print(f"   📊 NGBoost calibrated: {ngb_lambda:.0f} nm (raw={ngb_raw:.0f}, std={ngb_std[0]:.1f})")

            if analogue_prediction is not None:
                shifted_analogue = analogue_prediction + total_shift
                if analogue_similarity > 0.8:
                  analogue_conf = 1.0
                elif analogue_similarity > 0.4:
                  analogue_conf = 0.90
                elif analogue_similarity > 0.6:
                  analogue_conf = 0.9
                else: analogue_conf = analogue_similarity * 0.8
                candidates.append({'source': 'analogue', 'lambda': shifted_analogue,
                                   'similarity': analogue_similarity, 'confidence': analogue_conf})

            if shifted:
                sorted_by_intensity = sorted(shifted, key=lambda x: x[1], reverse=True)
                for i, (lam, inten, width) in enumerate(sorted_by_intensity[:2]):
                    candidates.append({'source': f'chromophore_{i}', 'lambda': lam,
                                       'intensity': inten, 'confidence': min(1.0, inten * 0.5)})
            if td_success and tddft_peaks:
                for lam, f, _ in tddft_peaks:
                    candidates.append({'source': 'tddft', 'lambda': lam,
                                       'oscillator': f, 'confidence': min(1.0, f * 2.0)})

            # ---- Add DR‑CNN predictions to candidates ----
            drcnn_pred, drcnn_conf = self.predictor.predict_drcnn(mol)
            if drcnn_pred is not None and 200 <= drcnn_pred <= 800:
                candidates.append({'source': 'drcnn', 'lambda': drcnn_pred,
                                   'confidence': drcnn_conf if drcnn_conf else 0.5})
                print(f"   🧠 DR‑CNN prediction: {drcnn_pred:.0f} nm (conf={drcnn_conf:.2f})")

            # ---- e3nn prediction ----
            e3nn_pred, e3nn_conf = self.predictor.e3nn_predictor.predict(mol)
            if e3nn_pred is not None and 200 <= e3nn_pred <= 800:
                candidates.append({'source': 'e3nn', 'lambda': e3nn_pred,
                                   'confidence': e3nn_conf if e3nn_conf else 0.5})
                print(f"   🧠 e3nn prediction: {e3nn_pred:.0f} nm (conf={e3nn_conf:.2f})")

            # ---- Hückel prediction ----
            huckel_pred, huckel_conf = self.predictor.huckel_predictor.predict(mol)
            if huckel_pred is not None and 200 <= huckel_pred <= 800:
                candidates.append({'source': 'huckel', 'lambda': huckel_pred,
                                   'confidence': huckel_conf if huckel_conf else 0.5})
                print(f"   🧮 Hückel prediction: {huckel_pred:.0f} nm (conf={huckel_conf:.2f})")

            # ---- Score and rank all candidates ----
            scored = []
            for cand in candidates:
                score = cand.get('confidence', 0.5)
                lam = cand['lambda']
                if lam > 600 and cand.get('oscillator', 0) < 0.5 and cand.get('similarity', 0) < 0.8:
                    score *= 0.2
                if lam < 220 and cand['source'] not in ('NN_spectrum', 'analogue'):
                    score *= 0.5
                scored.append((cand, score))
            scored.sort(key=lambda x: x[1], reverse=True)

            if scored:
                best_candidate = scored[0][0]
                final_lambda = best_candidate['lambda']
                if len(scored) > 1 and abs(scored[0][0]['lambda'] - scored[1][0]['lambda']) <= 15:
                    total_score = scored[0][1] + scored[1][1]
                    final_lambda = (scored[0][0]['lambda'] * scored[0][1] +
                                    scored[1][0]['lambda'] * scored[1][1]) / total_score
            else:
                final_lambda = 254.0

            # ---------- ACCURACY-FIRST META-COMBINER ----------
            if self.predictor.meta_combiner.fitted:
                try:
                    fallback_lambda = analogue_prediction if analogue_prediction is not None else (shifted[0][0] if shifted else 254.0)
                    fallback_lambda = float(np.clip(fallback_lambda, Config.WL_MIN, Config.WL_MAX))

                    feat = self.feature_extractor.extract(mol, self.current_ph).reshape(1, -1)
                    feat = np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0)
                    feat_scaled = self.predictor.scaler.transform(feat)

                    if np.any(~np.isfinite(feat_scaled)):
                        raise ValueError("Scaled features contain non-finite values.")

                    # -----------------------------------------------------
                    # Base-model predictions MUST match the five OOF features
                    # used to train MetaCombiner. NN is RAW here.
                    # Residual correction is applied AFTER MetaCombiner.
                    # -----------------------------------------------------
                    with torch.no_grad():
                        nn_pred_raw = float(
                            self.predictor.model(
                                torch.FloatTensor(feat_scaled).to(self.predictor.device)
                            ).cpu().item()
                        )
                    if not np.isfinite(nn_pred_raw):
                        nn_pred_raw = fallback_lambda

                    nn_pred = float(np.clip(nn_pred_raw, Config.WL_MIN, Config.WL_MAX))

                    if self.predictor.ensemble.trained:
                        ens_mean, _ = self.predictor.ensemble.predict(feat_scaled)
                        ens_pred = (
                            float(ens_mean[0])
                            if ens_mean is not None and len(ens_mean) > 0 and np.isfinite(ens_mean[0])
                            else nn_pred
                        )
                    else:
                        ens_pred = nn_pred

                    drcnn_pred, _ = self.predictor.predict_drcnn(mol)
                    if drcnn_pred is None or not np.isfinite(drcnn_pred):
                        drcnn_pred = nn_pred
                    drcnn_pred = float(drcnn_pred)

                    e3nn_pred, _ = self.predictor.e3nn_predictor.predict(mol)
                    if e3nn_pred is None or not np.isfinite(e3nn_pred):
                        e3nn_pred = nn_pred
                    e3nn_pred = float(e3nn_pred)

                    ngb_mean, _ = self.predictor.ngboost.predict(feat_scaled)
                    if ngb_mean is not None and len(ngb_mean) > 0 and np.isfinite(ngb_mean[0]):
                        ngb_pred = float(ngb_mean[0])
                    else:
                        ngb_pred = nn_pred

                    preds = np.array(
                        [[nn_pred, ens_pred, drcnn_pred, e3nn_pred, ngb_pred]],
                        dtype=np.float64
                    )
                    preds = np.nan_to_num(
                        preds, nan=fallback_lambda, posinf=fallback_lambda, neginf=fallback_lambda
                    )
                    preds = np.clip(preds, Config.WL_MIN, Config.WL_MAX)

                    # Analogue is intentionally NOT passed into MetaCombiner.
                    # It is an external chemical-neighbour prior only.
                    meta_final = float(self.predictor.meta_combiner.predict(preds)[0])
                    meta_final = float(np.clip(meta_final, Config.WL_MIN, Config.WL_MAX))

                    # -----------------------------------------------------
                    # Residual correction now corrects META error: actual-meta.
                    # This is aligned with how the residual model was trained.
                    # -----------------------------------------------------
                    residual_shift = 0.0
                    if (
                        self.predictor.use_residual
                        and self.predictor.residual_corrector.model is not None
                    ):
                        try:
                            residual_shift = float(
                                self.predictor.residual_corrector.predict(feat_scaled)[0]
                            )
                            if np.isfinite(residual_shift):
                                residual_shift = float(
                                    np.clip(residual_shift, -50.0, 50.0)
                                )
                                meta_corrected = meta_final + residual_shift
                                print(
                                    f"   🔧 Meta residual correction: {residual_shift:+.1f} nm"
                                )
                            else:
                                meta_corrected = meta_final
                        except Exception as e:
                            print(f"   ⚠️ Meta residual correction skipped: {e}")
                            meta_corrected = meta_final
                    else:
                        meta_corrected = meta_final

                    meta_corrected = float(
                        np.clip(meta_corrected, Config.WL_MIN, Config.WL_MAX)
                    )

                    # -----------------------------------------------------
                    # Analogue remains outside the learned stack.
                    # Use it only when similarity is meaningful.
                    # -----------------------------------------------------
                    use_analogue = (
                        analogue_prediction is not None
                        and analogue_similarity is not None
                        and analogue_similarity >= 0.40
                        and 200 <= analogue_prediction <= 800
                    )

                    if use_analogue:
                        # Similarity-adaptive prior weight; never replaces the
                        # learned model at moderate similarity.
                        if analogue_similarity >= 0.80:
                            analogue_weight = 0.70
                        elif analogue_similarity >= 0.65:
                            analogue_weight = 0.50
                        else:
                            analogue_weight = 0.30

                        final_lambda = (
                            analogue_weight * float(analogue_prediction)
                            + (1.0 - analogue_weight) * meta_corrected
                        )
                        print(
                            f"   🧩 Meta‑Combiner: {meta_corrected:.1f} nm | "
                            f"analogue: {analogue_prediction:.1f} nm (sim={analogue_similarity:.3f})"
                        )
                        print(
                            f"   ⭐ Analogue prior blend: {analogue_weight:.1f} analogue / "
                            f"{1.0-analogue_weight:.1f} meta"
                        )
                    else:
                        final_lambda = meta_corrected
                        print(
                            f"   🧩 Meta‑Combiner final (5 OOF features): "
                            f"{final_lambda:.1f} nm"
                        )

                except Exception as e:
                    fallback = analogue_prediction if analogue_prediction is not None else (shifted[0][0] if shifted else 254.0)
                    final_lambda = float(np.clip(fallback, Config.WL_MIN, Config.WL_MAX))
                    print(f"   ⚠️ Meta‑Combiner failed: {e}. Using fallback {final_lambda:.1f} nm.")

            # ---- Apply memory and empirical corrections ----
            if memory_correction != 0.0:
                final_lambda += memory_correction
                print(f"   🧠 Memory corrected: +{memory_correction:.0f} nm")
            if self.empirical_corrector is not None:
                emp_corr = self.empirical_corrector.get_correction(mol, final_lambda)
                if abs(emp_corr) > 0.5:
                    final_lambda += emp_corr
                    print(f"   📐 Empirical correction: {emp_corr:+.1f} nm")
            final_lambda = float(np.clip(final_lambda, Config.WL_MIN, Config.WL_MAX))
            primary_peak = (final_lambda, 1.0, 30)
            valid_candidates = [(final_lambda, 1.0, 30)]
            spec = self.spectrum_gen.from_peaks(valid_candidates)
            if primary_peak is not None and len(spec) > 0:
                primary_wl = primary_peak[0]
                idx = int(round((primary_wl - Config.WL_MIN) / Config.WL_STEP))
                if 0 <= idx < len(spec):
                    current_max = np.max(spec)
                    if current_max > 0:
                        spec = spec * (np.max(spec) / spec[idx])
                        spec = np.maximum(spec, 0)

            # ------------------------------------------------------------
            # CONFORMATIONAL CORRECTION with caching
            # ------------------------------------------------------------
            if self.predictor.use_conform and self.predictor.conform_corrector.model is not None:
                base_spec = np.copy(spec)
                base_primary_peak = primary_peak
                canon_smi = Chem.MolToSmiles(mol, canonical=True)

                # Generate or retrieve conformer ensemble
                if canon_smi not in self._conformer_cache:
                    if Config.USE_BOLTZMANN_AVERAGING:
                        ensemble = ConformerHandler.get_boltzmann_ensemble(
                            mol,
                            num_confs=Config.BOLTZMANN_NUM_CONFS,
                            temp=Config.BOLTZMANN_TEMP,
                            energy_cutoff=Config.BOLTZMANN_ENERGY_CUTOFF
                        )
                    else:
                        mol_h, conf_id = ConformerHandler.get_lowest_energy_conformer(mol)
                        ensemble = [(mol_h, conf_id, 0.0, 1.0)]
                    self._conformer_cache[canon_smi] = ensemble
                else:
                    ensemble = self._conformer_cache[canon_smi]

                if Config.USE_BOLTZMANN_AVERAGING:
                    if len(ensemble) == 1:
                        mol_h, conf_id, energy, weight = ensemble[0]
                        desc_3d = ConformerHandler.compute_3d_descriptors(mol_h, conf_id)
                        raw_shape_shift = self.predictor.conform_corrector.predict_shift(desc_3d)
                        num_atoms = mol.GetNumAtoms()
                        max_allowable_shift = min(80.0, 10.0 + (conj_len * 0.8) + (num_atoms * 0.15))
                        max_allowable_shift = min(max_allowable_shift, 30.0)
                        shape_shift = float(np.clip(raw_shape_shift, -max_allowable_shift, max_allowable_shift))
                        self._last_3d_desc = desc_3d
                        if not (analogue_prediction is not None and analogue_similarity > 0.4):
                            if abs(shape_shift) > 0.5:
                                spec = self.shift_spectrum(base_spec, shape_shift)
                                primary_peak = (base_primary_peak[0] + shape_shift,
                                                base_primary_peak[1], base_primary_peak[2])
                    else:
                        avg_spec = np.zeros_like(Config.wavelengths, dtype=float)
                        total_weight = 0.0
                        shifts = []
                        for mol_h, conf_id, energy, weight in ensemble:
                            desc_3d = ConformerHandler.compute_3d_descriptors(mol_h, conf_id)
                            raw_shift = self.predictor.conform_corrector.predict_shift(desc_3d)
                            num_atoms = mol.GetNumAtoms()
                            max_allowable_shift = min(80.0, 10.0 + (conj_len * 0.8) + (num_atoms * 0.15))
                            max_allowable_shift = min(max_allowable_shift, 30.0)
                            shape_shift = float(np.clip(raw_shift, -max_allowable_shift, max_allowable_shift))
                            shifts.append(shape_shift)
                            shifted_spec = self.shift_spectrum(base_spec, shape_shift)
                            avg_spec += weight * shifted_spec
                            total_weight += weight
                        if total_weight > 0:
                            avg_spec /= total_weight
                            spec = avg_spec
                            peak_idx = np.argmax(spec)
                            final_lambda_avg = Config.WL_MIN + peak_idx * Config.WL_STEP
                            primary_peak = (final_lambda_avg, np.max(spec), 30)
                            avg_shift = np.mean(shifts) if shifts else 0.0
                            print(f"   🔄 Boltzmann averaged over {len(ensemble)} conformers (avg shift: {avg_shift:.1f} nm)")
                        else:
                            spec = base_spec
                else:
                    # Single conformer (non-Boltzmann)
                    try:
                        mol_h, conf_id = ConformerHandler.get_lowest_energy_conformer(mol)
                        desc_3d = ConformerHandler.compute_3d_descriptors(mol_h, conf_id)
                        raw_shape_shift = self.predictor.conform_corrector.predict_shift(desc_3d)
                        num_atoms = mol.GetNumAtoms()
                        max_allowable_shift = min(80.0, 10.0 + (conj_len * 0.8) + (num_atoms * 0.15))
                        max_allowable_shift = min(max_allowable_shift, 30.0)
                        shape_shift = float(np.clip(raw_shape_shift, -max_allowable_shift, max_allowable_shift))
                        self._last_3d_desc = desc_3d
                        if not (analogue_prediction is not None and analogue_similarity > 0.4):
                            if abs(shape_shift) > 0.5:
                                spec = self.shift_spectrum(base_spec, shape_shift)
                                primary_peak = (base_primary_peak[0] + shape_shift,
                                                base_primary_peak[1], base_primary_peak[2])
                    except Exception as e:
                        print(f"   ⚠️ 3D correction skipped: {e}")

            # ---- Beer‑Lambert and instrument effects ----
            spec = self.spectrum_gen.apply_beer_lambert(spec, self.current_conc, self.instr.path_length)
            spec = self.instr.apply_instrument_effects(spec)
            if self.blank_spectrum is not None:
                spec = np.maximum(spec - self.blank_spectrum, 0)
            # Quantum correction
            if Config.USE_QUANTUM_CORRECTION and td_success and quantum_desc:
                integrator = QuantumIntegrator()
                spec = integrator.adjust_spectrum(spec, quantum_desc)
                quantum_confidence = integrator.compute_quantum_confidence(quantum_desc, Config.WL_MIN + np.argmax(spec))
                self._last_quantum_confidence = quantum_confidence
            else:
                self._last_quantum_confidence = 0.0
            # Final smoothing
            if Config.FINAL_SMOOTHING_WINDOW > 1 and len(spec) > Config.FINAL_SMOOTHING_WINDOW:
                spec = savgol_filter(spec, Config.FINAL_SMOOTHING_WINDOW, 3)
                spec = np.maximum(spec, 1e-7)

            method = "chromophore+ml+tddft" if use_tddft else "chromophore+ml+3D" if self.predictor.conform_corrector.model is not None else "chromophore+ml"
            final_lambda = Config.WL_MIN + np.argmax(spec) * Config.WL_STEP
            self._last_primary_peak = (final_lambda, np.max(spec), 30)

            # --- Interaction report (requirement #8) ---
            try:
                self._last_interaction_report = self.build_interaction_report(
                    mol,
                    solvent_props=_solvent_props_for_hb,
                    hbond=hbond_info,
                    shift_mech=getattr(self, "_last_solvent_shift_mechanism", {}),
                )
            except Exception as _rep_err:
                self._last_interaction_report = {"error": str(_rep_err)}

            return spec, method, shifted, chrom_classes, quantum_desc, final_lambda
        except Exception as e:
            print(f"❌ Critical error in predict: {e}")
            self._last_primary_peak = (254.0, 1.0, 30)
            return np.zeros_like(Config.wavelengths), "error", [], [], {}, None

    def predict_reaction_products(self, mol, name=None):
        """Score the reaction products identified by the last predict() call.

        Returns:
            list of dicts: {product_smiles, product_name, reaction_key,
                            score, spectrum, lambda}
        Never modifies the simulator's primary state.
        """
        out = []
        for cand in getattr(self, "_last_reaction_candidates", []):
            try:
                prod = cand["mol"]
                spec, _, _, cls, qd, lam = self.predict(prod, name or "product")
                out.append({
                    "product_smiles": Chem.MolToSmiles(prod, canonical=True),
                    "product_name": cand["name"],
                    "reaction_key": cand["key"],
                    "heuristic_score": cand["score"],
                    "spectrum": spec,
                    "lambda": lam,
                    "chromophore_classes": cls,
                })
            except Exception as e:
                out.append({"error": str(e), "reaction_key": cand.get("key")})
        return out

    def predict_with_refinement(self, mol, name, use_tddft=False, n_iterations=10):
        spectra = []
        lambda_predictions = []
        for i in range(n_iterations):
            spec, _, _, _, _, p_lambda = self.predict(mol, name, use_tddft=use_tddft)
            spectra.append(spec)
            if p_lambda is not None:
                lambda_predictions.append(float(p_lambda))
            print(f"  Iteration {i+1}: λmax = {p_lambda} nm")
        avg_spec = np.mean(spectra, axis=0)
        final_lambda = float(np.mean(lambda_predictions)) if lambda_predictions else float(Config.WL_MIN + np.argmax(avg_spec) * Config.WL_STEP)
        refinement_std_spec = np.std(spectra, axis=0)
        model_uncertainty = None
        ensemble = getattr(self.predictor, "ensemble", None)
        if ensemble is not None and getattr(ensemble, "trained", False):
            try:
                feat = self.predictor.featurize(mol, self.current_ph)
                feat_scaled = self.predictor.scaler.transform([feat])
                ensemble_mean, ensemble_std = ensemble.predict(feat_scaled)
                if ensemble_std is not None:
                    model_uncertainty = float(np.asarray(ensemble_std).ravel()[0])
            except Exception as e:
                print(f"⚠️ Ensemble uncertainty unavailable: {e}")
        if model_uncertainty is None:
            model_uncertainty = 0.0
        confidence = float(np.clip(1.0 / (1.0 + model_uncertainty), 0.0, 1.0))
        return avg_spec, f"refined_method_iter{n_iterations}", refinement_std_spec, confidence, final_lambda

    # ---------- evaluate_locked_test_set and run_ablation remain unchanged ----------
    def evaluate_locked_test_set(self):
        predictor = self.predictor
        test_records = []
        for idx in sorted(predictor.test_indices):
            if idx >= len(predictor.feedback_db): continue
            fb = predictor.feedback_db[idx]
            if fb.get('data_role') != 'test': continue
            if not fb.get('smiles') or fb.get('actual') is None: continue
            test_records.append(fb)
        if len(test_records) < 2:
            print("⚠️ Fewer than 2 valid TEST predictions.")
            return {}
        y_true, y_pred = [], []
        original_solvent = self.current_solvent
        original_ph = self.current_ph
        try:
            for fb in test_records:
                mol = Chem.MolFromSmiles(fb['smiles'])
                if mol is None: continue
                test_solvent = fb.get('solvent', original_solvent)
                if test_solvent in self.solvent_db.solvents:
                    self.set_solvent(test_solvent)
                else:
                    continue
                test_ph = fb.get('pH', self.current_ph)
                self.current_ph = float(test_ph) if isinstance(test_ph, (int, float)) else self.current_ph
                _, _, _, _, _, predicted_lambda = self.predict(
                    mol, fb.get('name', 'test'),
                    use_tddft=False,
                    extract_quantum=Config.EXTRACT_QUANTUM_DESCRIPTORS
                )
                if predicted_lambda is None: continue
                y_true.append(float(fb['actual']))
                y_pred.append(float(predicted_lambda))
        finally:
            self.set_solvent(original_solvent)
            self.current_ph = original_ph
        if len(y_true) < 2:
            print("⚠️ Fewer than 2 valid TEST predictions.")
            return {}
        y_true = np.array(y_true); y_pred = np.array(y_pred)
        n = len(y_true)
        errors = y_pred - y_true
        mae = np.mean(np.abs(errors))
        rmse = np.sqrt(np.mean(errors**2))
        ss_res = np.sum(errors**2)
        ss_tot = np.sum((y_true - np.mean(y_true))**2)
        r2 = 1.0 - ss_res/ss_tot if ss_tot > 0 else (1.0 if ss_res == 0 else 0.0)
        sep = np.sqrt(ss_res/(n-1)) if n>1 else None
        sd = np.std(y_true, ddof=1)
        rpd = sd/sep if (sep is not None and sep > 0) else None
        mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
        var_true, var_pred = np.var(y_true, ddof=1), np.var(y_pred, ddof=1)
        covariance = np.cov(y_true, y_pred, ddof=1)[0,1]
        ccc_denominator = var_true + var_pred + (mean_true - mean_pred)**2
        ccc = (2.0 * covariance) / ccc_denominator if ccc_denominator != 0 else 0.0
        me = np.mean(errors)
        mae_clipped = min(mae, 400.0)
        norm_score = 1.0 - (mae_clipped / 400.0)
        return {
            'n': n, 'MAE': float(mae), 'RMSE': float(rmse), 'R2': float(r2),
            'SEP': float(sep) if sep is not None else None,
            'RPD': float(rpd) if rpd is not None else None,
            'CCC': float(ccc), 'ME': float(me),
            'normalized_MAE_score': max(0.0, min(1.0, norm_score)),
            'n_skipped': 0
        }

    def run_ablation(self, test_records, use_tddft=False):
        results = {}
        variants = {
            'Base NN': {'residual': False, 'solvent': False, 'conform': False, 'ensemble': False, 'kt': False},
            'NN + Solvent': {'residual': False, 'solvent': True, 'conform': False, 'ensemble': False, 'kt': False},
            'NN + Residual': {'residual': True, 'solvent': False, 'conform': False, 'ensemble': False, 'kt': False},
            'NN + Conform': {'residual': False, 'solvent': False, 'conform': True, 'ensemble': False, 'kt': False},
            'NN + Solvent + Residual': {'residual': True, 'solvent': True, 'conform': False, 'ensemble': False, 'kt': False},
            'Full (no KT)': {'residual': True, 'solvent': True, 'conform': True, 'ensemble': True, 'kt': False},
            'Full + KT': {'residual': True, 'solvent': True, 'conform': True, 'ensemble': True, 'kt': True}
        }
        if not test_records:
            print("No test records available for ablation. Skipping.")
            return {}
        for variant_name, settings in variants.items():
            print(f"\nRunning ablation: {variant_name}")
            y_true, y_pred = [], []
            orig_res = self.predictor.use_residual
            orig_sol = self.predictor.use_dynamic_solvent
            orig_conf = self.predictor.use_conform
            orig_ens = self.predictor.use_ensemble
            orig_kt_predict = self.kt_model.predict
            try:
                self.predictor.use_residual = settings['residual']
                self.predictor.use_dynamic_solvent = settings['solvent']
                self.predictor.use_conform = settings['conform']
                self.predictor.use_ensemble = settings['ensemble']
                if not settings['kt']:
                    self.kt_model.predict = lambda *args, **kwargs: 0.0
                for record in test_records:
                    mol = Chem.MolFromSmiles(record['smiles'])
                    if mol is None: continue
                    name = record.get('name', 'unknown')
                    actual = float(record['actual'])
                    _, _, _, _, _, pred_lam = self.predict(mol, name, use_tddft=use_tddft)
                    if pred_lam is not None:
                        y_true.append(actual); y_pred.append(float(pred_lam))
            finally:
                self.predictor.use_residual = orig_res
                self.predictor.use_dynamic_solvent = orig_sol
                self.predictor.use_conform = orig_conf
                self.predictor.use_ensemble = orig_ens
                self.kt_model.predict = orig_kt_predict
            if len(y_true) < 2:
                results[variant_name] = 'Insufficient data'
                continue
            y_true = np.asarray(y_true, dtype=np.float64)
            y_pred = np.asarray(y_pred, dtype=np.float64)
            errors = y_pred - y_true
            n = len(y_true)
            mae = float(np.mean(np.abs(errors)))
            rmse = float(np.sqrt(np.mean(errors ** 2)))
            ss_res = float(np.sum(errors ** 2))
            ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
            r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
            sep = float(np.sqrt(ss_res / (n - 1))) if n > 1 else np.nan
            sd_true = float(np.std(y_true, ddof=1))
            rpd = float(sd_true / sep) if np.isfinite(sep) and sep > 0 and sd_true > 0 else np.nan
            mean_true = float(np.mean(y_true)); mean_pred = float(np.mean(y_pred))
            var_true = float(np.var(y_true, ddof=1)); var_pred = float(np.var(y_pred, ddof=1))
            cov = float(np.cov(y_true, y_pred, ddof=1)[0, 1])
            ccc_denom = var_true + var_pred + (mean_true - mean_pred) ** 2
            ccc = float((2.0 * cov) / ccc_denom) if ccc_denom > 0 else np.nan
            me = float(np.mean(errors))
            results[variant_name] = {'n': n, 'MAE': mae, 'RMSE': rmse, 'R2': r2,
                                     'SEP': sep, 'RPD': rpd, 'CCC': ccc, 'ME': me}
        return results


print("✅ All core classes loaded")

✅ All core classes loaded


In [ ]:
# @title 6.) Empirical Rule Corrector
"""
Professional implementation of classic UV‑Vis empirical rules.
These rules are applied as a post‑prediction correction.
They do NOT change the 1024‑dim feature vector.
"""

class EmpiricalRuleCorrector:
    """
    Computes λmax using:
      - Woodward‑Fieser (dienes, enones)
      - Fieser‑Kuhn (polyenes with >4 double bonds)
      - Scott’s Rule (aromatic carbonyls)
    Returns a correction shift (or direct λmax) that can be averaged
    with the ML prediction.
    """

    def __init__(self):
        # ---------- Woodward‑Fieser base values (nm) ----------
        self.WF_BASE = {
            "heteroannular_diene": 214,
            "homoannular_diene": 253,
            "enone_acyclic": 215,
            "enone_cyclic": 215,
            "enone_6member": 215,
            "enone_5member": 202,
        }
        # Increments (nm) for Woodward‑Fieser
        self.WF_INCREMENTS = {
            "alkyl_substituent": 5,
            "exocyclic_double_bond": 5,
            "ring_extension": 5,
            "double_bond_extension": 30,
            "acetate": 0,
            "ether": 6,
            "thioether": 30,
            "chlorine": 15,
            "bromine": 25,
            "hydroxyl": 35,
            "methoxyl": 35,
            "amine": 35,
            "acetamide": 20,
        }

    # ============================================================
    # 1. WOODWARD‑FIESER FOR DIENES
    # ============================================================
    def woodward_fieser_diene(self, mol: Chem.Mol) -> Optional[float]:
        """
        Apply Woodward‑Fieser rules for conjugated dienes.
        Returns λmax in nm, or None if no diene pattern found.
        """
        # Detect diene substructures
        # Heteroannular diene (two double bonds in different rings)
        hetero_pattern = Chem.MolFromSmarts("C=C-C=C")
        homo_pattern = Chem.MolFromSmarts("C=C-C=C")  # simplified; we distinguish via ring membership

        if not mol.HasSubstructMatch(hetero_pattern) and not mol.HasSubstructMatch(homo_pattern):
            return None

        # Simplified detection: count conjugated double bonds in the longest chain
        conj_atoms = self._get_conjugated_atoms(mol)
        if len(conj_atoms) < 4:
            return None

        # Determine if diene is homoannular (both double bonds in same ring) or heteroannular
        # Using ring info
        ring_info = mol.GetRingInfo()
        atom_rings = ring_info.AtomRings()
        is_homoannular = False
        for ring in atom_rings:
            ring_atoms = set(ring)
            # Check if at least 4 conjugated atoms are in the same ring
            if len(ring_atoms.intersection(conj_atoms)) >= 4:
                is_homoannular = True
                break

        base = self.WF_BASE["homoannular_diene"] if is_homoannular else self.WF_BASE["heteroannular_diene"]

        # Count alkyl substituents on the diene system
        alkyl_count = 0
        for atom_idx in conj_atoms:
            atom = mol.GetAtomWithIdx(atom_idx)
            for neighbor in atom.GetNeighbors():
                if neighbor.GetIdx() not in conj_atoms and neighbor.GetAtomicNum() == 6:
                    if neighbor.GetDegree() >= 3:  # sp3 carbon
                        alkyl_count += 1

        # Count exocyclic double bonds attached to the diene
        exocyclic_count = 0
        for bond in mol.GetBonds():
            if bond.GetBondType() == Chem.rdchem.BondType.DOUBLE:
                a1 = bond.GetBeginAtom()
                a2 = bond.GetEndAtom()
                if a1.GetIdx() in conj_atoms and a2.GetIdx() not in conj_atoms:
                    if any(a2.GetIdx() in ring for ring in atom_rings):
                        exocyclic_count += 1
                elif a2.GetIdx() in conj_atoms and a1.GetIdx() not in conj_atoms:
                    if any(a1.GetIdx() in ring for ring in atom_rings):
                        exocyclic_count += 1

        # Calculate λmax
        lam = base
        lam += alkyl_count * self.WF_INCREMENTS["alkyl_substituent"]
        lam += exocyclic_count * self.WF_INCREMENTS["exocyclic_double_bond"]

        return float(lam)

    # ============================================================
    # 2. WOODWARD‑FIESER FOR α,β‑UNSATURATED CARBONYLS (ENONES)
    # ============================================================
    def woodward_fieser_enone(self, mol: Chem.Mol) -> Optional[float]:
        """
        Woodward‑Fieser for α,β‑unsaturated carbonyls.
        """
        # Detect enone pattern: C=C-C=O
        enone_pattern = Chem.MolFromSmarts("C=CC=O")
        if not mol.HasSubstructMatch(enone_pattern):
            return None

        # Determine ring size if cyclic enone
        ring_info = mol.GetRingInfo()
        atom_rings = ring_info.AtomRings()
        ring_sizes = [len(r) for r in atom_rings]

        # Base value depends on ring size and acyclic/cyclic
        # Simplified: check if carbonyl carbon is in a ring
        carbonyl_c = None
        for atom in mol.GetAtoms():
            if atom.GetAtomicNum() == 6:
                for bond in atom.GetBonds():
                    if bond.GetBondType() == Chem.rdchem.BondType.DOUBLE:
                        nbr = bond.GetOtherAtom(atom)
                        if nbr.GetAtomicNum() == 8:
                            carbonyl_c = atom
                            break
                if carbonyl_c:
                    break

        is_cyclic = False
        ring_size = 0
        if carbonyl_c:
            for ring in atom_rings:
                if carbonyl_c.GetIdx() in ring:
                    is_cyclic = True
                    ring_size = len(ring)
                    break

        if is_cyclic:
            if ring_size == 5:
                base = self.WF_BASE["enone_5member"]
            else:
                base = self.WF_BASE["enone_6member"]
        else:
            base = self.WF_BASE["enone_acyclic"]

        # Count alkyl substituents on the C=C and carbonyl
        # Simplified: count all sp3 carbons attached to the conjugated system
        conj_atoms = self._get_conjugated_atoms(mol)
        alkyl_count = 0
        for idx in conj_atoms:
            atom = mol.GetAtomWithIdx(idx)
            for nbr in atom.GetNeighbors():
                if nbr.GetIdx() not in conj_atoms and nbr.GetAtomicNum() == 6:
                    if nbr.GetDegree() >= 3:
                        alkyl_count += 1

        # Add hydroxyl, methoxyl, etc. (only if directly attached to the chromophore)
        auxo_shift = 0
        auxo_smarts = {
            "OH": Chem.MolFromSmarts("[OH]"),
            "OCH3": Chem.MolFromSmarts("OC"),
            "NH2": Chem.MolFromSmarts("[NH2]"),
            "Cl": Chem.MolFromSmarts("[Cl]"),
            "Br": Chem.MolFromSmarts("[Br]"),
        }
        for name, patt in auxo_smarts.items():
            if patt and mol.HasSubstructMatch(patt):
                # Check if attached to the enone system
                matches = mol.GetSubstructMatches(patt)
                for match in matches:
                    for atom_idx in match:
                        atom = mol.GetAtomWithIdx(atom_idx)
                        for nbr in atom.GetNeighbors():
                            if nbr.GetIdx() in conj_atoms:
                                if name == "OH":
                                    auxo_shift += 35
                                elif name == "OCH3":
                                    auxo_shift += 35
                                elif name == "NH2":
                                    auxo_shift += 35
                                elif name == "Cl":
                                    auxo_shift += 15
                                elif name == "Br":
                                    auxo_shift += 25
                                break

        lam = base + alkyl_count * 5 + auxo_shift
        return float(lam)

    # ============================================================
    # 3. FIESER‑KUHN RULES (POLYENES)
    # ============================================================
    def fieser_kuhn(self, mol: Chem.Mol) -> Optional[float]:
        """
        Fieser‑Kuhn formula for polyenes with >4 conjugated double bonds.
        λmax = 114 + 5*M + n*(48.0 - 1.7*n) - 16.5*Rendo - 10*Reso
        where:
          M = number of alkyl substituents on the polyene chain
          n = number of conjugated double bonds
          Rendo = number of rings with endocyclic double bonds in the polyene
          Reso = number of rings with exocyclic double bonds in the polyene
        """
        # Find the longest conjugated chain
        conj_atoms = self._get_conjugated_atoms(mol)
        if len(conj_atoms) < 8:  # <4 double bonds
            return None

        # Count double bonds in the conjugated system
        double_bonds = 0
        for bond in mol.GetBonds():
            if bond.GetBondType() == Chem.rdchem.BondType.DOUBLE:
                if bond.GetBeginAtomIdx() in conj_atoms and bond.GetEndAtomIdx() in conj_atoms:
                    double_bonds += 1

        if double_bonds < 4:
            return None

        n = double_bonds
        # Count alkyl substituents on the conjugated system
        M = 0
        for idx in conj_atoms:
            atom = mol.GetAtomWithIdx(idx)
            for nbr in atom.GetNeighbors():
                if nbr.GetIdx() not in conj_atoms and nbr.GetAtomicNum() == 6:
                    if nbr.GetDegree() >= 3:
                        M += 1

        # Count rings with endocyclic/exocyclic double bonds in the polyene
        ring_info = mol.GetRingInfo()
        atom_rings = ring_info.AtomRings()
        Rendo = 0
        Reso = 0
        for ring in atom_rings:
            ring_set = set(ring)
            # Check if ring has at least 2 conjugated atoms (endocyclic)
            if len(ring_set.intersection(conj_atoms)) >= 2:
                Rendo += 1
            # Check for exocyclic double bond attached to ring
            for idx in ring:
                atom = mol.GetAtomWithIdx(idx)
                for bond in atom.GetBonds():
                    if bond.GetBondType() == Chem.rdchem.BondType.DOUBLE:
                        other = bond.GetOtherAtom(atom)
                        if other.GetIdx() not in ring_set and other.GetIdx() in conj_atoms:
                            Reso += 1

        # Fieser‑Kuhn formula
        lam = 114 + 5 * M + n * (48.0 - 1.7 * n) - 16.5 * Rendo - 10 * Reso
        return float(lam)

    # ============================================================
    # 4. SCOTT’S RULE (AROMATIC CARBONYLS)
    # ============================================================
    def scott_rule(self, mol: Chem.Mol) -> Optional[float]:
        """
        Scott’s rule for aromatic carbonyls (benzaldehydes, acetophenones, etc.)
        Base = 250 nm for benzaldehyde, + increments for substituents.
        """
        # Detect benzaldehyde or acetophenone
        benzaldehyde = Chem.MolFromSmarts("O=CC1=CC=CC=C1")
        acetophenone = Chem.MolFromSmarts("O=C(C)C1=CC=CC=C1")
        if not mol.HasSubstructMatch(benzaldehyde) and not mol.HasSubstructMatch(acetophenone):
            return None

        # Base
        base = 250.0
        # Increments for substituents on the aromatic ring (ortho/meta/para)
        # Simplified: count all substituents other than the carbonyl
        aromatic_atoms = set()
        for atom in mol.GetAtoms():
            if atom.GetIsAromatic():
                aromatic_atoms.add(atom.GetIdx())

        # Count substituents (non-H, non-carbonyl)
        subs = 0
        for idx in aromatic_atoms:
            atom = mol.GetAtomWithIdx(idx)
            # Check if it's the carbonyl attachment point
            is_carbonyl = False
            for nbr in atom.GetNeighbors():
                if nbr.GetAtomicNum() == 6:
                    for bond in nbr.GetBonds():
                        if bond.GetBondType() == Chem.rdchem.BondType.DOUBLE:
                            if bond.GetOtherAtom(nbr).GetAtomicNum() == 8:
                                is_carbonyl = True
                                break
                if is_carbonyl:
                    break
            if not is_carbonyl:
                for nbr in atom.GetNeighbors():
                    if nbr.GetAtomicNum() != 1 and nbr.GetIdx() not in aromatic_atoms:
                        subs += 1

        # Each substituent adds ~10 nm (simplified)
        lam = base + subs * 10.0
        return float(lam)

    # ============================================================
    # 5. HELPER: Get conjugated atoms
    # ============================================================
    def _get_conjugated_atoms(self, mol: Chem.Mol) -> set:
        """
        Returns a set of atom indices that are part of a conjugated system.
        """
        conj = set()
        # Find all double bonds and aromatic bonds
        for bond in mol.GetBonds():
            if bond.GetIsAromatic() or bond.GetBondType() in (Chem.rdchem.BondType.DOUBLE, Chem.rdchem.BondType.TRIPLE):
                conj.add(bond.GetBeginAtomIdx())
                conj.add(bond.GetEndAtomIdx())

        # Propagate through single bonds between sp2/sp atoms with lone pairs (heteroatoms)
        changed = True
        while changed:
            changed = False
            for bond in mol.GetBonds():
                if bond.GetBondType() == Chem.rdchem.BondType.SINGLE:
                    a1 = bond.GetBeginAtom()
                    a2 = bond.GetEndAtom()
                    if a1.GetIdx() in conj or a2.GetIdx() in conj:
                        # Check if the atom not in conj is sp2 or has lone pair
                        for atom in [a1, a2]:
                            if atom.GetIdx() not in conj:
                                if atom.GetHybridization() in (Chem.rdchem.HybridizationType.SP2, Chem.rdchem.HybridizationType.SP):
                                    if atom.GetAtomicNum() in (7, 8, 16, 9, 17, 35, 53):  # heteroatoms with lone pairs
                                        conj.add(atom.GetIdx())
                                        changed = True
        return conj

    # ============================================================
    # 6. MASTER PREDICT
    # ============================================================
    def predict(self, mol: Chem.Mol) -> Dict[str, float]:
        """
        Run all empirical rules and return the best estimate.
        Returns dict with 'lam' (best λmax) and 'method'.
        """
        results = []

        # Diene
        lam_diene = self.woodward_fieser_diene(mol)
        if lam_diene is not None:
            results.append((lam_diene, "Woodward-Fieser_diene"))

        # Enone
        lam_enone = self.woodward_fieser_enone(mol)
        if lam_enone is not None:
            results.append((lam_enone, "Woodward-Fieser_enone"))

        # Polyene
        lam_polyene = self.fieser_kuhn(mol)
        if lam_polyene is not None:
            results.append((lam_polyene, "Fieser-Kuhn"))

        # Scott
        lam_scott = self.scott_rule(mol)
        if lam_scott is not None:
            results.append((lam_scott, "Scott"))

        if not results:
            return {"lam": None, "method": "none"}

        # If multiple rules apply, take the average (or you could take the max)
        # For conjugated systems, the longest wavelength absorption is usually the most intense.
        # We'll take the highest λmax as the empirical estimate.
        best = max(results, key=lambda x: x[0])
        return {"lam": best[0], "method": best[1]}

    # ============================================================
    # 7. CORRECTION FACTOR (for integration with ML)
    # ============================================================
    def get_correction(self, mol: Chem.Mol, ml_lambda: float) -> float:
        """
        Returns a correction shift (empirical - ml) that can be added
        to the final prediction.
        """
        emp = self.predict(mol)
        if emp["lam"] is None:
            return 0.0
        # Weighted correction: only apply if empirical and ML are within reasonable range
        # If they differ by >100 nm, don't trust the empirical rule.
        diff = emp["lam"] - ml_lambda
        if abs(diff) > 100:
            return 0.0
        # Apply a damping factor (0.15) to avoid overcorrection
        return 0.10 * diff

print("✅ EmpiricalRuleCorrector loaded (Woodward‑Fieser, Fieser‑Kuhn, Scott).")

✅ EmpiricalRuleCorrector loaded (Woodward‑Fieser, Fieser‑Kuhn, Scott).


In [ ]:
# @title 7.) Helper functions

def set_global_seed(seed=Config.RANDOM_SEED):
    """
    Set all random seeds for reproducibility.
    Call this at the very beginning of main().
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # For Python's hash randomization
    os.environ['PYTHONHASHSEED'] = str(seed)

    print(f"✅ Global random seed set to {seed}")

def scaffold_split(feedback_db, test_size=0.15, val_size=0.15, random_seed=42):
    """
    Perform a scaffold-aware split of the feedback database.
    Returns: train_indices, val_indices, test_indices
    """
    mols = []
    for entry in feedback_db:
        smi = entry.get('smiles')
        if smi:
            mol = Chem.MolFromSmiles(smi)
            mols.append(mol) if mol else mols.append(None)
        else:
            mols.append(None)

    scaffolds = []
    for mol in mols:
        if mol is not None:
            try:
                scaffold = MurckoScaffold.GetScaffoldForMol(mol)
                scaffolds.append(Chem.MolToSmiles(scaffold))
            except:
                scaffolds.append(None)
        else:
            scaffolds.append(None)

    scaffold_groups = defaultdict(list)
    for idx, scaf in enumerate(scaffolds):
        if scaf is not None:
            scaffold_groups[scaf].append(idx)

    if len(scaffold_groups) < 3:
        print("⚠️ Too few scaffolds; falling back to random split.")
        indices = list(range(len(feedback_db)))
        import random
        random.seed(random_seed)
        random.shuffle(indices)
        n_test = int(len(indices) * test_size)
        n_val = int(len(indices) * val_size)
        return indices[n_test+n_val:], indices[:n_val], indices[n_val:n_test+n_val]

    import random
    random.seed(random_seed)
    scaffold_list = list(scaffold_groups.keys())
    random.shuffle(scaffold_list)
    n_test = max(1, int(len(scaffold_list) * test_size))
    n_val = max(1, int(len(scaffold_list) * val_size))
    test_scaffolds = set(scaffold_list[:n_test])
    val_scaffolds = set(scaffold_list[n_test:n_test+n_val])

    train_indices, val_indices, test_indices = [], [], []
    for idx, scaf in enumerate(scaffolds):
        if scaf in test_scaffolds:
            test_indices.append(idx)
        elif scaf in val_scaffolds:
            val_indices.append(idx)
        else:
            train_indices.append(idx)

    return train_indices, val_indices, test_indices


def build_artifact_manifest(sim):
    """Generate a manifest of the current data split and model metadata."""
    train_ids = sorted(str(fb.get('feedback_id', i)) for i, fb in enumerate(sim.predictor.feedback_db)
                       if fb.get('data_role') == 'train' and fb.get('training_eligible', True))
    test_ids = sorted(str(fb.get('feedback_id', i)) for i, fb in enumerate(sim.predictor.feedback_db)
                      if fb.get('data_role') == 'test')
    val_ids = sorted(str(fb.get('feedback_id', i)) for i, fb in enumerate(sim.predictor.feedback_db)
                     if fb.get('data_role') == 'validation')
    return {
        'manifest_version': '1.0',
        'model_version': Config.MODEL_VERSION,
        'feature_version': Config.FEATURE_VERSION,
        'feature_dim': sim.predictor.current_feature_dim,
        'n_feedbacks': len(sim.predictor.feedback_db),
        'n_train': len(train_ids),
        'n_validation': len(val_ids),
        'n_test': len(test_ids),
        'train_ids': train_ids,
        'validation_ids': val_ids,
        'test_ids': test_ids,
    }


def validate_artifact_manifest(sim, manifest):
    """Check if the manifest matches the current split. Returns (is_valid, message)."""
    if not isinstance(manifest, dict):
        return False, "Invalid manifest format"
    if manifest.get('manifest_version') != '1.0':
        return False, "Unsupported manifest version"
    if manifest.get('model_version') != Config.MODEL_VERSION:
        return False, f"model version mismatch: {manifest.get('model_version')} != {Config.MODEL_VERSION}"
    if manifest.get('feature_version') != Config.FEATURE_VERSION:
        return False, f"feature version mismatch: {manifest.get('feature_version')} != {Config.FEATURE_VERSION}"
    if manifest.get('feature_dim') != sim.predictor.current_feature_dim:
        return False, f"feature dimension mismatch: {manifest.get('feature_dim')} != {sim.predictor.current_feature_dim}"

    current_train_ids = sorted(str(fb.get('feedback_id', i)) for i, fb in enumerate(sim.predictor.feedback_db)
                               if fb.get('data_role') == 'train' and fb.get('training_eligible', True))
    current_test_ids = sorted(str(fb.get('feedback_id', i)) for i, fb in enumerate(sim.predictor.feedback_db)
                              if fb.get('data_role') == 'test')
    current_val_ids = sorted(str(fb.get('feedback_id', i)) for i, fb in enumerate(sim.predictor.feedback_db)
                             if fb.get('data_role') == 'validation')

    if manifest.get('train_ids', []) != current_train_ids:
        return False, "TRAIN membership changed"
    if manifest.get('test_ids', []) != current_test_ids:
        return False, "TEST membership changed"
    if manifest.get('validation_ids', []) != current_val_ids:
        return False, "VALIDATION membership changed"
    return True, "Artifact provenance verified"


def load_all(sim, base_dir='/content/drive/MyDrive/UVVis_Learner/'):
    """
    Load all model artifacts from Google Drive.
    If the manifest is invalid, it automatically rebuilds the split,
    updates the manifest, and loads secondary models.
    """
    print("\n📂 Loading from Google Drive...")
    drive.mount('/content/drive', force_remount=False)

    # --- Initialize ReactionLearner FIRST (after mount) ---
    try:
        sim.predictor.sim_ref = sim
        rml_path = (os.path.join(base_dir, 'reaction_feasibility.pkl')
                    if os.path.exists(base_dir) else "reaction_feasibility.pkl")
        sim.predictor.reaction_learner = ReactionLearner(
            model_path=rml_path,
            engine=ReactionEngine(),
            chromo=sim.chromo
        )
        print(f"✅ ReactionLearner attached "
              f"({len(sim.predictor.reaction_learner.real_X)} real samples)")
    except Exception as _e:
        print(f"⚠️ ReactionLearner init failed: {_e}")
        sim.predictor.reaction_learner = None

    if not os.path.exists(base_dir):
        print("No previous data found.")
        return

    # ---------- Load core artifacts ----------
    scaler_path = os.path.join(base_dir, 'scaler.pkl')
    if os.path.exists(scaler_path):
        with open(scaler_path, 'rb') as f:
            sim.predictor.scaler = pickle.load(f)
        print("✅ Scaler loaded")

    # Feedback DB
    fb_entries = []
    for fb_name in ['feedback.pkl', 'feedback_db.pkl']:
        fb_path = os.path.join(base_dir, fb_name)
        if os.path.exists(fb_path):
            with open(fb_path, 'rb') as f:
                fb = pickle.load(f)
                print(f"✅ Loaded {len(fb)} feedback records from {fb_name}")
                fb_entries.extend(fb)

    if fb_entries:
        for fb in fb_entries:
            fb.setdefault('chromophores', [])
            fb.setdefault('quantum', {})
            fb.setdefault('3d_descriptors', None)
            fb.setdefault('data_role', 'train')
            fb.setdefault('training_eligible', True)
        sim.predictor.feedback_db = fb_entries
        # Rebuild the frozen test set from persisted roles
        sim.predictor.frozen_test_smiles = set()
        for i, fb in enumerate(sim.predictor.feedback_db):
            if fb.get('data_role') == 'test':
                sim.predictor.frozen_test_smiles.add(fb.get('smiles'))
        print(f"🔒 Frozen TEST molecules: {len(sim.predictor.frozen_test_smiles)}")

        # Restore split from persisted roles (if any)
        persisted_test = {i for i, fb in enumerate(sim.predictor.feedback_db) if fb.get('data_role') == 'test'}
        persisted_validation = {i for i, fb in enumerate(sim.predictor.feedback_db) if fb.get('data_role') == 'validation'}

        if persisted_test:
            sim.predictor.test_indices = persisted_test
            sim.predictor.validation_indices = persisted_validation
            for i, fb in enumerate(sim.predictor.feedback_db):
                if i in sim.predictor.test_indices:
                    fb['data_role'] = 'test'
                    fb['training_eligible'] = False
                elif i in sim.predictor.validation_indices:
                    fb['data_role'] = 'validation'
                    fb['training_eligible'] = False
                else:
                    fb['data_role'] = 'train'
                    fb['training_eligible'] = True
            print(f"✅ Restored persisted split: {len(sim.predictor.feedback_db)-len(sim.predictor.test_indices)-len(sim.predictor.validation_indices)} train, {len(sim.predictor.validation_indices)} validation, {len(sim.predictor.test_indices)} test")
        else:
            # No persisted split – create a new scaffold split
            train_idx, val_idx, test_idx = scaffold_split(sim.predictor.feedback_db,
                                                          test_size=Config.TEST_SIZE,
                                                          val_size=Config.VALIDATION_SIZE)
            sim.predictor.test_indices = set(test_idx)
            sim.predictor.validation_indices = set(val_idx)
            for i, fb in enumerate(sim.predictor.feedback_db):
                if i in sim.predictor.test_indices:
                    fb['data_role'] = 'test'
                    fb['training_eligible'] = False
                elif i in sim.predictor.validation_indices:
                    fb['data_role'] = 'validation'
                    fb['training_eligible'] = False
                else:
                    fb['data_role'] = 'train'
                    fb['training_eligible'] = True
            print(f"✅ New scaffold split created: {len(train_idx)} train, {len(val_idx)} validation, {len(test_idx)} test")
    else:
        print("ℹ️ No feedback file found.")

    # Error history
    err_entries = []
    for err_name in ['errors.pkl', 'error_history.pkl']:
        err_path = os.path.join(base_dir, err_name)
        if os.path.exists(err_path):
            with open(err_path, 'rb') as f:
                err = pickle.load(f)
                print(f"✅ Loaded {len(err)} error records from {err_name}")
                err_entries.extend(err)
    if err_entries:
        sim.predictor.error_history = err_entries
    else:
        sim.predictor.error_history = []

    # Metrics log
    metrics_path = os.path.join(base_dir, 'metrics_log.pkl')
    if os.path.exists(metrics_path):
        with open(metrics_path, 'rb') as f:
            sim.predictor.metrics_log = pickle.load(f)
        print(f"✅ Loaded metrics log with {len(sim.predictor.metrics_log)} entries")
    else:
        sim.predictor.metrics_log = []

    # Metadata (solvent offsets)
    metadata_path = os.path.join(base_dir, 'metadata.json')
    if os.path.exists(metadata_path):
        with open(metadata_path, 'r') as f:
            meta = json.load(f)
            sim.predictor.solvent_offsets = meta.get("solvent_offsets", {})
        print("✅ Loaded solvent offsets from metadata.json")

    # Model weights
    model_path = os.path.join(base_dir, 'model_weights.pth')
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location='cpu')
        if isinstance(checkpoint, dict) and 'model_state' in checkpoint:
            saved_dim = checkpoint.get('feature_dim', Config.PADDED_FEAT_DIM)
            state_dict = checkpoint['model_state']
        else:
            saved_dim = Config.PADDED_FEAT_DIM
            state_dict = checkpoint

        if saved_dim == sim.predictor.current_feature_dim:
            sim.predictor.model = UVVisNN(
                sim.predictor.current_feature_dim,
                Config.HIDDEN_LAYERS,
                Config.DROPOUT
            ).to(sim.predictor.device)
            # Adapt old keys
            new_state_dict = {}
            for k, v in state_dict.items():
                if k.startswith('net.'):
                    if k.startswith('net.12.'):
                        new_k = 'spectrum_head.' + k[7:]
                    else:
                        new_k = 'backbone.' + k[4:]
                    new_state_dict[new_k] = v
                else:
                    new_state_dict[k] = v
            sim.predictor.model.load_state_dict(new_state_dict, strict=False)
            sim.predictor.model.eval()
            sim.predictor.is_trained = True
            print("✅ Model loaded (old architecture adapted to new UVVisNN)")
        else:
            print(f"⚠️ Model dimension mismatch; model not loaded.")
    else:
        print("ℹ️ model_weights.pth not found – starting fresh")

    # ---------- Manifest validation and auto‑repair ----------
    manifest_path = os.path.join(base_dir, 'artifact_manifest.json')
    artifacts_trusted = False
    manifest_mismatch = False

    if os.path.exists(manifest_path):
        try:
            with open(manifest_path, 'r') as f:
                manifest = json.load(f)
            artifacts_trusted, msg = validate_artifact_manifest(sim, manifest)
            if artifacts_trusted:
                print(f"✅ Learned-artifact provenance verified: {msg}")
            else:
                print(f"⚠️ Learned artifacts rejected: {msg}")
                manifest_mismatch = True
        except Exception as e:
            print(f"⚠️ Could not validate artifact manifest: {e}")
            manifest_mismatch = True
    else:
        print("⚠️ artifact_manifest.json not found – learned artifacts will NOT be loaded.")
        manifest_mismatch = True

    # If manifest is invalid, rebuild the split and save a fresh manifest
    if manifest_mismatch and len(sim.predictor.feedback_db) > 0:
        print("\n🔄 Manifest mismatch detected. Rebuilding split and updating manifest...")
        train_idx, val_idx, test_idx = scaffold_split(
            sim.predictor.feedback_db,
            test_size=Config.TEST_SIZE,
            val_size=Config.VALIDATION_SIZE
        )
        sim.predictor.test_indices = set(test_idx)
        sim.predictor.validation_indices = set(val_idx)
        for i, fb in enumerate(sim.predictor.feedback_db):
            if i in sim.predictor.test_indices:
                fb['data_role'] = 'test'
                fb['training_eligible'] = False
            elif i in sim.predictor.validation_indices:
                fb['data_role'] = 'validation'
                fb['training_eligible'] = False
            else:
                fb['data_role'] = 'train'
                fb['training_eligible'] = True
        print(f"✅ Split rebuilt: {len(train_idx)} train, {len(val_idx)} validation, {len(test_idx)} test")
        # Save the new manifest and all models
        save_all(sim)
        print("✅ Manifest updated. Now loading secondary models...")
        artifacts_trusted = True   # Now the manifest is consistent

    # ---------- Load secondary models (only if trusted) ----------
    if artifacts_trusted:
        res_path = os.path.join(base_dir, 'residual_corrector.pkl')
        if os.path.exists(res_path):
            sim.predictor.residual_corrector.load(res_path)
            print("✅ Residual corrector loaded")

        sol_path = os.path.join(base_dir, 'solvent_learner.pkl')
        if os.path.exists(sol_path):
            sim.predictor.solvent_learner.load(sol_path)
            print("✅ Solvent learner loaded")

        ens_path = os.path.join(base_dir, 'ensemble.pkl')
        if os.path.exists(ens_path):
            sim.predictor.ensemble.load(ens_path)
            print("✅ Ensemble loaded")

        out_path = os.path.join(base_dir, 'outlier.pkl')
        if os.path.exists(out_path):
            sim.predictor.outlier_detector.load(out_path)
            print("✅ Outlier detector loaded")

        deep_svdd_path = os.path.join(base_dir, 'deep_svdd.pth')
        if os.path.exists(deep_svdd_path):
            try:
                sim.predictor.deep_svdd.load(deep_svdd_path)
                sim.predictor.deep_svdd_trained = True
                print("✅ Deep SVDD loaded")
            except Exception as e:
                print(f"⚠️ Failed to load Deep SVDD: {e}")

        mem_path = os.path.join(base_dir, 'chemistry_memory.pkl')
        if os.path.exists(mem_path):
            sim.chemistry_memory.load(mem_path)
            print("✅ Chemistry Memory loaded")

        meta_path = os.path.join(base_dir, 'meta_combiner.pkl')
        if os.path.exists(meta_path):
            sim.predictor.meta_combiner.load(meta_path)
            print("✅ MetaCombiner loaded")

        conf_path = os.path.join(base_dir, 'conform_corrector.pkl')
        if os.path.exists(conf_path):
            sim.predictor.conform_corrector.load(conf_path)
            print("✅ Conformational corrector loaded")

        huckel_path = os.path.join(base_dir, 'huckel_predictor.pkl')
        if os.path.exists(huckel_path):
            sim.predictor.huckel_predictor.load(huckel_path)
            sim.predictor.huckel_trained = True
            print("✅ Hückel predictor loaded")

        ngb_path = os.path.join(base_dir, 'ngboost.pkl')
        if os.path.exists(ngb_path):
            try:
                sim.predictor.ngboost.load(ngb_path)
                print("✅ NGBoost loaded")
            except Exception as e:
                print(f"⚠️ Failed to load NGBoost: {e}")

        kt_path = os.path.join(base_dir, 'kamlet_taft_model.pkl')
        if os.path.exists(kt_path):
            sim.kt_model.load(kt_path)
            print("✅ Kamlet–Taft model loaded")

        mist_path = os.path.join(base_dir, 'mist_predictor.pkl')
        if os.path.exists(mist_path):
            sim.predictor.mist_predictor.load(mist_path)
            sim.predictor.mist_trained = True
            print("✅ MIST loaded")

        e3nn_path = os.path.join(base_dir, 'e3nn_predictor.pth')
        if os.path.exists(e3nn_path):
            sim.predictor.e3nn_predictor.load(e3nn_path)
            sim.predictor.e3nn_trained = True
            print("✅ e3nn loaded")

        ad_path = os.path.join(base_dir, 'ad_model.pkl')
        if os.path.exists(ad_path):
            with open(ad_path, 'rb') as f:
                ad_data = pickle.load(f)
                sim.ad_model.train_fps = ad_data.get('train_fps')
                sim.ad_model.nn = ad_data.get('nn')
                sim.ad_model.isolation_forest = ad_data.get('isolation_forest')
                sim.ad_model.iso_min = ad_data.get('iso_min')
                sim.ad_model.iso_max = ad_data.get('iso_max')
                sim.ad_model.fitted = True
            print("✅ Applicability Domain model loaded")
    else:
        print("ℹ️ Secondary models not loaded due to untrusted manifest.")

    # ---------- Rebuild Similarity Index from TRAIN data only ----------
    sim.similarity_index = SimilarityIndex()
    for fb in sim.predictor.feedback_db:
        if fb.get('data_role') == 'train' and fb.get('training_eligible', True) and fb.get('smiles') and fb.get('actual') is not None:
            mol = Chem.MolFromSmiles(fb['smiles'])
            if mol is not None:
                sim.similarity_index.add_molecule(mol, lam=fb['actual'], name=fb.get('name', 'unknown'))
    sim.similarity_index.build()
    print(f"✅ Similarity Index rebuilt from TRAIN data only: {len(sim.similarity_index.fingerprints)} molecules")

    # ---------- Load SMARTS dict if exists ----------
    smarts_path = os.path.join(base_dir, ConjugationAnalyzer._SMARTS_PKL_FILENAME)
    if os.path.exists(smarts_path):
        try:
            with open(smarts_path, 'rb') as f:
                ConjugationAnalyzer._SMARTS_DICT = pickle.load(f)
            print(f"✅ Chromophore SMARTS dictionary loaded ({len(ConjugationAnalyzer._SMARTS_DICT)} entries)")
        except Exception as e:
            print(f"⚠️ Failed to load SMARTS dictionary: {e}. Building fresh.")
            ConjugationAnalyzer._get_smarts_dict(base_dir)
    else:
        print("ℹ️ chromophore_smarts.pkl not found – building fresh.")
        ConjugationAnalyzer._get_smarts_dict(base_dir)

    # ReactionLearner — reload weights if pickle exists
    rml_path = os.path.join(base_dir, 'reaction_feasibility.pkl')
    if os.path.exists(rml_path):
        try:
            if getattr(sim.predictor, 'reaction_learner', None) is not None:
                rl = sim.predictor.reaction_learner
                rl.model_path = rml_path
                rl.base_ml.model_path = rml_path
                with open(rml_path, 'rb') as _f:
                    _b = pickle.load(_f)
                if _b.get("version") == rl.MODEL_VERSION:
                    rl.base_ml.model = _b["model"]
                    rl.base_ml.scaler = _b.get("scaler")
                    rl.base_ml.fitted = True
                    rl.real_X = [np.array(v, dtype=np.float32) for v in _b.get("real_X", [])]
                    rl.real_y = list(_b.get("real_y", []))
                    rl.real_keys = list(_b.get("real_keys", []))
                print(f"✅ ReactionLearner reloaded ({len(rl.real_X)} real samples)")
            else:
                sim.predictor.reaction_learner = ReactionLearner(
                    model_path=rml_path,
                    engine=ReactionEngine(),
                    chromo=sim.chromo
                )
                print(f"✅ ReactionLearner loaded ({len(sim.predictor.reaction_learner.real_X)} real samples)")
        except Exception as e:
            print(f"⚠️ Failed to load ReactionLearner: {e}")
    else:
        print("ℹ️ No reaction_feasibility.pkl yet — will train on first use.")

    # Empirical corrector
    emp_path = os.path.join(base_dir, 'empirical_corrector.pkl')
    if os.path.exists(emp_path):
        with open(emp_path, 'rb') as f:
            sim.empirical_corrector = pickle.load(f)
        print("✅ Empirical corrector loaded")
    else:
        sim.empirical_corrector = EmpiricalRuleCorrector()
        print("ℹ️ No empirical corrector found, using fresh instance.")

    # DR‑CNN
    drcnn_weights_path = os.path.join(base_dir, 'drcnn_weights.pth')
    drcnn_tokenizer_path = os.path.join(base_dir, 'drcnn_tokenizer.pkl')
    if os.path.exists(drcnn_weights_path) and os.path.exists(drcnn_tokenizer_path):
        try:
            with open(drcnn_tokenizer_path, 'rb') as f:
                sim.predictor.drcnn_tokenizer = pickle.load(f)
            checkpoint = torch.load(drcnn_weights_path, map_location='cpu')
            # Reconstruct model
            vocab_size = checkpoint.get('vocab_size', 0)
            if vocab_size > 0:
                sim.predictor.drcnn_model = DeepResidualCNN(
                    vocab_size=vocab_size,
                    embed_dim=Config.DRCNN_CHAR_DIM,
                    filters=Config.DRCNN_HIDDEN_FILTERS,
                    kernels=Config.DRCNN_KERNEL_SIZES,
                    dropout=Config.DRCNN_DROPOUT,
                    max_len=Config.DRCNN_MAX_SMILES_LEN
                ).to(sim.predictor.device)
                sim.predictor.drcnn_model.load_state_dict(checkpoint['model_state'], strict=False)
                sim.predictor.drcnn_model.eval()
                sim.predictor.is_trained = True  # keep primary NN flag
                print("✅ DR‑CNN model loaded")
            else:
                print("⚠️ DR‑CNN vocab size invalid, skipping load.")
        except Exception as e:
            print(f"⚠️ Failed to load DR‑CNN: {e}")
    else:
        print("ℹ️ No DR‑CNN artifacts found, starting fresh.")


def load_shared_model(sim):
    """
    Download and load the shared model from central Drive.
    (Uses the same auto‑repair logic as load_all for consistency.)
    """
    print("\n📥 Downloading shared model files...")
    file_ids = {
        "ad_model.pkl": "13rmBMTF_O4aEs-TwuyKxaOCd9ENzYIhl",
         "artifact_manifest.json":  "1MV-FiBZx5bB4nO4TH7_fnM5QzNFFavya",
          "chemistry_memory.pkl":    "1QqzY3MQ_Nv8IpKGWuttxm7oLXQRM3x7D",
          "chromophore_smarts.pkl":  "12YM8E6YPngaTYb5xJysowQu5vojG1Rgs",
          "conform_corrector.pkl":   "1mEMTEpd52mD2rvuAq1SiDBhLP9xnR_xX",
          "deep_svdd.pth":           "1vruYfNfPiPFTD70qUK5WvxiNFtoGFtyL",
          "drcnn_tokenizer.pkl": "1rDRU9g3KzkWX3m5kssbX_U_tJBfwn4Lc",
          "drcnn_weights.pth": "1xVWlNIGld2G3RXrjY8Wg7MhQQ9aSuAhx",
          "e3nn_predictor.pth":      "1qxmdaj7YyX4XYhhB1HaeAprRYk8xIS0v",
          "empirical_corrector.pkl": "1ntJNnGAUdhvTmHi9bfx_y52lUKPXW9yx",
          "ensemble.pkl":            "1xObqPthZKpOMolYdXn9ngT0QsNPaJrRP",
          "error_history.pkl":       "1m9jcZJrfyo6Rb-ISzOTzCBcL9mXiRD4I",
          "feedback_db.pkl":         "1aAghfPlph4vLmMBmyF8cPyZ4R2OmxxAA",
          "huckel_predictor.pkl":    "1b7EplHEP-ZYt4cAuiojFF7cTsIMEInM0",
          "kamlet_taft_model.pkl":   "14byvqakTLugonr6X1Dv9-wagpkwmh8zo",
          "meta_combiner.pkl":       "1JRbkyma9K-1YY2zKEK7r0QAIPCMKG2P7",
          "metadata.json":           "1TpfuUKO1T8h4sAjDrV3dHeNPaYUzxdgs",
          "metrics_log.pkl":         "1qCaWQwtdZY35fDndn3QLn8poAeVx6DBQ",
          "model_weights.pth":       "14bkrLhICs-CGbpFGCqs6P9ZRnGmyEpiS",
          "outlier.pkl":             "1afGicSC8URLrC_w0WG7h0yldaN6W98hF",
          "residual_corrector.pkl":  "1R2Nvho1UMibegbhGy4-B1jEvXrA4Xh9W",
          "scaler.pkl":              "1pg_6dfAWihy8yt0KogKfBe0vVC7iZXK7",
          "similarity_index.pkl":    "14Go4KibdH2a_i7zRPOvgRNhzs1v00FtM",
          "solvent_learner.pkl":     "1v8zz2ERhzqsKUOXfc77L9JqSub2k6zCG",
    }
    os.makedirs("shared_model", exist_ok=True)
    for filename, file_id in file_ids.items():
        if file_id is None or file_id.startswith("1NEW_"):
            print(f"ℹ️ Skipping {filename} (ID not provided)")
            continue
        output = f"shared_model/{filename}"
        if not os.path.exists(output):
            url = f'https://drive.google.com/uc?export=download&id={file_id}'
            gdown.download(url, output, quiet=False, fuzzy=True)
        else:
            print(f"✅ {filename} already exists.")


    load_all(sim, base_dir='shared_model')

    # The above will also rebuild the Similarity Index and load SMARTS.
    # We'll also print the startup verification as before.
    print("\n" + "="*40)
    print("STARTUP VERIFICATION")
    print("="*40)
    checks = [
        ("Primary NN", sim.predictor.is_trained),
        ("Scaler", sim.predictor.scaler is not None),
        ("Feedback DB", len(sim.predictor.feedback_db) > 0),
        ("Residual Corrector", sim.predictor.residual_corrector.model is not None),
        ("Solvent Learner", sim.predictor.solvent_learner.model is not None),
        ("Ensemble", sim.predictor.ensemble.trained),
        ("Outlier Detector", sim.predictor.outlier_detector.model is not None),
        ("Similarity Index", sim.similarity_index.is_built),
        ("Chemistry Memory", len(sim.chemistry_memory.global_stats) > 0),
        ("Conformational Corrector", sim.predictor.conform_corrector.model is not None),
        ("Test Set", len(sim.predictor.test_indices) > 0),
    ]
    for name, ok in checks:
        print(f"{'✓' if ok else '✗'} {name} {'loaded' if ok else 'not available'}")
    print("="*40 + "\n")


def save_all(sim, save_dir='/content/drive/MyDrive/UVVis_Learner/'):
    """
    Save all model artifacts to Google Drive.
    This includes the model weights, scaler, feedback database,
    error history, metrics log, metadata, and all secondary models.
    """
    print("\n💾 Saving to Google Drive...")
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(save_dir, exist_ok=True)

    if sim.predictor.model:
        torch.save({
            'model_state': sim.predictor.model.state_dict(),
            'feature_dim': sim.predictor.current_feature_dim,
            'use_deepchem': True,
            'model_version': Config.MODEL_VERSION,
            'feature_version': Config.FEATURE_VERSION
        }, os.path.join(save_dir, 'model_weights.pth'))
        print("✅ Model saved")

    with open(os.path.join(save_dir, 'scaler.pkl'), 'wb') as f:
        pickle.dump(sim.predictor.scaler, f)

    with open(os.path.join(save_dir, 'feedback_db.pkl'), 'wb') as f:
        pickle.dump(sim.predictor.feedback_db, f)

    with open(os.path.join(save_dir, 'error_history.pkl'), 'wb') as f:
        pickle.dump(sim.predictor.error_history, f)

    with open(os.path.join(save_dir, 'metrics_log.pkl'), 'wb') as f:
        pickle.dump(getattr(sim.predictor, 'metrics_log', []), f)

    n_3d = sum(1 for fb in sim.predictor.feedback_db if fb.get('3d_descriptors') is not None)
    metadata = {
        "last_saved": datetime.now().isoformat(),
        "n_feedbacks": len(sim.predictor.feedback_db),
        "n_errors": len(sim.predictor.error_history),
        "model_feature_dim": sim.predictor.current_feature_dim,
        "solvent_offsets": sim.predictor.solvent_offsets,
        "conform_corrector_trained": sim.predictor.conform_corrector.model is not None,
        "conform_corrector_samples": n_3d,
        "model_version": Config.MODEL_VERSION,
        "feature_version": Config.FEATURE_VERSION,
        "n_test": len(sim.predictor.test_indices),
        "n_validation": len(sim.predictor.validation_indices)
    }
    with open(os.path.join(save_dir, 'metadata.json'), 'w') as f:
        json.dump(metadata, f)
    print("✅ Scaler, feedback, errors, metrics, metadata saved")

    # Manifest
    manifest = build_artifact_manifest(sim)
    with open(os.path.join(save_dir, 'artifact_manifest.json'), 'w') as f:
        json.dump(manifest, f, indent=2)
    print("✅ Learned-artifact provenance manifest saved")

    # Secondary models
    if sim.predictor.residual_corrector.model is not None:
        sim.predictor.residual_corrector.save(os.path.join(save_dir, 'residual_corrector.pkl'))
        print("✅ Residual corrector saved")
    if sim.predictor.solvent_learner.model is not None:
        sim.predictor.solvent_learner.save(os.path.join(save_dir, 'solvent_learner.pkl'))
        print("✅ Solvent learner saved")
    if sim.predictor.ensemble.trained:
        sim.predictor.ensemble.save(os.path.join(save_dir, 'ensemble.pkl'))
        print("✅ Ensemble saved")
    if sim.predictor.outlier_detector.model is not None:
        sim.predictor.outlier_detector.save(os.path.join(save_dir, 'outlier.pkl'))
        print("✅ Outlier detector saved")

    if sim.predictor.huckel_trained:
        sim.predictor.huckel_predictor.save(os.path.join(save_dir, 'huckel_predictor.pkl'))
        print("✅ Hückel predictor saved")

    if sim.predictor.e3nn_trained:
        sim.predictor.e3nn_predictor.save(os.path.join(save_dir, 'e3nn_predictor.pth'))
        print("✅ e3nn saved")

    if sim.predictor.deep_svdd.is_trained:
        sim.predictor.deep_svdd.save(os.path.join(save_dir, 'deep_svdd.pth'))
        print("✅ Deep SVDD saved")

    if len(sim.similarity_index.fingerprints) > 0:
        sim.similarity_index.ensure_built()
        sim.similarity_index.save(os.path.join(save_dir, 'similarity_index.pkl'))
        print("✅ Similarity Index saved")

    if len(sim.chemistry_memory.global_stats) > 0:
        sim.chemistry_memory.save(os.path.join(save_dir, 'chemistry_memory.pkl'))
        print("✅ Chemistry Memory saved")

    if sim.predictor.ngboost.fitted:
        sim.predictor.ngboost.save(os.path.join(save_dir, 'ngboost.pkl'))
        print("✅ NGBoost saved")

    if sim.predictor.meta_combiner.fitted:
        sim.predictor.meta_combiner.save(os.path.join(save_dir, 'meta_combiner.pkl'))
        print("✅ MetaCombiner saved")

    if sim.predictor.conform_corrector.model is not None:
        sim.predictor.conform_corrector.save(os.path.join(save_dir, 'conform_corrector.pkl'))
        print("✅ Conformational corrector saved")

    # DR‑CNN model and tokenizer
    if hasattr(sim.predictor, 'drcnn_model') and sim.predictor.drcnn_model is not None:
        torch.save({
            'model_state': sim.predictor.drcnn_model.state_dict(),
            'vocab_size': sim.predictor.drcnn_tokenizer.vocab_size if sim.predictor.drcnn_tokenizer else 0,
        }, os.path.join(save_dir, 'drcnn_weights.pth'))
        print("✅ DR‑CNN weights saved")
    if hasattr(sim.predictor, 'drcnn_tokenizer') and sim.predictor.drcnn_tokenizer is not None:
        with open(os.path.join(save_dir, 'drcnn_tokenizer.pkl'), 'wb') as f:
            pickle.dump(sim.predictor.drcnn_tokenizer, f)
        print("✅ DR‑CNN tokenizer saved")

    if hasattr(ConjugationAnalyzer, '_SMARTS_DICT') and ConjugationAnalyzer._SMARTS_DICT:
        smarts_path = os.path.join(save_dir, ConjugationAnalyzer._SMARTS_PKL_FILENAME)
        try:
            with open(smarts_path, 'wb') as f:
                pickle.dump(ConjugationAnalyzer._SMARTS_DICT, f)
            print("✅ Chromophore SMARTS dictionary saved")
        except Exception as e:
            print(f"⚠️ Could not save SMARTS dictionary: {e}")

    if hasattr(sim, 'kt_model') and sim.kt_model.fitted:
        sim.kt_model.save(os.path.join(save_dir, 'kamlet_taft_model.pkl'))
        print("✅ Kamlet–Taft model saved")

    if hasattr(sim, 'ad_model') and sim.ad_model.fitted:
        with open(os.path.join(save_dir, 'ad_model.pkl'), 'wb') as f:
            pickle.dump({
                'train_fps': sim.ad_model.train_fps,
                'nn': sim.ad_model.nn,
                'isolation_forest': sim.ad_model.isolation_forest,
                'iso_min': sim.ad_model.iso_min,
                'iso_max': sim.ad_model.iso_max
            }, f)
        print("✅ Applicability Domain model saved")

    # Empirical corrector (rules are deterministic, but we save its parameters if any)
    if hasattr(sim, 'empirical_corrector'):
        with open(os.path.join(save_dir, 'empirical_corrector.pkl'), 'wb') as f:
            pickle.dump(sim.empirical_corrector, f)
        print("✅ Empirical corrector saved")

    if getattr(sim.predictor, 'reaction_learner', None) is not None:
        rl = sim.predictor.reaction_learner
        drive_path = os.path.join(save_dir, 'reaction_feasibility.pkl')
        rl.model_path = drive_path
        rl.base_ml.model_path = drive_path
        rl.save()
        print(f"✅ ReactionLearner saved ({len(rl.real_X)} real samples)")

print("✅ Helper functions loaded.")

✅ Helper functions loaded.


In [ ]:
# @title 8.) Main execution

def main():
    set_global_seed()
    print("="*60)
    print("SPECTRA ML v13.7.3 UV‑Vis Prediction tool")
    print("AI results can be used for research / educational purposes only. Not for medical or regulatory use.")
    print("Results may be inaccurate for molecules outside the training domain. Use with caution.")
    print("Use the tool ethically and responsibly. Do not use for illegal or harmful purposes. Do not attempt to reverse-engineer or misuse the code.")
    print("="*60)

    # Instantiate simulator
    sim = UVVisSimulator()
    sim.chromo = ChromophoreDetector()

    # Model source choice
    print("\n📂 Model source:")
    print("  1) Load shared model from central Drive")
    print("  2) Load from your personal Google Drive")
    print("  3) Start fresh (no pre‑trained model)")
    choice = input("Enter choice (1/2/3): ").strip()
    if choice == '1':
        load_shared_model(sim)
    elif choice == '2':
        load_all(sim)
    else:
        print("Starting with a fresh model.")

    # Ensure ReactionLearner exists (covers fresh-start branch)
    if getattr(sim.predictor, 'reaction_learner', None) is None:
        try:
            sim.predictor.sim_ref = sim
            sim.predictor.reaction_learner = ReactionLearner(
                model_path="reaction_feasibility.pkl",
                engine=ReactionEngine(),
                chromo=sim.chromo
            )
            print(f"✅ ReactionLearner attached "
                  f"({len(sim.predictor.reaction_learner.real_X)} real samples)")
        except Exception as _e:
            print(f"⚠️ ReactionLearner init failed: {_e}")

    # ============================================================
    #  OPTIONAL: Reset split ONCE to repair manifest corruption
    #  (only runs if a flag file is missing)
    # ============================================================
    reset_flag = os.path.join('/content/drive/MyDrive/UVVis_Learner/', 'split_reset_done.flag')
    if not os.path.exists(reset_flag) and len(sim.predictor.feedback_db) > 0:
        print("\n🔄 Resetting train/validation/test split to a clean scaffold split...")
        train_idx, val_idx, test_idx = scaffold_split(
            sim.predictor.feedback_db,
            test_size=Config.TEST_SIZE,
            val_size=Config.VALIDATION_SIZE
        )
        sim.predictor.test_indices = set(test_idx)
        sim.predictor.validation_indices = set(val_idx)
        for i, fb in enumerate(sim.predictor.feedback_db):
            if i in sim.predictor.test_indices:
                fb['data_role'] = 'test'
                fb['training_eligible'] = False
            elif i in sim.predictor.validation_indices:
                fb['data_role'] = 'validation'
                fb['training_eligible'] = False
            else:
                fb['data_role'] = 'train'
                fb['training_eligible'] = True
        print(f"✅ Split reset: {len(train_idx)} train, {len(val_idx)} validation, {len(test_idx)} test")
        # Immediately save to create a new manifest
        save_all(sim)
        print("✅ Manifest updated to the new split.")
        # Create flag file so this doesn't run again
        with open(reset_flag, 'w') as f:
            f.write('done')
        print("ℹ️ Split reset will not run again (flag file created).")

    # Solvent selection
    print("\n📋 Available Solvents (enter the ID code):")
    for entry in sim.solvent_db.list_solvents():
        # entry is like "water (01)"
        name, code = entry.rsplit(" (", 1)
        code = code.rstrip(")")
        print(f"  {code}. {name}")
    solv = input("Enter solvent code: ").strip().lower()
    sim.set_solvent(solv)

    # User parameters
    try:
        ph = float(input(f"pH (default {sim.current_ph:.1f}): ") or sim.current_ph)
        sim.current_ph = ph
    except:
        pass
    try:
        temp = float(input("Temperature K (default 298.15): ") or "298.15")
        sim.current_temp = temp
    except:
        pass
    try:
        conc = float(input("Concentration M (default 0.01): ") or "0.01")
        sim.current_conc = conc
    except:
        pass

    # Upload SDF
    print("\n📤 Upload SDF file:")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    suppl = Chem.SDMolSupplier(filename)

    use_tddft = input("\n🔬 Run TD‑DFT for each molecule? (y/n, slow!): ").strip().lower() == 'y'
    refine_iter = input("🔄 Use iterative refinement? Enter number of iterations (0 to skip, default 1): ").strip() or "1"
    n_iter = int(refine_iter) if refine_iter.isdigit() else 0

    mols = [m for m in suppl if m is not None]
    print(f"\n🧪 Processing {len(mols)} molecules...")
    spectra = []
    names = []
    std_devs = []
    confidences = []
    mols_processed = []
    chrom_classes_all = []
    quantum_data = []
    family_analyses = []
    true_peaks = []

    for i, mol in enumerate(mols[:50]):
        name = mol.GetProp('_Name') if mol.HasProp('_Name') else f"mol_{i}"
        names.append(name)
        mols_processed.append(mol)

        ad_sim, ad_iso, ad_status = sim.ad_model.evaluate(mol) if sim.ad_model.fitted else (0.5, 0.5, 'Unknown')
        print(f"AD for {name}: similarity={ad_sim:.3f}, iso={ad_iso:.3f}, status={ad_status}")

        if n_iter > 0:
            spec, method, std_spec, conf, primary_lambda = sim.predict_with_refinement(
                mol, name, use_tddft=use_tddft, n_iterations=n_iter
            )
            std_devs.append(std_spec)
            confidences.append(conf)
            family_analysis = getattr(sim, '_last_family_analysis', None)
            family_analyses.append(family_analysis)
            _, _, _, chrom_classes, _, _ = sim.predict(mol, name, use_tddft=use_tddft)
            chrom_classes_all.append(chrom_classes)
            quantum_data.append({})
            print(f"{i+1}. {name}: λmax = {primary_lambda} nm (confidence: {conf:.2f})")
            true_peaks.append(primary_lambda)
        else:
            spec, method, _, chrom_classes, qdesc, primary_lambda = sim.predict(
                mol, name, use_tddft=use_tddft,
                extract_quantum=Config.EXTRACT_QUANTUM_DESCRIPTORS
            )
            std_devs.append(np.zeros_like(spec))
            family_analysis = getattr(sim, '_last_family_analysis', None)
            family_analyses.append(family_analysis)

            quantum_confidence = getattr(sim, '_last_quantum_confidence', 0.0)
            conf_score, cat, color, details = sim.predictor.compute_confidence(
                mol, sim.current_ph,
                family_analysis=family_analysis,
                solvent=sim.current_solvent,
                chromophore_classes=chrom_classes,
                quantum_confidence=quantum_confidence,
                ad_score=ad_sim
            )
            conf = conf_score / 100.0
            confidences.append(conf)
            chrom_classes_all.append(chrom_classes)
            quantum_data.append(qdesc)

            print(f"{i+1}. {name}: λmax = {primary_lambda} nm ({method})")
            # ---- SHAP explainability ----
            if Config.USE_SHAP and sim.predictor.is_trained:
                try:
                    shap_vals, exp_val, fnames, top_feats = sim.predictor.explain_prediction(
                        mol, sim.current_ph, nsamples=50
                    )
                    if shap_vals is not None:
                        print("  🔍 SHAP top features:")
                        for feat_name, val in top_feats[:5]:
                            print(f"      {feat_name}: {val:+.3f}")
                except Exception as e:
                    print(f"  ⚠️ SHAP explanation failed: {e}")
            true_peaks.append(primary_lambda)

            if family_analysis:
                print(f"  Chromophore Family: {family_analysis['family']} (novelty: {family_analysis['novelty']:.3f})")
                top3 = family_analysis['top_similarities']
                print(f"  Top similarities: {top3[0][0]} ({top3[0][1]:.2f}), {top3[1][0]} ({top3[1][1]:.2f}), {top3[2][0]} ({top3[2][1]:.2f})")
            analogues = getattr(sim, '_last_analogues', None)
            if analogues:
                print("  Nearest analogues (SMILES, λmax, similarity):")
                for smi, lam, sim_score, aname in analogues[:2]:
                    print(f"    {aname} (λmax={lam:.0f} nm, sim={sim_score:.3f})")
            if qdesc and qdesc.get('gap'):
                print(f"  Quantum gap: {qdesc['gap']:.2f} eV (basis: {qdesc.get('basis','N/A')}, functional: {qdesc.get('functional','N/A')})")
            if quantum_confidence > 0:
                print(f"  Quantum agreement: {quantum_confidence:.2f}")
            if sim._last_3d_desc is not None and sim.predictor.conform_corrector.model is not None:
                print(f"  3D descriptors: dipole={sim._last_3d_desc.get('dipole',0):.2f} D, SASA={sim._last_3d_desc.get('sasa',0):.0f} Å²")

            # --- Solute–solvent interaction report (v13.7.4) ---
            inter = getattr(sim, "_last_interaction_report", None)
            if inter and "modeled" in inter:
                print(f"  🌐 Solute–solvent interaction report "
                      f"(reliability={inter.get('reliability', 0.0):.2f}):")
                for line in inter["modeled"]:
                    print(f"     ✅ Modelled:     {line}")
                for line in inter["approximated"]:
                    print(f"     ≈  Approximated: {line}")
                for line in inter["omitted"]:
                    print(f"     ⊘  Omitted:      {line}")

            print(f"  Confidence: {conf_score:.1f}% ({cat})")
            print(f"  AD status: {ad_status}")

        spectra.append(spec)

    spectra = np.array(spectra)
    mixture = spectra.sum(axis=0)

    visualizer = SpectraVisualizer(Config.wavelengths)

    print("\n" + "="*60)
    print("SIMULATION RESULTS")
    print("="*60)

    df = pd.DataFrame({"Wavelength (nm)": Config.wavelengths})
    for name, spec in zip(names, spectra):
        df[f"{name} (Absorbance)"] = spec
        df[f"{name} (Transmittance)"] = 10 ** (-spec)
    df["Mixture (Absorbance)"] = mixture
    df["Mixture (Transmittance)"] = 10 ** (-mixture)

    def get_top_peaks(y, n=5, height_ratio=0.001):
        max_y = np.max(y)
        if max_y == 0:
            return []
        peaks, props = find_peaks(y, height=max_y * height_ratio, distance=max(1, int(20 / Config.WL_STEP)))
        if len(peaks) == 0:
            return []
        heights = props['peak_heights']
        sorted_idx = np.argsort(heights)[::-1][:n]
        return [(peaks[i], heights[i]) for i in sorted_idx]

    print("\n--- Top 5 Absorbance Peaks for Mixture ---")
    y_mix = df["Mixture (Absorbance)"].values
    top_mix = get_top_peaks(y_mix, n=5)
    if top_mix:
        peak_data = []
        for idx, height in top_mix:
            wl = df["Wavelength (nm)"].iloc[idx]
            trans = df["Mixture (Transmittance)"].iloc[idx]
            peak_data.append((wl, height, trans))
        peak_df = pd.DataFrame(peak_data, columns=["Wavelength (nm)", "Absorbance", "Transmittance"])
        display(peak_df)
    else:
        print("No significant peaks found in mixture.")

    solvent_name = sim.solvent_db.get(sim.current_solvent)['name']
    title_str = f"UV‑Vis in {solvent_name} (pH={sim.current_ph:.1f}, T={sim.current_temp:.1f}K)"
    visualizer.plot_spectra(
        spectra=np.array(spectra),
        labels=names,
        mixture=mixture,
        title=title_str,
        save_path=None,
    )

    # ============================================================
    # FEEDBACK COLLECTION – with user choice and manifest update
    # ============================================================
    print("\n" + "="*40)
    print("FEEDBACK COLLECTION")
    print("="*40)

    if input("Collect feedback? (y/n): ").strip().lower() == 'y':
        indices = list(range(len(names)))
        indices.sort(key=lambda i: confidences[i])
        for idx in indices[:3]:
            name = names[idx]
            spec = spectra[idx]
            mol = mols_processed[idx]
            conf = confidences[idx]
            chrom_classes = chrom_classes_all[idx]
            qdesc = quantum_data[idx] if idx < len(quantum_data) else {}
            family_analysis = family_analyses[idx] if idx < len(family_analyses) else None

            print(f"\n--- {name} (confidence: {conf:.2f}) ---")
            pred_max = true_peaks[idx]
            print(f"Predicted λmax = {pred_max} nm")

            # Print info
            if chrom_classes:
                display_classes = [c for c in chrom_classes if c not in ['ml_predicted', 'default_error']]
                if any('analogue' in c for c in chrom_classes):
                    analogue_name = next((c.replace('analogue_', '') for c in chrom_classes if 'analogue_' in c), None)
                    if analogue_name and analogue_name != 'unknown' and hasattr(sim, '_last_analogue_lambda'):
                        print(f"  📌 Based on analogue: {analogue_name} (λmax={sim._last_analogue_lambda:.0f} nm)")
                if display_classes:
                    print(f"  Chromophore features: {', '.join(display_classes[:5])}")
            if family_analysis:
                print(f"Chromophore Family: {family_analysis['family']} (novelty: {family_analysis['novelty']:.2f})")
            analogues = getattr(sim, '_last_analogues', None)
            if analogues:
                print("  Nearest analogue:", analogues[0][3], f"(λmax={analogues[0][1]:.0f} nm, sim={analogues[0][2]:.3f})")
            if qdesc and qdesc.get('gap'):
                print(f"  Quantum gap: {qdesc['gap']:.2f} eV")
            if sim.predictor.confidence_scorer is not None:
                quantum_confidence = getattr(sim, '_last_quantum_confidence', 0.0)
                ad_sim, _, _ = sim.ad_model.evaluate(mol) if sim.ad_model.fitted else (0.5, 0.5, 'Unknown')
                conf_score, cat, color, details = sim.predictor.compute_confidence(
                    mol, sim.current_ph,
                    family_analysis=family_analysis,
                    solvent=sim.current_solvent,
                    chromophore_classes=chrom_classes,
                    quantum_confidence=quantum_confidence,
                    ad_score=ad_sim
                )
                print(f"Confidence: {conf_score:.1f}% ({cat})")

            visualizer.plot_individual_spectrum(spec, name, conf)

            correct = input("Correct? (y/n): ").strip().lower()

            if correct == 'y':
                # Ask for role (default to 'train' for new molecules, but allow choice)
                current_role = sim.predictor.get_frozen_data_role(mol)
                if current_role is not None:
                    print(f"ℹ️ This molecule currently has role: {current_role.upper()}")
                role = input("📂 Data role (train / validation / test) [train]: ").strip().lower() or "train"
                while role not in {"train", "validation", "test"}:
                    print("⚠️ Invalid option. Choose: train, validation, or test.")
                    role = input("📂 Data role (train / validation / test) [train]: ").strip().lower() or "train"
                if current_role is not None and role != current_role:
                    print(f"⚠️ You are changing the role from {current_role.upper()} to {role.upper()} – the manifest will be updated.")
                sim.predictor.receive_feedback(
                    mol, spec, pred_max, pred_max,
                    sim.current_ph, sim.current_solvent,
                    chromophore_classes=chrom_classes,
                    quantum_desc=qdesc,
                    data_role=role
                )
                print(f"✅ Positive feedback recorded (role={role})")
            else:
                actual = float(input("Enter actual λmax: "))
                current_role = sim.predictor.get_frozen_data_role(mol)
                if current_role is not None:
                    print(f"ℹ️ This molecule currently has role: {current_role.upper()}")
                role = input("📂 Data role (train / validation / test) [train]: ").strip().lower() or "train"
                while role not in {"train", "validation", "test"}:
                    print("⚠️ Invalid option. Choose: train, validation, or test.")
                    role = input("📂 Data role (train / validation / test) [train]: ").strip().lower() or "train"
                if current_role is not None and role != current_role:
                    print(f"⚠️ You are changing the role from {current_role.upper()} to {role.upper()} – the manifest will be updated.")
                sim.predictor.receive_feedback(
                    mol, spec, pred_max, actual,
                    sim.current_ph, sim.current_solvent,
                    chromophore_classes=chrom_classes,
                    quantum_desc=qdesc,
                    data_role=role
                )

        # After all feedback, retrain and rebuild the manifest
        sim.predictor.retrain()
        sim._fit_kt_model_from_feedback()
        sim._fit_ad_model_from_feedback()

        # ============================================================
        #  Update the manifest to reflect the current roles
        # ============================================================
        print("\n🔄 Updating manifest to match current data roles...")
        # Rebuild indices from current feedback_db roles
        train_indices = [i for i, fb in enumerate(sim.predictor.feedback_db) if fb.get('data_role') == 'train']
        val_indices   = [i for i, fb in enumerate(sim.predictor.feedback_db) if fb.get('data_role') == 'validation']
        test_indices  = [i for i, fb in enumerate(sim.predictor.feedback_db) if fb.get('data_role') == 'test']
        sim.predictor.train_indices = set(train_indices)   # not used elsewhere, but for consistency
        sim.predictor.validation_indices = set(val_indices)
        sim.predictor.test_indices = set(test_indices)
        print("✅ Manifest updated in memory (will be saved later).")

    # Locked test benchmark
    if len(sim.predictor.test_indices) > 0:
        import contextlib
        with open('/dev/null', 'w') as fnull:
            with contextlib.redirect_stdout(fnull):
                locked_metrics = sim.evaluate_locked_test_set()
        if locked_metrics:
            print("\n--- LOCKED TEST BENCHMARK ---")
            print(f"Number of samples (n):              {locked_metrics.get('n', 0)}")
            print(f"Mean Absolute Error (MAE):          {locked_metrics.get('MAE', 0):.2f} nm")
            print(f"Root Mean Squared Error (RMSE):     {locked_metrics.get('RMSE', 0):.2f} nm")
            print(f"Coefficient of Determination (R²):   {locked_metrics.get('R2', 0):.3f}")
            sep = locked_metrics.get('SEP')
            if sep is not None:
                print(f"Standard Error of Prediction (SEP): {sep:.2f} nm")
            rpd = locked_metrics.get('RPD')
            if rpd is not None:
                print(f"Ratio of Performance to Deviation (RPD): {rpd:.2f}")
            print(f"Concordance Correlation Coefficient (CCC): {locked_metrics.get('CCC', 0):.3f}")
            print(f"Mean Error (ME / bias):             {locked_metrics.get('ME', 0):.2f} nm")
            norm_score = locked_metrics.get('normalized_MAE_score')
            if norm_score is not None:
                print(f"Normalized MAE Score:               {norm_score:.3f}")
            if locked_metrics.get('n_skipped', 0) > 0:
                print(f"⚠️ TEST records skipped: {locked_metrics['n_skipped']}")

    # Learning curve
    if sim.predictor.error_history:
        visualizer.plot_learning_curve(sim.predictor.error_history)
        avg_err = np.mean(sim.predictor.error_history[-20:])
        print(f"Average error (last 20): {avg_err:.1f} nm")
        full_metrics = sim.predictor.compute_full_metrics(n_last=20)
        print("Full metrics (last 20):")
        for k, v in full_metrics.items():
            if v is not None:
                if k == 'MAE':
                    print(f"  Mean Absolute Error (MAE): {v:.3f}")
                elif k == 'RMSE':
                    print(f"  Root Mean Squared Error (RMSE): {v:.3f}")
                elif k == 'R2':
                    print(f"  Coefficient of Determination (R²): {v:.3f}")
                elif k == 'SEP':
                    print(f"  Standard Error of Prediction (SEP): {v:.3f}")
                elif k == 'RPD':
                    print(f"  Ratio of Performance to Deviation (RPD): {v:.3f}")
                elif k == 'CCC':
                    print(f"  Concordance Correlation Coefficient (CCC): {v:.3f}")
                elif k == 'ME':
                    print(f"  Mean Error (ME): {v:.3f}")
                elif k == 'normalized_MAE_score':
                    print(f"  Normalized MAE Score: {v:.3f}")
                else:
                    print(f"  {k}: {v}")

    # Optional detailed breakdowns
    if len(sim.predictor.test_indices) > 5:
      if input("\nRun detailed evaluation by solvent, wavelength region, and chemical class? (y/n): ").strip().lower() == 'y':
        by_sol = sim.predictor.evaluate_by_solvent()
        by_reg = sim.predictor.evaluate_by_wavelength_region()
        by_class = sim.predictor.evaluate_by_class()

        # Helper to convert dict of dicts to a clean DataFrame
        def format_metrics(data_dict, title):
            if not data_dict:
                print(f"\n--- {title} ---")
                print("No data available.")
                return
            # Build a list of rows
            rows = []
            for key, m in data_dict.items():
                rows.append({
                    'Group': key,
                    'n': m.get('n', 0),
                    'MAE (nm)': m.get('MAE', np.nan),
                    'RMSE (nm)': m.get('RMSE', np.nan),
                    'R²': m.get('R2', np.nan)
                })
            df = pd.DataFrame(rows)
            # Round numeric columns
            num_cols = ['MAE (nm)', 'RMSE (nm)', 'R²']
            df[num_cols] = df[num_cols].round(2)
            print(f"\n--- {title} ---")
            print(df.to_string(index=False))
            print()

        format_metrics(by_sol, "Evaluation by Solvent")
        format_metrics(by_reg, "Evaluation by Wavelength Region")

        # ---- Special formatting for Chemical Class (sorted, limited) ----
        if by_class:
            # Convert to DataFrame and sort by MAE
            rows = []
            for key, m in by_class.items():
                rows.append({
                    'Group': key,
                    'n': m.get('n', 0),
                    'MAE (nm)': m.get('MAE', np.nan),
                    'RMSE (nm)': m.get('RMSE', np.nan),
                    'R²': m.get('R2', np.nan)
                })
            df_class = pd.DataFrame(rows)
            num_cols = ['MAE (nm)', 'RMSE (nm)', 'R²']
            df_class[num_cols] = df_class[num_cols].round(2)

            # Sort by MAE descending (worst first)
            df_class_sorted = df_class.sort_values('MAE (nm)', ascending=False)

            print("\n--- Evaluation by Chemical Class (worst 15 by MAE) ---")
            print(df_class_sorted.head(15).to_string(index=False))

            print("\n--- Evaluation by Chemical Class (best 15 by MAE) ---")
            print(df_class_sorted.tail(15).to_string(index=False))
            print(f"Total chemical classes: {len(df_class)}")
        else:
            print("\n--- Evaluation by Chemical Class ---")
            print("No data available.")
        print("✅ Detailed evaluation completed.")

      if input("\nRun ablation study? (y/n): ").strip().lower() == 'y':
          # ---- Ask user for data source ----
          print("\n📊 Ablation data source:")
          print("  1) Locked test set (current held-out molecules)")
          print("  2) Last 20 feedback records (most recent corrections)")
          print("  3) Random 10 from the last 20 feedback records")
          source_choice = input("Enter choice (1/2/3): ").strip()

          test_records = []

          if source_choice == '1':
              # ---- Original locked test set ----
              for idx in sim.predictor.test_indices:
                  if idx >= len(sim.predictor.feedback_db):
                      continue
                  fb = sim.predictor.feedback_db[idx]
                  if fb.get('data_role') == 'test' and fb.get('actual') is not None and fb.get('smiles'):
                      test_records.append({
                          'feedback_id': fb.get('feedback_id', idx),
                          'smiles': fb['smiles'],
                          'actual': fb['actual'],
                          'name': fb.get('name', f'test_{idx}')
                      })
              if test_records and len(test_records) > 8:
                  import random
                  random.seed(42)
                  test_records = random.sample(test_records, 8)

          elif source_choice == '2' or source_choice == '3':
              # ---- Use the last 20 feedback records (any role) ----
              all_fb = sim.predictor.feedback_db
              # Filter only those with actual and predicted values
              valid_fb = [fb for fb in all_fb if fb.get('actual') is not None and fb.get('smiles')]
              if len(valid_fb) < 2:
                  print("⚠️ Not enough feedback records with actual values.")
              else:
                  # Take the last 20 (or all if fewer than 20)
                  last_20 = valid_fb[-20:]
                  if source_choice == '3':
                      # Randomly select 10 from the last 20
                      import random
                      random.seed(42)
                      sample_size = min(10, len(last_20))
                      selected = random.sample(last_20, sample_size)
                      print(f"✅ Using {sample_size} randomly selected from the last {len(last_20)} feedback records.")
                  else:
                      selected = last_20
                      print(f"✅ Using all {len(selected)} records from the last 20 feedback entries.")

                  for fb in selected:
                      test_records.append({
                          'feedback_id': fb.get('feedback_id', -1),
                          'smiles': fb['smiles'],
                          'actual': fb['actual'],
                          'name': fb.get('name', 'unknown')
                      })

          else:
              print("⚠️ Invalid choice. Skipping ablation.")

          # ---- Run the ablation (ONLY if we have records) ----
          if test_records:
              import contextlib
              with open('/dev/null', 'w') as fnull:
                  with contextlib.redirect_stdout(fnull):
                      ablation_results = sim.run_ablation(test_records)
              print("\n--- Ablation Results ---")
              for name, res in ablation_results.items():
                  if isinstance(res, dict):
                      print(f"{name}: MAE={res['MAE']:.2f}, RMSE={res['RMSE']:.2f}, R2={res['R2']:.3f}")
                  else:
                      print(f"{name}: {res}")
          else:
              print("⚠️ No valid records found for ablation. Skipping.")

    # Save all (if not already saved)
    if input("\nSave all learning data to Google Drive? (y/n): ").strip().lower() == 'y':
        save_all(sim)

    print("\n✅ Done!")

if __name__ == "__main__":
    main()

✅ Global random seed set to 42
SPECTRA ML v13.7.3 UV‑Vis Prediction tool
AI results can be used for research / educational purposes only. Not for medical or regulatory use.
Results may be inaccurate for molecules outside the training domain. Use with caution.
Use the tool ethically and responsibly. Do not use for illegal or harmful purposes. Do not attempt to reverse-engineer or misuse the code.

📂 Model source:
  1) Load shared model from central Drive
  2) Load from your personal Google Drive
  3) Start fresh (no pre‑trained model)
Enter choice (1/2/3): 2

📂 Loading from Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ ReactionEngine loaded (50 rules).
✅ ReactionFeasibilityML loaded
✅ ReactionLearner history restored: 2 real samples
✅ ReactionLearner attached (2 real samples)
✅ Scaler loaded
✅ Loaded 775 feedback records from feedback_db.pkl
🔒 Frozen TEST molecules: 74
✅ Restored persist

Saving Aspirin.sdf to Aspirin.sdf

🔬 Run TD‑DFT for each molecule? (y/n, slow!): n
🔄 Use iterative refinement? Enter number of iterations (0 to skip, default 1): 2

🧪 Processing 1 molecules...
AD for 2244: similarity=1.000, iso=0.168, status=High
   ℹ️ Score 1.00 but SMARTS produced no product — ignoring.
   ✅ Using analogue λmax: 275 nm (sim=1.000)
   🧠 DR‑CNN prediction: 245 nm (conf=0.78)
   🧠 e3nn prediction: 226 nm (conf=0.13)
   🧮 Hückel prediction: 283 nm (conf=0.00)
   🔧 Meta residual correction: -14.8 nm
   🧩 Meta‑Combiner: 260.4 nm | analogue: 275.0 nm (sim=1.000)
   ⭐ Analogue prior blend: 0.7 analogue / 0.3 meta
  Iteration 1: λmax = 270.0 nm
   ℹ️ Score 1.00 but SMARTS produced no product — ignoring.
   ✅ Using analogue λmax: 275 nm (sim=1.000)
   🧠 DR‑CNN prediction: 245 nm (conf=0.78)
   🧠 e3nn prediction: 226 nm (conf=0.13)
   🧮 Hückel prediction: 283 nm (conf=0.00)
   🔧 Meta residual correction: -14.8 nm
   🧩 Meta‑Combiner: 260.4 nm | analogue: 275.0 nm (sim=1.000)
   

,Wavelength (nm),Absorbance,Transmittance
0,270.5,0.010033,0.977163
1,201.5,0.000701,0.998388
2,345.5,0.000500,0.998850
3,536.5,0.000109,0.999749
4,369.0,0.000099,0.999772



FEEDBACK COLLECTION
Collect feedback? (y/n): y

--- 2244 (confidence: 0.19) ---
Predicted λmax = 271.0 nm
  📌 Based on analogue: 2244 (λmax=275 nm)
  Chromophore features: Insulator: ester_linker, Base: benzene, Base: carbonyl, Base: ketone, Steric Penalty: ortho_di_sub_benzene
Chromophore Family: novel/unknown (novelty: 0.95)
  Nearest analogue: 2244 (λmax=275 nm, sim=1.000)


Correct? (y/n): n
Enter actual λmax: 275
ℹ️ This molecule currently has role: TRAIN
📂 Data role (train / validation / test) [train]: train
   🔬 ReactionLearner recorded: prob=0.05 key=None (3 total, 1/20)
   🔬 ReactionLearner recorded: prob=0.05 key=None (4 total, 2/20)
📝 Feedback recorded: error = 4.0 nm (role=train)
[Operational Metrics] MAE: 30.75 nm, RMSE: 48.35 nm, R²: 0.607, SEP: 49.61, RPD: 1.59, CCC: 0.775, ME: 22.00 nm (last 20 feedback records)
⚠️ 74 TEST molecule(s) are frozen. Retraining only on TRAIN data.
✅ Primary NN retrained.
✅ Solvent learner trained.
✅ CatBoost trained.
✅ Deep SVDD trained.
🧪 Training Hückel predictor...
✅ Hückel predictor trained on 606 samples.
🧪 Training RadialMLPPredictor...
✅ RadialMLP trained on 606 molecules.
🔄 Generating OOF predictions...
✅ RadialMLP trained on 484 molecules.
✅ RadialMLP trained on 484 molecules.
✅ RadialMLP trained on 486 molecules.
✅ RadialMLP trained on 485 molecules.
✅ RadialMLP trained on 485 molecules.
✅ Accuracy-first

Average error (last 20): 32.1 nm
Full metrics (last 20):
  n: 20
  Mean Absolute Error (MAE): 30.750
  Root Mean Squared Error (RMSE): 48.354
  Coefficient of Determination (R²): 0.607
  Standard Error of Prediction (SEP): 49.610
  Ratio of Performance to Deviation (RPD): 1.594
  Concordance Correlation Coefficient (CCC): 0.775
  Mean Error (ME): 22.000
  Normalized MAE Score: 0.923

Run detailed evaluation by solvent, wavelength region, and chemical class? (y/n): n

Run ablation study? (y/n): n

Save all learning data to Google Drive? (y/n): y

💾 Saving to Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Model saved
✅ Scaler, feedback, errors, metrics, metadata saved
✅ Learned-artifact provenance manifest saved
✅ Residual corrector saved
✅ Solvent learner saved
✅ Ensemble saved
✅ Outlier detector saved
✅ Hückel predictor saved
✅ e3nn saved
✅ Deep SVDD saved
✅ Similarity Index saved
✅ Chem